In [1]:
import torch
import subprocess

print("===== KIỂM TRA MÔI TRƯỜNG =====")
print("Phiên bản PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("Tên GPU:", gpu.name)
    print("Dung lượng VRAM:", round(gpu.total_memory / 1024**3, 2), "GiB")
else:
    print("CHƯA CÓ GPU - CẦN KIỂM TRA LẠI!")

print("\n===== NVIDIA-SMI =====")
result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)
print(result.stdout if result.returncode == 0 else result.stderr)

===== KIỂM TRA MÔI TRƯỜNG =====
Phiên bản PyTorch: 2.14.1+cu130
CUDA khả dụng: True
Tên GPU: Tesla T4
Dung lượng VRAM: 14.56 GiB

===== NVIDIA-SMI =====
Sun Nov 22 16:58:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       3MiB /  15360MiB |      0%   

In [2]:
from google.colab import drive
import os

# Kết nối Google Drive nếu chưa kết nối
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

# Đường dẫn thư mục bài báo nghiên cứu
PARENT = (
    "/content/drive/MyDrive/"
    "0. Classroom/"
    "3. NĂM 3/"
    "HK1(3)/"
    "Nhập môn Xử lý ngôn ngữ tự nhiên/"
    "bài báo nghiên cứu"
)

# Kiểm tra đường dẫn cha có tồn tại không
if not os.path.isdir(PARENT):
    raise FileNotFoundError(
        f"Không tìm thấy thư mục:\n{PARENT}\n"
        "Hãy kiểm tra lại tên các thư mục trên Drive."
    )

# Thư mục dành riêng cho thực nghiệm
BASE = os.path.join(PARENT, "mac_sql_lite_v3")

# Tạo các thư mục con
folders = [
    "data",
    "prompts",
    "configs",
    "results",
    "logs",
    "figures"
]

for folder in folders:
    os.makedirs(os.path.join(BASE, folder), exist_ok=True)

print("===== KIỂM TRA THƯ MỤC =====")
print("Đường dẫn:", BASE)

for folder in sorted(os.listdir(BASE)):
    print("✓", folder)

print("\nTHIẾT LẬP THÀNH CÔNG!")

Mounted at /content/drive
===== KIỂM TRA THƯ MỤC =====
Đường dẫn: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3
✓ configs
✓ data
✓ figures
✓ logs
✓ prompts
✓ results

THIẾT LẬP THÀNH CÔNG!


In [3]:
import os
import sys
import json
import uuid
import glob
import shutil
import platform
import subprocess
import datetime
import torch

# 1. Tạo mã định danh cho phiên Colab hiện tại
if "RUN_ID" not in globals():
    RUN_ID = (
        datetime.datetime.now(datetime.timezone.utc)
        .strftime("%Y%m%dT%H%M%SZ")
        + "_" + uuid.uuid4().hex[:6]
    )

print("===== PHIÊN THỰC NGHIỆM =====")
print("RUN_ID:", RUN_ID)

# 2. Kiểm tra dung lượng ổ đĩa
print("\n===== DUNG LƯỢNG LƯU TRỮ =====")

for name, path in [
    ("Ổ đĩa tạm Colab", "/content"),
    ("Google Drive", "/content/drive/MyDrive")
]:
    total, used, free = shutil.disk_usage(path)
    print(
        f"{name}: còn trống {free / 1024**3:.2f} GiB"
    )

# 3. Tìm file Spider trong thư mục môn học
print("\n===== TÌM FILE SPIDER =====")

spider_files = glob.glob(
    os.path.join(PARENT, "**", "*spider*.zip"),
    recursive=True
)

if spider_files:
    for path in spider_files:
        print("Tìm thấy:", path)
else:
    print("Chưa tìm thấy file ZIP có tên chứa 'spider'.")
    print("Nếu file ở thư mục khác, ta sẽ xác định sau.")

# 4. Thu thập thông tin môi trường
print("\n===== THÔNG TIN MÔI TRƯỜNG =====")

gpu_name = None
gpu_vram = None

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_name = props.name
    gpu_vram = round(props.total_memory / 1024**3, 2)

environment = {
    "run_id": RUN_ID,
    "time_utc": datetime.datetime.now(
        datetime.timezone.utc
    ).isoformat(),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "pytorch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_build": torch.version.cuda,
    "gpu": gpu_name,
    "gpu_vram_gib": gpu_vram
}

print(json.dumps(environment, indent=2, ensure_ascii=False))

# 5. Lưu thông tin môi trường vào Drive
logs_dir = os.path.join(BASE, "logs")
os.makedirs(logs_dir, exist_ok=True)

environment_path = os.path.join(
    logs_dir, f"environment_{RUN_ID}.json"
)

with open(environment_path, "w", encoding="utf-8") as f:
    json.dump(environment, f, indent=2, ensure_ascii=False)

# 6. Lưu danh sách thư viện
freeze_path = os.path.join(
    logs_dir, f"pip_freeze_{RUN_ID}.txt"
)

result = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    raise RuntimeError(result.stderr)

with open(freeze_path, "w", encoding="utf-8") as f:
    f.write(result.stdout)

# 7. Cập nhật nhật ký các phiên
manifest_path = os.path.join(
    logs_dir, "run_manifest.json"
)

if os.path.exists(manifest_path):
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
else:
    manifest = {}

manifest.setdefault("sessions", [])

existing_ids = {
    session["run_id"]
    for session in manifest["sessions"]
}

if RUN_ID not in existing_ids:
    manifest["sessions"].append(environment)

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("\n===== KẾT QUẢ =====")
print("Đã lưu thông tin môi trường.")
print("Đã lưu danh sách thư viện.")
print("Đã cập nhật run_manifest.json.")
print("Số phiên đã ghi:", len(manifest["sessions"]))
print("\nHOÀN THÀNH!")

===== PHIÊN THỰC NGHIỆM =====
RUN_ID: 20261122T165935Z_9c6c0d

===== DUNG LƯỢNG LƯU TRỮ =====
Ổ đĩa tạm Colab: còn trống 112.54 GiB
Google Drive: còn trống 106.91 GiB

===== TÌM FILE SPIDER =====
Tìm thấy: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/spider.zip

===== THÔNG TIN MÔI TRƯỜNG =====
{
  "run_id": "20261122T165935Z_9c6c0d",
  "time_utc": "2026-11-22T16:59:35.049250+00:00",
  "python": "3.13.16",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "pytorch": "2.14.1+cu130",
  "cuda_available": true,
  "cuda_build": "13.0",
  "gpu": "Tesla T4",
  "gpu_vram_gib": 14.56
}

===== KẾT QUẢ =====
Đã lưu thông tin môi trường.
Đã lưu danh sách thư viện.
Đã cập nhật run_manifest.json.
Số phiên đã ghi: 1

HOÀN THÀNH!


In [4]:
import os
import json

print("===== TẠO CẤU HÌNH THỰC NGHIỆM =====")

config = {
    "name": "MAC-SQL Lite v3",
    "status": "DRAFT_INITIAL",

    "research": {
        "topic": "Multi-Agent Text-to-SQL based on MAC-SQL",
        "type": "Resource-Constrained Reimplementation",
        "main_hypothesis": "EX(C5) > EX(C1)"
    },

    "model": {
        "id": "Qwen/Qwen2.5-Coder-3B-Instruct",
        "revision": None,
        "precision": "fp16",
        "fine_tuning": False
    },

    "decoding": {
        "do_sample": False,
        "seed": 42,
        "max_new_tokens": None
    },

    "dataset": {
        "name": "Spider 1.0",
        "train_debug_samples": 30,
        "dev_samples_target": 300,
        "sample_seed": 42
    },

    "main_configs": [
        "C1", "C2", "C3", "C4", "C5"
    ],

    "optional_configs": [
        "C2b", "C6"
    ],

    "agents": {
        "selector": {
            "type": "heuristic",
            "activation_policy": None
        },

        "decomposer": {
            "type": "LLM",
            "parser_fallback": None
        },

        "refiner": {
            "type": "LLM",
            "refine_on_exception_only": True,
            "max_refinements": 3
        }
    },

    "evaluation": {
        "primary_metric": "Execution Accuracy",
        "evaluator": None,
        "bootstrap": True,
        "mcnemar": True
    }
}

# Đường dẫn lưu cấu hình
config_path = os.path.join(
    BASE,
    "configs",
    "experiment_config_draft.json"
)

# Tránh vô tình ghi đè file đã có
if os.path.exists(config_path):
    print("File cấu hình nháp đã tồn tại.")
    print("Không ghi đè.")
else:
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(
            config,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("Đã tạo file cấu hình mới.")

print("\nĐường dẫn:")
print(config_path)

print("\n===== NỘI DUNG CẤU HÌNH =====")

with open(config_path, "r", encoding="utf-8") as f:
    saved_config = json.load(f)

print(json.dumps(saved_config, indent=2, ensure_ascii=False))

print("\nHOÀN THÀNH!")

===== TẠO CẤU HÌNH THỰC NGHIỆM =====
Đã tạo file cấu hình mới.

Đường dẫn:
/content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/configs/experiment_config_draft.json

===== NỘI DUNG CẤU HÌNH =====
{
  "name": "MAC-SQL Lite v3",
  "status": "DRAFT_INITIAL",
  "research": {
    "topic": "Multi-Agent Text-to-SQL based on MAC-SQL",
    "type": "Resource-Constrained Reimplementation",
    "main_hypothesis": "EX(C5) > EX(C1)"
  },
  "model": {
    "id": "Qwen/Qwen2.5-Coder-3B-Instruct",
    "revision": null,
    "precision": "fp16",
    "fine_tuning": false
  },
  "decoding": {
    "do_sample": false,
    "seed": 42,
    "max_new_tokens": null
  },
  "dataset": {
    "name": "Spider 1.0",
    "train_debug_samples": 30,
    "dev_samples_target": 300,
    "sample_seed": 42
  },
  "main_configs": [
    "C1",
    "C2",
    "C3",
    "C4",
    "C5"
  ],
  "optional_configs": [
    "C2b",
    "C6"
  ],
  "agents": {
    "selector": 

In [5]:
from pathlib import Path
import hashlib
import json
import datetime

# Đường dẫn file đặc tả
spec_path = Path(BASE) / "configs" / "SPEC_v3_initial.md"

spec_text = """# MAC-SQL Lite v3 — Initial Experiment Specification

## Research topic
Nghiên cứu kiến trúc cộng tác đa tác tử cho bài toán
Text-to-SQL dựa trên mô hình MAC-SQL.

## Experiment type
Resource-Constrained Reimplementation.
Đây không phải bản tái lập nguyên xi MAC-SQL gốc.

## Research hypothesis
H1: EX(C5) > EX(C1).

## Model
Qwen/Qwen2.5-Coder-3B-Instruct.
FP16, greedy decoding, seed 42.
No fine-tuning. No paid inference API.

## Dataset
Spider 1.0.
30 train samples for debugging and prompt development.
Target: 300 Spider dev samples, sampled with seed 42.
No gold SQL in inference prompts.
No prompt tuning on dev.

## Main configurations
C1: Full Schema -> Direct SQL.
C2: Heuristic Selector -> Direct SQL.
C3: Full Schema -> Decomposer -> SQL.
C4: C1 SQL -> Exception-only Refiner.
C5: Selector -> Decomposer -> Refiner.

## Optional configurations
C2b: Forced-pruning stress test.
C6: C5 initial SQL before Refiner.

## Agents
Selector: Heuristic schema pruning, preserve PK/FK
and necessary JOIN paths, with fallback.

Decomposer: LLM-based stepwise decomposition
and final SQL generation.

Refiner: SQLite exception-triggered correction,
maximum 3 refinement attempts.

## Evaluation
Primary metric: Spider Execution Accuracy (EX).
Main comparison: C5 versus C1.
Paired bootstrap 95% CI and McNemar test.

Additional analyses:
Schema token reduction, latency, LLM calls,
execution repair, EX repair, error categories.

## Reproducibility
Same model, dataset, evaluator and decoding policy
for all main configurations.
Store sample IDs, prompts, configuration, predictions,
execution logs and evaluation results.

## Pending decisions
- Exact model revision.
- Spider evaluator implementation/version.
- Selector activation threshold.
- Decomposer parser and fallback.
- Maximum generation tokens.
- Final prompt versions.

All pending decisions must be resolved using
train data before official dev evaluation.

## Research integrity
Do not reuse old notebook results.
Do not select dev examples based on performance.
Do not report BIRD/VES as locally reproduced results.
Report negative results and experimental limitations.
"""

# Không ghi đè đặc tả đã tồn tại
if not spec_path.exists():
    spec_path.write_text(spec_text, encoding="utf-8")
    print("Đã tạo đặc tả ban đầu.")
else:
    print("Đặc tả đã tồn tại. Không ghi đè.")

# Tính SHA-256 cho nội dung đã lưu
spec_bytes = spec_path.read_bytes()
spec_hash = hashlib.sha256(spec_bytes).hexdigest()

# Cập nhật manifest
manifest_path = Path(BASE) / "logs" / "run_manifest.json"

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

if "spec_initial" not in manifest:
    manifest["spec_initial"] = {
        "file": "configs/SPEC_v3_initial.md",
        "sha256": spec_hash,
        "size_bytes": len(spec_bytes),
        "recorded_utc": datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat(),
        "status": "INITIAL_NOT_FROZEN"
    }

    with manifest_path.open("w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    print("Đã ghi thông tin đặc tả vào manifest.")
else:
    print("Manifest đã có bản ghi đặc tả ban đầu.")

print("\n===== KIỂM TRA =====")
print("File:", spec_path)
print("SHA-256:", spec_hash)
print("Kích thước:", len(spec_bytes), "bytes")
print("Trạng thái: INITIAL_NOT_FROZEN")
print("\n HOÀN THÀNH!")

Đã tạo đặc tả ban đầu.
Đã ghi thông tin đặc tả vào manifest.

===== KIỂM TRA =====
File: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/configs/SPEC_v3_initial.md
SHA-256: 726225b8037f39ab16fbbe3aa02708a1258a9f775de29b946073e1db7f243d05
Kích thước: 2173 bytes
Trạng thái: INITIAL_NOT_FROZEN

 HOÀN THÀNH!


In [6]:
import subprocess
import json
import datetime
from pathlib import Path

print("===== KIỂM TRA REPOSITORY MAC-SQL =====")

repo_url = "https://github.com/wbbeyourself/MAC-SQL.git"

# Lấy commit hash của HEAD từ GitHub
result = subprocess.run(
    ["git", "ls-remote", repo_url, "HEAD"],
    capture_output=True,
    text=True,
    timeout=60
)

if result.returncode != 0:
    print("Không thể truy cập repository.")
    print("Chi tiết lỗi:", result.stderr)
    raise RuntimeError("Không lấy được commit hash.")

if not result.stdout.strip():
    raise RuntimeError("GitHub không trả về commit hash.")

commit_hash = result.stdout.split()[0]

print("Repository:", repo_url)
print("Commit hash:", commit_hash)

# Đường dẫn nhật ký
manifest_path = (
    Path(BASE) / "logs" / "run_manifest.json"
)

# Đọc manifest hiện tại
with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

# Thêm bản ghi kiểm tra repository
manifest.setdefault("macsql_repo_checks", [])

record = {
    "url": repo_url,
    "head_commit_at_check": commit_hash,
    "checked_utc": datetime.datetime.now(
        datetime.timezone.utc
    ).isoformat(),
    "source_code_inspected": False,
    "note": (
        "Chỉ ghi nhận commit hash. "
        "Chưa kiểm tra trực tiếp mã nguồn."
    )
}

manifest["macsql_repo_checks"].append(record)

# Lưu manifest
with manifest_path.open("w", encoding="utf-8") as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n===== KẾT QUẢ =====")
print("Đã ghi commit hash vào run_manifest.json.")
print("Đã kiểm tra mã nguồn:", False)
print("\nHOÀN THÀNH!")

===== KIỂM TRA REPOSITORY MAC-SQL =====
Repository: https://github.com/wbbeyourself/MAC-SQL.git
Commit hash: 31a9df5e0d520be4769be57a4b9022e5e34a14f4

===== KẾT QUẢ =====
Đã ghi commit hash vào run_manifest.json.
Đã kiểm tra mã nguồn: False

HOÀN THÀNH!


In [7]:
import os
import zipfile
import hashlib
import json
import posixpath
from collections import Counter
from pathlib import Path

print("===== CELL 7: KIỂM TRA SPIDER ZIP =====")

# 1. Xác định file Spider
SPIDER_ZIP = os.path.join(PARENT, "spider.zip")

if not os.path.isfile(SPIDER_ZIP):
    raise FileNotFoundError(
        f"Không tìm thấy file: {SPIDER_ZIP}"
    )

if not zipfile.is_zipfile(SPIDER_ZIP):
    raise ValueError("File không phải ZIP hợp lệ!")

# 2. Hàm tính SHA-256
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(block)
    return h.hexdigest()

ZIP_SHA256 = sha256_file(SPIDER_ZIP)

# Đường dẫn nhật ký thực nghiệm
MANIFEST_PATH = (
    Path(BASE) / "logs" / "run_manifest.json"
)

print("File:", SPIDER_ZIP)
print(
    "Dung lượng:",
    round(os.path.getsize(SPIDER_ZIP) / 1024**2, 2),
    "MiB"
)
print("SHA-256:", ZIP_SHA256)

# 3. Kiểm tra cấu trúc ZIP
NEEDED = [
    "train_spider.json",
    "dev.json",
    "tables.json"
]

with zipfile.ZipFile(SPIDER_ZIP, "r") as z:

    infos = z.infolist()
    names = [info.filename for info in infos]

    print("\n===== CẤU TRÚC ZIP =====")
    print("Tổng số mục:", len(names))

    tops = Counter(
        name.split("/")[0] for name in names
    )

    print("\nCác mục cấp cao nhất:")
    for name, count in tops.most_common(8):
        print(f"- {name}: {count} mục")

    # 4. Chuẩn hóa và kiểm tra đường dẫn
    normalized = [
        posixpath.normpath(
            name.replace("\\", "/")
        )
        for name in names
    ]

    unsafe = []

    for original, norm in zip(names, normalized):
        raw = original.replace("\\", "/")
        raw_parts = raw.split("/")

        if (
            raw.startswith("/")
            or ".." in raw_parts
            or norm.startswith("/")
            or norm == ".."
            or norm.startswith("../")
            or (
                len(norm) > 1
                and norm[1] == ":"
            )
        ):
            unsafe.append(original)

    files_only = [
        norm
        for norm, info in zip(normalized, infos)
        if not info.is_dir()
    ]

    duplicates = [
        name
        for name, count in Counter(files_only).items()
        if count > 1
    ]

    print("\n===== KIỂM TRA AN TOÀN =====")
    print(
        "Đường dẫn không an toàn:",
        unsafe[:5] if unsafe else "không"
    )
    print(
        "Tên file trùng:",
        duplicates[:5] if duplicates else "không"
    )

    # 5. Kiểm tra các file bắt buộc
    base_names = {
        posixpath.basename(name)
        for name in files_only
    }

    print("\n===== FILE QUAN TRỌNG =====")

    for filename in NEEDED:
        status = (
            "CÓ" if filename in base_names
            else "THIẾU"
        )
        print(f"{filename}: {status}")

    # 6. Đếm SQLite
    n_sqlite = sum(
        name.lower().endswith(".sqlite")
        for name in files_only
    )

    print("\n===== DATABASE =====")
    print("Số file SQLite:", n_sqlite)

    total_bytes = sum(
        info.file_size for info in infos
    )

    print(
        "Dung lượng sau giải nén:",
        round(total_bytes / 1024**3, 2),
        "GiB"
    )

    # 7. Kiểm tra tính toàn vẹn CRC
    print("\n===== KIỂM TRA CRC =====")

    crc_bad = z.testzip()

    print(
        "testzip (None = tốt):",
        crc_bad
    )

# 8. Tổng hợp kết quả
ZIP_OK = (
    not unsafe
    and not duplicates
    and all(
        name in base_names
        for name in NEEDED
    )
    and n_sqlite > 0
    and crc_bad is None
)

print("\n===== KẾT QUẢ CUỐI =====")
print("ZIP_OK =", ZIP_OK)

if not ZIP_OK:
    raise RuntimeError(
        "ZIP chưa đạt kiểm tra. "
        "Không được giải nén ở bước tiếp theo."
    )

print("\nHOÀN THÀNH!")

===== CELL 7: KIỂM TRA SPIDER ZIP =====
File: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/spider.zip
Dung lượng: 196.27 MiB
SHA-256: 00636695dabed6b5f4b8328a16b13e069a2f16591d5efcce57660669c85b121b

===== CẤU TRÚC ZIP =====
Tổng số mục: 2625

Các mục cấp cao nhất:
- spider_data: 1313 mục
- __MACOSX: 1312 mục

===== KIỂM TRA AN TOÀN =====
Đường dẫn không an toàn: không
Tên file trùng: không

===== FILE QUAN TRỌNG =====
train_spider.json: CÓ
dev.json: CÓ
tables.json: CÓ

===== DATABASE =====
Số file SQLite: 744
Dung lượng sau giải nén: 1.71 GiB

===== KIỂM TRA CRC =====
testzip (None = tốt): None

===== KẾT QUẢ CUỐI =====
ZIP_OK = True

HOÀN THÀNH!


In [8]:
import os
import json
import time
import shutil
import zipfile
import tempfile
from pathlib import Path

print("===== GIẢI NÉN SPIDER =====")

# Kiểm tra kết quả Cell 7
assert ZIP_OK, "Cell 7 chưa đạt kiểm tra ZIP!"

EXTRACT_DIR = Path("/content/spider_data")
MARKER = EXTRACT_DIR / ".extract_complete.json"

# Chỉ lấy dữ liệu Spider, bỏ qua metadata macOS
def is_spider_entry(name):
    normalized = name.replace("\\", "/")
    return normalized.startswith("spider_data/")

def find_root(directory):
    for root, dirs, files in os.walk(directory):
        if (
            all(name in files for name in NEEDED)
            and "database" in dirs
        ):
            return Path(root)
    return None

# Đếm file SQLite thực tế
def count_sqlite(directory):
    return sum(
        1
        for root, dirs, files in os.walk(directory)
        for name in files
        if name.lower().endswith(".sqlite")
    )

# Đếm SQLite trong phần dữ liệu cần giải nén
with zipfile.ZipFile(SPIDER_ZIP, "r") as z:
    selected = [
        info for info in z.infolist()
        if is_spider_entry(info.filename)
    ]

expected_sqlite = sum(
    info.filename.lower().endswith(".sqlite")
    for info in selected
    if not info.is_dir()
)

print("Số SQLite cần giải nén:", expected_sqlite)

# Kiểm tra dữ liệu đã giải nén trước đó
def marker_valid():
    if not MARKER.is_file():
        return False

    try:
        with MARKER.open("r", encoding="utf-8") as f:
            record = json.load(f)

        return (
            record.get("zip_sha256") == ZIP_SHA256
            and record.get("n_sqlite") == expected_sqlite
            and find_root(EXTRACT_DIR) is not None
            and count_sqlite(EXTRACT_DIR) == expected_sqlite
        )
    except (OSError, ValueError, TypeError):
        return False

if marker_valid():
    print("Dữ liệu đã giải nén và kiểm tra trước đó.")
    print("Bỏ qua giải nén lại.")

else:
    print("\nBắt đầu giải nén...")
    start = time.monotonic()

    # Giải nén vào thư mục tạm trước
    temp_dir = Path(
        tempfile.mkdtemp(
            prefix="spider_extract_",
            dir="/content"
        )
    )

    try:
        with zipfile.ZipFile(SPIDER_ZIP, "r") as z:
            root_resolved = temp_dir.resolve()

            for info in selected:
                # Chặn đường dẫn bất thường
                relative = Path(
                    info.filename.replace("\\", "/")
                )

                destination = (
                    temp_dir / relative
                ).resolve()

                if not destination.is_relative_to(
                    root_resolved
                ):
                    raise ValueError(
                        "Đường dẫn không an toàn: "
                        + info.filename
                    )

                if info.is_dir():
                    destination.mkdir(
                        parents=True,
                        exist_ok=True
                    )
                else:
                    destination.parent.mkdir(
                        parents=True,
                        exist_ok=True
                    )

                    with z.open(info, "r") as source:
                        with open(destination, "wb") as target:
                            shutil.copyfileobj(source, target)

        # Xác minh dữ liệu tạm
        spider_root = find_root(temp_dir)
        actual_sqlite = count_sqlite(temp_dir)

        assert spider_root is not None, (
            "Không tìm thấy đủ file JSON và database/"
        )

        assert actual_sqlite == expected_sqlite, (
            f"Số SQLite không khớp: "
            f"{actual_sqlite}/{expected_sqlite}"
        )

        # Chỉ thay thư mục chính khi dữ liệu tạm đã đạt
        if EXTRACT_DIR.exists():
            shutil.rmtree(EXTRACT_DIR)

        temp_dir.rename(EXTRACT_DIR)

        with MARKER.open("w", encoding="utf-8") as f:
            json.dump(
                {
                    "zip_sha256": ZIP_SHA256,
                    "n_sqlite": expected_sqlite,
                    "status": "extracted"
                },
                f,
                indent=2
            )

        print(
            "Giải nén xong sau:",
            round(time.monotonic() - start, 1),
            "giây"
        )

    finally:
        # Dọn thư mục tạm nếu còn tồn tại
        if temp_dir.exists():
            shutil.rmtree(temp_dir)

print("\n===== KIỂM TRA SAU GIẢI NÉN =====")

print("Thư mục:", EXTRACT_DIR)
print("Nội dung cấp 1:")

for name in sorted(os.listdir(EXTRACT_DIR)):
    print("-", name)

print("Số SQLite:", count_sqlite(EXTRACT_DIR))
print("Marker hợp lệ:", marker_valid())

assert marker_valid(), (
    "Dữ liệu giải nén chưa đạt kiểm tra!"
)

print("\nHOÀN THÀNH!")

===== GIẢI NÉN SPIDER =====
Số SQLite cần giải nén: 372

Bắt đầu giải nén...
Giải nén xong sau: 14.6 giây

===== KIỂM TRA SAU GIẢI NÉN =====
Thư mục: /content/spider_data
Nội dung cấp 1:
- .extract_complete.json
- spider_data
Số SQLite: 372
Marker hợp lệ: True

HOÀN THÀNH!


In [9]:
import os
from pathlib import Path

print("===== XÁC ĐỊNH SPIDER_ROOT =====")

# Hàm find_root đã được tạo trong Cell 8
SPIDER_ROOT = find_root(EXTRACT_DIR)

if SPIDER_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục Spider có đủ "
        "train_spider.json, dev.json, "
        "tables.json và database/"
    )

SPIDER_ROOT = Path(SPIDER_ROOT)
DB_DIR = SPIDER_ROOT / "database"

print("\nSPIDER_ROOT =", SPIDER_ROOT)
print("DB_DIR      =", DB_DIR)

print("\n===== NỘI DUNG THƯ MỤC GỐC =====")
for item in sorted(SPIDER_ROOT.iterdir()):
    print(
        "[DIR] " if item.is_dir() else "[FILE]",
        item.name
    )

print("\n===== KIỂM TRA FILE BẮT BUỘC =====")

required_files = [
    "train_spider.json",
    "dev.json",
    "tables.json"
]

for filename in required_files:
    path = SPIDER_ROOT / filename

    if not path.is_file():
        raise FileNotFoundError(
            f"Thiếu file: {path}"
        )

    print("✓", filename)

if not DB_DIR.is_dir():
    raise FileNotFoundError(
        f"Thiếu thư mục database: {DB_DIR}"
    )

print("✓ database/")

print("\nHOÀN THÀNH!")

===== XÁC ĐỊNH SPIDER_ROOT =====

SPIDER_ROOT = /content/spider_data/spider_data
DB_DIR      = /content/spider_data/spider_data/database

===== NỘI DUNG THƯ MỤC GỐC =====
[FILE] .DS_Store
[FILE] README.txt
[DIR]  database
[FILE] dev.json
[FILE] dev_gold.sql
[FILE] tables.json
[FILE] test.json
[DIR]  test_database
[FILE] test_gold.sql
[FILE] test_tables.json
[FILE] train_gold.sql
[FILE] train_others.json
[FILE] train_spider.json

===== KIỂM TRA FILE BẮT BUỘC =====
✓ train_spider.json
✓ dev.json
✓ tables.json
✓ database/

HOÀN THÀNH!


In [10]:
import json
from collections import Counter

print("===== THỐNG KÊ SPIDER =====")

# 1. Đọc dữ liệu JSON
with open(
    SPIDER_ROOT / "train_spider.json",
    "r",
    encoding="utf-8"
) as f:
    train = json.load(f)

with open(
    SPIDER_ROOT / "dev.json",
    "r",
    encoding="utf-8"
) as f:
    dev = json.load(f)

with open(
    SPIDER_ROOT / "tables.json",
    "r",
    encoding="utf-8"
) as f:
    tables = json.load(f)

# 2. Thống kê số mẫu
print("\n===== SỐ LƯỢNG DỮ LIỆU =====")

print("Số mẫu train:", len(train))
print("Số mẫu dev:", len(dev))
print("Số database trong tables.json:", len(tables))

# 3. Thống kê database
train_dbs = sorted({
    item["db_id"] for item in train
})

dev_dbs = sorted({
    item["db_id"] for item in dev
})

print("\n===== DATABASE =====")

print("Số database khác nhau trong train:", len(train_dbs))
print("Số database khác nhau trong dev:", len(dev_dbs))

print(
    "Database xuất hiện ở cả train và dev:",
    len(set(train_dbs) & set(dev_dbs))
)

# 4. Kiểm tra cấu trúc mẫu
print("\n===== CẤU TRÚC MẪU DEV =====")

print("Các trường dữ liệu:")
print(list(dev[0].keys()))

print("\nVí dụ một câu hỏi:")
print("db_id:", dev[0]["db_id"])
print("question:", dev[0]["question"])

# Không in gold SQL trong ví dụ dev
print("\nTrường 'query' có tồn tại:",
      "query" in dev[0])

# 5. Kiểm tra metadata
print("\n===== CẤU TRÚC TABLES.JSON =====")

print("Các trường metadata:")
print(list(tables[0].keys()))

# 6. Kiểm tra tính hợp lệ cơ bản
assert len(train) > 0, "Tập train rỗng!"
assert len(dev) > 0, "Tập dev rỗng!"
assert len(tables) > 0, "tables.json rỗng!"

print("\nHOÀN THÀNH!")

===== THỐNG KÊ SPIDER =====

===== SỐ LƯỢNG DỮ LIỆU =====
Số mẫu train: 7000
Số mẫu dev: 1034
Số database trong tables.json: 166

===== DATABASE =====
Số database khác nhau trong train: 140
Số database khác nhau trong dev: 20
Database xuất hiện ở cả train và dev: 0

===== CẤU TRÚC MẪU DEV =====
Các trường dữ liệu:
['db_id', 'query', 'query_toks', 'query_toks_no_value', 'question', 'question_toks', 'sql']

Ví dụ một câu hỏi:
db_id: concert_singer
question: How many singers do we have?

Trường 'query' có tồn tại: True

===== CẤU TRÚC TABLES.JSON =====
Các trường metadata:
['column_names', 'column_names_original', 'column_types', 'db_id', 'foreign_keys', 'primary_keys', 'table_names', 'table_names_original']

HOÀN THÀNH!


In [11]:
import os
import sqlite3
import time
from pathlib import Path

print("===== KIỂM TRA SQLITE =====")

# 1. Xác định đường dẫn database
def db_path(db_id):
    return DB_DIR / db_id / f"{db_id}.sqlite"

# 2. Mở SQLite ở chế độ chỉ đọc
def open_ro(db_id):
    path = db_path(db_id)

    if not path.is_file():
        raise FileNotFoundError(
            f"Không tìm thấy database: {path}"
        )

    return sqlite3.connect(
        path.resolve().as_uri() + "?mode=ro",
        uri=True
    )

# 3. Kiểm tra từng database
def check_db(db_id):
    try:
        with open_ro(db_id) as con:
            n_tables = con.execute(
                "SELECT COUNT(*) "
                "FROM sqlite_master "
                "WHERE type='table'"
            ).fetchone()[0]

            result = con.execute(
                "PRAGMA quick_check"
            ).fetchone()

            if result is None or result[0] != "ok":
                return {
                    "ok": False,
                    "reason": f"quick_check: {result}"
                }

            return {
                "ok": True,
                "tables": n_tables
            }

    except Exception as e:
        return {
            "ok": False,
            "reason": f"{type(e).__name__}: {e}"
        }

# 4. Kiểm tra toàn bộ database thuộc train và dev
all_dbs = sorted(set(train_dbs) | set(dev_dbs))

print("Số database cần kiểm tra:", len(all_dbs))

start = time.monotonic()

db_results = {}

for i, db_id in enumerate(all_dbs, start=1):
    db_results[db_id] = check_db(db_id)

    if i % 20 == 0 or i == len(all_dbs):
        print(
            f"Đã kiểm tra {i}/{len(all_dbs)} database"
        )

elapsed = time.monotonic() - start

# 5. Tổng hợp database lỗi
bad_train = [
    db_id for db_id in train_dbs
    if not db_results[db_id]["ok"]
]

bad_dev = [
    db_id for db_id in dev_dbs
    if not db_results[db_id]["ok"]
]

print("\n===== KẾT QUẢ SQLITE =====")

print("Thời gian:", round(elapsed, 2), "giây")
print("Database train lỗi:", bad_train or "không")
print("Database dev lỗi:", bad_dev or "không")

# 6. Kiểm tra metadata
schema_ids = {
    item["db_id"] for item in tables
}

missing_schema = sorted(
    set(all_dbs) - schema_ids
)

schema_complete = len(missing_schema) == 0

print("\n===== KIỂM TRA SCHEMA =====")

print(
    "Database thiếu trong tables.json:",
    missing_schema or "không"
)

# 7. Tổng kết
SQLITE_OK = (
    len(bad_train) == 0
    and len(bad_dev) == 0
    and schema_complete
)

print("\n===== KẾT QUẢ CUỐI =====")
print("SQLITE_OK =", SQLITE_OK)

if not SQLITE_OK:
    print("\nCHI TIẾT DATABASE LỖI:")

    for db_id in sorted(set(bad_train + bad_dev)):
        print(db_id, ":", db_results[db_id])

    raise RuntimeError(
        "Có database không đạt kiểm tra. "
        "Không chuyển sang Cell 12."
    )

print("\nHOÀN THÀNH!")

===== KIỂM TRA SQLITE =====
Số database cần kiểm tra: 160
Đã kiểm tra 20/160 database
Đã kiểm tra 40/160 database
Đã kiểm tra 60/160 database
Đã kiểm tra 80/160 database
Đã kiểm tra 100/160 database
Đã kiểm tra 120/160 database
Đã kiểm tra 140/160 database
Đã kiểm tra 160/160 database

===== KẾT QUẢ SQLITE =====
Thời gian: 0.53 giây
Database train lỗi: không
Database dev lỗi: không

===== KIỂM TRA SCHEMA =====
Database thiếu trong tables.json: không

===== KẾT QUẢ CUỐI =====
SQLITE_OK = True

HOÀN THÀNH!


In [12]:
import time
import sqlite3

print("===== KIỂM TRA THỰC THI SQL =====")

# 1. Hàm chạy SQL với giới hạn thời gian
def run_sql(db_id, sql, timeout_s=20):

    con = open_ro(db_id)

    # Xử lý một số chuỗi không chuẩn trong Spider
    con.text_factory = lambda b: b.decode(
        "utf-8", errors="ignore"
    )

    start = time.monotonic()

    # Ngắt truy vấn nếu vượt quá thời gian
    def progress_handler():
        elapsed = time.monotonic() - start
        return 1 if elapsed > timeout_s else 0

    con.set_progress_handler(
        progress_handler,
        10000
    )

    try:
        cursor = con.execute(sql)
        rows = cursor.fetchall()

        return {
            "ok": True,
            "rows": rows
        }

    except Exception as e:
        return {
            "ok": False,
            "error": (
                f"{type(e).__name__}: {e}"
            )
        }

    finally:
        con.close()

# 2. Lấy 5 mẫu đầu tiên từ tập TRAIN
smoke = train[:5]

smoke_ok = 0
smoke_results = []

print("\n===== THỰC THI 5 GOLD SQL (TRAIN) =====")

for i, example in enumerate(smoke, start=1):

    db_id = example["db_id"]
    gold_sql = example["query"]

    result = run_sql(
        db_id,
        gold_sql,
        timeout_s=20
    )

    smoke_results.append({
        "index": i,
        "db_id": db_id,
        "ok": result["ok"],
        "error": result.get("error")
    })

    if result["ok"]:
        smoke_ok += 1

        print(
            f"Mẫu {i} | {db_id} | "
            f"OK | {len(result['rows'])} dòng"
        )

    else:
        print(
            f"Mẫu {i} | {db_id} | "
            f"LỖI: {result['error']}"
        )

# 3. Tổng hợp kết quả
print("\n===== KẾT QUẢ CUỐI =====")

print(
    f"Gold SQL thực thi thành công: "
    f"{smoke_ok}/{len(smoke)}"
)

SMOKE_OK = smoke_ok == len(smoke)

print("SMOKE_OK =", SMOKE_OK)

if not SMOKE_OK:
    raise RuntimeError(
        "Có gold SQL không thực thi được. "
        "Cần kiểm tra trước khi tiếp tục."
    )

print("\nHOÀN THÀNH!")

===== KIỂM TRA THỰC THI SQL =====

===== THỰC THI 5 GOLD SQL (TRAIN) =====
Mẫu 1 | department_management | OK | 1 dòng
Mẫu 2 | department_management | OK | 10 dòng
Mẫu 3 | department_management | OK | 15 dòng
Mẫu 4 | department_management | OK | 1 dòng
Mẫu 5 | department_management | OK | 1 dòng

===== KẾT QUẢ CUỐI =====
Gold SQL thực thi thành công: 5/5
SMOKE_OK = True

HOÀN THÀNH!


In [13]:
import json
import os
import datetime
from pathlib import Path

print("===== GHI MANIFEST =====")

# 1. Xác định file manifest trên Google Drive
MANIFEST_PATH = (
    Path(BASE) / "logs" / "run_manifest.json"
)

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy manifest: {MANIFEST_PATH}"
    )

# 2. Tổng hợp các điều kiện kiểm tra
checks = {
    "zip_ok": bool(ZIP_OK),
    "extraction_marker_valid": bool(marker_valid()),
    "required_files_present": all(
        (SPIDER_ROOT / name).is_file()
        for name in NEEDED
    ),
    "sqlite_all_ok": bool(SQLITE_OK),
    "schema_complete": bool(schema_complete),
    "gold_smoke_train_5of5": bool(SMOKE_OK)
}

PHASE2_PASSED = all(checks.values())

print("\n===== KẾT QUẢ KIỂM TRA =====")

for name, passed in checks.items():
    print(
        f"{name}: {'PASS' if passed else 'FAIL'}"
    )

print("\nPHASE2_PASSED =", PHASE2_PASSED)

# 3. Đọc manifest hiện có
with MANIFEST_PATH.open(
    "r", encoding="utf-8"
) as f:
    manifest = json.load(f)

# 4. Tạo bản ghi Giai đoạn 2
record = {
    "phase": "2_spider_verification",
    "passed": PHASE2_PASSED,
    "time_utc": datetime.datetime.now(
        datetime.timezone.utc
    ).isoformat(),

    "checks": checks,

    "dataset": {
        "name": "Spider 1.0",
        "zip_path": str(SPIDER_ZIP),
        "zip_sha256": ZIP_SHA256,
        "spider_root": str(SPIDER_ROOT),

        "n_train": len(train),
        "n_dev": len(dev),
        "n_schema_databases": len(tables),

        "n_train_databases": len(train_dbs),
        "n_dev_databases": len(dev_dbs),
        "n_checked_databases": len(db_results),

        "bad_train_databases": bad_train,
        "bad_dev_databases": bad_dev,
        "missing_schema_ids": missing_schema
    },

    "integrity_check": {
        "method": "SQLite PRAGMA quick_check",
        "scope": "All databases used by train and dev",
        "database_count": len(db_results)
    },

    "smoke_test": {
        "split": "train_spider.json",
        "sample_count": len(smoke),
        "successful_count": smoke_ok,
        "all_passed": SMOKE_OK,
        "results": smoke_results
    },

    "file_hashes": {
        "train_spider_json": sha256_file(
            SPIDER_ROOT / "train_spider.json"
        ),
        "dev_json": sha256_file(
            SPIDER_ROOT / "dev.json"
        ),
        "tables_json": sha256_file(
            SPIDER_ROOT / "tables.json"
        )
    }
}

# 5. Lưu bản ghi, tránh ghi trùng khi chạy lại
dataset_checks = manifest.setdefault(
    "dataset_checks", []
)

dataset_checks = [
    item for item in dataset_checks
    if item.get("phase") != "2_spider_verification"
]

dataset_checks.append(record)
manifest["dataset_checks"] = dataset_checks

# 6. Ghi an toàn qua file tạm
temp_path = MANIFEST_PATH.with_suffix(".tmp")

with temp_path.open(
    "w", encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

os.replace(temp_path, MANIFEST_PATH)

print("\n===== ĐÃ LƯU MANIFEST =====")
print("Đường dẫn:", MANIFEST_PATH)

# 7. Đọc lại để xác nhận đã lưu
with MANIFEST_PATH.open(
    "r", encoding="utf-8"
) as f:
    saved = json.load(f)

saved_record = next(
    (
        item for item in saved["dataset_checks"]
        if item.get("phase")
        == "2_spider_verification"
    ),
    None
)

assert saved_record is not None, (
    "Không tìm thấy bản ghi Giai đoạn 2!"
)

assert saved_record["passed"] == PHASE2_PASSED

print(
    "Trạng thái đã lưu:",
    saved_record["passed"]
)

if not PHASE2_PASSED:
    raise RuntimeError(
        "chưa đạt tất cả điều kiện."
    )

print("\nHOÀN THÀNH!")

===== GHI MANIFEST =====

===== KẾT QUẢ KIỂM TRA =====
zip_ok: PASS
extraction_marker_valid: PASS
required_files_present: PASS
sqlite_all_ok: PASS
schema_complete: PASS
gold_smoke_train_5of5: PASS

PHASE2_PASSED = True

===== ĐÃ LƯU MANIFEST =====
Đường dẫn: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/logs/run_manifest.json
Trạng thái đã lưu: True

HOÀN THÀNH!


In [14]:
# CELL 14 — SURVEY SPIDER EVALUATORS
# Chỉ khảo sát mã nguồn, chưa chọn evaluator.

import ast
import sys
import json
import os
import sqlite3
import hashlib
import subprocess
import datetime
from pathlib import Path

print("===== KHẢO SÁT EVALUATOR =====")

REPOS = {
    "spider": "https://github.com/taoyds/spider.git",
    "test_suite": (
        "https://github.com/"
        "taoyds/test-suite-sql-eval.git"
    )
}

EVAL_ROOT = Path("/content/evaluators")
EVAL_ROOT.mkdir(parents=True, exist_ok=True)

# 1. Hàm chạy lệnh Git
def git_run(args, cwd=None):
    result = subprocess.run(
        ["git"] + args,
        cwd=cwd,
        capture_output=True,
        text=True,
        timeout=300
    )

    if result.returncode != 0:
        raise RuntimeError(
            "Git command failed:\n"
            + " ".join(args)
            + "\n"
            + result.stderr[-1500:]
        )

    return result.stdout.strip()

# 2. Hàm tính SHA-256
def hash_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

# 3. Kiểm tra môi trường
print("\n===== MÔI TRƯỜNG =====")
print("Python:", sys.version.split()[0])
print("SQLite:", sqlite3.sqlite_version)

survey = {}

# 4. Khảo sát từng repository
for name, url in REPOS.items():

    print(f"\n===== REPOSITORY: {name} =====")

    repo_dir = EVAL_ROOT / name

    if not repo_dir.exists():
        print("Đang clone:", url)

        git_run([
            "clone",
            "--depth", "1",
            url,
            str(repo_dir)
        ])

    # Không chấp nhận thư mục hỏng
    if not (repo_dir / ".git").exists():
        raise RuntimeError(
            f"Repository không hợp lệ: {repo_dir}"
        )

    # Xác minh nguồn Git
    actual_url = git_run(
        ["remote", "get-url", "origin"],
        cwd=repo_dir
    )

    if actual_url.rstrip("/") != url.rstrip("/"):
        raise RuntimeError(
            f"Git remote không khớp: {actual_url}"
        )

    # Lấy commit
    commit = git_run(
        ["rev-parse", "HEAD"],
        cwd=repo_dir
    )

    commit_date = git_run(
        ["log", "-1", "--format=%cI"],
        cwd=repo_dir
    )

    print("HEAD:", commit)
    print("Commit date:", commit_date)

    # 5. Liệt kê file Python
    py_files = sorted(
        p for p in repo_dir.rglob("*.py")
        if ".git" not in p.parts
    )

    print("Số file Python:", len(py_files))

    file_records = {}
    evaluation_candidates = []

    for path in py_files:
        relative = str(
            path.relative_to(repo_dir)
        )

        record = {
            "sha256": hash_file(path),
            "syntax_ok": False
        }

        try:
            source = path.read_text(
                encoding="utf-8"
            )

            tree = ast.parse(source)

            record["syntax_ok"] = True

            # Thu thập hàm và class
            functions = sorted({
                node.name
                for node in ast.walk(tree)
                if isinstance(
                    node,
                    (
                        ast.FunctionDef,
                        ast.AsyncFunctionDef
                    )
                )
            })

            classes = sorted({
                node.name
                for node in ast.walk(tree)
                if isinstance(
                    node,
                    ast.ClassDef
                )
            })

            record["functions"] = functions
            record["classes"] = classes

        except Exception as e:
            record["syntax_error"] = (
                f"{type(e).__name__}: {e}"
            )

        file_records[relative] = record

        # Tìm các file có thể liên quan đánh giá
        if any(
            word in path.name.lower()
            for word in [
                "eval",
                "exec",
                "test_suite"
            ]
        ):
            evaluation_candidates.append(relative)

    print("\nFile liên quan evaluator:")
    for item in evaluation_candidates[:30]:
        print(" -", item)

    # 6. Đọc README, tìm thông tin metric
    readmes = sorted(
        p for p in repo_dir.iterdir()
        if p.is_file()
        and p.name.lower().startswith("readme")
    )

    readme_hits = []

    for readme in readmes:
        text = readme.read_text(
            encoding="utf-8",
            errors="replace"
        )

        for line in text.splitlines():
            if any(
                keyword in line.lower()
                for keyword in [
                    "execution",
                    "test suite",
                    "test-suite",
                    "exact match",
                    "database",
                    "download",
                    "evaluate"
                ]
            ):
                cleaned = line.strip()

                if cleaned:
                    readme_hits.append(cleaned[:250])

    print("\nThông tin từ README:")
    for line in readme_hits[:25]:
        print(" -", line)

    survey[name] = {
        "url": url,
        "commit": commit,
        "commit_date": commit_date,
        "python_files": file_records,
        "evaluation_candidates": (
            evaluation_candidates
        ),
        "readme_hits": readme_hits[:100]
    }

# 7. Kiểm tra dữ liệu test-suite hiện có
print("\n===== KIỂM TRA TEST-SUITE DATA =====")

search_roots = [
    Path(PARENT),
    Path("/content")
]

data_candidates = []

for root in search_roots:
    if not root.exists():
        continue

    # Chỉ kiểm tra các tên thư mục gợi ý
    for pattern in [
        "*test*suite*",
        "database_ts",
        "*test*database*"
    ]:
        for path in root.glob(pattern):
            data_candidates.append(str(path))

data_candidates = sorted(set(data_candidates))

print("Đường dẫn ứng viên:")
if data_candidates:
    for path in data_candidates[:20]:
        print(" -", path)
else:
    print("Chưa phát hiện bằng cách tìm tên.")

print(
    "Lưu ý: Không tìm thấy không có nghĩa "
    "chắc chắn thiếu test-suite data."
)

# 8. Lưu báo cáo khảo sát riêng
SURVEY_PATH = (
    Path(BASE)
    / "logs"
    / "evaluator_survey_cell14.json"
)

survey_record = {
    "phase": "3_cell14_survey",
    "time_utc": datetime.datetime.now(
        datetime.timezone.utc
    ).isoformat(),
    "python_version": sys.version.split()[0],
    "sqlite_version": sqlite3.sqlite_version,
    "repositories": survey,
    "data_candidates": data_candidates,
    "decision_made": False
}

with SURVEY_PATH.open(
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        survey_record,
        f,
        ensure_ascii=False,
        indent=2
    )

# 9. Kiểm tra đã ghi thành công
with SURVEY_PATH.open(
    "r",
    encoding="utf-8"
) as f:
    saved = json.load(f)

assert saved["decision_made"] is False
assert len(saved["repositories"]) == 2

print("\n===== KẾT QUẢ CUỐI =====")
print("Số repository khảo sát:", len(survey))
print("Đã chọn evaluator:", saved["decision_made"])
print("Báo cáo:", SURVEY_PATH)

CELL14_OK = True

print("\nHOÀN THÀNH!")

===== KHẢO SÁT EVALUATOR =====

===== MÔI TRƯỜNG =====
Python: 3.13.16
SQLite: 3.45.1

===== REPOSITORY: spider =====
Đang clone: https://github.com/taoyds/spider.git
HEAD: b7b5b8c890cd30e35427348bb9eb8c6d1350ca7c
Commit date: 2024-05-29T11:16:42+08:00
Số file Python: 182


<unknown>:48: SyntaxWarning: invalid escape sequence '\d'
<unknown>:89: SyntaxWarning: invalid escape sequence '\d'
<unknown>:92: SyntaxWarning: invalid escape sequence '\d'
<unknown>:100: SyntaxWarning: invalid escape sequence '\d'
<unknown>:17: SyntaxWarning: invalid escape sequence '\.'
<unknown>:17: SyntaxWarning: invalid escape sequence '\.'



File liên quan evaluator:
 - baselines/nl2code/evaluation.py
 - evaluation.py

Thông tin từ README:
 - Spider is a large human-labeled dataset for complex and cross-domain semantic parsing and text-to-SQL task (natural language interfaces for relational databases). It is released along with our EMNLP 2018 paper: [Spider: A Large-Scale Human-Labeled Da
 - -`11/15/2020` We will use [Test Suite Accuracy](https://arxiv.org/abs/2010.02840) as our official evaluation metric for Spider, SParC, and CoSQL. Please find the evaluation code from [here](https://github.com/taoyds/test-suite-sql-eval).
 - - `08/03/2020` Corrected `column_name` and `column_name_original` mismatches in 2 dbs (`scholar` and `formula_1`) in `tables.json`, and reparsed SQL queries (this only affects some models (e.g. RATSQL) which use our parsed SQL as the SQL input). Plea
 - - `06/07/2020` We corrected some annotation errors and label mismatches (not errors) in Spider dev and test sets (~4% of dev examples updated, clic

<unknown>:127: SyntaxWarning: invalid escape sequence '\s'
<unknown>:11: SyntaxWarning: invalid escape sequence '\s'
<unknown>:57: SyntaxWarning: invalid escape sequence '\d'
<unknown>:62: SyntaxWarning: invalid escape sequence '\d'
<unknown>:70: SyntaxWarning: invalid escape sequence '\d'
<unknown>:206: SyntaxWarning: invalid escape sequence '\.'



===== KẾT QUẢ CUỐI =====
Số repository khảo sát: 2
Đã chọn evaluator: False
Báo cáo: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/logs/evaluator_survey_cell14.json

HOÀN THÀNH!


In [15]:
# KIỂM TRA API EVALUATOR
# Chỉ đọc mã nguồn, chưa chốt metric.

import ast
import re
import subprocess
from pathlib import Path

print("===== KIỂM TRA EVALUATOR =====")

expected_commits = {
    "spider": (
        "b7b5b8c890cd30e35427348bb9eb8c6d1350ca7c"
    ),
    "test_suite": (
        "e97acc546ecbee8fa27fa8dbf025ef61493a876c"
    )
}

for name, expected in expected_commits.items():

    repo = EVAL_ROOT / name

    actual = subprocess.check_output(
        ["git", "-C", str(repo), "rev-parse", "HEAD"],
        text=True
    ).strip()

    assert actual == expected, (
        f"Commit không khớp ở {name}: {actual}"
    )

    print(f"\n===== {name.upper()} =====")
    print("Commit:", actual)

    evaluation_file = repo / "evaluation.py"

    assert evaluation_file.is_file(), (
        f"Thiếu evaluation.py: {evaluation_file}"
    )

    source = evaluation_file.read_text(
        encoding="utf-8"
    )

    tree = ast.parse(source)

    print("\nClass:")
    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            print(" -", node.name)

            methods = [
                child.name
                for child in node.body
                if isinstance(
                    child,
                    (
                        ast.FunctionDef,
                        ast.AsyncFunctionDef
                    )
                )
            ]

            print("   Methods:", methods[:30])

    print("\nHàm cấp module:")
    for node in tree.body:
        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef
            )
        ):
            if any(
                key in node.name.lower()
                for key in [
                    "eval",
                    "exec",
                    "match",
                    "score"
                ]
            ):
                print(" -", node.name)

    print("\nCác dòng liên quan đến metric:")

    keywords = (
        r"etype|exec_match|eval_exec|"
        r"plug_value|keep_distinct|"
        r"test_suite|sqlite3.connect"
    )

    matches = [
        (i, line.strip())
        for i, line in enumerate(
            source.splitlines(), start=1
        )
        if re.search(keywords, line, re.I)
    ]

    for line_no, line in matches[:35]:
        print(f" L{line_no}: {line[:170]}")

    print("\nREADME: hướng dẫn dữ liệu")

    readmes = [
        p for p in repo.iterdir()
        if p.is_file()
        and p.name.lower().startswith("readme")
    ]

    for readme in readmes[:2]:
        lines = readme.read_text(
            encoding="utf-8",
            errors="replace"
        ).splitlines()

        for i, line in enumerate(lines):
            if any(
                word in line.lower()
                for word in [
                    "download the test suites",
                    "database/",
                    "plug_value",
                    "keep_distinct"
                ]
            ):
                print(
                    f" README L{i+1}: "
                    f"{line.strip()[:220]}"
                )

print("\n===== KIỂM TRA DỮ LIỆU =====")

print("Spider database:", DB_DIR)
print("Tồn tại:", DB_DIR.is_dir())

print("\nThư mục database trong test-suite repo:")

ts_db = EVAL_ROOT / "test_suite" / "database"

print("Đường dẫn:", ts_db)
print("Tồn tại:", ts_db.is_dir())

if ts_db.is_dir():
    print(
        "Số thư mục con:",
        sum(1 for p in ts_db.iterdir() if p.is_dir())
    )

print("\nHOÀN THÀNH!")

===== KIỂM TRA EVALUATOR =====

===== SPIDER =====
Commit: b7b5b8c890cd30e35427348bb9eb8c6d1350ca7c

Class:
 - Evaluator
   Methods: ['__init__', 'eval_hardness', 'eval_exact_match', 'eval_partial_match']

Hàm cấp module:
 - get_scores
 - eval_sel
 - eval_where
 - eval_group
 - eval_having
 - eval_order
 - eval_and_or
 - eval_nested
 - eval_IUEN
 - eval_keywords
 - print_scores
 - evaluate
 - eval_exec_match

Các dòng liên quan đến metric:
 L435: conn = sqlite3.connect(db)
 L444: def print_scores(scores, etype):
 L453: if etype in ["all", "exec"]:
 L458: if etype in ["all", "match"]:
 L478: def evaluate(gold, predict, db_dir, etype, kmaps):
 L546: if etype in ["all", "exec"]:
 L547: exec_score = eval_exec_match(db, p_str, g_str, p_sql, g_sql)
 L552: if etype in ["all", "match"]:
 L588: if etype in ["all", "exec"]:
 L591: if etype in ["all", "match"]:
 L611: print_scores(scores, etype)
 L614: def eval_exec_match(db, p_str, g_str, pred, gold):
 L619: conn = sqlite3.connect(db)
 L855: par

In [16]:
# KIỂM TRA TEST-SUITE EXECUTION
# Chỉ đọc mã nguồn, không thay đổi dữ liệu.

import ast
import re
from pathlib import Path

print("===== KIỂM TRA EXECUTION =====")

ts_repo = Path("/content/evaluators/test_suite")

files_to_inspect = [
    ts_repo / "exec_eval.py",
    ts_repo / "exec_subprocess.py",
]

for file_path in files_to_inspect:

    print(f"\n===== {file_path.name} =====")

    if not file_path.is_file():
        print("KHÔNG TÌM THẤY FILE")
        continue

    source = file_path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    tree = ast.parse(source)

    print("Số dòng:", len(source.splitlines()))

    print("\nHàm và class:")

    for node in tree.body:
        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
                ast.ClassDef
            )
        ):
            print(" -", node.name)

    print("\nCác dòng liên quan:")

    keywords = (
        r"def eval_exec_match|"
        r"def eval_exec|"
        r"sqlite3.connect|"
        r"db_dir|"
        r"test_suite|"
        r"timeout|"
        r"subprocess|"
        r"multiprocessing|"
        r"plug_value|"
        r"keep_distinct|"
        r"database"
    )

    matches = [
        (i, line.strip())
        for i, line in enumerate(
            source.splitlines(), start=1
        )
        if re.search(keywords, line, re.I)
    ]

    for line_no, line in matches[:60]:
        print(f"L{line_no}: {line[:180]}")

print("\n===== README: HƯỚNG DẪN DATABASE =====")

readme = ts_repo / "README.md"

if readme.is_file():

    lines = readme.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    for i, line in enumerate(lines):
        if any(
            key in line.lower()
            for key in [
                "download the test suites",
                "database/",
                "test suite accuracy",
                "execution accuracy with values"
            ]
        ):
            start = max(0, i - 1)
            end = min(len(lines), i + 3)

            print(f"\nREADME dòng {i+1}:")

            for j in range(start, end):
                print(f"  {j+1}: {lines[j][:220]}")

print("\n===== KIỂM TRA DATABASE =====")

spider_db = Path(DB_DIR)
test_suite_db = ts_repo / "database"

print("Spider DB tồn tại:", spider_db.is_dir())
print("Test-suite DB tồn tại:", test_suite_db.is_dir())

if test_suite_db.is_dir():
    entries = list(test_suite_db.iterdir())

    print("Số mục trong test-suite/database:",
          len(entries))

    print("Một số mục đầu:")
    for item in entries[:10]:
        print(" -", item.name)

print("\nHOÀN THÀNH!")

===== KIỂM TRA EXECUTION =====

===== exec_eval.py =====
Số dòng: 245

Hàm và class:
 - permute_tuple
 - unorder_row
 - quick_rej
 - multiset_eq
 - get_constraint_permutation
 - result_eq
 - replace_cur_year
 - get_cursor_from_path
 - exec_on_db_
 - exec_on_db
 - postprocess
 - eval_exec_match

Các dòng liên quan:
L14: import subprocess
L20: TIMEOUT = 60
L131: # get the database cursor for a sqlite database path
L136: connection = sqlite3.connect(sqlite_path)
L160: sqlite_path: str, query: str, process_id: str = "", timeout: int = TIMEOUT
L163: return await asyncio.wait_for(exec_on_db_(sqlite_path, query), timeout)
L164: except asyncio.TimeoutError:
L165: return ('exception', TimeoutError)
L178: # db is the database path
L179: # we are going to evaluate whether they are equivalent in all the databases
L184: def eval_exec_match(db: str, p_str: str, g_str: str, plug_value: bool, keep_distinct: bool, progress_bar_for_each_datapoint: bool) -> int:
L188: if not keep_distinct:
L199: # find a

<unknown>:127: SyntaxWarning: invalid escape sequence '\s'
<unknown>:11: SyntaxWarning: invalid escape sequence '\s'


In [17]:

# KIỂM TRA YÊU CẦU TEST-SUITE
# Chỉ đọc file và kiểm tra thư viện.

from pathlib import Path
import ast
import importlib.util
import re

print("===== TEST-SUITE REQUIREMENTS =====")

repo = Path("/content/evaluators/test_suite")

# 1. Đọc hướng dẫn trong database/
readme = repo / "database" / "readme.txt"

print("\n===== DATABASE README =====")

if readme.is_file():
    print(
        readme.read_text(
            encoding="utf-8",
            errors="replace"
        )[:5000]
    )
else:
    print("Không tìm thấy database/readme.txt")

# 2. Kiểm tra thư viện
print("\n===== THƯ VIỆN =====")

for name in [
    "sqlparse",
    "nltk",
    "tqdm"
]:
    spec = importlib.util.find_spec(name)
    print(
        f"{name}:",
        "Đã cài" if spec else "Chưa cài"
    )

# 3. Đọc cấu trúc exec_eval.py
source_file = repo / "exec_eval.py"

print("\n===== EXEC_EVAL.PY =====")

source = source_file.read_text(
    encoding="utf-8"
)

tree = ast.parse(source)

for node in tree.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef
        )
    ):

        print(
            f"\nFUNCTION: {node.name}"
            f" (lines {node.lineno}-{node.end_lineno})"
        )

        if any(
            key in node.name.lower()
            for key in [
                "exec",
                "db",
                "suite"
            ]
        ):

            lines = source.splitlines()

            for i in range(
                node.lineno - 1,
                min(node.end_lineno, node.lineno + 35)
            ):
                print(
                    f"{i+1}: {lines[i][:180]}"
                )

# 4. Tìm các dòng chỉ định database
print("\n===== DATABASE PATH LOGIC =====")

for i, line in enumerate(
    source.splitlines(),
    start=1
):
    if re.search(
        r"glob|os\.listdir|os\.walk|"
        r"db_dir|db_path|test_suite|"
        r"sqlite|database",
        line,
        re.I
    ):
        print(f"L{i}: {line.strip()[:180]}")

print("\nHOÀN THÀNH!")

===== TEST-SUITE REQUIREMENTS =====

===== DATABASE README =====
Please download the database from the goolge drive link mentioned in the repo-level readme and decompress in this directory.
After this step, "test-suite-sql-eval/database/atis/atis.sqlite" should be a valid file path.

===== THƯ VIỆN =====
sqlparse: Đã cài
nltk: Đã cài
tqdm: Đã cài

===== EXEC_EVAL.PY =====

FUNCTION: permute_tuple (lines 23-25)

FUNCTION: unorder_row (lines 28-29)

FUNCTION: quick_rej (lines 36-42)

FUNCTION: multiset_eq (lines 46-56)

FUNCTION: get_constraint_permutation (lines 59-73)

FUNCTION: result_eq (lines 77-122)

FUNCTION: replace_cur_year (lines 125-128)

FUNCTION: get_cursor_from_path (lines 132-142)

FUNCTION: exec_on_db_ (lines 145-157)
145: async def exec_on_db_(sqlite_path: str, query: str) -> Tuple[str, Any]:
146:     query = replace_cur_year(query)
147:     cursor = get_cursor_from_path(sqlite_path)
148:     try:
149:         cursor.execute(query)
150:         result = cursor.fetchall()

<unknown>:127: SyntaxWarning: invalid escape sequence '\s'


In [18]:

# KIỂM TRA DUNG LƯỢNG
# Chưa tải test-suite database.

import shutil
from pathlib import Path

print("===== STORAGE CHECK =====")

colab_usage = shutil.disk_usage("/content")

print("\n===== COLAB =====")
print(
    "Dung lượng trống:",
    round(colab_usage.free / (1024**3), 2),
    "GiB"
)

drive_usage = shutil.disk_usage(BASE)

print("\n===== GOOGLE DRIVE =====")
print(
    "Dung lượng trống:",
    round(drive_usage.free / (1024**3), 2),
    "GiB"
)

print("\n===== VỊ TRÍ DỰ KIẾN =====")

target = (
    Path(BASE)
    / "data"
    / "test_suite_archive"
)

print("Nơi lưu file nén:", target)

extract_target = Path(
    "/content/test_suite_databases"
)

print("Nơi giải nén:", extract_target)

print("\nHOÀN THÀNH!")

===== STORAGE CHECK =====

===== COLAB =====
Dung lượng trống: 110.5 GiB

===== GOOGLE DRIVE =====
Dung lượng trống: 104.97 GiB

===== VỊ TRÍ DỰ KIẾN =====
Nơi lưu file nén: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/data/test_suite_archive
Nơi giải nén: /content/test_suite_databases

HOÀN THÀNH!


In [20]:

# CELL 15E — RESTORE TEST-SUITE TỪ GOOGLE DRIVE CÁ NHÂN

from pathlib import Path
import hashlib
import zipfile

print("===== KHÔI PHỤC TEST-SUITE =====")

# Kết nối Drive nếu chưa kết nối
from google.colab import drive

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

# Thư mục chứa bản sao test-suite đã lưu
parent = Path(
    "/content/drive/MyDrive/0. Classroom/3. NĂM 3/"
    "HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/"
    "bài báo nghiên cứu"
)

archive_path = parent / "Bản sao của testsuitedatabases.zip"

assert archive_path.is_file(), (
    f"Không tìm thấy bản sao test-suite:\n{archive_path}"
)

print("Đã tìm thấy:", archive_path.name)
print(
    "Kích thước:",
    round(archive_path.stat().st_size / 1024**3, 3),
    "GiB"
)

# Xác minh bản sao đúng với file đã kiểm tra trước đó
expected_sha256 = (
    "9ec24ea8debc6bd04abfe137b5f1a739"
    "b5a8836f32c0464e4dfc94eb7f41da96"
)

h = hashlib.sha256()

with archive_path.open("rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        h.update(chunk)

actual_sha256 = h.hexdigest()

print("SHA-256:", actual_sha256)

assert actual_sha256 == expected_sha256, (
    "SHA-256 không khớp. Dừng để tránh sử dụng sai dữ liệu."
)

print("Kiểm tra SHA-256: PASS")

# Giải nén vào runtime
extract_dir = Path("/content/test_suite_databases")
suite_root = extract_dir / "database"

if suite_root.is_dir() and any(suite_root.rglob("*.sqlite")):
    print("Test-suite đã được giải nén, bỏ qua.")
else:
    print("Đang giải nén test-suite...")

    with zipfile.ZipFile(archive_path) as zf:
        bad_file = zf.testzip()
        assert bad_file is None, f"File ZIP bị lỗi: {bad_file}"

        zf.extractall(extract_dir)

# Kiểm tra kết quả
assert suite_root.is_dir(), (
    "Không tìm thấy thư mục database sau giải nén"
)

sqlite_files = list(suite_root.rglob("*.sqlite"))

print("\n===== KẾT QUẢ =====")
print("Thư mục:", suite_root)
print("Số file SQLite:", len(sqlite_files))

assert len(sqlite_files) == 3194, (
    "Số file SQLite không khớp với lần khôi phục trước"
)

print("\nHOÀN THÀNH!")

===== KHÔI PHỤC TEST-SUITE =====
Đã tìm thấy: Bản sao của testsuitedatabases.zip
Kích thước: 1.182 GiB
SHA-256: 9ec24ea8debc6bd04abfe137b5f1a739b5a8836f32c0464e4dfc94eb7f41da96
Kiểm tra SHA-256: PASS
Đang giải nén test-suite...

===== KẾT QUẢ =====
Thư mục: /content/test_suite_databases/database
Số file SQLite: 3194

HOÀN THÀNH!


In [22]:

# CELL 15F — ZIP STRUCTURE & SAFETY AUDIT
# Kiểm tra metadata, KHÔNG giải nén.

from pathlib import Path, PurePosixPath
from collections import Counter
import zipfile
import stat
import re

print("===== ZIP AUDIT =====")

# Dùng bản sao ZIP trên Google Drive
archive = Path(
    "/content/drive/MyDrive/0. Classroom/3. NĂM 3/"
    "HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/"
    "bài báo nghiên cứu/Bản sao của testsuitedatabases.zip"
)

assert archive.is_file(), f"Không tìm thấy ZIP: {archive}"
assert zipfile.is_zipfile(archive), "File không phải ZIP hợp lệ"

with zipfile.ZipFile(archive, "r") as zf:
    entries = zf.infolist()

    files = [x for x in entries if not x.is_dir()]
    sqlite_files = [
        x for x in files
        if x.filename.lower().endswith(".sqlite")
    ]

    total_size = sum(x.file_size for x in files)
    compressed_size = sum(x.compress_size for x in files)

    unsafe = []
    names_seen = set()
    duplicate_names = []

    for entry in entries:
        name = entry.filename
        path = PurePosixPath(name)

        bad_path = (
            path.is_absolute()
            or ".." in path.parts
            or "\\" in name
            or bool(re.match(r"^[A-Za-z]:", name))
            or "\x00" in name
        )

        mode = entry.external_attr >> 16
        is_symlink = stat.S_ISLNK(mode)

        if bad_path or is_symlink:
            unsafe.append(name)

        if name in names_seen:
            duplicate_names.append(name)

        names_seen.add(name)

    top_levels = Counter(
        PurePosixPath(x.filename).parts[0]
        for x in files
        if PurePosixPath(x.filename).parts
    )

    sqlite_by_parent = Counter(
        str(PurePosixPath(x.filename).parent)
        for x in sqlite_files
    )

    print("\n===== THỐNG KÊ =====")
    print("Tổng entries:", len(entries))
    print("Tổng file:", len(files))
    print("File SQLite:", len(sqlite_files))
    print(
        "Dung lượng giải nén:",
        round(total_size / 1024**3, 3),
        "GiB"
    )
    print(
        "Dung lượng nén cộng dồn:",
        round(compressed_size / 1024**3, 3),
        "GiB"
    )

    print("\n===== THƯ MỤC CẤP CAO =====")
    for name, count in top_levels.most_common(20):
        print(f"{name}: {count} file")

    print("\n===== 15 SQLITE ĐẦU TIÊN =====")
    for entry in sqlite_files[:15]:
        print(
            entry.filename,
            "|",
            round(entry.file_size / 1024**2, 2),
            "MiB"
        )

    print("\n===== THƯ MỤC CHỨA NHIỀU SQLITE =====")
    for name, count in sqlite_by_parent.most_common(15):
        print(f"{name}: {count}")

    print("\n===== KIỂM TRA AN TOÀN =====")
    print("Đường dẫn nguy hiểm:", len(unsafe))
    print("Tên trùng lặp:", len(duplicate_names))

    for name in unsafe[:10]:
        print("UNSAFE:", name)

    for name in duplicate_names[:10]:
        print("DUPLICATE:", name)

    if not sqlite_files:
        raise RuntimeError(
            "Không tìm thấy file SQLite trong ZIP."
        )

    if unsafe or duplicate_names:
        raise RuntimeError(
            "ZIP có đường dẫn bất thường hoặc trùng lặp."
        )

print("\nHOÀN THÀNH!")


===== ZIP AUDIT =====

===== THỐNG KÊ =====
Tổng entries: 3999
Tổng file: 3944
File SQLite: 3889
Dung lượng giải nén: 4.801 GiB
Dung lượng nén cộng dồn: 1.181 GiB

===== THƯ MỤC CẤP CAO =====
database: 3211 file
__MACOSX: 733 file

===== 15 SQLITE ĐẦU TIÊN =====
database/voter_1/voter_1v515patch1.sqlite | 0.02 MiB
__MACOSX/database/voter_1/._voter_1v515patch1.sqlite | 0.0 MiB
database/voter_1/voter_1v515nightpatch3.sqlite | 0.02 MiB
__MACOSX/database/voter_1/._voter_1v515nightpatch3.sqlite | 0.0 MiB
database/voter_1/voter_1v5083.sqlite | 0.02 MiB
__MACOSX/database/voter_1/._voter_1v5083.sqlite | 0.0 MiB
database/voter_1/voter_1v5081.sqlite | 0.04 MiB
__MACOSX/database/voter_1/._voter_1v5081.sqlite | 0.0 MiB
database/voter_1/voter_1.sqlite | 0.02 MiB
__MACOSX/database/voter_1/._voter_1.sqlite | 0.0 MiB
database/voter_1/voter_1v515patch3.sqlite | 0.02 MiB
__MACOSX/database/voter_1/._voter_1v515patch3.sqlite | 0.0 MiB
database/voter_1/voter_1v515nightpatch1.sqlite | 0.03 MiB
__MACOSX/data

In [24]:


# CELL 15G — SAFE TEST-SUITE EXTRACTION
# Tương thích với CELL 15E mới
# Không giải nén lại nếu dữ liệu đã đầy đủ

from pathlib import Path, PurePosixPath
from collections import Counter
import zipfile
import shutil
import stat
import sqlite3

print("===== SAFE TEST-SUITE EXTRACTION =====")

# 1. Đường dẫn bản ZIP đã lưu trên Google Drive
archive = Path(
    "/content/drive/MyDrive/0. Classroom/3. NĂM 3/"
    "HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/"
    "bài báo nghiên cứu/Bản sao của testsuitedatabases.zip"
)

target = Path("/content/test_suite_databases")
database_root = target / "database"

EXPECTED_SQLITE = 3194
EXPECTED_GROUPS = 28

assert archive.is_file(), "Không tìm thấy ZIP trên Google Drive"
assert zipfile.is_zipfile(archive), "ZIP không hợp lệ"

# 2. Kiểm tra dữ liệu đã giải nén
existing_sqlite = (
    list(database_root.rglob("*.sqlite"))
    if database_root.is_dir()
    else []
)

if len(existing_sqlite) == EXPECTED_SQLITE:
    print("Test-suite đã có đủ 3194 file SQLite.")
    print("Bỏ qua giải nén để tránh ghi đè.")

else:
    print("Số SQLite hiện có:", len(existing_sqlite))

    # Không giải nén đè lên dữ liệu chưa đầy đủ
    if target.exists() and any(target.iterdir()):
        raise RuntimeError(
            "Thư mục đích đã có dữ liệu nhưng chưa đầy đủ. "
            "Dừng để tránh ghi đè hoặc trộn dữ liệu."
        )

    print("Bắt đầu giải nén an toàn...")

    target.mkdir(parents=True, exist_ok=True)

    selected = []
    total_bytes = 0

    with zipfile.ZipFile(archive, "r") as zf:

        for info in zf.infolist():

            if info.is_dir():
                continue

            name = info.filename
            parts = PurePosixPath(name).parts

            if not parts or parts[0] != "database":
                continue

            # Chặn đường dẫn không an toàn
            if (
                name.startswith("/")
                or "\\" in name
                or ".." in parts
                or any(p.startswith("._") for p in parts)
                or ".DS_Store" in parts
            ):
                raise RuntimeError(
                    "Đường dẫn không an toàn: " + name
                )

            mode = info.external_attr >> 16

            if stat.S_ISLNK(mode):
                raise RuntimeError(
                    "Phát hiện symlink: " + name
                )

            selected.append(info)
            total_bytes += info.file_size

        print("File sẽ giải nén:", len(selected))
        print(
            "Dung lượng dự kiến:",
            round(total_bytes / 1024**3, 3),
            "GiB"
        )

        if total_bytes > 12 * 1024**3:
            raise RuntimeError(
                "Dung lượng vượt ngưỡng an toàn 12 GiB."
            )

        for index, info in enumerate(selected, start=1):

            relative = PurePosixPath(info.filename)
            destination = target.joinpath(*relative.parts)

            destination.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            with zf.open(info, "r") as src:
                with destination.open("xb") as dst:
                    shutil.copyfileobj(
                        src,
                        dst,
                        length=1024 * 1024
                    )

            if index % 500 == 0:
                print(
                    f"Đã giải nén {index}/{len(selected)} file"
                )

# 3. Kiểm tra kết quả
print("\n===== KIỂM TRA SAU GIẢI NÉN =====")

assert database_root.is_dir(), (
    "Thiếu thư mục database"
)

sqlite_paths = sorted(
    database_root.rglob("*.sqlite")
)

print("Số SQLite:", len(sqlite_paths))

db_groups = {
    p.relative_to(database_root).parts[0]
    for p in sqlite_paths
}

print("Số nhóm database:", len(db_groups))

assert len(sqlite_paths) == EXPECTED_SQLITE, (
    "Số file SQLite không khớp"
)

assert len(db_groups) == EXPECTED_GROUPS, (
    "Số nhóm database không khớp"
)

group_counts = Counter(
    p.relative_to(database_root).parts[0]
    for p in sqlite_paths
)

print("\n10 nhóm có nhiều SQLite nhất:")

for name, count in group_counts.most_common(10):
    print(f" - {name}: {count}")

# 4. Kiểm tra tính toàn vẹn SQLite
print("\n===== SQLITE QUICK CHECK =====")

for path in sqlite_paths[:5]:

    conn = sqlite3.connect(
        f"file:{path.as_posix()}?mode=ro",
        uri=True
    )

    try:
        result = conn.execute(
            "PRAGMA quick_check"
        ).fetchone()
    finally:
        conn.close()

    status = result[0] if result else "UNKNOWN"

    print(path.name, "=>", status)

    assert status == "ok", (
        f"SQLite không hợp lệ: {path}"
    )

print("\n===== KẾT LUẬN =====")
print("Test-suite: PASS")
print("SQLite:", len(sqlite_paths))
print("Database groups:", len(db_groups))
print("HOÀN THÀNH!")

===== SAFE TEST-SUITE EXTRACTION =====
Test-suite đã có đủ 3194 file SQLite.
Bỏ qua giải nén để tránh ghi đè.

===== KIỂM TRA SAU GIẢI NÉN =====
Số SQLite: 3194
Số nhóm database: 28

10 nhóm có nhiều SQLite nhất:
 - atis: 663
 - advising: 532
 - academic: 412
 - yelp: 283
 - scholar: 208
 - imdb: 201
 - geography: 109
 - restaurants: 91
 - car_1: 60
 - flight_2: 56

===== SQLITE QUICK CHECK =====
academic.sqlite => ok
academicv1223round10group0.sqlite => ok
academicv1223round10group1.sqlite => ok
academicv1223round10group10.sqlite => ok
academicv1223round10group11.sqlite => ok

===== KẾT LUẬN =====
Test-suite: PASS
SQLite: 3194
Database groups: 28
HOÀN THÀNH!


In [25]:

# CELL 15H — SPIDER DEV TEST-SUITE COVERAGE
# Kiểm tra toàn bộ database của Spider dev.

from pathlib import Path
from collections import Counter
import sqlite3
import time

print("===== SPIDER DEV COVERAGE =====")

test_suite_root = (
    Path("/content/test_suite_databases")
    / "database"
)

assert test_suite_root.is_dir(), (
    "Không tìm thấy test-suite database."
)

assert isinstance(dev, list) and len(dev) > 0, (
    "Biến dev chưa được tải."
)

# 1. Danh sách database trong Spider dev
dev_db_counts = Counter(
    item["db_id"]
    for item in dev
)

dev_db_ids = sorted(dev_db_counts)

print("\n===== SPIDER DEV =====")
print("Số câu hỏi dev:", len(dev))
print("Số db_id:", len(dev_db_ids))

# 2. Đối chiếu thư mục test-suite
print("\n===== TEST-SUITE COVERAGE =====")

coverage = []
missing = []

for db_id in dev_db_ids:

    folder = test_suite_root / db_id

    sqlite_files = (
        sorted(folder.glob("*.sqlite"))
        if folder.is_dir()
        else []
    )

    original = folder / f"{db_id}.sqlite"

    record = {
        "db_id": db_id,
        "dev_questions": dev_db_counts[db_id],
        "sqlite_count": len(sqlite_files),
        "original_exists": original.is_file()
    }

    coverage.append(record)

    if not sqlite_files:
        missing.append(db_id)

    print(
        f"{db_id:35s}"
        f" questions={record['dev_questions']:3d}"
        f" sqlite={record['sqlite_count']:3d}"
        f" original={record['original_exists']}"
    )

print("\n===== TỔNG KẾT COVERAGE =====")
print("Database dev:", len(dev_db_ids))
print(
    "Database có test-suite:",
    sum(x["sqlite_count"] > 0 for x in coverage)
)
print("Database thiếu:", missing)

# 3. Kiểm tra SQLite chỉ đọc
print("\n===== SQLITE INTEGRITY =====")

checked = 0
failed = []
start = time.monotonic()

for record in coverage:

    db_id = record["db_id"]
    folder = test_suite_root / db_id

    if not folder.is_dir():
        continue

    for path in sorted(folder.glob("*.sqlite")):

        try:
            # URI read-only: không sửa database
            uri = path.resolve().as_uri() + "?mode=ro"

            conn = sqlite3.connect(
                uri,
                uri=True,
                timeout=5
            )

            try:
                result = conn.execute(
                    "PRAGMA quick_check"
                ).fetchone()
            finally:
                conn.close()

            if not result or result[0] != "ok":
                failed.append(
                    (str(path), str(result))
                )

        except Exception as exc:
            failed.append(
                (
                    str(path),
                    f"{type(exc).__name__}: {exc}"
                )
            )

        checked += 1

        if checked % 100 == 0:
            print(
                f"Đã kiểm tra {checked} file..."
            )

elapsed = time.monotonic() - start

print("\n===== KẾT QUẢ CUỐI =====")
print("SQLite đã kiểm tra:", checked)
print("SQLite lỗi:", len(failed))
print("Thời gian:", round(elapsed, 2), "giây")

for path, error in failed[:10]:
    print("LỖI:", path, error)

if missing:
    print(
        "\nCẢNH BÁO: Thiếu test-suite cho:",
        missing
    )

if failed:
    print(
        "\nCẢNH BÁO: Có SQLite không vượt qua quick_check."
    )

CELL15H_OK = (
    len(missing) == 0
    and len(failed) == 0
    and all(
        x["original_exists"]
        for x in coverage
    )
)

print("\nCELL15H_OK =", CELL15H_OK)
print("HOÀN THÀNH!")

===== SPIDER DEV COVERAGE =====

===== SPIDER DEV =====
Số câu hỏi dev: 1034
Số db_id: 20

===== TEST-SUITE COVERAGE =====
battle_death                        questions= 16 sqlite= 28 original=True
car_1                               questions= 92 sqlite= 60 original=True
concert_singer                      questions= 45 sqlite= 32 original=True
course_teach                        questions= 30 sqlite= 25 original=True
cre_Doc_Template_Mgt                questions= 84 sqlite= 25 original=True
dog_kennels                         questions= 82 sqlite= 48 original=True
employee_hire_evaluation            questions= 38 sqlite= 27 original=True
flight_2                            questions= 80 sqlite= 56 original=True
museum_visit                        questions= 18 sqlite= 35 original=True
network_1                           questions= 56 sqlite= 29 original=True
orchestra                           questions= 40 sqlite= 28 original=True
pets_1                              questions= 42 sq

In [26]:

# CELL 15I-FIX — EVALUATOR SUBPROCESS SMOKE TEST

from pathlib import Path
import subprocess
import sys
import json
import time

print("===== FIX =====")

repo = Path("/content/evaluators/test_suite")
suite_root = Path("/content/test_suite_databases/database")

assert (repo / "exec_eval.py").is_file()
assert suite_root.is_dir()

# Tạo script chạy ngoài Jupyter event loop
smoke_script = repo / "_mac_sql_smoke_test.py"

smoke_script.write_text(
    """
import sys
import json

sys.path.insert(0, "/content/evaluators/test_suite")

from exec_eval import eval_exec_match

db_path = (
    "/content/test_suite_databases/database/"
    "concert_singer/concert_singer.sqlite"
)

score = eval_exec_match(
    db_path,
    "SELECT 1",
    "SELECT 1",
    plug_value=False,
    keep_distinct=True,
    progress_bar_for_each_datapoint=False
)

print("MAC_SQL_SMOKE_RESULT=" + json.dumps({
    "score": score
}))
""",
    encoding="utf-8"
)

print("Đang kiểm thử evaluator trong subprocess...")

start = time.monotonic()

try:
    proc = subprocess.run(
        [sys.executable, str(smoke_script)],
        capture_output=True,
        text=True,
        timeout=60,
        cwd=str(repo)
    )
except subprocess.TimeoutExpired:
    raise RuntimeError(
        "Evaluator vượt quá 60 giây. "
        "Cần kiểm tra cơ chế thực thi trước khi tiếp tục."
    )

elapsed = time.monotonic() - start

print("Return code:", proc.returncode)
print("Elapsed:", round(elapsed, 3), "seconds")

if proc.stderr.strip():
    print("\nSTDERR:")
    print(proc.stderr[-2000:])

if proc.returncode != 0:
    print("\nSTDOUT:")
    print(proc.stdout[-2000:])
    raise RuntimeError(
        "Evaluator subprocess thất bại."
    )

marker = "MAC_SQL_SMOKE_RESULT="

result_lines = [
    line
    for line in proc.stdout.splitlines()
    if line.startswith(marker)
]

assert len(result_lines) == 1, (
    "Không tìm thấy kết quả evaluator hợp lệ. "
    f"STDOUT: {proc.stdout[-2000:]}"
)

result = json.loads(
    result_lines[0][len(marker):]
)

score = result["score"]

print("\n===== OFFICIAL EVALUATOR SMOKE TEST =====")
print("Predicted SQL: SELECT 1")
print("Gold SQL: SELECT 1")
print("Score:", score)

assert score == 1, (
    "Evaluator không trả về 1 "
    "cho hai câu SQL giống nhau."
)

# Ghi cấu hình chỉ sau khi smoke test đạt
commit = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"],
    text=True
).strip()

config = {
    "project": "MAC-SQL Lite v3.1",
    "dataset": "Spider 1.0",
    "metric": "Test Suite Execution Accuracy",
    "evaluator_repository": "taoyds/test-suite-sql-eval",
    "evaluator_commit": commit,
    "evaluator_path": str(repo),
    "test_suite_root": str(suite_root),
    "plug_value": False,
    "keep_distinct": True,
    "execution_method": "subprocess",
    "smoke_test": {
        "db_id": "concert_singer",
        "sql": "SELECT 1",
        "score": score,
        "elapsed_seconds": elapsed
    },
    "status": "smoke_test_passed"
}

config_path = (
    Path(BASE)
    / "configs"
    / "evaluator_config.json"
)

config_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

config_path.write_text(
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("\nConfig saved:", config_path)
print("CELL HOÀN THÀNH!")

===== FIX =====
Đang kiểm thử evaluator trong subprocess...
Return code: 0
Elapsed: 0.298 seconds

STDERR:
/content/evaluators/test_suite/exec_eval.py:127: SyntaxWarning: invalid escape sequence '\s'
  "YEAR\s*\(\s*CURDATE\s*\(\s*\)\s*\)\s*", "2020", query, flags=re.IGNORECASE
/content/evaluators/test_suite/parse.py:57: SyntaxWarning: invalid escape sequence '\d'
  float_nums = re.findall("[-+]?\d*\.\d+", query)
/content/evaluators/test_suite/parse.py:62: SyntaxWarning: invalid escape sequence '\d'
  int_nums = [i.strip() for i in re.findall("[^tT]\d+", query)]
/content/evaluators/test_suite/parse.py:70: SyntaxWarning: invalid escape sequence '\d'
  table = re.findall("[Tt]\d+\.", tok)
/content/evaluators/test_suite/parse.py:206: SyntaxWarning: invalid escape sequence '\.'
  for table, col, val1, val2 in re.findall('(?:([^\.\s]*)\.)?([^\.\s]+) between ([^\s;]+) and ([^\s;]+)', query, re.IGNORECASE):


===== OFFICIAL EVALUATOR SMOKE TEST =====
Predicted SQL: SELECT 1
Gold SQL: SELECT 1


In [27]:

# CELL 16 — SAFE SQL EXECUTION
# MAC-SQL Lite v3.1

from pathlib import Path
import subprocess
import sys
import json
import time
import textwrap

print("===== SAFE SQL EXECUTION =====")

WORKER_PATH = Path("/content/mac_sql_safe_worker.py")

WORKER_CODE = r'''
import sys
import json
import sqlite3
from pathlib import Path

def main():
    payload = json.load(sys.stdin)

    db_path = Path(payload["db_path"]).resolve()
    sql = payload["sql"]
    max_rows = int(payload["max_rows"])

    if not db_path.is_file():
        raise FileNotFoundError(str(db_path))

    if not isinstance(sql, str) or not sql.strip():
        raise ValueError("Empty SQL")

    if not (1 <= max_rows <= 10000):
        raise ValueError("Invalid max_rows")

    uri = db_path.as_uri() + "?mode=ro"

    conn = sqlite3.connect(
        uri,
        uri=True,
        timeout=2
    )

    try:
        conn.execute("PRAGMA query_only = ON")

        # Chỉ cho phép thao tác đọc và một số
        # thao tác nội bộ cần cho SELECT.
        allowed = {
            sqlite3.SQLITE_SELECT,
            sqlite3.SQLITE_READ,
            sqlite3.SQLITE_FUNCTION,
            sqlite3.SQLITE_RECURSIVE,
        }

        def authorizer(action, arg1, arg2, db, source):
            if action in allowed:
                return sqlite3.SQLITE_OK
            return sqlite3.SQLITE_DENY

        conn.set_authorizer(authorizer)

        # execute() chỉ chấp nhận một statement.
        cursor = conn.execute(sql)

        if cursor.description is None:
            raise ValueError("Query has no result set")

        rows = cursor.fetchmany(max_rows + 1)

        if len(rows) > max_rows:
            raise ValueError("Result row limit exceeded")

        print(json.dumps({
            "status": "ok",
            "rows": rows,
            "row_count": len(rows)
        }, ensure_ascii=False, default=str))

    finally:
        conn.close()

if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(json.dumps({
            "status": "error",
            "error_type": type(exc).__name__,
            "message": str(exc)[:500]
        }))
'''

WORKER_PATH.write_text(
    WORKER_CODE,
    encoding="utf-8"
)

def safe_execute_sql(
    db_path,
    sql,
    timeout_seconds=10,
    max_rows=10000
):
    """Chạy SQL chỉ đọc trong subprocess."""

    payload = {
        "db_path": str(db_path),
        "sql": str(sql),
        "max_rows": max_rows
    }

    start = time.monotonic()

    try:
        proc = subprocess.run(
            [sys.executable, str(WORKER_PATH)],
            input=json.dumps(payload),
            capture_output=True,
            text=True,
            timeout=timeout_seconds
        )

        elapsed = time.monotonic() - start

        if proc.returncode != 0:
            return {
                "status": "error",
                "error_type": "WorkerProcessError",
                "message": proc.stderr[-500:],
                "elapsed_seconds": elapsed
            }

        result = json.loads(proc.stdout)
        result["elapsed_seconds"] = elapsed

        return result

    except subprocess.TimeoutExpired:
        return {
            "status": "timeout",
            "elapsed_seconds": time.monotonic() - start
        }

    except Exception as exc:
        return {
            "status": "error",
            "error_type": type(exc).__name__,
            "message": str(exc)[:500],
            "elapsed_seconds": time.monotonic() - start
        }


# ===== SMOKE TEST =====

sample_db = (
    Path(DB_DIR)
    / "concert_singer"
    / "concert_singer.sqlite"
)

assert sample_db.is_file()

tests = {
    "valid_select": "SELECT 1",
    "valid_with": (
        "WITH t AS (SELECT 1 AS x) "
        "SELECT x FROM t"
    ),
    "invalid_table": (
        "SELECT * FROM nonexistent_table"
    ),
    "blocked_delete": "DELETE FROM singer",
    "blocked_create": (
        "CREATE TABLE test_table (id INTEGER)"
    ),
    "multiple_statements": "SELECT 1; SELECT 2"
}

expected = {
    "valid_select": "ok",
    "valid_with": "ok",
    "invalid_table": "error",
    "blocked_delete": "error",
    "blocked_create": "error",
    "multiple_statements": "error"
}

print("\n===== SMOKE TEST =====")

for name, sql in tests.items():
    result = safe_execute_sql(
        sample_db,
        sql,
        timeout_seconds=10
    )

    status = result["status"]
    passed = status == expected[name]

    print(
        f"{name}: {status} "
        f"| expected={expected[name]} "
        f"| passed={passed}"
    )

    if not passed:
        print("Details:", result)

    assert passed, f"Test failed: {name}"

print("\nWorker:", WORKER_PATH)
print("HOÀN THÀNH!")

===== SAFE SQL EXECUTION =====

===== SMOKE TEST =====
valid_select: ok | expected=ok | passed=True
valid_with: ok | expected=ok | passed=True
invalid_table: error | expected=error | passed=True
blocked_delete: error | expected=error | passed=True
blocked_create: error | expected=error | passed=True
multiple_statements: error | expected=error | passed=True

Worker: /content/mac_sql_safe_worker.py
HOÀN THÀNH!


In [28]:

# CELL 17 — SAFE EXECUTOR EDGE-CASE TESTS
# MAC-SQL Lite v3.1

from pathlib import Path
import sqlite3
import tempfile
import time

print("===== EDGE-CASE TESTS =====")

assert callable(safe_execute_sql)
assert WORKER_PATH.is_file()

results = []

def check(name, sql, expected_status, **kwargs):
    result = safe_execute_sql(
        test_db,
        sql,
        timeout_seconds=kwargs.pop("timeout_seconds", 5),
        **kwargs
    )
    passed = result["status"] == expected_status
    results.append((name, passed, result))
    print(
        f"{name}: {result['status']} "
        f"| expected={expected_status} "
        f"| passed={passed}"
    )
    if not passed:
        print("  Details:", result)

# Database kiểm thử riêng, không đụng Spider.
with tempfile.TemporaryDirectory() as tmp:
    test_db = Path(tmp) / "edge_cases.sqlite"

    conn = sqlite3.connect(test_db)
    conn.executescript("""
        CREATE TABLE items (
            id INTEGER,
            value REAL,
            label TEXT
        );

        INSERT INTO items VALUES (1, 1.5, 'A');
        INSERT INTO items VALUES (2, NULL, 'A');
        INSERT INTO items VALUES (3, 2.75, 'B');
        INSERT INTO items VALUES (4, 2.75, 'B');
    """)
    conn.commit()
    conn.close()

    check(
        "null_values",
        "SELECT value FROM items WHERE id = 2",
        "ok"
    )

    check(
        "floating_point",
        "SELECT value FROM items WHERE id = 1",
        "ok"
    )

    check(
        "duplicate_rows",
        "SELECT label FROM items ORDER BY id",
        "ok"
    )

    check(
        "empty_result",
        "SELECT * FROM items WHERE id = -1",
        "ok"
    )

    check(
        "row_limit",
        "SELECT * FROM items",
        "error",
        max_rows=2
    )

    check(
        "commented_select",
        "-- harmless comment\nSELECT 1",
        "ok"
    )

    check(
        "blocked_pragma",
        "PRAGMA user_version",
        "error"
    )

    check(
        "blocked_attach",
        "ATTACH DATABASE ':memory:' AS extra",
        "error"
    )

    check(
        "recursive_timeout",
        """
        WITH RECURSIVE nums(x) AS (
            SELECT 1
            UNION ALL
            SELECT x + 1 FROM nums
        )
        SELECT SUM(x) FROM nums
        """,
        "timeout",
        timeout_seconds=1
    )

print("\n===== SUMMARY =====")
passed_count = sum(ok for _, ok, _ in results)

print(f"Passed: {passed_count}/{len(results)}")

for name, passed, result in results:
    if not passed:
        print(
            "FAILED:",
            name,
            result
        )

assert passed_count == len(results), (
    "Có kiểm thử chưa đạt. "
    "Cần phân tích trước khi tiếp tục."
)

print("HOÀN THÀNH!")

===== EDGE-CASE TESTS =====
null_values: ok | expected=ok | passed=True
floating_point: ok | expected=ok | passed=True
duplicate_rows: ok | expected=ok | passed=True
empty_result: ok | expected=ok | passed=True
row_limit: error | expected=error | passed=True
commented_select: ok | expected=ok | passed=True
blocked_pragma: error | expected=error | passed=True
blocked_attach: error | expected=error | passed=True
recursive_timeout: timeout | expected=timeout | passed=True

===== SUMMARY =====
Passed: 9/9
HOÀN THÀNH!


In [29]:

# CELL 17B — SQL RESULT CORRECTNESS

import sqlite3
import tempfile
from pathlib import Path

print("===== RESULT CORRECTNESS =====")

with tempfile.TemporaryDirectory() as tmp:
    db = Path(tmp) / "result_test.sqlite"

    conn = sqlite3.connect(db)
    conn.executescript("""
        CREATE TABLE items (
            id INTEGER,
            value REAL,
            label TEXT
        );

        INSERT INTO items VALUES (1, 1.5, 'A');
        INSERT INTO items VALUES (2, NULL, 'A');
        INSERT INTO items VALUES (3, 2.75, 'B');
        INSERT INTO items VALUES (4, 2.75, 'B');
    """)
    conn.commit()
    conn.close()

    cases = {
        "null": (
            "SELECT value FROM items WHERE id = 2",
            [[None]]
        ),
        "float": (
            "SELECT value FROM items WHERE id = 1",
            [[1.5]]
        ),
        "duplicates": (
            "SELECT label FROM items ORDER BY id",
            [["A"], ["A"], ["B"], ["B"]]
        ),
        "empty": (
            "SELECT id FROM items WHERE id = -1",
            []
        ),
        "aggregate": (
            "SELECT COUNT(*) FROM items",
            [[4]]
        )
    }

    for name, (sql, expected) in cases.items():
        result = safe_execute_sql(db, sql)

        assert result["status"] == "ok", (
            name, result
        )

        assert result["rows"] == expected, (
            name,
            result["rows"],
            expected
        )

        print(f"{name}: PASS")

    # Kiểm tra tính xác định
    sql = "SELECT id, label FROM items ORDER BY id"

    first = safe_execute_sql(db, sql)
    second = safe_execute_sql(db, sql)

    assert first["status"] == "ok"
    assert second["status"] == "ok"
    assert first["rows"] == second["rows"]

    print("determinism: PASS")

print("\nHOÀN THÀNH!")

===== RESULT CORRECTNESS =====
null: PASS
float: PASS
duplicates: PASS
empty: PASS
aggregate: PASS
determinism: PASS

HOÀN THÀNH!


In [30]:

# CELL 17C — OFFICIAL EVALUATOR CORRECTNESS
# MAC-SQL Lite v3.1

from pathlib import Path
import subprocess
import sys
import json
import textwrap
import time

print("===== EVALUATOR CORRECTNESS =====")

repo = Path("/content/evaluators/test_suite")
suite_root = Path("/content/test_suite_databases/database")
db_id = "concert_singer"
db_path = suite_root / db_id / f"{db_id}.sqlite"

assert (repo / "exec_eval.py").is_file()
assert db_path.is_file()

cases = [
    {
        "name": "identical",
        "pred": "SELECT 1",
        "gold": "SELECT 1",
        "expected": 1
    },
    {
        "name": "equivalent",
        "pred": "SELECT 1 + 1",
        "gold": "SELECT 2",
        "expected": 1
    },
    {
        "name": "incorrect",
        "pred": "SELECT 3",
        "gold": "SELECT 2",
        "expected": 0
    },
    {
        "name": "wrong_order",
        "pred": "SELECT 2 UNION ALL SELECT 1 ORDER BY 1 DESC",
        "gold": "SELECT 1 UNION ALL SELECT 2 ORDER BY 1 ASC",
        "expected": 0
    }
]

script_path = Path("/content/mac_sql_eval_check.py")

script_path.write_text(
    textwrap.dedent("""
    import sys
    import json

    sys.path.insert(0, "/content/evaluators/test_suite")
    from exec_eval import eval_exec_match

    payload = json.load(sys.stdin)
    results = []

    for case in payload["cases"]:
        score = eval_exec_match(
            payload["db_path"],
            case["pred"],
            case["gold"],
            plug_value=False,
            keep_distinct=True,
            progress_bar_for_each_datapoint=False
        )
        results.append({
            "name": case["name"],
            "score": score,
            "expected": case["expected"]
        })

    print("MAC_SQL_RESULTS=" + json.dumps(results))
    """),
    encoding="utf-8"
)

start = time.monotonic()

try:
    proc = subprocess.run(
        [sys.executable, str(script_path)],
        input=json.dumps({
            "db_path": str(db_path),
            "cases": cases
        }),
        capture_output=True,
        text=True,
        timeout=60,
        cwd=str(repo)
    )
except subprocess.TimeoutExpired:
    raise RuntimeError("Evaluator timeout sau 60 giây")

if proc.returncode != 0:
    print("STDOUT:", proc.stdout[-2000:])
    print("STDERR:", proc.stderr[-2000:])
    raise RuntimeError("Evaluator subprocess failed")

marker = "MAC_SQL_RESULTS="
lines = [
    line for line in proc.stdout.splitlines()
    if line.startswith(marker)
]

assert len(lines) == 1, (
    "Không tìm thấy kết quả evaluator: "
    + proc.stdout[-2000:]
)

results = json.loads(lines[0][len(marker):])

passed = 0

for item in results:
    ok = item["score"] == item["expected"]
    passed += int(ok)
    print(
        f"{item['name']}: "
        f"score={item['score']} "
        f"expected={item['expected']} "
        f"passed={ok}"
    )

print("\nPassed:", passed, "/", len(results))
print("Elapsed:", round(time.monotonic() - start, 3), "seconds")

assert passed == len(results), (
    "Evaluator chưa vượt qua toàn bộ kiểm thử."
)

print(" HOÀN THÀNH!")

===== EVALUATOR CORRECTNESS =====
identical: score=1 expected=1 passed=True
equivalent: score=1 expected=1 passed=True
incorrect: score=0 expected=0 passed=True
wrong_order: score=0 expected=0 passed=True

Passed: 4 / 4
Elapsed: 0.273 seconds
 HOÀN THÀNH!


In [31]:

# CELL 18 — FIXED TRAIN DEVELOPMENT SAMPLE
# MAC-SQL Lite v3.1

from pathlib import Path
from collections import Counter
import hashlib
import json
import random

print("===== TRAIN SAMPLE SELECTION =====")

SEED = 42
N_TRAIN_DEV = 30

assert isinstance(train, list)
assert len(train) >= N_TRAIN_DEV

def stable_sample_id(split, index):
    return f"{split}_{index:05d}"

def canonical_hash(obj):
    payload = json.dumps(
        obj,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":")
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()

rng = random.Random(SEED)

selected_indices = sorted(
    rng.sample(
        range(len(train)),
        N_TRAIN_DEV
    )
)

samples = []

for index in selected_indices:
    item = train[index]

    sample = {
        "sample_id": stable_sample_id("train", index),
        "source_index": index,
        "db_id": item["db_id"],
        "question": item["question"],
        "gold_sql": item["query"]
    }

    samples.append(sample)

assert len(samples) == N_TRAIN_DEV

assert len({
    s["sample_id"] for s in samples
}) == N_TRAIN_DEV

assert len({
    s["source_index"] for s in samples
}) == N_TRAIN_DEV

# Không đưa gold SQL vào prompt mô hình.
# Gold SQL chỉ dùng để đánh giá sau khi dự đoán.
output = {
    "dataset": "Spider 1.0",
    "split": "train",
    "purpose": "development_only",
    "seed": SEED,
    "sample_count": N_TRAIN_DEV,
    "selection_method": "random.sample_without_replacement",
    "samples": samples
}

output["sha256"] = canonical_hash(samples)

output_path = (
    Path(BASE)
    / "data"
    / "train_dev_30_seed42.json"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Không âm thầm ghi đè mẫu đã cố định.
if output_path.exists():
    previous = json.loads(
        output_path.read_text(encoding="utf-8")
    )

    assert previous == output, (
        "File mẫu train đã tồn tại nhưng nội dung khác. "
        "Dừng để bảo vệ tính tái lập."
    )

    print("Đã có file mẫu trùng khớp, không ghi đè.")
else:
    output_path.write_text(
        json.dumps(
            output,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

print("\n===== SAMPLE SUMMARY =====")
print("Train population:", len(train))
print("Selected:", len(samples))
print("Unique DBs:", len(set(
    s["db_id"] for s in samples
)))
print("SHA256:", output["sha256"])

print("\n5 mẫu đầu tiên:")
for sample in samples[:5]:
    print(
        sample["sample_id"],
        "|",
        sample["db_id"],
        "|",
        sample["question"][:85]
    )

print("\nSaved:", output_path)
print("HOÀN THÀNH!")

===== TRAIN SAMPLE SELECTION =====

===== SAMPLE SUMMARY =====
Train population: 7000
Selected: 30
Unique DBs: 27
SHA256: 0a0374706ab2b9b308dd8ea0e4fd2f0909a30f095cc8b6d1fb38cd4f21c335e8

5 mẫu đầu tiên:
train_00204 | bike_1 | What are the different ids and names of the stations that have had more than 12 bikes
train_00217 | book_2 | What are the title and issues of the books?
train_00244 | musical | What is the duration of the oldest actor?
train_00260 | musical | Show different nominees and the number of musicals they have been nominated.
train_00712 | customers_card_transactions | What are the different customer ids, and how many cards does each one hold?

Saved: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/data/train_dev_30_seed42.json
HOÀN THÀNH!


In [32]:

# CELL 19 — FIXED SPIDER DEV EVALUATION SAMPLE
# MAC-SQL Lite v3.1

from pathlib import Path
import hashlib
import json
import random

print("===== DEV SAMPLE SELECTION =====")

SEED = 42
N_DEV_EVAL = 300

assert isinstance(dev, list)
assert len(dev) == 1034

rng = random.Random(SEED)

selected_indices = sorted(
    rng.sample(range(len(dev)), N_DEV_EVAL)
)

eval_samples = []
gold_samples = []

for index in selected_indices:
    item = dev[index]
    sample_id = f"dev_{index:05d}"

    eval_samples.append({
        "sample_id": sample_id,
        "source_index": index,
        "db_id": item["db_id"],
        "question": item["question"]
    })

    gold_samples.append({
        "sample_id": sample_id,
        "db_id": item["db_id"],
        "gold_sql": item["query"]
    })

assert len(eval_samples) == N_DEV_EVAL
assert len(gold_samples) == N_DEV_EVAL

assert len({
    s["sample_id"] for s in eval_samples
}) == N_DEV_EVAL

assert all(
    a["sample_id"] == b["sample_id"]
    for a, b in zip(eval_samples, gold_samples)
)

assert all(
    "gold_sql" not in s
    for s in eval_samples
)

def sha256_json(data):
    encoded = json.dumps(
        data,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":")
    ).encode("utf-8")

    return hashlib.sha256(encoded).hexdigest()

eval_hash = sha256_json(eval_samples)
gold_hash = sha256_json(gold_samples)

data_dir = Path(BASE) / "data"
data_dir.mkdir(parents=True, exist_ok=True)

eval_path = (
    data_dir / "dev_eval_300_seed42.json"
)

gold_path = (
    data_dir / "dev_gold_300_seed42.json"
)

eval_output = {
    "dataset": "Spider 1.0",
    "split": "dev",
    "purpose": "official_evaluation_inputs",
    "seed": SEED,
    "sample_count": N_DEV_EVAL,
    "selection_method": "random.sample_without_replacement",
    "sha256": eval_hash,
    "samples": eval_samples
}

gold_output = {
    "dataset": "Spider 1.0",
    "split": "dev",
    "purpose": "evaluation_only",
    "seed": SEED,
    "sample_count": N_DEV_EVAL,
    "sha256": gold_hash,
    "samples": gold_samples
}

def save_without_overwrite(path, data):
    if path.exists():
        existing = json.loads(
            path.read_text(encoding="utf-8")
        )

        assert existing == data, (
            f"File đã tồn tại nhưng nội dung khác: {path}"
        )

        print("Verified existing:", path.name)

    else:
        path.write_text(
            json.dumps(
                data,
                ensure_ascii=False,
                indent=2
            ),
            encoding="utf-8"
        )

        print("Saved:", path.name)

save_without_overwrite(eval_path, eval_output)
save_without_overwrite(gold_path, gold_output)

print("\n===== DEV EVALUATION SUMMARY =====")

print("Dev population:", len(dev))
print("Selected:", len(eval_samples))
print("Unique DBs:", len({
    s["db_id"] for s in eval_samples
}))

print("Eval SHA256:", eval_hash)
print("Gold SHA256:", gold_hash)

print("\nFirst 5 samples:")

for sample in eval_samples[:5]:
    print(
        sample["sample_id"],
        "|",
        sample["db_id"],
        "|",
        sample["question"][:85]
    )

print("\nGold SQL in inference file:",
      any("gold_sql" in s for s in eval_samples))

print("\nEval file:", eval_path)
print("Gold file:", gold_path)

print("\nHOÀN THÀNH!")

===== DEV SAMPLE SELECTION =====
Saved: dev_eval_300_seed42.json
Saved: dev_gold_300_seed42.json

===== DEV EVALUATION SUMMARY =====
Dev population: 1034
Selected: 300
Unique DBs: 19
Eval SHA256: a1d1da15f800e0b7bf335efcb90bf28a86ed57542c6f0eed8ae4afd65c90b658
Gold SHA256: 49e6eeddff457816554004899a16ba4cfb279af02d9707e89055918d7136fd5e

First 5 samples:
dev_00000 | concert_singer | How many singers do we have?
dev_00003 | concert_singer | What are the names, countries, and ages for every singer in descending order of age?
dev_00006 | concert_singer | Show the name and the release year of the song by the youngest singer.
dev_00007 | concert_singer | What are the names and release years for all the songs of the youngest singer?
dev_00011 | concert_singer | How many singers are from each country?

Gold SQL in inference file: False

Eval file: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/data/dev_eval_300_seed42.js

In [33]:

# CELL 20 — DATA INTEGRITY & LEAKAGE AUDIT
# MAC-SQL Lite v3.1

from pathlib import Path
from collections import Counter
import hashlib
import json

print("===== DATA INTEGRITY AUDIT =====")

data_dir = Path(BASE) / "data"

train_path = data_dir / "train_dev_30_seed42.json"
eval_path = data_dir / "dev_eval_300_seed42.json"
gold_path = data_dir / "dev_gold_300_seed42.json"

for path in [train_path, eval_path, gold_path]:
    assert path.is_file(), f"Missing: {path}"

train_data = json.loads(
    train_path.read_text(encoding="utf-8")
)
eval_data = json.loads(
    eval_path.read_text(encoding="utf-8")
)
gold_data = json.loads(
    gold_path.read_text(encoding="utf-8")
)

train_samples = train_data["samples"]
eval_samples = eval_data["samples"]
gold_samples = gold_data["samples"]

def hash_json(items):
    raw = json.dumps(
        items,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

checks = {}

# 1. Kiểm tra số lượng và hash
checks["sample_counts"] = (
    len(train_samples) == 30
    and len(eval_samples) == 300
    and len(gold_samples) == 300
)

checks["hashes"] = (
    hash_json(train_samples) == train_data["sha256"]
    and hash_json(eval_samples) == eval_data["sha256"]
    and hash_json(gold_samples) == gold_data["sha256"]
)

# 2. Sample ID không trùng
checks["unique_ids"] = (
    len({s["sample_id"] for s in train_samples}) == 30
    and len({s["sample_id"] for s in eval_samples}) == 300
    and len({s["sample_id"] for s in gold_samples}) == 300
)

# 3. Không có gold SQL trong file inference
checks["no_gold_in_inference"] = all(
    set(s.keys()) == {
        "sample_id",
        "source_index",
        "db_id",
        "question"
    }
    for s in eval_samples
)

# 4. Kiểm tra khớp với Spider gốc
checks["train_source_match"] = all(
    s["sample_id"] == f"train_{s['source_index']:05d}"
    and s["db_id"] == train[s["source_index"]]["db_id"]
    and s["question"] == train[s["source_index"]]["question"]
    and s["gold_sql"] == train[s["source_index"]]["query"]
    for s in train_samples
)

checks["dev_source_match"] = all(
    s["sample_id"] == f"dev_{s['source_index']:05d}"
    and s["db_id"] == dev[s["source_index"]]["db_id"]
    and s["question"] == dev[s["source_index"]]["question"]
    for s in eval_samples
)

# 5. Inference và gold cùng thứ tự
checks["eval_gold_alignment"] = all(
    a["sample_id"] == b["sample_id"]
    and a["db_id"] == b["db_id"]
    and b["gold_sql"] == dev[a["source_index"]]["query"]
    for a, b in zip(eval_samples, gold_samples)
)

# 6. Train/dev không chia sẻ database
train_db_ids = {s["db_id"] for s in train}
dev_db_ids = {s["db_id"] for s in dev}

checks["disjoint_database_splits"] = (
    len(train_db_ids & dev_db_ids) == 0
)

# 7. Kiểm tra file SQLite gốc
checks["dev_databases_exist"] = all(
    (
        Path(DB_DIR)
        / s["db_id"]
        / f"{s['db_id']}.sqlite"
    ).is_file()
    for s in eval_samples
)

# 8. Kiểm tra test-suite tương ứng
suite_root = (
    Path("/content/test_suite_databases")
    / "database"
)

checks["test_suite_coverage"] = all(
    (
        suite_root
        / s["db_id"]
        / f"{s['db_id']}.sqlite"
    ).is_file()
    for s in eval_samples
)

# 9. Kiểm tra tính tái lập của cách lấy mẫu
import random

checks["train_selection_reproducible"] = (
    [s["source_index"] for s in train_samples]
    == sorted(
        random.Random(42).sample(range(len(train)), 30)
    )
)

checks["dev_selection_reproducible"] = (
    [s["source_index"] for s in eval_samples]
    == sorted(
        random.Random(42).sample(range(len(dev)), 300)
    )
)

print("\n===== AUDIT RESULTS =====")

for name, passed in checks.items():
    print(
        f"{name}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

passed_count = sum(checks.values())

print(
    f"\nPassed: {passed_count}/{len(checks)}"
)

assert all(checks.values()), (
    "Audit chưa đạt. Không được chuyển sang Cell 21."
)

print("\nTrain DB count:", len(train_db_ids))
print("Dev DB count:", len(dev_db_ids))
print(
    "Selected dev DB count:",
    len({s["db_id"] for s in eval_samples})
)

print("\nHOÀN THÀNH!")

===== DATA INTEGRITY AUDIT =====

===== AUDIT RESULTS =====
sample_counts: PASS
hashes: PASS
unique_ids: PASS
no_gold_in_inference: PASS
train_source_match: PASS
dev_source_match: PASS
eval_gold_alignment: PASS
disjoint_database_splits: PASS
dev_databases_exist: PASS
test_suite_coverage: PASS
train_selection_reproducible: PASS
dev_selection_reproducible: PASS

Passed: 12/12

Train DB count: 140
Dev DB count: 20
Selected dev DB count: 19

HOÀN THÀNH!


In [34]:

# CELL 21 — FREEZE EVALUATION PROTOCOL
# MAC-SQL Lite v3.1

from pathlib import Path
import json
import hashlib
import subprocess
import datetime

print("===== PROTOCOL FREEZE =====")

base = Path(BASE)
data_dir = base / "data"
config_dir = base / "configs"
logs_dir = base / "logs"

config_dir.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "train": data_dir / "train_dev_30_seed42.json",
    "dev_eval": data_dir / "dev_eval_300_seed42.json",
    "dev_gold": data_dir / "dev_gold_300_seed42.json",
    "evaluator": config_dir / "evaluator_config.json"
}

for name, path in paths.items():
    assert path.is_file(), f"Missing {name}: {path}"

def load_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

train_cfg = load_json(paths["train"])
eval_cfg = load_json(paths["dev_eval"])
gold_cfg = load_json(paths["dev_gold"])
evaluator_cfg = load_json(paths["evaluator"])

assert len(train_cfg["samples"]) == 30
assert len(eval_cfg["samples"]) == 300
assert len(gold_cfg["samples"]) == 300

assert evaluator_cfg["status"] == "smoke_test_passed"
assert evaluator_cfg["smoke_test"]["score"] == 1

assert evaluator_cfg["plug_value"] is False
assert evaluator_cfg["keep_distinct"] is True

assert evaluator_cfg["evaluator_commit"] == (
    "e97acc546ecbee8fa27fa8dbf025ef61493a876c"
)

assert all(
    "gold_sql" not in item
    for item in eval_cfg["samples"]
)

assert [
    s["sample_id"] for s in eval_cfg["samples"]
] == [
    s["sample_id"] for s in gold_cfg["samples"]
]

# Xác minh mã nguồn evaluator vẫn ở commit đã chọn.
repo = Path(evaluator_cfg["evaluator_path"])

actual_commit = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"],
    text=True
).strip()

assert actual_commit == evaluator_cfg["evaluator_commit"]

# Xác minh toàn bộ database được chọn có test-suite.
suite_root = Path(evaluator_cfg["test_suite_root"])

selected_db_ids = sorted({
    s["db_id"] for s in eval_cfg["samples"]
})

assert all(
    (suite_root / db_id / f"{db_id}.sqlite").is_file()
    for db_id in selected_db_ids
)

protocol = {
    "project": "MAC-SQL Lite v3.1",
    "protocol_version": "1.0",
    "dataset": {
        "name": "Spider",
        "version": "1.0",
        "train_population": len(train),
        "dev_population": len(dev),
        "train_development_size": 30,
        "dev_evaluation_size": 300,
        "selected_dev_database_count": len(selected_db_ids),
        "seed": 42,
        "selection_method": "random.sample_without_replacement"
    },
    "evaluation": {
        "primary_metric": "Test Suite Execution Accuracy",
        "evaluator_repository": "taoyds/test-suite-sql-eval",
        "evaluator_commit": actual_commit,
        "plug_value": False,
        "keep_distinct": True,
        "execution_method": "subprocess",
        "test_suite_root": str(suite_root),
        "official_evaluator_smoke_test": "passed",
        "safe_executor_tests": "19/19 passed"
    },
    "model_plan": {
        "model": "Qwen/Qwen2.5-Coder-3B-Instruct",
        "precision": "FP16",
        "decoding": "greedy",
        "fine_tuning": False,
        "paid_api": False,
        "seed": 42
    },
    "configurations": {
        "C1": "Baseline with full schema",
        "C2": "Baseline + Selector",
        "C3": "Baseline + Decomposer",
        "C4": "Baseline + Refiner",
        "C5": "Selector + Decomposer + Refiner"
    },
    "experimental_rules": {
        "same_dev_samples_for_all_configurations": True,
        "no_dev_gold_in_model_prompts": True,
        "no_dev_based_prompt_tuning": True,
        "c4_uses_saved_c1_sql": True,
        "c4_does_not_regenerate_baseline_sql": True,
        "freeze_model_and_agent_code_before_dev_evaluation": True,
        "paired_bootstrap_confidence_intervals": True,
        "mcnemar_test": True
    },
    "files": {
        name: {
            "path": str(path),
            "sha256_file": sha256_file(path)
        }
        for name, path in paths.items()
    },
    "sample_hashes": {
        "train": train_cfg["sha256"],
        "dev_eval": eval_cfg["sha256"],
        "dev_gold": gold_cfg["sha256"]
    },
    "status": "evaluation_protocol_frozen"
}

protocol_path = (
    config_dir / "evaluation_protocol_v1.json"
)

# Không âm thầm thay đổi giao thức đã đóng băng.
if protocol_path.exists():
    existing = load_json(protocol_path)
    assert existing == protocol, (
        "Giao thức đã tồn tại nhưng nội dung khác. "
        "Không được ghi đè."
    )
    print("Existing protocol verified.")
else:
    protocol_path.write_text(
        json.dumps(
            protocol,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )
    print("Protocol saved.")

print("\n===== FREEZE SUMMARY =====")
print("Protocol:", protocol_path)
print("Train development:", 30)
print("Dev evaluation:", 300)
print("Dev databases:", len(selected_db_ids))
print("Evaluator commit:", actual_commit)
print("Metric:", protocol["evaluation"]["primary_metric"])
print("Configurations:", len(protocol["configurations"]))
print("Status:", protocol["status"])

print("\nHOÀN THÀNH!")

===== PROTOCOL FREEZE =====
Protocol saved.

===== FREEZE SUMMARY =====
Protocol: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/configs/evaluation_protocol_v1.json
Train development: 30
Dev evaluation: 300
Dev databases: 19
Evaluator commit: e97acc546ecbee8fa27fa8dbf025ef61493a876c
Metric: Test Suite Execution Accuracy
Configurations: 5
Status: evaluation_protocol_frozen

HOÀN THÀNH!


In [35]:

# CELL 22 — STAGE 3.5: KIỂM TRA THÀNH PHẦN ĐÁNH GIÁ

import json
import sys
import subprocess
from pathlib import Path

print("===== ENVIRONMENT =====")

# Sử dụng BASE từ giai đoạn trước nếu còn hợp lệ
if "BASE" not in globals() or not Path(BASE).is_dir():
    BASE = Path(
        "/content/drive/MyDrive/0. Classroom/3. NĂM 3/"
        "HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/"
        "bài báo nghiên cứu/mac_sql_lite_v3"
    )

BASE = Path(BASE)

assert BASE.is_dir(), "Không tìm thấy thư mục dự án"

# Đọc giao thức đã đóng băng
protocol_path = BASE / "configs" / "evaluation_protocol_v1.json"

assert protocol_path.is_file(), "Thiếu giao thức đánh giá"

protocol = json.loads(
    protocol_path.read_text(encoding="utf-8")
)

assert protocol["status"] == "evaluation_protocol_frozen"

print("Protocol:", protocol["status"])

# Kiểm tra evaluator
repo = Path("/content/test-suite-sql-eval")
expected_commit = protocol["evaluation"]["evaluator_commit"]

if not (repo / ".git").is_dir():
    print("Đang khôi phục evaluator...")

    subprocess.run(
        [
            "git", "clone",
            "https://github.com/taoyds/test-suite-sql-eval.git",
            str(repo)
        ],
        check=True
    )

subprocess.run(
    ["git", "-C", str(repo), "checkout", "--detach",
     expected_commit],
    check=True,
    capture_output=True,
    text=True
)

actual_commit = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"],
    text=True
).strip()

assert actual_commit == expected_commit
assert (repo / "evaluation.py").is_file()
assert (repo / "exec_eval.py").is_file()

print("Official evaluator: PASS")
print("Evaluator commit:", actual_commit[:12])

# Kiểm tra test-suite
suite_root = Path(protocol["evaluation"]["test_suite_root"])

assert suite_root.is_dir(), "Thiếu test-suite database"

print("Test-suite: PASS")

# Kiểm tra Safe Worker
worker_path = Path("/content/mac_sql_safe_worker.py")

if worker_path.is_file():
    print("Safe Worker: FOUND")
else:
    print("Safe Worker: MISSING — cần tái tạo ở Cell 23")

# Kiểm tra mẫu dev
dev_path = Path(protocol["files"]["dev_eval"]["path"])
gold_path = Path(protocol["files"]["dev_gold"]["path"])

assert dev_path.is_file(), "Thiếu dữ liệu dev"
assert gold_path.is_file(), "Thiếu dữ liệu gold"

print("Dev evaluation: PASS")
print("Dev gold: PASS")

print("\n===== KẾT LUẬN =====")
print("Giao thức và evaluator sẵn sàng.")
print("HOÀN THÀNH!")

===== ENVIRONMENT =====
Protocol: evaluation_protocol_frozen
Đang khôi phục evaluator...
Official evaluator: PASS
Evaluator commit: e97acc546ecb
Test-suite: PASS
Safe Worker: FOUND
Dev evaluation: PASS
Dev gold: PASS

===== KẾT LUẬN =====
Giao thức và evaluator sẵn sàng.
HOÀN THÀNH!


In [38]:

# CELL 23 — SAFE WORKER FUNCTIONAL & TIMEOUT TEST
# MAC-SQL Lite v3.1 — Stage 3.5
# Kiểm thử trên SQLite tạm, không thay đổi dữ liệu Spider.

import json
import sqlite3
import subprocess
import sys
import tempfile
import time
from pathlib import Path

print("===== SAFE WORKER TEST =====")

WORKER = Path("/content/mac_sql_safe_worker.py")
assert WORKER.is_file(), "Không tìm thấy Safe Worker"

def run_worker(db_path, sql, timeout=10):
    request = {
        "db_path": str(db_path),
        "sql": sql,
        "max_rows": 10000,
        "row_limit": 10000
    }

    start = time.monotonic()

    try:
        proc = subprocess.run(
            [sys.executable, str(WORKER)],
            input=json.dumps(request),
            text=True,
            capture_output=True,
            timeout=timeout,
            check=False
        )
    except subprocess.TimeoutExpired:
        return {
            "status": "timeout",
            "elapsed": time.monotonic() - start
        }

    if proc.returncode != 0:
        return {
            "status": "worker_error",
            "error": proc.stderr[-1000:]
        }

    try:
        result = json.loads(proc.stdout)
    except json.JSONDecodeError:
        return {
            "status": "worker_error",
            "error": "Invalid JSON",
            "stdout": proc.stdout[-500:]
        }

    result["elapsed"] = time.monotonic() - start
    return result


with tempfile.TemporaryDirectory() as temp_dir:
    db_path = Path(temp_dir) / "test.sqlite"

    # Tạo database thử nghiệm
    conn = sqlite3.connect(db_path)
    conn.execute(
        "CREATE TABLE students (id INTEGER, name TEXT)"
    )
    conn.executemany(
        "INSERT INTO students VALUES (?, ?)",
        [(1, "An"), (2, "Binh"), (3, "Chi")]
    )
    conn.commit()
    conn.close()

    passed = 0

    # TEST 1 — SELECT hợp lệ
    result = run_worker(
        db_path,
        "SELECT COUNT(*) FROM students"
    )

    ok = (
        result.get("status") == "ok"
        and result.get("rows") == [[3]]
        and result.get("row_count") == 1
    )

    print(
        f"{'PASS' if ok else 'FAIL'} | "
        f"SELECT hợp lệ | {result}"
    )
    passed += int(ok)

    # TEST 2 — SQL sai cú pháp
    result = run_worker(
        db_path,
        "SELEC * FROM students"
    )

    ok = (
        result.get("status") == "error"
        and result.get("error_type") == "OperationalError"
    )

    print(
        f"{'PASS' if ok else 'FAIL'} | "
        f"SQL sai cú pháp | {result}"
    )
    passed += int(ok)

    # TEST 3–5 — Chặn các lệnh thay đổi database
    blocked_queries = [
        ("DELETE", "DELETE FROM students"),
        ("UPDATE", "UPDATE students SET name='X'"),
        ("DROP", "DROP TABLE students")
    ]

    for name, sql in blocked_queries:
        result = run_worker(db_path, sql)

        ok = (
            result.get("status") == "error"
            and result.get("error_type") == "DatabaseError"
            and "not authorized" in result.get("message", "").lower()
        )

        print(
            f"{'PASS' if ok else 'FAIL'} | "
            f"Chặn {name} | {result}"
        )
        passed += int(ok)

    # TEST 6 — SQL timeout thực tế
    heavy_sql = """
    WITH RECURSIVE seq(x) AS (
        SELECT 1
        UNION ALL
        SELECT x + 1
        FROM seq
        WHERE x < 100000000
    )
    SELECT SUM(x) FROM seq
    """

    result = run_worker(
        db_path,
        heavy_sql,
        timeout=1
    )

    ok = result.get("status") == "timeout"

    print(
        f"{'PASS' if ok else 'FAIL'} | "
        f"SQL timeout | {result}"
    )
    passed += int(ok)

    # TEST 7 — Database không bị thay đổi
    conn = sqlite3.connect(db_path)

    try:
        rows = conn.execute(
            "SELECT id, name FROM students ORDER BY id"
        ).fetchall()

        table_exists = conn.execute(
            "SELECT COUNT(*) FROM sqlite_master "
            "WHERE type='table' AND name='students'"
        ).fetchone()[0] == 1
    finally:
        conn.close()

    ok = (
        table_exists
        and rows == [
            (1, "An"),
            (2, "Binh"),
            (3, "Chi")
        ]
    )

    print(
        f"{'PASS' if ok else 'FAIL'} | "
        f"Database không bị thay đổi"
    )
    passed += int(ok)

print("\n===== KẾT QUẢ =====")
print(f"Passed: {passed}/7")

assert passed == 7, (
    "Safe Worker chưa đạt đầy đủ kiểm thử."
)

print("HOÀN THÀNH!")

===== SAFE WORKER TEST =====
PASS | SELECT hợp lệ | {'status': 'ok', 'rows': [[3]], 'row_count': 1, 'elapsed': 0.11130118300002323}
PASS | SQL sai cú pháp | {'status': 'error', 'error_type': 'OperationalError', 'message': 'near "SELEC": syntax error', 'elapsed': 0.1568527230001564}
PASS | Chặn DELETE | {'status': 'error', 'error_type': 'DatabaseError', 'message': 'not authorized', 'elapsed': 0.11430559700011145}
PASS | Chặn UPDATE | {'status': 'error', 'error_type': 'DatabaseError', 'message': 'not authorized', 'elapsed': 0.1060334049998346}
PASS | Chặn DROP | {'status': 'error', 'error_type': 'DatabaseError', 'message': 'not authorized', 'elapsed': 0.1070352090000597}
PASS | SQL timeout | {'status': 'timeout', 'elapsed': 1.0081797519999327}
PASS | Database không bị thay đổi

===== KẾT QUẢ =====
Passed: 7/7
HOÀN THÀNH!


In [39]:

# CELL 24 — GOLD-VS-GOLD PREFLIGHT
# Kiểm tra định dạng dữ liệu và evaluator trước khi chấm 300 mẫu.

import json
import sys
import inspect
from pathlib import Path

print("===== GOLD-VS-GOLD PREFLIGHT =====")

BASE = Path(
    "/content/drive/MyDrive/0. Classroom/3. NĂM 3/"
    "HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/"
    "bài báo nghiên cứu/mac_sql_lite_v3"
)

protocol_path = BASE / "configs" / "evaluation_protocol_v1.json"
assert protocol_path.is_file(), "Thiếu protocol"

protocol = json.loads(protocol_path.read_text(encoding="utf-8"))

assert protocol.get("status") == "evaluation_protocol_frozen"

dev_path = Path(protocol["files"]["dev_eval"]["path"])
gold_path = Path(protocol["files"]["dev_gold"]["path"])

dev_data = json.loads(dev_path.read_text(encoding="utf-8"))
gold_data = json.loads(gold_path.read_text(encoding="utf-8"))

print("Dev type:", type(dev_data).__name__)
print("Gold type:", type(gold_data).__name__)

def describe(data, name):
    if isinstance(data, list):
        print(f"{name} length:", len(data))
        if data:
            first = data[0]
            print(f"{name} first item type:", type(first).__name__)
            if isinstance(first, dict):
                print(f"{name} first item keys:", list(first.keys()))
            else:
                print(f"{name} first item preview:", repr(first)[:250])

    elif isinstance(data, dict):
        print(f"{name} top-level keys:", list(data.keys())[:20])

describe(dev_data, "DEV")
describe(gold_data, "GOLD")

repo = Path("/content/test-suite-sql-eval")
assert (repo / "evaluation.py").is_file(), "Thiếu evaluator"

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import evaluation

print("\n===== OFFICIAL EVALUATOR =====")
print("evaluate signature:", inspect.signature(evaluation.evaluate))
print("Evaluator signature:", inspect.signature(evaluation.Evaluator))

suite_root = Path(protocol["evaluation"]["test_suite_root"])
print("Test-suite root:", suite_root)
print("Test-suite exists:", suite_root.is_dir())

assert suite_root.is_dir(), "Không tìm thấy test-suite"

print("\nPREFLIGHT HOÀN THÀNH!")

===== GOLD-VS-GOLD PREFLIGHT =====
Dev type: dict
Gold type: dict
DEV top-level keys: ['dataset', 'split', 'purpose', 'seed', 'sample_count', 'selection_method', 'sha256', 'samples']
GOLD top-level keys: ['dataset', 'split', 'purpose', 'seed', 'sample_count', 'sha256', 'samples']

===== OFFICIAL EVALUATOR =====
evaluate signature: (gold, predict, db_dir, etype, kmaps, plug_value, keep_distinct, progress_bar_for_each_datapoint)
Evaluator signature: ()
Test-suite root: /content/test_suite_databases/database
Test-suite exists: True

PREFLIGHT HOÀN THÀNH!


/content/test-suite-sql-eval/exec_eval.py:127: SyntaxWarning: invalid escape sequence '\s'
  "YEAR\s*\(\s*CURDATE\s*\(\s*\)\s*\)\s*", "2020", query, flags=re.IGNORECASE
/content/test-suite-sql-eval/parse.py:57: SyntaxWarning: invalid escape sequence '\d'
  float_nums = re.findall("[-+]?\d*\.\d+", query)
/content/test-suite-sql-eval/parse.py:62: SyntaxWarning: invalid escape sequence '\d'
  int_nums = [i.strip() for i in re.findall("[^tT]\d+", query)]
/content/test-suite-sql-eval/parse.py:70: SyntaxWarning: invalid escape sequence '\d'
  table = re.findall("[Tt]\d+\.", tok)
/content/test-suite-sql-eval/parse.py:206: SyntaxWarning: invalid escape sequence '\.'
  for table, col, val1, val2 in re.findall('(?:([^\.\s]*)\.)?([^\.\s]+) between ([^\s;]+) and ([^\s;]+)', query, re.IGNORECASE):


In [40]:

# CELL 25 — GOLD-VS-GOLD DATA VALIDATION
# Không sửa dữ liệu, không chạy mô hình.

import json
from pathlib import Path
from collections import Counter

print("===== GOLD-VS-GOLD DATA VALIDATION =====")

BASE = Path(
    "/content/drive/MyDrive/0. Classroom/3. NĂM 3/"
    "HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/"
    "bài báo nghiên cứu/mac_sql_lite_v3"
)

protocol = json.loads(
    (BASE / "configs" / "evaluation_protocol_v1.json")
    .read_text(encoding="utf-8")
)

dev_data = json.loads(
    Path(protocol["files"]["dev_eval"]["path"])
    .read_text(encoding="utf-8")
)

gold_data = json.loads(
    Path(protocol["files"]["dev_gold"]["path"])
    .read_text(encoding="utf-8")
)

dev_samples = dev_data["samples"]
gold_samples = gold_data["samples"]

assert isinstance(dev_samples, list)
assert isinstance(gold_samples, list)
assert len(dev_samples) == len(gold_samples) == 300

print("DEV samples:", len(dev_samples))
print("GOLD samples:", len(gold_samples))

print("\n===== SAMPLE STRUCTURE =====")

for name, samples in [
    ("DEV", dev_samples),
    ("GOLD", gold_samples)
]:
    print(f"\n{name} — First sample:")

    first = samples[0]
    print("Type:", type(first).__name__)

    if isinstance(first, dict):
        print("Keys:", list(first.keys()))

        for key, value in first.items():
            if isinstance(value, (str, int, float, bool)):
                preview = str(value)[:180]
            else:
                preview = f"<{type(value).__name__}>"

            print(f"  {key}: {preview}")
    else:
        print("Preview:", repr(first)[:300])

print("\n===== METADATA =====")

for key in ["dataset", "split", "seed", "sample_count"]:
    print(
        key,
        "| DEV:", dev_data.get(key),
        "| GOLD:", gold_data.get(key)
    )

assert dev_data.get("sample_count") == 300
assert gold_data.get("sample_count") == 300
assert dev_data.get("seed") == gold_data.get("seed")

print("\n===== FIELD CONSISTENCY =====")

dev_keysets = Counter(
    tuple(sorted(item.keys()))
    for item in dev_samples
    if isinstance(item, dict)
)

gold_keysets = Counter(
    tuple(sorted(item.keys()))
    for item in gold_samples
    if isinstance(item, dict)
)

print("DEV field patterns:", dict(dev_keysets))
print("GOLD field patterns:", dict(gold_keysets))

print("\nDATA VALIDATION HOÀN THÀNH!")

===== GOLD-VS-GOLD DATA VALIDATION =====
DEV samples: 300
GOLD samples: 300

===== SAMPLE STRUCTURE =====

DEV — First sample:
Type: dict
Keys: ['sample_id', 'source_index', 'db_id', 'question']
  sample_id: dev_00000
  source_index: 0
  db_id: concert_singer
  question: How many singers do we have?

GOLD — First sample:
Type: dict
Keys: ['sample_id', 'db_id', 'gold_sql']
  sample_id: dev_00000
  db_id: concert_singer
  gold_sql: SELECT count(*) FROM singer

===== METADATA =====
dataset | DEV: Spider 1.0 | GOLD: Spider 1.0
split | DEV: dev | GOLD: dev
seed | DEV: 42 | GOLD: 42
sample_count | DEV: 300 | GOLD: 300

===== FIELD CONSISTENCY =====
DEV field patterns: {('db_id', 'question', 'sample_id', 'source_index'): 300}
GOLD field patterns: {('db_id', 'gold_sql', 'sample_id'): 300}

DATA VALIDATION HOÀN THÀNH!


In [45]:

# CELL 26 — GOLD-VS-GOLD OFFICIAL EVALUATION
# MAC-SQL Lite v3.1
# Sửa định dạng GOLD và chạy evaluator bằng subprocess.

import json
import sys
import subprocess
import tempfile
import textwrap
from pathlib import Path

print("===== GOLD-VS-GOLD OFFICIAL EVALUATION =====")

BASE = Path(
    "/content/drive/MyDrive/0. Classroom/3. NĂM 3/"
    "HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/"
    "bài báo nghiên cứu/mac_sql_lite_v3"
)

protocol = json.loads(
    (BASE / "configs" / "evaluation_protocol_v1.json")
    .read_text(encoding="utf-8")
)

assert protocol["status"] == "evaluation_protocol_frozen"

dev = json.loads(
    Path(protocol["files"]["dev_eval"]["path"])
    .read_text(encoding="utf-8")
)["samples"]

gold = json.loads(
    Path(protocol["files"]["dev_gold"]["path"])
    .read_text(encoding="utf-8")
)["samples"]

assert len(dev) == len(gold) == 300

gold_by_id = {x["sample_id"]: x for x in gold}
assert len(gold_by_id) == 300

aligned = []

for item in dev:
    sid = item["sample_id"]
    assert sid in gold_by_id

    g = gold_by_id[sid]
    assert item["db_id"] == g["db_id"]

    sql = g["gold_sql"].strip()
    assert sql and "\n" not in sql and "\t" not in sql

    aligned.append((sid, item["db_id"], sql))

print("Aligned samples:", len(aligned))

repo = Path("/content/test-suite-sql-eval")
suite_root = Path(protocol["evaluation"]["test_suite_root"])
tables_path = Path("/content/spider_data/spider_data/tables.json")

assert (repo / "evaluation.py").is_file()
assert suite_root.is_dir()
assert tables_path.is_file()

runner_code = textwrap.dedent("""
import json
import sys
from pathlib import Path

repo = Path(sys.argv[1])
tables_path = Path(sys.argv[2])
gold_file = sys.argv[3]
pred_file = sys.argv[4]
suite_root = sys.argv[5]
plug_value = sys.argv[6] == "true"
keep_distinct = sys.argv[7] == "true"

sys.path.insert(0, str(repo))

import evaluation
from evaluation import build_foreign_key_map

tables = json.loads(
    tables_path.read_text(encoding="utf-8")
)

kmaps = {
    schema["db_id"]: build_foreign_key_map(schema)
    for schema in tables
}

result = evaluation.evaluate(
    gold=gold_file,
    predict=pred_file,
    db_dir=suite_root,
    etype="exec",
    kmaps=kmaps,
    plug_value=plug_value,
    keep_distinct=keep_distinct,
    progress_bar_for_each_datapoint=False
)

print("\\n===== OFFICIAL RESULT =====")
print(repr(result))
""")

with tempfile.TemporaryDirectory() as temp_dir:
    tmp = Path(temp_dir)

    gold_file = tmp / "gold.txt"
    pred_file = tmp / "pred.txt"
    runner_file = tmp / "run_evaluator.py"

    # Dùng dấu TAB thật giữa SQL và db_id
    gold_lines = [
        sql + "\t" + db_id
        for _, db_id, sql in aligned
    ]

    pred_lines = [
        sql for _, _, sql in aligned
    ]

    gold_file.write_text(
        "\n".join(gold_lines) + "\n",
        encoding="utf-8"
    )

    pred_file.write_text(
        "\n".join(pred_lines) + "\n",
        encoding="utf-8"
    )

    runner_file.write_text(
        runner_code,
        encoding="utf-8"
    )

    # Kiểm tra đầu vào trước khi thực thi
    check_gold = gold_file.read_text(
        encoding="utf-8"
    ).splitlines()

    check_pred = pred_file.read_text(
        encoding="utf-8"
    ).splitlines()

    assert len(check_gold) == 300
    assert len(check_pred) == 300

    for index, line in enumerate(check_gold):
        parts = line.split("\t")

        assert len(parts) == 2, (
            f"GOLD dòng {index + 1} sai định dạng"
        )

        assert parts[0].strip()
        assert parts[1].strip()

    print("Gold samples:", len(check_gold))
    print("Prediction samples:", len(check_pred))
    print("Gold format: PASS")
    print("Evaluator mode: subprocess")
    print("Plug value:", protocol["evaluation"]["plug_value"])
    print("Keep distinct:", protocol["evaluation"]["keep_distinct"])

    command = [
        sys.executable,
        str(runner_file),
        str(repo),
        str(tables_path),
        str(gold_file),
        str(pred_file),
        str(suite_root),
        str(protocol["evaluation"]["plug_value"]).lower(),
        str(protocol["evaluation"]["keep_distinct"]).lower()
    ]

    print("\nĐang chạy official evaluator...")

    try:
        process = subprocess.run(
            command,
            text=True,
            capture_output=True,
            timeout=900,
            check=False
        )
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(
            "Evaluator vượt quá 900 giây."
        ) from exc

    print("\n===== STDOUT =====")
    print(process.stdout[-15000:] or "(trống)")

    if process.stderr.strip():
        print("\n===== STDERR =====")
        print(process.stderr[-5000:])

    print("\nExit code:", process.returncode)

    assert process.returncode == 0, (
        "Official evaluator chưa chạy thành công."
    )

print("\nHOÀN THÀNH!")

===== GOLD-VS-GOLD OFFICIAL EVALUATION =====
Aligned samples: 300
Gold samples: 300
Prediction samples: 300
Gold format: PASS
Evaluator mode: subprocess
Plug value: False
Keep distinct: True

Đang chạy official evaluator...

===== STDOUT =====
                     easy                 medium               hard                 extra                all                 
count                75                   131                  46                   48                   300                 
=====================   EXECUTION ACCURACY     =====================
execution            1.000                1.000                1.000                1.000                1.000               

===== OFFICIAL RESULT =====
None


Exit code: 0

HOÀN THÀNH!


In [42]:

# CELL 26A — FIX NLTK TOKENIZER DEPENDENCIES
# Bổ sung tài nguyên NLTK cho official evaluator.
# Không thay đổi protocol, dataset hoặc evaluator.

import nltk

print("===== NLTK DEPENDENCY CHECK =====")

resources = [
    ("tokenizers/punkt", "punkt"),
    ("tokenizers/punkt_tab/english/", "punkt_tab")
]

for resource_path, package_name in resources:
    try:
        nltk.data.find(resource_path)
        print(f"PASS | {package_name}: đã có")

    except LookupError:
        print(f"Đang tải {package_name}...")

        success = nltk.download(
            package_name,
            quiet=False,
            raise_on_error=True
        )

        assert success, (
            f"Không tải được tài nguyên {package_name}"
        )

        nltk.data.find(resource_path)
        print(f"PASS | {package_name}: đã cài đặt")

# Kiểm thử tokenizer thực tế
tokens = nltk.word_tokenize(
    "SELECT COUNT(*) FROM singer"
)

print("\n===== TOKENIZER TEST =====")
print("Tokens:", tokens)

assert len(tokens) > 0

print("\nHOÀN THÀNH!")

===== NLTK DEPENDENCY CHECK =====
Đang tải punkt...


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


PASS | punkt: đã cài đặt
Đang tải punkt_tab...


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


PASS | punkt_tab: đã cài đặt

===== TOKENIZER TEST =====
Tokens: ['SELECT', 'COUNT', '(', '*', ')', 'FROM', 'singer']

HOÀN THÀNH!


In [46]:

# CELL 27 — STAGE 4.1: GPU & MODEL ENVIRONMENT CHECK
# MAC-SQL Lite v3.1
# Chỉ kiểm tra môi trường, không cài đặt hoặc tải mô hình.

import sys
import platform
import importlib.metadata as metadata
from pathlib import Path

print("===== GPU ENVIRONMENT =====")

# 1. Kiểm tra Python
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

# 2. Kiểm tra PyTorch
import torch

print("\n===== PYTORCH =====")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "Không phát hiện GPU CUDA. "
    "Hãy chọn GPU T4 trong cài đặt Colab."
)

gpu_name = torch.cuda.get_device_name(0)
gpu_properties = torch.cuda.get_device_properties(0)

total_vram = gpu_properties.total_memory / (1024**3)

print("GPU:", gpu_name)
print("Total VRAM:", round(total_vram, 2), "GiB")
print("CUDA version:", torch.version.cuda)

# 3. Kiểm tra bộ nhớ GPU hiện tại
free_bytes, total_bytes = torch.cuda.mem_get_info()

print("\n===== GPU MEMORY =====")
print("Free VRAM:", round(free_bytes / 1024**3, 2), "GiB")
print("Total VRAM:", round(total_bytes / 1024**3, 2), "GiB")

# 4. Kiểm tra thư viện
print("\n===== LIBRARIES =====")

packages = [
    "transformers",
    "accelerate",
    "tokenizers",
    "safetensors",
    "huggingface_hub"
]

missing = []

for package in packages:
    try:
        version = metadata.version(package)
        print(f"PASS | {package}: {version}")
    except metadata.PackageNotFoundError:
        print(f"MISSING | {package}")
        missing.append(package)

# 5. Kiểm tra khả năng dùng FP16
print("\n===== MODEL CONFIGURATION =====")

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
MODEL_DTYPE = torch.float16
SEED = 42

print("Model:", MODEL_ID)
print("Precision:", MODEL_DTYPE)
print("Seed:", SEED)

assert not missing, (
    "Thiếu thư viện: " + ", ".join(missing)
)

# 6. Kiểm tra dữ liệu nghiên cứu vẫn tồn tại
print("\n===== EXPERIMENT DATA =====")

BASE = Path(
    "/content/drive/MyDrive/0. Classroom/3. NĂM 3/"
    "HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/"
    "bài báo nghiên cứu/mac_sql_lite_v3"
)

protocol_path = BASE / "configs" / "evaluation_protocol_v1.json"

assert protocol_path.is_file(), (
    "Không tìm thấy giao thức thực nghiệm đã đóng băng."
)

print("Frozen protocol: PASS")
print("Project directory: PASS")

print("\n===== FINAL STATUS =====")
print("GPU:", gpu_name)
print("Available VRAM:", round(free_bytes / 1024**3, 2), "GiB")
print("Libraries: PASS")
print("Protocol: PASS")

print("\nHOÀN THÀNH!")

===== GPU ENVIRONMENT =====
Python: 3.13.16
Platform: Linux-6.6.159+-x86_64-with-glibc2.39

===== PYTORCH =====
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
Total VRAM: 14.56 GiB
CUDA version: 13.0

===== GPU MEMORY =====
Free VRAM: 14.46 GiB
Total VRAM: 14.56 GiB

===== LIBRARIES =====
PASS | transformers: 5.22.1
PASS | accelerate: 1.15.0
PASS | tokenizers: 0.23.3
PASS | safetensors: 0.8.0
PASS | huggingface_hub: 1.33.0

===== MODEL CONFIGURATION =====
Model: Qwen/Qwen2.5-Coder-3B-Instruct
Precision: torch.float16
Seed: 42

===== EXPERIMENT DATA =====
Frozen protocol: PASS
Project directory: PASS

===== FINAL STATUS =====
GPU: Tesla T4
Available VRAM: 14.46 GiB
Libraries: PASS
Protocol: PASS

HOÀN THÀNH!


In [47]:

# CELL 28 — LOAD QWEN2.5-CODER-3B-INSTRUCT
# MAC-SQL Lite v3.1 | Stage 4.2
# FP16 | Tesla T4 | Seed 42

import gc
import random
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

print("===== LOAD QWEN MODEL =====")

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
SEED = 42

# 1. Cố định seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 2. Kiểm tra GPU
assert torch.cuda.is_available(), "Không tìm thấy CUDA GPU"

device = torch.device("cuda:0")

free_bytes, total_bytes = torch.cuda.mem_get_info()

print("GPU:", torch.cuda.get_device_name(0))
print("Free VRAM before loading:",
      round(free_bytes / 1024**3, 2), "GiB")

# 3. Tải tokenizer
print("\n===== LOADING TOKENIZER =====")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=False
)

assert tokenizer.eos_token_id is not None

print("Tokenizer: PASS")
print("EOS token ID:", tokenizer.eos_token_id)

# 4. Tải mô hình FP16
print("\n===== LOADING MODEL FP16 =====")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=False
)

# Chuyển toàn bộ mô hình lên GPU
model = model.to(device)
model.eval()

# 5. Kiểm tra cấu hình
print("\n===== MODEL VALIDATION =====")

assert next(model.parameters()).device.type == "cuda"
assert next(model.parameters()).dtype == torch.float16
assert not model.training

print("Model:", MODEL_ID)
print("Device:", next(model.parameters()).device)
print("Precision:", next(model.parameters()).dtype)
print("Evaluation mode:", not model.training)

# 6. Thống kê tham số
param_count = sum(p.numel() for p in model.parameters())

print("Parameters:", f"{param_count:,}")
print("Parameters (billions):",
      round(param_count / 1e9, 3))

# 7. Kiểm tra VRAM
torch.cuda.synchronize()

free_bytes, total_bytes = torch.cuda.mem_get_info()

allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3
free_vram = free_bytes / 1024**3

print("\n===== GPU MEMORY AFTER LOADING =====")
print("Allocated:", round(allocated, 2), "GiB")
print("Reserved:", round(reserved, 2), "GiB")
print("Free:", round(free_vram, 2), "GiB")

assert free_vram > 1.0, (
    "VRAM còn quá ít để thực hiện suy luận."
)

print("\n===== FINAL STATUS =====")
print("Tokenizer: PASS")
print("Model loading: PASS")
print("FP16: PASS")
print("CUDA: PASS")
print("Eval mode: PASS")

print("\nHOÀN THÀNH!")

/usr/local/lib/python3.13/dist-packages/transformers/__init__.py:29: RuntimeWarning: coroutine 'exec_on_db' was never awaited
  from . import dependency_versions_check


===== LOAD QWEN MODEL =====
GPU: Tesla T4
Free VRAM before loading: 14.46 GiB

===== LOADING TOKENIZER =====


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (4 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]


===== MODEL VALIDATION =====
Model: Qwen/Qwen2.5-Coder-3B-Instruct
Device: cuda:0
Precision: torch.float16
Evaluation mode: True
Parameters: 3,085,938,688
Parameters (billions): 3.086

===== GPU MEMORY AFTER LOADING =====
Allocated: 5.85 GiB
Reserved: 6.0 GiB
Free: 8.46 GiB

===== FINAL STATUS =====
Tokenizer: PASS
Model loading: PASS
FP16: PASS
CUDA: PASS
Eval mode: PASS

HOÀN THÀNH!


In [49]:

# CELL 29 — QWEN SQL GENERATION SMOKE TEST
# MAC-SQL Lite v3.1 | Stage 4.3
# Compatible with Transformers 5.x
# FP16 | Greedy decoding | No dev gold

import time
import torch

print("===== QWEN SQL GENERATION TEST =====")

assert "model" in globals(), "Chưa nạp model"
assert "tokenizer" in globals(), "Chưa nạp tokenizer"
assert not model.training
assert next(model.parameters()).device.type == "cuda"

# 1. Ví dụ kiểm tra kỹ thuật
schema = """
CREATE TABLE singer (
    singer_id INTEGER PRIMARY KEY,
    name TEXT,
    age INTEGER
);
"""

question = "How many singers are there?"

messages = [
    {
        "role": "system",
        "content": (
            "You are a Text-to-SQL assistant. "
            "Generate a valid SQLite SELECT query. "
            "Return only SQL without explanation or markdown."
        )
    },
    {
        "role": "user",
        "content": (
            f"Database schema:\n{schema}\n\n"
            f"Question: {question}\n\n"
            "SQL:"
        )
    }
]

# 2. Tokenization
encoded = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

# Hỗ trợ cả BatchEncoding và Tensor
if isinstance(encoded, torch.Tensor):
    input_ids = encoded.to("cuda")
    attention_mask = torch.ones_like(input_ids)

else:
    input_ids = encoded["input_ids"].to("cuda")

    attention_mask = encoded.get("attention_mask")

    if attention_mask is None:
        attention_mask = torch.ones_like(input_ids)
    else:
        attention_mask = attention_mask.to("cuda")

print("Model: Qwen/Qwen2.5-Coder-3B-Instruct")
print("Input tokens:", input_ids.shape[-1])
print("Question:", question)

# 3. Greedy inference
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

start = time.perf_counter()

with torch.inference_mode():
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

# 4. Chỉ giải mã token mới
generated_ids = outputs[0, input_ids.shape[-1]:]

generated_sql = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

print("\n===== GENERATED SQL =====")
print(generated_sql)

print("\n===== PERFORMANCE =====")
print("Generated tokens:", len(generated_ids))
print("Inference time:", round(elapsed, 2), "seconds")
print(
    "Peak GPU memory:",
    round(
        torch.cuda.max_memory_allocated() / 1024**3,
        2
    ),
    "GiB"
)

assert generated_sql, "Mô hình không sinh nội dung"

print("\nHOÀN THÀNH!")

===== QWEN SQL GENERATION TEST =====
Model: Qwen/Qwen2.5-Coder-3B-Instruct
Input tokens: 70
Question: How many singers are there?

===== GENERATED SQL =====
SELECT COUNT(*) FROM singer

===== PERFORMANCE =====
Generated tokens: 6
Inference time: 1.54 seconds
Peak GPU memory: 5.87 GiB

HOÀN THÀNH!


In [50]:

# CELL 30 — TRAIN DATA & SPIDER SCHEMA INSPECTION
# MAC-SQL Lite v3.1 | Stage 4.4
# Read-only | No dev gold | No model generation

import json
from pathlib import Path

print("===== TRAIN DATA INSPECTION =====")

BASE = Path(
    "/content/drive/MyDrive/0. Classroom/3. NĂM 3/"
    "HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/"
    "bài báo nghiên cứu/mac_sql_lite_v3"
)

train_path = BASE / "data" / "train_dev_30_seed42.json"
tables_path = Path("/content/spider_data/spider_data/tables.json")
db_root = Path("/content/spider_data/spider_data/database")

assert train_path.is_file(), "Không tìm thấy train data"
assert tables_path.is_file(), "Không tìm thấy tables.json"
assert db_root.is_dir(), "Không tìm thấy Spider databases"

train_data = json.loads(train_path.read_text(encoding="utf-8"))
tables_data = json.loads(tables_path.read_text(encoding="utf-8"))

print("Train data type:", type(train_data).__name__)

if isinstance(train_data, dict):
    print("Top-level keys:", list(train_data.keys()))
    assert "samples" in train_data
    samples = train_data["samples"]
else:
    samples = train_data

assert isinstance(samples, list)
assert len(samples) == 30

print("Train samples:", len(samples))
print("Spider schemas:", len(tables_data))

schema_by_id = {
    schema["db_id"]: schema
    for schema in tables_data
}

print("\n===== TRAIN SAMPLE STRUCTURE =====")

for i, sample in enumerate(samples[:3], start=1):
    print(f"\nSample {i}")
    print("Keys:", list(sample.keys()))
    print("Sample ID:", sample.get("sample_id"))
    print("DB ID:", sample.get("db_id"))
    print("Question:", sample.get("question"))
    print("SQL field present:",
          [k for k in ("query", "sql", "gold_sql")
           if k in sample])

print("\n===== SCHEMA STRUCTURE =====")

first_db = samples[0]["db_id"]
assert first_db in schema_by_id

schema = schema_by_id[first_db]

print("Database:", first_db)
print("Schema keys:", list(schema.keys()))
print("Tables:", schema["table_names_original"][:10])
print("Columns:", schema["column_names_original"][:15])
print("Foreign keys:", schema["foreign_keys"][:10])

db_path = db_root / first_db / f"{first_db}.sqlite"
print("SQLite file exists:", db_path.is_file())

assert db_path.is_file()

print("\n===== FINAL STATUS =====")
print("Train dataset: PASS")
print("Spider schema: PASS")
print("SQLite database: PASS")
print("\nHOÀN THÀNH!")

===== TRAIN DATA INSPECTION =====
Train data type: dict
Top-level keys: ['dataset', 'split', 'purpose', 'seed', 'sample_count', 'selection_method', 'samples', 'sha256']
Train samples: 30
Spider schemas: 166

===== TRAIN SAMPLE STRUCTURE =====

Sample 1
Keys: ['sample_id', 'source_index', 'db_id', 'question', 'gold_sql']
Sample ID: train_00204
DB ID: bike_1
Question: What are the different ids and names of the stations that have had more than 12 bikes available?
SQL field present: ['gold_sql']

Sample 2
Keys: ['sample_id', 'source_index', 'db_id', 'question', 'gold_sql']
Sample ID: train_00217
DB ID: book_2
Question: What are the title and issues of the books?
SQL field present: ['gold_sql']

Sample 3
Keys: ['sample_id', 'source_index', 'db_id', 'question', 'gold_sql']
Sample ID: train_00244
DB ID: musical
Question: What is the duration of the oldest actor?
SQL field present: ['gold_sql']

===== SCHEMA STRUCTURE =====
Database: bike_1
Schema keys: ['column_names', 'column_names_original

In [51]:

# CELL 31 — SPIDER SCHEMA TO DDL
# MAC-SQL Lite v3.1 | Stage 4.4
# Baseline C1: Full schema, no pruning

import json
from pathlib import Path

print("===== SPIDER SCHEMA BUILDER =====")

tables_path = Path(
    "/content/spider_data/spider_data/tables.json"
)

tables_data = json.loads(
    tables_path.read_text(encoding="utf-8")
)

schema_by_id = {
    item["db_id"]: item
    for item in tables_data
}

def quote_identifier(name):
    """Quote SQLite identifiers safely."""
    return '"' + str(name).replace('"', '""') + '"'


def spider_schema_to_ddl(db_id):
    """
    Convert Spider tables.json metadata to CREATE TABLE DDL.

    Includes:
      - All tables
      - All columns
      - Column types
      - Primary keys
      - Foreign keys

    Does not prune schema.
    """

    if db_id not in schema_by_id:
        raise KeyError(f"Unknown database: {db_id}")

    schema = schema_by_id[db_id]

    table_names = schema["table_names_original"]
    columns = schema["column_names_original"]
    column_types = schema["column_types"]

    primary_keys = schema["primary_keys"]
    foreign_keys = schema["foreign_keys"]

    # Spider column index -> metadata
    column_lookup = {
        index: (table_index, column_name)
        for index, (table_index, column_name)
        in enumerate(columns)
        if table_index >= 0
    }

    # Primary keys grouped by table
    pk_by_table = {}

    for column_index in primary_keys:
        if column_index not in column_lookup:
            continue

        table_index, column_name = column_lookup[column_index]

        pk_by_table.setdefault(
            table_index, []
        ).append(column_name)

    # Foreign keys grouped by source table
    fk_by_table = {}

    for source_index, target_index in foreign_keys:

        if (
            source_index not in column_lookup
            or target_index not in column_lookup
        ):
            continue

        source_table, source_column = column_lookup[source_index]
        target_table, target_column = column_lookup[target_index]

        fk_by_table.setdefault(
            source_table, []
        ).append(
            (
                source_column,
                table_names[target_table],
                target_column
            )
        )

    ddl_statements = []

    for table_index, table_name in enumerate(table_names):

        definitions = []

        # Columns
        for column_index, (owner, column_name) in enumerate(columns):

            if owner != table_index:
                continue

            col_type = column_types[column_index].lower()

            type_mapping = {
                "text": "TEXT",
                "number": "REAL",
                "time": "TEXT",
                "boolean": "INTEGER",
                "others": "TEXT"
            }

            sql_type = type_mapping.get(
                col_type, "TEXT"
            )

            definitions.append(
                f"    {quote_identifier(column_name)} {sql_type}"
            )

        # Primary key
        pk_columns = pk_by_table.get(table_index, [])

        if pk_columns:
            pk_sql = ", ".join(
                quote_identifier(col)
                for col in pk_columns
            )

            definitions.append(
                f"    PRIMARY KEY ({pk_sql})"
            )

        # Foreign keys
        for source_col, target_table, target_col in fk_by_table.get(
            table_index, []
        ):

            definitions.append(
                "    FOREIGN KEY "
                f"({quote_identifier(source_col)}) "
                f"REFERENCES {quote_identifier(target_table)}"
                f"({quote_identifier(target_col)})"
            )

        statement = (
            f"CREATE TABLE {quote_identifier(table_name)} (\n"
            + ",\n".join(definitions)
            + "\n);"
        )

        ddl_statements.append(statement)

    return "\n\n".join(ddl_statements)


# Kiểm tra trên database đầu tiên của train
test_db = samples[0]["db_id"]

ddl = spider_schema_to_ddl(test_db)

print("Database:", test_db)
print("Tables:", len(schema_by_id[test_db]["table_names_original"]))

print("\n===== GENERATED DDL =====")
print(ddl[:12000])

assert "CREATE TABLE" in ddl
assert "PRIMARY KEY" in ddl
assert len(ddl.strip()) > 0

print("\n===== FINAL STATUS =====")
print("Schema builder: PASS")
print("Full schema: PASS")
print("No schema pruning: PASS")

print("\nHOÀN THÀNH!")

===== SPIDER SCHEMA BUILDER =====
Database: bike_1
Tables: 4

===== GENERATED DDL =====
CREATE TABLE "station" (
    "id" REAL,
    "name" TEXT,
    "lat" REAL,
    "long" REAL,
    "dock_count" REAL,
    "city" TEXT,
    "installation_date" TEXT,
    PRIMARY KEY ("id")
);

CREATE TABLE "status" (
    "station_id" REAL,
    "bikes_available" REAL,
    "docks_available" REAL,
    "time" TEXT,
    FOREIGN KEY ("station_id") REFERENCES "station"("id")
);

CREATE TABLE "trip" (
    "id" REAL,
    "duration" REAL,
    "start_date" TEXT,
    "start_station_name" TEXT,
    "start_station_id" REAL,
    "end_date" TEXT,
    "end_station_name" TEXT,
    "end_station_id" REAL,
    "bike_id" REAL,
    "subscription_type" TEXT,
    "zip_code" REAL,
    PRIMARY KEY ("id")
);

CREATE TABLE "weather" (
    "date" TEXT,
    "max_temperature_f" REAL,
    "mean_temperature_f" REAL,
    "min_temperature_f" REAL,
    "max_dew_point_f" REAL,
    "mean_dew_point_f" REAL,
    "min_dew_point_f" REAL,
    "max_

In [52]:

# CELL 32 — C1 SCHEMA VALIDATION & PROMPT BUILDER
# MAC-SQL Lite v3.1 | Stage 4.4c
# Read-only | Train only | No dev gold

import json
import sqlite3
from pathlib import Path

print("===== PROMPT PREPARATION =====")

# 1. Kiểm tra các đối tượng từ Cell trước
assert "samples" in globals()
assert "schema_by_id" in globals()
assert "spider_schema_to_ddl" in globals()
assert "tokenizer" in globals()

DB_ROOT = Path("/content/spider_data/spider_data/database")

# 2. Đối chiếu schema Spider với SQLite
def validate_spider_schema(db_id):
    schema = schema_by_id[db_id]
    db_path = DB_ROOT / db_id / f"{db_id}.sqlite"

    assert db_path.is_file(), f"Missing DB: {db_id}"

    connection = sqlite3.connect(
        f"file:{db_path.as_posix()}?mode=ro",
        uri=True
    )

    mismatches = []

    try:
        actual_tables = {
            row[0].lower(): row[0]
            for row in connection.execute(
                "SELECT name FROM sqlite_master "
                "WHERE type='table'"
            )
        }

        for table_index, table_name in enumerate(
            schema["table_names_original"]
        ):
            if table_name.lower() not in actual_tables:
                mismatches.append(
                    f"Missing table: {table_name}"
                )
                continue

            actual_name = actual_tables[table_name.lower()]

            pragma_sql = (
                "PRAGMA table_info("
                + quote_identifier(actual_name)
                + ")"
            )

            actual_columns = {
                row[1].lower()
                for row in connection.execute(pragma_sql)
            }

            expected_columns = {
                col_name.lower()
                for owner, col_name
                in schema["column_names_original"]
                if owner == table_index
            }

            missing_columns = (
                expected_columns - actual_columns
            )

            for column in sorted(missing_columns):
                mismatches.append(
                    f"{table_name}.{column}: missing"
                )

    finally:
        connection.close()

    return mismatches


train_db_ids = sorted({
    sample["db_id"] for sample in samples
})

validation_results = {}

for db_id in train_db_ids:
    validation_results[db_id] = (
        validate_spider_schema(db_id)
    )

failed = {
    db_id: errors
    for db_id, errors in validation_results.items()
    if errors
}

print("Train databases checked:", len(train_db_ids))
print("Databases with mismatches:", len(failed))

if failed:
    for db_id, errors in failed.items():
        print(f"\n{db_id}:")
        for error in errors[:10]:
            print(" -", error)

assert not failed, (
    "Có khác biệt giữa Spider metadata và SQLite. "
    "Cần kiểm tra trước khi cố định prompt."
)

print("Schema consistency: PASS")

# 3. Định nghĩa prompt Baseline C1
C1_SYSTEM_PROMPT = (
    "You are an expert Text-to-SQL assistant. "
    "Translate the user's question into one valid "
    "SQLite SELECT query. "
    "Use only the provided database schema. "
    "Return only SQL, without explanations, "
    "markdown, or code fences."
)

def build_c1_messages(question, db_id):
    ddl = spider_schema_to_ddl(db_id)

    user_prompt = (
        "Database schema:\n"
        f"{ddl}\n\n"
        "Question:\n"
        f"{question}\n\n"
        "Write the SQLite query:"
    )

    return [
        {
            "role": "system",
            "content": C1_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]


# 4. Kiểm tra prompt trên train
token_counts = []

for sample in samples:
    messages = build_c1_messages(
        question=sample["question"],
        db_id=sample["db_id"]
    )

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True
    )

    # Tương thích nhiều phiên bản Transformers
    if hasattr(encoded, "keys"):
        ids = encoded["input_ids"]
    else:
        ids = encoded

    if hasattr(ids, "tolist"):
        ids = ids.tolist()

    if ids and isinstance(ids[0], list):
        ids = ids[0]

    token_counts.append(len(ids))

print("\n===== PROMPT TOKEN STATISTICS =====")
print("Samples:", len(token_counts))
print("Min tokens:", min(token_counts))
print("Max tokens:", max(token_counts))
print(
    "Mean tokens:",
    round(sum(token_counts) / len(token_counts), 2)
)

# 5. Hiển thị prompt mẫu
example = samples[0]

messages = build_c1_messages(
    example["question"],
    example["db_id"]
)

print("\n===== C1 PROMPT PREVIEW =====")
print("Sample ID:", example["sample_id"])
print("Database:", example["db_id"])
print("Question:", example["question"])
print("\nSystem prompt:")
print(messages[0]["content"])
print("\nUser prompt preview:")
print(messages[1]["content"][:1500])

print("\n===== FINAL STATUS =====")
print("Schema validation: PASS")
print("C1 prompt builder: PASS")
print("Train-only inspection: PASS")
print("No schema pruning: PASS")
print("No model inference: PASS")

print("\nHOÀN THÀNH!")

===== PROMPT PREPARATION =====
Train databases checked: 27
Databases with mismatches: 0
Schema consistency: PASS

===== PROMPT TOKEN STATISTICS =====
Samples: 30
Min tokens: 169
Max tokens: 823
Mean tokens: 369.4

===== C1 PROMPT PREVIEW =====
Sample ID: train_00204
Database: bike_1
Question: What are the different ids and names of the stations that have had more than 12 bikes available?

System prompt:
You are an expert Text-to-SQL assistant. Translate the user's question into one valid SQLite SELECT query. Use only the provided database schema. Return only SQL, without explanations, markdown, or code fences.

User prompt preview:
Database schema:
CREATE TABLE "station" (
    "id" REAL,
    "name" TEXT,
    "lat" REAL,
    "long" REAL,
    "dock_count" REAL,
    "city" TEXT,
    "installation_date" TEXT,
    PRIMARY KEY ("id")
);

CREATE TABLE "status" (
    "station_id" REAL,
    "bikes_available" REAL,
    "docks_available" REAL,
    "time" TEXT,
    FOREIGN KEY ("station_id") REFER

In [53]:

# CELL 33 — C1 BASELINE TRAIN SMOKE TEST
# MAC-SQL Lite v3.1 | Stage 4.4
# 3 train samples | FP16 | Greedy decoding
# No dev data | No official evaluation

import time
import torch

print("===== C1 TRAIN SMOKE TEST =====")

assert "model" in globals()
assert "tokenizer" in globals()
assert "samples" in globals()
assert "build_c1_messages" in globals()

assert not model.training
assert next(model.parameters()).device.type == "cuda"

# Chọn cố định 3 mẫu đầu tiên của tập train
test_samples = samples[:3]

results = []

for index, sample in enumerate(test_samples, start=1):

    print(f"\n===== SAMPLE {index}/3 =====")
    print("Sample ID:", sample["sample_id"])
    print("Database:", sample["db_id"])
    print("Question:", sample["question"])

    messages = build_c1_messages(
        question=sample["question"],
        db_id=sample["db_id"]
    )

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    if isinstance(encoded, torch.Tensor):
        input_ids = encoded.to("cuda")
        attention_mask = torch.ones_like(input_ids)
    else:
        input_ids = encoded["input_ids"].to("cuda")
        attention_mask = encoded.get("attention_mask")

        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        else:
            attention_mask = attention_mask.to("cuda")

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    generated_ids = output[
        0, input_ids.shape[-1]:
    ]

    predicted_sql = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    result = {
        "sample_id": sample["sample_id"],
        "db_id": sample["db_id"],
        "question": sample["question"],
        "predicted_sql": predicted_sql,
        "gold_sql": sample["gold_sql"],
        "input_tokens": input_ids.shape[-1],
        "output_tokens": len(generated_ids),
        "inference_seconds": round(elapsed, 3)
    }

    results.append(result)

    print("\nPREDICTED SQL:")
    print(predicted_sql)

    print("\nGOLD SQL (TRAIN ONLY):")
    print(sample["gold_sql"])

    print("\nInput tokens:", result["input_tokens"])
    print("Output tokens:", result["output_tokens"])
    print("Inference time:", result["inference_seconds"], "s")

    assert predicted_sql, (
        f"Empty prediction: {sample['sample_id']}"
    )

print("\n===== SUMMARY =====")
print("Samples tested:", len(results))
print(
    "Total inference time:",
    round(
        sum(x["inference_seconds"] for x in results),
        2
    ),
    "seconds"
)

print("\nHOÀN THÀNH!")

===== C1 TRAIN SMOKE TEST =====

===== SAMPLE 1/3 =====
Sample ID: train_00204
Database: bike_1
Question: What are the different ids and names of the stations that have had more than 12 bikes available?

PREDICTED SQL:
SELECT DISTINCT station_id, name FROM status WHERE bikes_available > 12

GOLD SQL (TRAIN ONLY):
SELECT DISTINCT T1.id ,  T1.name FROM station AS T1 JOIN status AS T2 ON T1.id  =  T2.station_id WHERE T2.bikes_available  >  12

Input tokens: 505
Output tokens: 16
Inference time: 0.941 s

===== SAMPLE 2/3 =====
Sample ID: train_00217
Database: book_2
Question: What are the title and issues of the books?

PREDICTED SQL:
SELECT Title, Issues FROM book

GOLD SQL (TRAIN ONLY):
SELECT Title ,  Issues FROM book

Input tokens: 176
Output tokens: 7
Inference time: 0.309 s

===== SAMPLE 3/3 =====
Sample ID: train_00244
Database: musical
Question: What is the duration of the oldest actor?

PREDICTED SQL:
SELECT MAX(Duration) FROM actor

GOLD SQL (TRAIN ONLY):
SELECT Duration FROM act

In [54]:

# CELL 34 — C1 TRAIN EXECUTION DIAGNOSTICS
# MAC-SQL Lite v3.1 | Stage 4.4
# Train only | Read-only | No dev gold

import sqlite3
from pathlib import Path

print("===== C1 EXECUTION DIAGNOSTICS =====")

assert "results" in globals(), "Cần chạy Cell 33 trước"
assert len(results) == 3

DB_ROOT = Path("/content/spider_data/spider_data/database")

def execute_train_sql(db_id, sql):
    """
    Kiểm tra SQL trên SQLite chỉ đọc.
    Dùng SQLite progress handler để giới hạn
    thời gian thực thi và tránh truy vấn kéo dài.
    """
    import time

    db_path = DB_ROOT / db_id / f"{db_id}.sqlite"
    assert db_path.is_file()

    conn = sqlite3.connect(
        f"file:{db_path.as_posix()}?mode=ro",
        uri=True,
        timeout=2
    )

    conn.execute("PRAGMA query_only = ON")

    deadline = time.monotonic() + 3.0

    def progress():
        return 1 if time.monotonic() > deadline else 0

    conn.set_progress_handler(progress, 1000)

    try:
        cursor = conn.execute(sql)

        if cursor.description is None:
            return {
                "status": "error",
                "error": "Not a result-producing query"
            }

        rows = cursor.fetchmany(1001)

        if len(rows) > 1000:
            return {
                "status": "truncated",
                "error": "Result exceeds 1000 rows"
            }

        return {
            "status": "ok",
            "rows": rows
        }

    except sqlite3.Error as exc:
        return {
            "status": "error",
            "error": str(exc)
        }

    finally:
        conn.close()


diagnostics = []

for index, item in enumerate(results, start=1):
    db_id = item["db_id"]

    predicted = execute_train_sql(
        db_id,
        item["predicted_sql"]
    )

    gold = execute_train_sql(
        db_id,
        item["gold_sql"]
    )

    if predicted["status"] != "ok":
        verdict = "PREDICTION_ERROR"

    elif gold["status"] != "ok":
        verdict = "GOLD_EXECUTION_ERROR"

    elif sorted(map(repr, predicted["rows"])) == sorted(
        map(repr, gold["rows"])
    ):
        verdict = "MATCH_ON_ORIGINAL_DB"

    else:
        verdict = "RESULT_MISMATCH"

    diagnostics.append({
        "sample_id": item["sample_id"],
        "db_id": db_id,
        "verdict": verdict,
        "predicted_status": predicted["status"],
        "gold_status": gold["status"]
    })

    print(f"\n===== SAMPLE {index}/3 =====")
    print("Sample ID:", item["sample_id"])
    print("Database:", db_id)
    print("Verdict:", verdict)
    print("Prediction status:", predicted["status"])
    print("Gold status:", gold["status"])

    if predicted["status"] == "error":
        print("Prediction error:", predicted["error"])

    if gold["status"] == "error":
        print("Gold error:", gold["error"])

    if predicted["status"] == "ok":
        print("Predicted rows:", len(predicted["rows"]))

    if gold["status"] == "ok":
        print("Gold rows:", len(gold["rows"]))

print("\n===== SUMMARY =====")

for verdict in sorted(set(x["verdict"] for x in diagnostics)):
    count = sum(
        x["verdict"] == verdict
        for x in diagnostics
    )
    print(f"{verdict}: {count}")

print("\nHOÀN THÀNH!")

===== C1 EXECUTION DIAGNOSTICS =====

===== SAMPLE 1/3 =====
Sample ID: train_00204
Database: bike_1
Verdict: PREDICTION_ERROR
Prediction status: error
Gold status: ok
Prediction error: no such column: name
Gold rows: 9

===== SAMPLE 2/3 =====
Sample ID: train_00217
Database: book_2
Verdict: MATCH_ON_ORIGINAL_DB
Prediction status: ok
Gold status: ok
Predicted rows: 10
Gold rows: 10

===== SAMPLE 3/3 =====
Sample ID: train_00244
Database: musical
Verdict: RESULT_MISMATCH
Prediction status: ok
Gold status: ok
Predicted rows: 1
Gold rows: 1

===== SUMMARY =====
MATCH_ON_ORIGINAL_DB: 1
PREDICTION_ERROR: 1
RESULT_MISMATCH: 1

HOÀN THÀNH!


In [55]:

# CELL 35 — FREEZE C1 BASELINE CONFIGURATION
# MAC-SQL Lite v3.1 | Stage 4.4
# Freeze prompt and decoding configuration

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

print("===== FREEZE C1 BASELINE =====")

assert "C1_SYSTEM_PROMPT" in globals()
assert "build_c1_messages" in globals()
assert "spider_schema_to_ddl" in globals()
assert "model" in globals()
assert "tokenizer" in globals()

BASE = Path(
    "/content/drive/MyDrive/0. Classroom/3. NĂM 3/"
    "HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/"
    "bài báo nghiên cứu/mac_sql_lite_v3"
)

config_dir = BASE / "configs"
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "c1_baseline_v1.json"

config = {
    "experiment": "MAC-SQL Lite v3.1",
    "configuration": "C1",
    "status": "frozen",
    "model": {
        "model_id": "Qwen/Qwen2.5-Coder-3B-Instruct",
        "precision": "float16",
        "seed": 42,
        "evaluation_mode": True
    },
    "prompt": {
        "system": C1_SYSTEM_PROMPT,
        "user_template": (
            "Database schema:\n"
            "{full_schema_ddl}\n\n"
            "Question:\n"
            "{question}\n\n"
            "Write the SQLite query:"
        ),
        "schema_source": "Spider tables.json",
        "schema_representation": "Full schema DDL",
        "schema_pruning": False,
        "chain_of_thought": False,
        "self_refinement": False
    },
    "generation": {
        "do_sample": False,
        "max_new_tokens": 256,
        "pad_token": "eos",
        "decoding": "greedy"
    },
    "evaluation": {
        "dataset": "Spider 1.0",
        "split": "dev",
        "sample_count": 300,
        "official_evaluator": "test-suite-sql-eval",
        "gold_access_during_generation": False
    },
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

# Tạo hash từ cấu hình cốt lõi, không tính timestamp
canonical = {
    k: v for k, v in config.items()
    if k != "created_at_utc"
}

serialized = json.dumps(
    canonical,
    sort_keys=True,
    ensure_ascii=False
)

config_hash = hashlib.sha256(
    serialized.encode("utf-8")
).hexdigest()

config["config_sha256"] = config_hash

# Không ghi đè cấu hình đã đóng băng
if config_path.exists():
    existing = json.loads(
        config_path.read_text(encoding="utf-8")
    )

    assert existing.get("config_sha256") == config_hash, (
        "C1 đã được đóng băng với cấu hình khác. "
        "Không được ghi đè."
    )

    print("Existing frozen config: VERIFIED")

else:
    config_path.write_text(
        json.dumps(
            config,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print("New frozen config: SAVED")

print("\n===== FROZEN CONFIG =====")
print("Path:", config_path)
print("Model:", config["model"]["model_id"])
print("Precision:", config["model"]["precision"])
print("Decoding:", config["generation"]["decoding"])
print("Max new tokens:", config["generation"]["max_new_tokens"])
print("Schema pruning:", config["prompt"]["schema_pruning"])
print("CoT:", config["prompt"]["chain_of_thought"])
print("Refiner:", config["prompt"]["self_refinement"])
print("SHA256:", config_hash)

print("\nHOÀN THÀNH!")

===== FREEZE C1 BASELINE =====
New frozen config: SAVED

===== FROZEN CONFIG =====
Path: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/configs/c1_baseline_v1.json
Model: Qwen/Qwen2.5-Coder-3B-Instruct
Precision: float16
Decoding: greedy
Max new tokens: 256
Schema pruning: False
CoT: False
Refiner: False
SHA256: 27fb1030c8d4da73c06eb42e673e6bb4f30c8e92fbbec4926d4763faab0c9bbe

HOÀN THÀNH!


In [56]:

# CELL 36 — C1 DEV PROMPT LENGTH AUDIT
# MAC-SQL Lite v3.1 | Stage 4.5
# Read-only dev | No gold | No generation

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

print("===== DEV PROMPT LENGTH AUDIT =====")

BASE = Path(
    "/content/drive/MyDrive/0. Classroom/3. NĂM 3/"
    "HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/"
    "bài báo nghiên cứu/mac_sql_lite_v3"
)

# 1. Đọc cấu hình C1 đã đóng băng
config_path = BASE / "configs" / "c1_baseline_v1.json"
protocol_path = BASE / "configs" / "evaluation_protocol_v1.json"

assert config_path.is_file()
assert protocol_path.is_file()

config = json.loads(
    config_path.read_text(encoding="utf-8")
)

protocol = json.loads(
    protocol_path.read_text(encoding="utf-8")
)

assert config["status"] == "frozen"
assert protocol["status"] == "evaluation_protocol_frozen"

assert config["model"]["model_id"] == (
    "Qwen/Qwen2.5-Coder-3B-Instruct"
)

assert config["generation"]["do_sample"] is False
assert config["generation"]["max_new_tokens"] == 256

# 2. Kiểm tra hàm prompt
assert "build_c1_messages" in globals()
assert "tokenizer" in globals()
assert "model" in globals()

assert C1_SYSTEM_PROMPT == config["prompt"]["system"]

# 3. Đọc dev không có gold
dev_path = Path(protocol["files"]["dev_eval"]["path"])

dev_data = json.loads(
    dev_path.read_text(encoding="utf-8")
)

dev_samples = dev_data["samples"]

assert len(dev_samples) == 300

allowed_keys = {
    "sample_id",
    "source_index",
    "db_id",
    "question"
}

assert all(
    set(sample.keys()) == allowed_keys
    for sample in dev_samples
), "DEV có trường dữ liệu ngoài dự kiến"

assert len({
    x["sample_id"] for x in dev_samples
}) == 300

print("Frozen config: PASS")
print("DEV samples:", len(dev_samples))
print("Gold SQL access: NONE")

# 4. Đếm token prompt
audit = []

for index, sample in enumerate(dev_samples):

    messages = build_c1_messages(
        question=sample["question"],
        db_id=sample["db_id"]
    )

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True
    )

    if hasattr(encoded, "keys"):
        ids = encoded["input_ids"]
    else:
        ids = encoded

    if hasattr(ids, "tolist"):
        ids = ids.tolist()

    if ids and isinstance(ids[0], list):
        ids = ids[0]

    token_count = len(ids)

    audit.append({
        "sample_id": sample["sample_id"],
        "db_id": sample["db_id"],
        "input_tokens": token_count
    })

    if (index + 1) % 50 == 0:
        print(f"Audited: {index + 1}/300")

# 5. Thống kê
counts = sorted(
    item["input_tokens"] for item in audit
)

maximum = max(counts)
minimum = min(counts)
mean = sum(counts) / len(counts)

def percentile(sorted_values, p):
    position = (len(sorted_values) - 1) * p
    lower = int(position)
    upper = min(lower + 1, len(sorted_values) - 1)
    weight = position - lower

    return (
        sorted_values[lower] * (1 - weight)
        + sorted_values[upper] * weight
    )

p95 = percentile(counts, 0.95)

print("\n===== TOKEN STATISTICS =====")
print("Minimum:", minimum)
print("Mean:", round(mean, 2))
print("P95:", round(p95, 2))
print("Maximum:", maximum)

# 6. Kiểm tra giới hạn ngữ cảnh
max_new_tokens = config["generation"]["max_new_tokens"]

# Qwen2.5-Coder-3B-Instruct:
# dùng giới hạn ngữ cảnh cấu hình thực tế của model
context_limit = getattr(
    model.config,
    "max_position_embeddings",
    None
)

print("\n===== CONTEXT CHECK =====")
print("Model context limit:", context_limit)
print("Reserved output tokens:", max_new_tokens)

if context_limit is not None:
    over_limit = [
        x for x in audit
        if x["input_tokens"] + max_new_tokens
        > context_limit
    ]

    print("Over context limit:", len(over_limit))

    assert not over_limit, (
        "Có prompt vượt giới hạn ngữ cảnh."
    )
else:
    print(
        "Không xác định được giới hạn ngữ cảnh; "
        "cần kiểm tra cấu hình mô hình."
    )

# 7. Hiển thị 5 prompt dài nhất
largest = sorted(
    audit,
    key=lambda x: x["input_tokens"],
    reverse=True
)[:5]

print("\n===== TOP 5 LONGEST PROMPTS =====")

for item in largest:
    print(
        item["sample_id"],
        "|",
        item["db_id"],
        "|",
        item["input_tokens"],
        "tokens"
    )

# 8. Lưu báo cáo
report_dir = BASE / "results" / "c1"
report_dir.mkdir(parents=True, exist_ok=True)

report_path = report_dir / "dev_prompt_audit_v1.json"

report = {
    "experiment": "MAC-SQL Lite v3.1",
    "configuration": "C1",
    "audit_type": "dev_prompt_length",
    "sample_count": 300,
    "min_tokens": minimum,
    "mean_tokens": round(mean, 2),
    "p95_tokens": round(p95, 2),
    "max_tokens": maximum,
    "context_limit": context_limit,
    "max_new_tokens": max_new_tokens,
    "samples": audit
}

serialized = json.dumps(
    report,
    ensure_ascii=False,
    indent=2
)

if report_path.exists():
    existing = json.loads(
        report_path.read_text(encoding="utf-8")
    )
    assert existing == report, (
        "Báo cáo audit đã tồn tại nhưng khác nội dung."
    )
else:
    report_path.write_text(
        serialized,
        encoding="utf-8"
    )

print("\n===== FINAL STATUS =====")
print("DEV audit: PASS")
print("Frozen C1: PASS")
print("No gold access: PASS")
print("No model generation: PASS")
print("Report saved:", report_path)

print("\nHOÀN THÀNH!")

===== DEV PROMPT LENGTH AUDIT =====
Frozen config: PASS
DEV samples: 300
Gold SQL access: NONE
Audited: 50/300
Audited: 100/300
Audited: 150/300
Audited: 200/300
Audited: 250/300
Audited: 300/300

===== TOKEN STATISTICS =====
Minimum: 193
Mean: 375.48
P95: 860.05
Maximum: 881

===== CONTEXT CHECK =====
Model context limit: 32768
Reserved output tokens: 256
Over context limit: 0

===== TOP 5 LONGEST PROMPTS =====
dev_00541 | student_transcripts_tracking | 881 tokens
dev_00584 | student_transcripts_tracking | 879 tokens
dev_00549 | student_transcripts_tracking | 872 tokens
dev_00562 | student_transcripts_tracking | 868 tokens
dev_00529 | student_transcripts_tracking | 865 tokens

===== FINAL STATUS =====
DEV audit: PASS
Frozen C1: PASS
No gold access: PASS
No model generation: PASS
Report saved: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/results/c1/dev_prompt_audit_v1.json

HOÀN THÀNH!


In [57]:

# CELL 37 — C1 BASELINE DEV INFERENCE
# MAC-SQL Lite v3.1
# 300 dev samples | Greedy | Checkpoint resume
# No gold access | No prompt modification

import json
import time
import hashlib
import torch
from pathlib import Path

print("===== C1 DEV INFERENCE =====")

assert "model" in globals()
assert "tokenizer" in globals()
assert "build_c1_messages" in globals()
assert "dev_samples" in globals()
assert len(dev_samples) == 300
assert not model.training

BASE = Path(
    "/content/drive/MyDrive/0. Classroom/3. NĂM 3/"
    "HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/"
    "bài báo nghiên cứu/mac_sql_lite_v3"
)

config = json.loads(
    (BASE / "configs" / "c1_baseline_v1.json")
    .read_text(encoding="utf-8")
)

assert config["status"] == "frozen"
assert config["generation"]["do_sample"] is False
assert config["generation"]["max_new_tokens"] == 256
assert C1_SYSTEM_PROMPT == config["prompt"]["system"]

output_dir = BASE / "results" / "c1"
output_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = output_dir / "c1_dev_predictions_v1.jsonl"

def canonical_json(obj):
    return json.dumps(
        obj,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":")
    )

# Khóa checkpoint với dữ liệu và cấu hình hiện tại
dataset_hash = hashlib.sha256(
    canonical_json(dev_samples).encode("utf-8")
).hexdigest()

run_signature = hashlib.sha256(
    (
        config["config_sha256"]
        + dataset_hash
    ).encode("utf-8")
).hexdigest()

# Đọc checkpoint cũ nếu có
saved = {}

if checkpoint_path.exists():
    with checkpoint_path.open(
        "r", encoding="utf-8"
    ) as f:
        for line in f:
            if not line.strip():
                continue

            record = json.loads(line)

            assert record["run_signature"] == run_signature, (
                "Checkpoint thuộc cấu hình/dataset khác."
            )

            sample_id = record["sample_id"]

            assert sample_id not in saved, (
                f"Duplicate checkpoint: {sample_id}"
            )

            saved[sample_id] = record

valid_ids = {s["sample_id"] for s in dev_samples}
assert set(saved).issubset(valid_ids)

print("Total dev samples:", len(dev_samples))
print("Already completed:", len(saved))
print("Remaining:", len(dev_samples) - len(saved))
print("Checkpoint:", checkpoint_path)

device = next(model.parameters()).device

start_all = time.perf_counter()

for index, sample in enumerate(dev_samples, start=1):

    sample_id = sample["sample_id"]

    if sample_id in saved:
        continue

    messages = build_c1_messages(
        question=sample["question"],
        db_id=sample["db_id"]
    )

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    if isinstance(encoded, torch.Tensor):
        input_ids = encoded.to(device)
        attention_mask = torch.ones_like(input_ids)
    else:
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded.get("attention_mask")

        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        else:
            attention_mask = attention_mask.to(device)

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    generated_ids = output[
        0, input_ids.shape[-1]:
    ]

    predicted_sql = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    record = {
        "sample_id": sample_id,
        "db_id": sample["db_id"],
        "predicted_sql": predicted_sql,
        "input_tokens": int(input_ids.shape[-1]),
        "output_tokens": int(len(generated_ids)),
        "inference_seconds": round(elapsed, 4),
        "run_signature": run_signature
    }

    # Ghi ngay sau mỗi mẫu để tránh mất tiến độ
    with checkpoint_path.open(
        "a", encoding="utf-8"
    ) as f:
        f.write(canonical_json(record) + "\n")
        f.flush()

    saved[sample_id] = record

    if index % 25 == 0 or index == len(dev_samples):
        print(
            f"Progress: {len(saved)}/300 "
            f"| Last: {sample_id} "
            f"| {elapsed:.2f}s"
        )

# Kiểm tra toàn bộ checkpoint
assert len(saved) == 300

ordered_predictions = [
    saved[s["sample_id"]]
    for s in dev_samples
]

assert len(ordered_predictions) == 300
assert len({
    x["sample_id"] for x in ordered_predictions
}) == 300

empty_count = sum(
    not x["predicted_sql"]
    for x in ordered_predictions
)

total_inference = sum(
    x["inference_seconds"]
    for x in ordered_predictions
)

print("\n===== INFERENCE SUMMARY =====")
print("Completed:", len(ordered_predictions))
print("Empty predictions:", empty_count)
print("Total model inference:", round(total_inference, 2), "s")
print("Current session elapsed:", round(
    time.perf_counter() - start_all, 2
), "s")
print("Checkpoint:", checkpoint_path)

print("\nHOÀN THÀNH!")

===== C1 DEV INFERENCE =====
Total dev samples: 300
Already completed: 0
Remaining: 300
Checkpoint: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/results/c1/c1_dev_predictions_v1.jsonl
Progress: 25/300 | Last: dev_00080 | 2.16s
Progress: 50/300 | Last: dev_00163 | 0.82s
Progress: 75/300 | Last: dev_00235 | 1.60s
Progress: 100/300 | Last: dev_00331 | 1.11s
Progress: 125/300 | Last: dev_00432 | 0.37s
Progress: 150/300 | Last: dev_00526 | 2.45s
Progress: 175/300 | Last: dev_00603 | 0.85s
Progress: 200/300 | Last: dev_00687 | 0.64s
Progress: 225/300 | Last: dev_00768 | 3.18s
Progress: 250/300 | Last: dev_00849 | 0.68s
Progress: 275/300 | Last: dev_00947 | 3.03s
Progress: 300/300 | Last: dev_01028 | 1.29s

===== INFERENCE SUMMARY =====
Completed: 300
Empty predictions: 0
Total model inference: 468.88 s
Current session elapsed: 471.1 s
Checkpoint: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn 

In [58]:

# CELL 38 — OFFICIAL C1 TEST-SUITE EVALUATION
# MAC-SQL Lite v3.1 | Spider dev 300
# Gold used only for evaluation

import json
import subprocess
import sys
import hashlib
from pathlib import Path

print("===== OFFICIAL C1 EVALUATION =====")

BASE = Path(
    "/content/drive/MyDrive/0. Classroom/3. NĂM 3/"
    "HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/"
    "bài báo nghiên cứu/mac_sql_lite_v3"
)

protocol = json.loads(
    (BASE / "configs" / "evaluation_protocol_v1.json")
    .read_text(encoding="utf-8")
)

config = json.loads(
    (BASE / "configs" / "c1_baseline_v1.json")
    .read_text(encoding="utf-8")
)

assert protocol["status"] == "evaluation_protocol_frozen"
assert config["status"] == "frozen"

dev = json.loads(
    Path(protocol["files"]["dev_eval"]["path"])
    .read_text(encoding="utf-8")
)["samples"]

gold = json.loads(
    Path(protocol["files"]["dev_gold"]["path"])
    .read_text(encoding="utf-8")
)["samples"]

prediction_path = (
    BASE / "results" / "c1" /
    "c1_dev_predictions_v1.jsonl"
)

predictions = [
    json.loads(line)
    for line in prediction_path.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

assert len(dev) == len(gold) == len(predictions) == 300

# Validate frozen run signature
def canonical_json(obj):
    return json.dumps(
        obj,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":")
    )

dataset_hash = hashlib.sha256(
    canonical_json(dev).encode("utf-8")
).hexdigest()

expected_signature = hashlib.sha256(
    (
        config["config_sha256"] + dataset_hash
    ).encode("utf-8")
).hexdigest()

prediction_map = {}

for p in predictions:
    assert p["run_signature"] == expected_signature
    assert p["sample_id"] not in prediction_map
    prediction_map[p["sample_id"]] = p

gold_map = {}

for g in gold:
    assert g["sample_id"] not in gold_map
    gold_map[g["sample_id"]] = g

assert len(prediction_map) == 300
assert len(gold_map) == 300

for sample in dev:
    sid = sample["sample_id"]
    assert sid in prediction_map
    assert sid in gold_map
    assert prediction_map[sid]["db_id"] == sample["db_id"]
    assert gold_map[sid]["db_id"] == sample["db_id"]

print("Dataset alignment: PASS")
print("Checkpoint signature: PASS")

# Prepare evaluator files
eval_dir = BASE / "results" / "c1" / "official_eval"
eval_dir.mkdir(parents=True, exist_ok=True)

gold_txt = eval_dir / "gold.txt"
pred_txt = eval_dir / "pred.txt"
result_txt = eval_dir / "evaluation_output.txt"

gold_lines = []
pred_lines = []

for sample in dev:
    sid = sample["sample_id"]
    db_id = sample["db_id"]

    gold_sql = gold_map[sid]["gold_sql"]
    predicted_sql = prediction_map[sid]["predicted_sql"]

    assert "\n" not in gold_sql
    assert "\n" not in predicted_sql
    assert "\t" not in gold_sql

    gold_lines.append(f"{gold_sql}\t{db_id}")
    pred_lines.append(predicted_sql)

gold_txt.write_text(
    "\n".join(gold_lines) + "\n",
    encoding="utf-8"
)

pred_txt.write_text(
    "\n".join(pred_lines) + "\n",
    encoding="utf-8"
)

print("Evaluator input files: PASS")

# Run official evaluator in subprocess
eval_repo = Path("/content/test-suite-sql-eval")
suite_root = Path(
    protocol["evaluation"]["test_suite_root"]
)

assert eval_repo.is_dir()
assert suite_root.is_dir()

evaluation_script = r'''
import sys
from pathlib import Path
import evaluation

gold_path = sys.argv[1]
pred_path = sys.argv[2]
db_dir = sys.argv[3]

evaluation.evaluate(
    gold=gold_path,
    predict=pred_path,
    db_dir=db_dir,
    etype="exec",
    kmaps=None,
    plug_value=False,
    keep_distinct=True,
    progress_bar_for_each_datapoint=False
)
'''

process = subprocess.run(
    [
        sys.executable,
        "-c",
        evaluation_script,
        str(gold_txt),
        str(pred_txt),
        str(suite_root)
    ],
    cwd=str(eval_repo),
    capture_output=True,
    text=True,
    timeout=1800
)

combined_output = (
    process.stdout + "\n" + process.stderr
)

result_txt.write_text(
    combined_output,
    encoding="utf-8"
)

print("\n===== OFFICIAL EVALUATOR OUTPUT =====")
print(combined_output[-12000:])

print("Evaluator exit code:", process.returncode)
print("Saved output:", result_txt)

assert process.returncode == 0, (
    "Official evaluator failed. "
    "Inspect evaluation_output.txt"
)

print("\nHOÀN THÀNH!")

===== OFFICIAL C1 EVALUATION =====
Dataset alignment: PASS
Checkpoint signature: PASS
Evaluator input files: PASS

===== OFFICIAL EVALUATOR OUTPUT =====
                     easy                 medium               hard                 extra                all                 
count                75                   131                  46                   48                   300                 
=====================   EXECUTION ACCURACY     =====================
execution            0.760                0.603                0.304                0.417                0.567               


Evaluator exit code: 0
Saved output: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/results/c1/official_eval/evaluation_output.txt

HOÀN THÀNH!


In [59]:

# CELL 39 — C1 RESULT INTEGRITY & SUMMARY
# MAC-SQL Lite v3.1 | Stage 4
# No model generation | No re-evaluation

import json
import re
import hashlib
import statistics
from pathlib import Path
from datetime import datetime, timezone

print("===== C1 RESULT INTEGRITY =====")

BASE = Path(
    "/content/drive/MyDrive/0. Classroom/3. NĂM 3/"
    "HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/"
    "bài báo nghiên cứu/mac_sql_lite_v3"
)

config_path = BASE / "configs" / "c1_baseline_v1.json"
protocol_path = BASE / "configs" / "evaluation_protocol_v1.json"
pred_path = BASE / "results" / "c1" / "c1_dev_predictions_v1.jsonl"
eval_path = BASE / "results" / "c1" / "official_eval" / "evaluation_output.txt"

for path in [config_path, protocol_path, pred_path, eval_path]:
    assert path.is_file(), f"Missing file: {path}"

config = json.loads(config_path.read_text(encoding="utf-8"))
protocol = json.loads(protocol_path.read_text(encoding="utf-8"))

assert config["status"] == "frozen"
assert protocol["status"] == "evaluation_protocol_frozen"

# Verify config SHA256
canonical_config = {
    k: v for k, v in config.items()
    if k not in ("created_at_utc", "config_sha256")
}

def canonical_json(obj):
    return json.dumps(
        obj,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":")
    )

# Cell 35 used default JSON separators
config_serialized = json.dumps(
    canonical_config,
    sort_keys=True,
    ensure_ascii=False
)

calculated_config_hash = hashlib.sha256(
    config_serialized.encode("utf-8")
).hexdigest()

assert calculated_config_hash == config["config_sha256"], (
    "Frozen C1 config hash mismatch"
)

print("Frozen config integrity: PASS")

# Load frozen dev
dev_path = Path(protocol["files"]["dev_eval"]["path"])
dev_data = json.loads(dev_path.read_text(encoding="utf-8"))
dev_samples = dev_data["samples"]

assert len(dev_samples) == 300

# Verify checkpoint run signature
dataset_hash = hashlib.sha256(
    canonical_json(dev_samples).encode("utf-8")
).hexdigest()

expected_signature = hashlib.sha256(
    (config["config_sha256"] + dataset_hash).encode("utf-8")
).hexdigest()

predictions = [
    json.loads(line)
    for line in pred_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

assert len(predictions) == 300

prediction_map = {}

for p in predictions:
    sid = p["sample_id"]

    assert sid not in prediction_map, f"Duplicate: {sid}"
    assert p["run_signature"] == expected_signature
    assert isinstance(p["predicted_sql"], str)
    assert p["predicted_sql"].strip()

    prediction_map[sid] = p

for sample in dev_samples:
    sid = sample["sample_id"]

    assert sid in prediction_map, f"Missing: {sid}"
    assert prediction_map[sid]["db_id"] == sample["db_id"]

assert len(prediction_map) == 300

print("Prediction integrity: PASS")
print("Checkpoint signature: PASS")
print("Empty predictions: 0")

# Parse official evaluator output
eval_text = eval_path.read_text(encoding="utf-8")

match = re.search(
    r"^\s*execution\s+"
    r"([0-9.]+)\s+([0-9.]+)\s+"
    r"([0-9.]+)\s+([0-9.]+)\s+"
    r"([0-9.]+)\s*$",
    eval_text,
    flags=re.MULTILINE
)

assert match, "Cannot parse official execution accuracy"

accuracy = {
    "easy": float(match.group(1)),
    "medium": float(match.group(2)),
    "hard": float(match.group(3)),
    "extra": float(match.group(4)),
    "all": float(match.group(5))
}

assert all(0 <= value <= 1 for value in accuracy.values())

# Check official count row
count_match = re.search(
    r"^\s*count\s+(\d+)\s+(\d+)\s+"
    r"(\d+)\s+(\d+)\s+(\d+)\s*$",
    eval_text,
    flags=re.MULTILINE
)

assert count_match, "Cannot parse official count row"

counts = {
    "easy": int(count_match.group(1)),
    "medium": int(count_match.group(2)),
    "hard": int(count_match.group(3)),
    "extra": int(count_match.group(4)),
    "all": int(count_match.group(5))
}

assert counts["all"] == 300
assert sum(counts[k] for k in ("easy", "medium", "hard", "extra")) == 300

# Inference statistics
times = [float(p["inference_seconds"]) for p in predictions]
output_tokens = [int(p["output_tokens"]) for p in predictions]
input_tokens = [int(p["input_tokens"]) for p in predictions]

summary = {
    "experiment": "MAC-SQL Lite v3.1",
    "configuration": "C1",
    "status": "evaluated",
    "dataset": "Spider 1.0 dev",
    "sample_count": 300,
    "model": config["model"]["model_id"],
    "precision": config["model"]["precision"],
    "decoding": config["generation"]["decoding"],
    "metric": "Test Suite Execution Accuracy",
    "execution_accuracy": accuracy,
    "difficulty_counts": counts,
    "inference": {
        "total_seconds": round(sum(times), 4),
        "mean_seconds": round(statistics.mean(times), 4),
        "median_seconds": round(statistics.median(times), 4),
        "total_output_tokens": sum(output_tokens),
        "mean_output_tokens": round(statistics.mean(output_tokens), 2),
        "mean_input_tokens": round(statistics.mean(input_tokens), 2)
    },
    "integrity": {
        "config_sha256": config["config_sha256"],
        "dataset_sha256": dataset_hash,
        "run_signature": expected_signature,
        "prediction_count": len(predictions),
        "empty_predictions": 0
    },
    "artifacts": {
        "predictions": str(pred_path),
        "official_evaluator_output": str(eval_path),
        "frozen_config": str(config_path),
        "frozen_protocol": str(protocol_path)
    }
}

summary_path = BASE / "results" / "c1" / "c1_summary_v1.json"

if summary_path.exists():
    existing = json.loads(summary_path.read_text(encoding="utf-8"))
    assert existing == summary, (
        "Existing summary differs. Refusing overwrite."
    )
else:
    summary_path.write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

print("\n===== OFFICIAL EXECUTION ACCURACY =====")
for level in ("easy", "medium", "hard", "extra", "all"):
    print(
        f"{level.upper():8s}: "
        f"{accuracy[level] * 100:5.1f}% "
        f"(n={counts[level]})"
    )

print("\n===== INFERENCE STATISTICS =====")
print("Total seconds:", summary["inference"]["total_seconds"])
print("Mean seconds/sample:", summary["inference"]["mean_seconds"])
print("Median seconds/sample:", summary["inference"]["median_seconds"])
print("Mean output tokens:", summary["inference"]["mean_output_tokens"])
print("Mean input tokens:", summary["inference"]["mean_input_tokens"])

print("\n===== FINAL STATUS =====")
print("Frozen configuration: PASS")
print("Dataset alignment: PASS")
print("Predictions: 300/300")
print("Official evaluation parsed: PASS")
print("Summary saved:", summary_path)

print("\nHOÀN THÀNH!")

===== C1 RESULT INTEGRITY =====
Frozen config integrity: PASS
Prediction integrity: PASS
Checkpoint signature: PASS
Empty predictions: 0

===== OFFICIAL EXECUTION ACCURACY =====
EASY    :  76.0% (n=75)
MEDIUM  :  60.3% (n=131)
HARD    :  30.4% (n=46)
EXTRA   :  41.7% (n=48)
ALL     :  56.7% (n=300)

===== INFERENCE STATISTICS =====
Total seconds: 468.8769
Mean seconds/sample: 1.5629
Median seconds/sample: 1.4253
Mean output tokens: 34.01
Mean input tokens: 375.48

===== FINAL STATUS =====
Frozen configuration: PASS
Dataset alignment: PASS
Predictions: 300/300
Official evaluation parsed: PASS
Summary saved: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/results/c1/c1_summary_v1.json

HOÀN THÀNH!


In [60]:

# CELL 40 — C1 FINAL MANIFEST
# MAC-SQL Lite v3.1 | End of Stage 4
# No generation | No evaluation | No overwrites

import json
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import torch
import transformers

print("===== CELL 40 — C1 FINAL MANIFEST =====")

BASE = Path(
    "/content/drive/MyDrive/0. Classroom/3. NĂM 3/"
    "HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/"
    "bài báo nghiên cứu/mac_sql_lite_v3"
)

assert "model" in globals()
assert "tokenizer" in globals()

files = {
    "evaluation_protocol":
        BASE / "configs" / "evaluation_protocol_v1.json",
    "c1_config":
        BASE / "configs" / "c1_baseline_v1.json",
    "dev_dataset":
        BASE / "data" / "dev_eval_300_seed42.json",
    "dev_gold":
        BASE / "data" / "dev_gold_300_seed42.json",
    "predictions":
        BASE / "results" / "c1" /
        "c1_dev_predictions_v1.jsonl",
    "evaluation_output":
        BASE / "results" / "c1" /
        "official_eval" / "evaluation_output.txt",
    "summary":
        BASE / "results" / "c1" /
        "c1_summary_v1.json"
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)
    return digest.hexdigest()

file_manifest = {}

for name, path in files.items():
    assert path.is_file(), f"Missing: {path}"

    file_manifest[name] = {
        "path": str(path),
        "sha256": sha256_file(path),
        "size_bytes": path.stat().st_size
    }

print("Artifact integrity: PASS")
print("Files verified:", len(file_manifest))

summary = json.loads(
    files["summary"].read_text(encoding="utf-8")
)

assert summary["configuration"] == "C1"
assert summary["sample_count"] == 300
assert summary["execution_accuracy"]["all"] == 0.567

# Capture evaluator commit
eval_repo = Path("/content/test-suite-sql-eval")

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=eval_repo,
    text=True
).strip()

protocol = json.loads(
    files["evaluation_protocol"].read_text(
        encoding="utf-8"
    )
)

assert commit == protocol["evaluation"]["evaluator_commit"], (
    "Evaluator commit differs from frozen protocol"
)

print("Evaluator commit: PASS")

# Model revision is not necessarily pinned locally.
# Record available metadata without claiming exact weight hash.
model_metadata = {
    "model_id": model.config._name_or_path,
    "architecture": type(model).__name__,
    "dtype": str(next(model.parameters()).dtype),
    "device": str(next(model.parameters()).device),
    "transformers_version": transformers.__version__,
    "torch_version": torch.__version__,
    "exact_weight_revision_verified": False
}

manifest = {
    "experiment": "MAC-SQL Lite v3.1",
    "stage": 4,
    "configuration": "C1",
    "status": "completed",
    "sample_count": 300,
    "metric": "Test Suite Execution Accuracy",
    "accuracy": summary["execution_accuracy"],
    "model": model_metadata,
    "evaluator_commit": commit,
    "artifacts": file_manifest,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

manifest_path = (
    BASE / "results" / "c1" /
    "c1_final_manifest_v1.json"
)

if manifest_path.exists():
    existing = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )

    # Compare stable fields; ignore creation timestamp
    existing_stable = {
        k: v for k, v in existing.items()
        if k != "created_at_utc"
    }

    current_stable = {
        k: v for k, v in manifest.items()
        if k != "created_at_utc"
    }

    assert existing_stable == current_stable, (
        "Existing manifest differs. Refusing overwrite."
    )

    print("Existing manifest: VERIFIED")

else:
    manifest_path.write_text(
        json.dumps(
            manifest,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    print("New manifest: SAVED")

print("\n===== FINAL C1 STATUS =====")
print("Samples:", manifest["sample_count"])
print("Execution Accuracy:",
      manifest["accuracy"]["all"] * 100, "%")
print("Artifacts:", len(file_manifest))
print("Model:", model_metadata["model_id"])
print("Evaluator commit:", commit)
print("Manifest:", manifest_path)

print("\nCOMPLETED!")

===== CELL 40 — C1 FINAL MANIFEST =====
Artifact integrity: PASS
Files verified: 7
Evaluator commit: PASS
New manifest: SAVED

===== FINAL C1 STATUS =====
Samples: 300
Execution Accuracy: 56.699999999999996 %
Artifacts: 7
Model: Qwen/Qwen2.5-Coder-3B-Instruct
Evaluator commit: e97acc546ecbee8fa27fa8dbf025ef61493a876c
Manifest: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/results/c1/c1_final_manifest_v1.json

COMPLETED!


In [61]:

# CELL 41 — SELECTOR TABLE SELECTION SMOKE TEST
# MAC-SQL Lite v3.1 | Stage 4 | C2
# Train only | No dev gold | No SQL generation

import json
import re
import torch
import time

print("===== SELECTOR SMOKE TEST =====")

assert "model" in globals()
assert "tokenizer" in globals()
assert "samples" in globals()
assert "schema_by_id" in globals()
assert "spider_schema_to_ddl" in globals()

assert not model.training

SELECTOR_SYSTEM_PROMPT = (
    "You are a database schema selection agent. "
    "Given a natural language question and a database schema, "
    "identify the tables required to answer the question. "
    "Include tables needed for joins. "
    "Return only a JSON object with the key 'tables', "
    "containing a list of table names. "
    "Do not generate SQL or explanations."
)

def build_selector_messages(question, db_id):
    ddl = spider_schema_to_ddl(db_id)

    return [
        {
            "role": "system",
            "content": SELECTOR_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Database schema:\n{ddl}\n\n"
                f"Question:\n{question}\n\n"
                "Select the relevant tables:"
            )
        }
    ]

def parse_selector_tables(text, db_id):
    schema = schema_by_id[db_id]
    available = schema["table_names_original"]

    match = re.search(r"\{.*\}", text, re.DOTALL)

    if not match:
        return None, "NO_JSON"

    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None, "INVALID_JSON"

    if not isinstance(parsed, dict):
        return None, "INVALID_STRUCTURE"

    tables = parsed.get("tables")

    if not isinstance(tables, list):
        return None, "MISSING_TABLE_LIST"

    if not tables or not all(
        isinstance(t, str) for t in tables
    ):
        return None, "INVALID_TABLE_LIST"

    canonical = {
        name.casefold(): name
        for name in available
    }

    selected = []

    for table in tables:
        key = table.casefold()

        if key not in canonical:
            return None, f"UNKNOWN_TABLE: {table}"

        name = canonical[key]

        if name not in selected:
            selected.append(name)

    return selected, "PASS"


# Dùng đúng 3 mẫu train đã kiểm tra ở Cell 33
selector_train_samples = samples[:3]
selector_smoke_results = []

device = next(model.parameters()).device

for index, sample in enumerate(
    selector_train_samples, start=1
):
    db_id = sample["db_id"]

    messages = build_selector_messages(
        sample["question"], db_id
    )

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    if isinstance(encoded, torch.Tensor):
        input_ids = encoded.to(device)
        attention_mask = torch.ones_like(input_ids)
    else:
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded.get("attention_mask")

        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        else:
            attention_mask = attention_mask.to(device)

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    generated = output[0, input_ids.shape[-1]:]

    raw_output = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    selected_tables, status = parse_selector_tables(
        raw_output, db_id
    )

    record = {
        "sample_id": sample["sample_id"],
        "db_id": db_id,
        "question": sample["question"],
        "raw_output": raw_output,
        "selected_tables": selected_tables,
        "parse_status": status,
        "inference_seconds": round(elapsed, 4)
    }

    selector_smoke_results.append(record)

    print(f"\n===== SAMPLE {index}/3 =====")
    print("Sample ID:", sample["sample_id"])
    print("Database:", db_id)
    print("Question:", sample["question"])
    print("Available tables:",
          schema_by_id[db_id]["table_names_original"])
    print("Raw selector output:", raw_output)
    print("Selected tables:", selected_tables)
    print("Parse status:", status)
    print("Inference seconds:", round(elapsed, 3))

print("\n===== SUMMARY =====")
print("Samples:", len(selector_smoke_results))
print(
    "Valid selections:",
    sum(
        r["parse_status"] == "PASS"
        for r in selector_smoke_results
    )
)
print(
    "Total selector inference seconds:",
    round(
        sum(
            r["inference_seconds"]
            for r in selector_smoke_results
        ),
        3
    )
)

print("\nHOÀN THÀNH!")

===== SELECTOR SMOKE TEST =====

===== SAMPLE 1/3 =====
Sample ID: train_00204
Database: bike_1
Question: What are the different ids and names of the stations that have had more than 12 bikes available?
Available tables: ['station', 'status', 'trip', 'weather']
Raw selector output: ```json
{
  "tables": ["station", "status"]
}
```
Selected tables: ['station', 'status']
Parse status: PASS
Inference seconds: 1.439

===== SAMPLE 2/3 =====
Sample ID: train_00217
Database: book_2
Question: What are the title and issues of the books?
Available tables: ['publication', 'book']
Raw selector output: ```json
{
  "tables": ["book"]
}
```
Selected tables: ['book']
Parse status: PASS
Inference seconds: 1.262

===== SAMPLE 3/3 =====
Sample ID: train_00244
Database: musical
Question: What is the duration of the oldest actor?
Available tables: ['musical', 'actor']
Raw selector output: ```json
{
  "tables": ["actor"]
}
```
Selected tables: ['actor']
Parse status: PASS
Inference seconds: 0.589

===== SUM

In [62]:

# CELL 42 — SELECTOR FOREIGN KEY CONNECTIVITY AUDIT
# MAC-SQL Lite v3.1 | Stage 4 | C2
# Train only | No model inference | No dev access

from collections import deque

print("===== SELECTOR JOIN CONNECTIVITY =====")

assert "schema_by_id" in globals()
assert "selector_smoke_results" in globals()

def get_fk_graph(db_id):
    schema = schema_by_id[db_id]
    table_names = schema["table_names_original"]
    columns = schema["column_names_original"]

    graph = {
        table_name: set()
        for table_name in table_names
    }

    fk_details = []

    for source_col, target_col in schema["foreign_keys"]:
        source_table_idx, source_name = columns[source_col]
        target_table_idx, target_name = columns[target_col]

        if source_table_idx < 0 or target_table_idx < 0:
            continue

        source_table = table_names[source_table_idx]
        target_table = table_names[target_table_idx]

        if source_table == target_table:
            continue

        graph[source_table].add(target_table)
        graph[target_table].add(source_table)

        fk_details.append({
            "source_table": source_table,
            "source_column": source_name,
            "target_table": target_table,
            "target_column": target_name
        })

    return graph, fk_details


def shortest_table_path(graph, start, end):
    queue = deque([(start, [start])])
    visited = {start}

    while queue:
        current, path = queue.popleft()

        if current == end:
            return path

        for neighbor in sorted(graph[current]):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(
                    (neighbor, path + [neighbor])
                )

    return None


def audit_selected_tables(db_id, selected_tables):
    graph, fk_details = get_fk_graph(db_id)

    if not selected_tables:
        return {
            "status": "EMPTY_SELECTION",
            "connected": False,
            "expanded_tables": [],
            "added_bridge_tables": []
        }

    # Kết nối từng bảng được chọn với thành phần
    # đã kết nối, ưu tiên đường đi ngắn nhất.
    expanded = {selected_tables[0]}
    remaining = set(selected_tables[1:])
    added_bridges = set()

    while remaining:
        candidates = []

        for target in sorted(remaining):
            for source in sorted(expanded):
                path = shortest_table_path(
                    graph, source, target
                )

                if path is not None:
                    candidates.append(
                        (len(path), target, path)
                    )

        if not candidates:
            return {
                "status": "DISCONNECTED",
                "connected": False,
                "expanded_tables": sorted(expanded),
                "added_bridge_tables": sorted(added_bridges)
            }

        _, target, best_path = min(candidates)

        for table in best_path:
            if table not in selected_tables:
                added_bridges.add(table)
            expanded.add(table)

        remaining.remove(target)

    return {
        "status": "PASS",
        "connected": True,
        "expanded_tables": sorted(expanded),
        "added_bridge_tables": sorted(added_bridges),
        "foreign_keys": [
            fk for fk in fk_details
            if (
                fk["source_table"] in expanded
                and fk["target_table"] in expanded
            )
        ]
    }


selector_join_audits = []

for record in selector_smoke_results:
    assert record["parse_status"] == "PASS"

    audit = audit_selected_tables(
        record["db_id"],
        record["selected_tables"]
    )

    selector_join_audits.append({
        "sample_id": record["sample_id"],
        "db_id": record["db_id"],
        **audit
    })

    print("\nSample:", record["sample_id"])
    print("Database:", record["db_id"])
    print("Original tables:", record["selected_tables"])
    print("Expanded tables:", audit["expanded_tables"])
    print("Added bridge tables:", audit["added_bridge_tables"])
    print("Connectivity:", audit["status"])

    for fk in audit.get("foreign_keys", []):
        print(
            "FK:",
            f'{fk["source_table"]}.{fk["source_column"]}',
            "->",
            f'{fk["target_table"]}.{fk["target_column"]}'
        )

print("\n===== SUMMARY =====")
print("Samples audited:", len(selector_join_audits))
print(
    "Connected selections:",
    sum(
        x["connected"]
        for x in selector_join_audits
    )
)

print("\nHOÀN THÀNH!")

===== SELECTOR JOIN CONNECTIVITY =====

Sample: train_00204
Database: bike_1
Original tables: ['station', 'status']
Expanded tables: ['station', 'status']
Added bridge tables: []
Connectivity: PASS
FK: status.station_id -> station.id

Sample: train_00217
Database: book_2
Original tables: ['book']
Expanded tables: ['book']
Added bridge tables: []
Connectivity: PASS

Sample: train_00244
Database: musical
Original tables: ['actor']
Expanded tables: ['actor']
Added bridge tables: []
Connectivity: PASS

===== SUMMARY =====
Samples audited: 3
Connected selections: 3

HOÀN THÀNH!


In [63]:

# CELL 43 — C2 TABLE-LEVEL SCHEMA PRUNING
# MAC-SQL Lite v3.1 | Stage 4 | C2
# Train only | No model inference | No dev access

import sqlite3
import statistics

print("===== CELL 43 — C2 SCHEMA PRUNING =====")

assert "schema_by_id" in globals()
assert "spider_schema_to_ddl" in globals()
assert "selector_smoke_results" in globals()
assert "selector_join_audits" in globals()
assert "tokenizer" in globals()
assert "quote_identifier" in globals()

def build_pruned_schema_ddl(db_id, selected_tables):
    schema = schema_by_id[db_id]

    table_names = schema["table_names_original"]
    column_names = schema["column_names_original"]
    column_types = schema["column_types"]

    available = set(table_names)
    selected = set(selected_tables)

    assert selected, "Empty table selection"
    assert selected.issubset(available), "Unknown table"

    type_mapping = {
        "text": "TEXT",
        "number": "REAL",
        "time": "TEXT",
        "boolean": "INTEGER",
        "others": "TEXT"
    }

    primary_keys = set(schema["primary_keys"])
    statements = []

    for table_index, table_name in enumerate(table_names):

        if table_name not in selected:
            continue

        definitions = []

        for col_index, (owner, col_name) in enumerate(column_names):

            if owner != table_index:
                continue

            sql_type = type_mapping.get(
                column_types[col_index], "TEXT"
            )

            definitions.append(
                f"    {quote_identifier(col_name)} {sql_type}"
            )

        table_pks = [
            col_index
            for col_index in primary_keys
            if column_names[col_index][0] == table_index
        ]

        if table_pks:
            pk_columns = ", ".join(
                quote_identifier(column_names[i][1])
                for i in sorted(table_pks)
            )

            definitions.append(
                f"    PRIMARY KEY ({pk_columns})"
            )

        for source_idx, target_idx in schema["foreign_keys"]:

            source_owner, source_col = column_names[source_idx]
            target_owner, target_col = column_names[target_idx]

            if source_owner != table_index:
                continue

            if target_owner < 0:
                continue

            target_table = table_names[target_owner]

            if target_table not in selected:
                continue

            definitions.append(
                "    FOREIGN KEY "
                f"({quote_identifier(source_col)}) "
                "REFERENCES "
                f"{quote_identifier(target_table)}"
                f"({quote_identifier(target_col)})"
            )

        statement = (
            f"CREATE TABLE {quote_identifier(table_name)} (\n"
            + ",\n".join(definitions)
            + "\n);"
        )

        statements.append(statement)

    return "\n\n".join(statements)


def count_tokens(text):
    encoded = tokenizer(
        text,
        add_special_tokens=False
    )
    return len(encoded["input_ids"])


audit_map = {
    record["sample_id"]: record
    for record in selector_join_audits
}

pruning_results = []

for record in selector_smoke_results:

    sample_id = record["sample_id"]
    db_id = record["db_id"]

    assert record["parse_status"] == "PASS"
    assert sample_id in audit_map
    assert audit_map[sample_id]["connected"]

    selected_tables = audit_map[sample_id]["expanded_tables"]

    full_ddl = spider_schema_to_ddl(db_id)

    pruned_ddl = build_pruned_schema_ddl(
        db_id,
        selected_tables
    )

    # Validate DDL syntax in an isolated in-memory database
    connection = sqlite3.connect(":memory:")

    try:
        connection.executescript(pruned_ddl)

        actual_tables = {
            row[0]
            for row in connection.execute(
                "SELECT name FROM sqlite_master "
                "WHERE type='table'"
            )
        }

        assert actual_tables == set(selected_tables)

    finally:
        connection.close()

    full_tokens = count_tokens(full_ddl)
    pruned_tokens = count_tokens(pruned_ddl)

    reduction = (
        (full_tokens - pruned_tokens) / full_tokens * 100
        if full_tokens else 0
    )

    result = {
        "sample_id": sample_id,
        "db_id": db_id,
        "selected_tables": selected_tables,
        "full_schema_tokens": full_tokens,
        "pruned_schema_tokens": pruned_tokens,
        "reduction_percent": round(reduction, 2),
        "pruned_ddl": pruned_ddl
    }

    pruning_results.append(result)

    print(f"\n===== {sample_id} =====")
    print("Database:", db_id)
    print("Selected tables:", selected_tables)
    print("Full schema tokens:", full_tokens)
    print("Pruned schema tokens:", pruned_tokens)
    print("Token reduction:", round(reduction, 2), "%")
    print("DDL validation: PASS")

    if sample_id == "train_00204":
        print("\nPRUNED DDL PREVIEW:")
        print(pruned_ddl)

print("\n===== SUMMARY =====")
print("Samples:", len(pruning_results))

print(
    "Mean token reduction:",
    round(
        statistics.mean(
            r["reduction_percent"]
            for r in pruning_results
        ),
        2
    ),
    "%"
)

print(
    "DDL validation:",
    "PASS" if len(pruning_results) == 3 else "CHECK"
)

print("\nCELL 43 HOÀN THÀNH!")


===== CELL 43 — C2 SCHEMA PRUNING =====

===== train_00204 =====
Database: bike_1
Selected tables: ['station', 'status']
Full schema tokens: 419
Pruned schema tokens: 107
Token reduction: 74.46 %
DDL validation: PASS

PRUNED DDL PREVIEW:
CREATE TABLE "station" (
    "id" REAL,
    "name" TEXT,
    "lat" REAL,
    "long" REAL,
    "dock_count" REAL,
    "city" TEXT,
    "installation_date" TEXT,
    PRIMARY KEY ("id")
);

CREATE TABLE "status" (
    "station_id" REAL,
    "bikes_available" REAL,
    "docks_available" REAL,
    "time" TEXT,
    FOREIGN KEY ("station_id") REFERENCES "station"("id")
);

===== train_00217 =====
Database: book_2
Selected tables: ['book']
Full schema tokens: 101
Pruned schema tokens: 39
Token reduction: 61.39 %
DDL validation: PASS

===== train_00244 =====
Database: musical
Selected tables: ['actor']
Full schema tokens: 130
Pruned schema tokens: 69
Token reduction: 46.92 %
DDL validation: PASS

===== SUMMARY =====
Samples: 3
Mean token reduction: 60.92 %
DDL 

In [64]:

# CELL 44 — C2 END-TO-END TRAIN SMOKE TEST
# MAC-SQL Lite v3.1 | Stage 4 | C2
# Train only | 3 samples | No dev evaluation

import time
import re
import sqlite3
import torch
from pathlib import Path

print("===== CELL 44 — C2 END-TO-END SMOKE TEST =====")

required = [
    "model", "tokenizer", "samples",
    "pruning_results", "C1_SYSTEM_PROMPT"
]
for name in required:
    assert name in globals(), f"Missing global: {name}"

assert not model.training

SPIDER_DB_ROOT = Path(
    "/content/spider_data/spider_data/database"
)

pruning_map = {
    r["sample_id"]: r
    for r in pruning_results
}

def build_c2_sql_messages(question, pruned_ddl):
    return [
        {
            "role": "system",
            "content": C1_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Database schema:\n{pruned_ddl}\n\n"
                f"Question:\n{question}\n\n"
                "Write the SQLite query:"
            )
        }
    ]


def generate_c2_sql(messages):
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    device = next(model.parameters()).device

    if isinstance(encoded, torch.Tensor):
        input_ids = encoded.to(device)
        attention_mask = torch.ones_like(input_ids)
    else:
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded.get("attention_mask")
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        else:
            attention_mask = attention_mask.to(device)

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    generated = output[0, input_ids.shape[-1]:]

    raw_sql = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    sql = re.sub(
        r"^```(?:sql)?\s*",
        "",
        raw_sql,
        flags=re.IGNORECASE
    )
    sql = re.sub(r"\s*```$", "", sql).strip()

    return {
        "sql": sql,
        "raw_output": raw_sql,
        "input_tokens": int(input_ids.shape[-1]),
        "output_tokens": int(generated.numel()),
        "inference_seconds": elapsed
    }


def readonly_sqlite_check(db_id, sql):
    db_path = (
        SPIDER_DB_ROOT / db_id / f"{db_id}.sqlite"
    )

    assert db_path.is_file(), f"Missing DB: {db_path}"

    # Smoke test only. Use Safe Worker for
    # robust timeout/isolation in later evaluation.
    connection = sqlite3.connect(
        f"file:{db_path}?mode=ro",
        uri=True,
        timeout=2
    )

    try:
        connection.execute("PRAGMA query_only = ON")

        # Only allow one SELECT/WITH statement.
        stripped = sql.strip().rstrip(";").strip()

        if not re.match(
            r"^(SELECT|WITH)\b",
            stripped,
            flags=re.IGNORECASE
        ):
            return "REJECTED_NON_SELECT", None

        cursor = connection.execute(stripped)
        rows = cursor.fetchmany(3)

        return "EXECUTED", rows

    except sqlite3.Error as exc:
        return "SQLITE_ERROR", str(exc)

    finally:
        connection.close()


c2_train_smoke_results = []

for sample in samples[:3]:
    sample_id = sample["sample_id"]
    db_id = sample["db_id"]

    assert sample_id in pruning_map

    pruned_ddl = pruning_map[sample_id]["pruned_ddl"]

    messages = build_c2_sql_messages(
        sample["question"],
        pruned_ddl
    )

    generation = generate_c2_sql(messages)

    execution_status, execution_preview = (
        readonly_sqlite_check(
            db_id,
            generation["sql"]
        )
    )

    result = {
        "sample_id": sample_id,
        "db_id": db_id,
        "question": sample["question"],
        "selected_tables":
            pruning_map[sample_id]["selected_tables"],
        "sql": generation["sql"],
        "execution_status": execution_status,
        "execution_preview": execution_preview,
        "input_tokens": generation["input_tokens"],
        "output_tokens": generation["output_tokens"],
        "inference_seconds":
            round(generation["inference_seconds"], 4)
    }

    c2_train_smoke_results.append(result)

    print(f"\n===== {sample_id} =====")
    print("Question:", sample["question"])
    print("Selected tables:", result["selected_tables"])
    print("Generated SQL:", result["sql"])
    print("Execution status:", execution_status)
    print("Execution preview:", execution_preview)
    print("SQL input tokens:", result["input_tokens"])
    print("SQL output tokens:", result["output_tokens"])
    print(
        "SQL inference seconds:",
        result["inference_seconds"]
    )

print("\n===== SUMMARY =====")
print("Samples:", len(c2_train_smoke_results))
print(
    "Executed:",
    sum(
        r["execution_status"] == "EXECUTED"
        for r in c2_train_smoke_results
    )
)
print(
    "Mean SQL input tokens:",
    round(
        sum(
            r["input_tokens"]
            for r in c2_train_smoke_results
        ) / len(c2_train_smoke_results),
        2
    )
)

print("\nCELL 44 HOÀN THÀNH!")


===== CELL 44 — C2 END-TO-END SMOKE TEST =====

===== train_00204 =====
Question: What are the different ids and names of the stations that have had more than 12 bikes available?
Selected tables: ['station', 'status']
Generated SQL: SELECT DISTINCT T1.id, T1.name FROM station AS T1 INNER JOIN status AS T2 ON T1.id = T2.station_id WHERE T2.bikes_available > 12
Execution status: EXECUTED
Execution preview: [(45, 'Commercial at Montgomery'), (48, 'Embarcadero at Vallejo'), (56, 'Beale at Market')]
SQL input tokens: 193
SQL output tokens: 40
SQL inference seconds: 1.6667

===== train_00217 =====
Question: What are the title and issues of the books?
Selected tables: ['book']
Generated SQL: SELECT Title, Issues FROM book
Execution status: EXECUTED
Execution preview: [('The Black Lamb', 6.0), ('Bloody Mary', 4.0), ('Bloody Mary : Lady Liberty', 4.0)]
SQL input tokens: 114
SQL output tokens: 7
SQL inference seconds: 0.2919

===== train_00244 =====
Question: What is the duration of the oldest a

In [65]:

# CELL 45 — C2 SELECTOR TRAIN-30 AUDIT
# MAC-SQL Lite v3.1 | Stage 4
# Train only | No dev gold | No SQL generation

import time
import statistics
import torch

print("===== SELECTOR TRAIN-30 AUDIT =====")

required = [
    "samples",
    "selector_smoke_results",
    "build_selector_messages",
    "parse_selector_tables",
    "audit_selected_tables",
    "schema_by_id",
    "model",
    "tokenizer"
]

for name in required:
    assert name in globals(), f"Missing: {name}"

assert len(samples) == 30
assert not model.training

device = next(model.parameters()).device

selector_train30_results = {
    r["sample_id"]: dict(r)
    for r in selector_smoke_results
}

assert len(selector_train30_results) == 3

for sample in samples:
    sample_id = sample["sample_id"]

    if sample_id in selector_train30_results:
        continue

    messages = build_selector_messages(
        sample["question"],
        sample["db_id"]
    )

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    if isinstance(encoded, torch.Tensor):
        input_ids = encoded.to(device)
        attention_mask = torch.ones_like(input_ids)
    else:
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded.get("attention_mask")

        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        else:
            attention_mask = attention_mask.to(device)

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    generated = output[0, input_ids.shape[-1]:]

    raw_output = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    selected_tables, parse_status = (
        parse_selector_tables(
            raw_output,
            sample["db_id"]
        )
    )

    selector_train30_results[sample_id] = {
        "sample_id": sample_id,
        "db_id": sample["db_id"],
        "question": sample["question"],
        "raw_output": raw_output,
        "selected_tables": selected_tables,
        "parse_status": parse_status,
        "inference_seconds": round(elapsed, 4)
    }

    print(
        f"{len(selector_train30_results):02d}/30",
        sample_id,
        parse_status,
        selected_tables
    )


# Preserve original train sample order.
selector_train30_results = [
    selector_train30_results[s["sample_id"]]
    for s in samples
]

assert len(selector_train30_results) == 30

for record in selector_train30_results:

    available = schema_by_id[
        record["db_id"]
    ]["table_names_original"]

    record["available_table_count"] = len(available)

    if record["parse_status"] != "PASS":
        record["join_status"] = "NOT_EVALUATED"
        record["expanded_tables"] = None
        record["selected_table_count"] = 0
        continue

    audit = audit_selected_tables(
        record["db_id"],
        record["selected_tables"]
    )

    record["join_status"] = audit["status"]
    record["expanded_tables"] = audit["expanded_tables"]
    record["selected_table_count"] = len(
        audit["expanded_tables"]
    )


valid = [
    r for r in selector_train30_results
    if r["parse_status"] == "PASS"
]

connected = [
    r for r in valid
    if r["join_status"] == "PASS"
]

print("\n===== SELECTOR TRAIN-30 SUMMARY =====")

print("Total samples:", len(selector_train30_results))
print("JSON parse PASS:", len(valid), "/30")
print("FK connectivity PASS:", len(connected), "/30")

print(
    "Mean selector inference seconds:",
    round(
        statistics.mean(
            r["inference_seconds"]
            for r in selector_train30_results
        ),
        3
    )
)

if connected:
    print(
        "Mean retained tables:",
        round(
            statistics.mean(
                r["selected_table_count"]
                for r in connected
            ),
            2
        )
    )

    print(
        "Mean available tables:",
        round(
            statistics.mean(
                r["available_table_count"]
                for r in connected
            ),
            2
        )
    )

print("\n===== FAILURES =====")

failures = [
    r for r in selector_train30_results
    if (
        r["parse_status"] != "PASS"
        or r["join_status"] != "PASS"
    )
]

if not failures:
    print("None")
else:
    for r in failures:
        print(
            r["sample_id"],
            r["parse_status"],
            r["join_status"],
            r["raw_output"][:200]
        )

print("\nHOÀN THÀNH!")

===== SELECTOR TRAIN-30 AUDIT =====
04/30 train_00260 PASS ['musical']
05/30 train_00712 PASS ['Customers', 'Customers_Cards']
06/30 train_00767 PASS ['race']
07/30 train_00839 PASS ['Album', 'Track']
08/30 train_00912 PASS ['Customers', 'Customers_Policies', 'Available_Policies']
09/30 train_01143 PASS ['mountain', 'climber']
10/30 train_01628 PASS ['Products', 'Order_Items']
11/30 train_01791 PASS ['ACCOUNTS', 'CHECKING', 'SAVINGS']
12/30 train_01828 PASS ['browser']
13/30 train_01905 PASS ['School', 'budget']
14/30 train_02006 PASS ['company', 'gas_station']
15/30 train_02253 PASS ['technician', 'machine', 'repair_assignment']
16/30 train_03456 PASS ['countries', 'locations']
17/30 train_04139 PASS ['film_market_estimation', 'market']
18/30 train_04464 PASS ['Person', 'PersonFriend']
19/30 train_04467 PASS ['Person', 'PersonFriend']
20/30 train_04597 PASS ['Products', 'Products_in_Events']
21/30 train_04837 PASS ['pilot', 'match']
22/30 train_04931 PASS ['product', 'store_product']


In [66]:

# CELL 46 — INVESTIGATE DISCONNECTED SELECTOR CASES
# MAC-SQL Lite v3.1 | Stage 4 | C2
# Train only | Read-only | No inference

import sqlite3
from pathlib import Path
from collections import deque

print("===== CELL 46 — DISCONNECTED CASE INVESTIGATION =====")

assert "samples" in globals()
assert "schema_by_id" in globals()
assert "selector_train30_results" in globals()
assert "get_fk_graph" in globals()

DB_ROOT = Path(
    "/content/spider_data/spider_data/database"
)

train_by_id = {
    s["sample_id"]: s
    for s in samples
}

failures = [
    r for r in selector_train30_results
    if r["join_status"] == "DISCONNECTED"
]

assert len(failures) == 2, (
    f"Expected 2 disconnected cases, got {len(failures)}"
)


def connected_components(graph):
    remaining = set(graph)
    components = []

    while remaining:
        start = sorted(remaining)[0]
        queue = deque([start])
        visited = {start}

        while queue:
            current = queue.popleft()

            for neighbor in graph[current]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)

        components.append(sorted(visited))
        remaining -= visited

    return components


def inspect_sqlite_schema(db_id, table_names):
    db_path = DB_ROOT / db_id / f"{db_id}.sqlite"

    assert db_path.is_file(), f"Missing DB: {db_path}"

    conn = sqlite3.connect(
        f"file:{db_path}?mode=ro",
        uri=True
    )

    try:
        conn.execute("PRAGMA query_only = ON")

        for table in table_names:
            print(f"\n  TABLE: {table}")

            # Read actual CREATE TABLE definition
            row = conn.execute(
                "SELECT sql FROM sqlite_master "
                "WHERE type='table' AND name=?",
                (table,)
            ).fetchone()

            print("  CREATE SQL:")
            print(" ", row[0] if row else "NOT FOUND")

            escaped = table.replace('"', '""')

            columns = conn.execute(
                f'PRAGMA table_info("{escaped}")'
            ).fetchall()

            print("  COLUMNS:")
            for col in columns:
                print(
                    "   ",
                    col[1],
                    "| type:", col[2],
                    "| PK:", col[5]
                )

            fks = conn.execute(
                f'PRAGMA foreign_key_list("{escaped}")'
            ).fetchall()

            print("  SQLITE FOREIGN KEYS:")
            if fks:
                for fk in fks:
                    print(
                        "   ",
                        fk[3], "->",
                        fk[2] + "." + fk[4]
                    )
            else:
                print("    None declared")

    finally:
        conn.close()


for record in failures:
    sample_id = record["sample_id"]
    db_id = record["db_id"]
    sample = train_by_id[sample_id]

    schema = schema_by_id[db_id]
    graph, fk_details = get_fk_graph(db_id)

    print("\n" + "=" * 65)
    print("SAMPLE:", sample_id)
    print("DATABASE:", db_id)
    print("QUESTION:", sample["question"])
    print("SELECTED TABLES:", record["selected_tables"])

    print("\nALL TABLES:")
    for name in schema["table_names_original"]:
        print(" -", name)

    print("\nSPIDER FOREIGN KEYS:")
    if fk_details:
        for fk in fk_details:
            print(
                f' {fk["source_table"]}.'
                f'{fk["source_column"]} -> '
                f'{fk["target_table"]}.'
                f'{fk["target_column"]}'
            )
    else:
        print(" None declared")

    print("\nFK GRAPH COMPONENTS:")
    for component in connected_components(graph):
        print(" ", component)

    print("\nSQLITE SCHEMA DETAILS:")
    inspect_sqlite_schema(
        db_id,
        schema["table_names_original"]
    )

print("\n===== SUMMARY =====")
print("Disconnected cases inspected:", len(failures))
print("No database modifications performed.")
print("\nCELL 46 HOÀN THÀNH!")


===== CELL 46 — DISCONNECTED CASE INVESTIGATION =====

SAMPLE: train_00712
DATABASE: customers_card_transactions
QUESTION: What are the different customer ids, and how many cards does each one hold?
SELECTED TABLES: ['Customers', 'Customers_Cards']

ALL TABLES:
 - Accounts
 - Customers
 - Customers_Cards
 - Financial_Transactions

SPIDER FOREIGN KEYS:
 Financial_Transactions.account_id -> Accounts.account_id
 Financial_Transactions.card_id -> Customers_Cards.card_id

FK GRAPH COMPONENTS:
  ['Accounts', 'Customers_Cards', 'Financial_Transactions']
  ['Customers']

SQLITE SCHEMA DETAILS:

  TABLE: Accounts
  CREATE SQL:
  CREATE TABLE `Accounts` (
`account_id` INTEGER PRIMARY KEY,
`customer_id` INTEGER NOT NULL,
`account_name` VARCHAR(50),
`other_account_details` VARCHAR(255)
)
  COLUMNS:
    account_id | type: INTEGER | PK: 1
    customer_id | type: INTEGER | PK: 0
    account_name | type: VARCHAR(50) | PK: 0
    other_account_details | type: VARCHAR(255) | PK: 0
  SQLITE FOREIGN KEYS:


In [67]:

# CELL 47 — FK METADATA RECONCILIATION
# MAC-SQL Lite v3.1 | Stage 4 | C2
# Train only | Read-only | No inference

import sqlite3
from pathlib import Path
from collections import deque

print("===== CELL 47 — FK METADATA RECONCILIATION =====")

assert "schema_by_id" in globals()
assert "selector_train30_results" in globals()
assert "get_fk_graph" in globals()

DB_ROOT = Path(
    "/content/spider_data/spider_data/database"
)


def canonical_table_lookup(db_id):
    names = schema_by_id[db_id]["table_names_original"]

    lookup = {}

    for name in names:
        key = name.casefold()
        assert key not in lookup, (
            f"Ambiguous case-insensitive table name: {db_id}"
        )
        lookup[key] = name

    return lookup


def sqlite_declared_foreign_keys(db_id):
    db_path = DB_ROOT / db_id / f"{db_id}.sqlite"
    assert db_path.is_file()

    lookup = canonical_table_lookup(db_id)
    relationships = []

    conn = sqlite3.connect(
        f"file:{db_path}?mode=ro",
        uri=True
    )

    try:
        conn.execute("PRAGMA query_only = ON")

        for table in schema_by_id[db_id][
            "table_names_original"
        ]:
            escaped = table.replace('"', '""')

            rows = conn.execute(
                f'PRAGMA foreign_key_list("{escaped}")'
            ).fetchall()

            for row in rows:
                target_raw = row[2]
                target = lookup.get(
                    target_raw.casefold()
                )

                if target is None:
                    continue

                relationships.append({
                    "source_table": table,
                    "source_column": row[3],
                    "target_table": target,
                    "target_column": row[4],
                    "source": "sqlite_declared"
                })

    finally:
        conn.close()

    return relationships


def combined_fk_graph(db_id):
    table_names = schema_by_id[db_id][
        "table_names_original"
    ]

    graph = {
        table: set()
        for table in table_names
    }

    _, spider_fks = get_fk_graph(db_id)

    relationships = []
    seen = set()

    for fk in spider_fks:
        record = dict(fk)
        record["source"] = "spider_metadata"

        key = (
            record["source_table"].casefold(),
            record["source_column"].casefold(),
            record["target_table"].casefold(),
            record["target_column"].casefold()
        )

        seen.add(key)
        relationships.append(record)

    sqlite_fks = sqlite_declared_foreign_keys(db_id)

    for record in sqlite_fks:
        key = (
            record["source_table"].casefold(),
            record["source_column"].casefold(),
            record["target_table"].casefold(),
            record["target_column"].casefold()
        )

        if key not in seen:
            seen.add(key)
            relationships.append(record)

    for fk in relationships:
        a = fk["source_table"]
        b = fk["target_table"]

        if a == b:
            continue

        graph[a].add(b)
        graph[b].add(a)

    return graph, relationships


def shortest_path(graph, start, end):
    queue = deque([(start, [start])])
    visited = {start}

    while queue:
        node, path = queue.popleft()

        if node == end:
            return path

        for neighbor in sorted(graph[node]):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(
                    (neighbor, path + [neighbor])
                )

    return None


def audit_combined_fk(db_id, selected_tables):
    graph, relationships = combined_fk_graph(db_id)

    if not selected_tables:
        return {
            "status": "EMPTY",
            "expanded_tables": [],
            "added_bridges": [],
            "relationships": relationships
        }

    expanded = {selected_tables[0]}
    remaining = set(selected_tables[1:])
    added_bridges = set()

    while remaining:
        candidates = []

        for target in sorted(remaining):
            for source in sorted(expanded):
                path = shortest_path(
                    graph, source, target
                )

                if path:
                    candidates.append(
                        (len(path), target, path)
                    )

        if not candidates:
            return {
                "status": "UNRESOLVED",
                "expanded_tables": sorted(
                    expanded | remaining
                ),
                "added_bridges": sorted(added_bridges),
                "relationships": relationships
            }

        _, target, path = min(candidates)

        for table in path:
            if table not in selected_tables:
                added_bridges.add(table)
            expanded.add(table)

        remaining.remove(target)

    return {
        "status": "PASS",
        "expanded_tables": sorted(expanded),
        "added_bridges": sorted(added_bridges),
        "relationships": relationships
    }


target_ids = {
    "train_00712",
    "train_04931"
}

c2_fk_reconciliation_results = []

for record in selector_train30_results:
    if record["sample_id"] not in target_ids:
        continue

    db_id = record["db_id"]

    result = audit_combined_fk(
        db_id,
        record["selected_tables"]
    )

    c2_fk_reconciliation_results.append({
        "sample_id": record["sample_id"],
        "db_id": db_id,
        "selected_tables": record["selected_tables"],
        **result
    })

    print("\nSAMPLE:", record["sample_id"])
    print("DATABASE:", db_id)
    print("SELECTED:", record["selected_tables"])
    print("STATUS:", result["status"])
    print("EXPANDED:", result["expanded_tables"])
    print("BRIDGES:", result["added_bridges"])

    print("RELATIONSHIPS:")
    for fk in result["relationships"]:
        print(
            f'  {fk["source_table"]}.'
            f'{fk["source_column"]} -> '
            f'{fk["target_table"]}.'
            f'{fk["target_column"]} '
            f'[{fk["source"]}]'
        )

assert len(c2_fk_reconciliation_results) == 2

print("\n===== SUMMARY =====")
print(
    "Resolved:",
    sum(
        r["status"] == "PASS"
        for r in c2_fk_reconciliation_results
    ),
    "/2"
)
print(
    "Unresolved:",
    sum(
        r["status"] == "UNRESOLVED"
        for r in c2_fk_reconciliation_results
    ),
    "/2"
)

print("\nCELL 47 HOÀN THÀNH!")


===== CELL 47 — FK METADATA RECONCILIATION =====

SAMPLE: train_00712
DATABASE: customers_card_transactions
SELECTED: ['Customers', 'Customers_Cards']
STATUS: UNRESOLVED
EXPANDED: ['Customers', 'Customers_Cards']
BRIDGES: []
RELATIONSHIPS:
  Financial_Transactions.account_id -> Accounts.account_id [spider_metadata]
  Financial_Transactions.card_id -> Customers_Cards.card_id [spider_metadata]

SAMPLE: train_04931
DATABASE: store_product
SELECTED: ['product', 'store_product']
STATUS: PASS
EXPANDED: ['product', 'store_product']
BRIDGES: []
RELATIONSHIPS:
  store_product.Store_ID -> store.Store_ID [spider_metadata]
  store_district.District_ID -> district.District_ID [spider_metadata]
  store_district.Store_ID -> store.Store_ID [spider_metadata]
  store_product.Product_ID -> product.Product_ID [sqlite_declared]

===== SUMMARY =====
Resolved: 1 /2
Unresolved: 1 /2

CELL 47 HOÀN THÀNH!


In [68]:

# CELL 48 — C2 RECONCILED SCHEMA PRUNING
# MAC-SQL Lite v3.1 | Stage 4 | C2
# Train only | No LLM inference | No dev access

import sqlite3
import statistics
from collections import Counter

print("===== CELL 48 — RECONCILED SCHEMA PRUNING =====")

required = [
    "schema_by_id",
    "selector_train30_results",
    "combined_fk_graph",
    "audit_combined_fk",
    "spider_schema_to_ddl",
    "quote_identifier",
    "tokenizer"
]

for name in required:
    assert name in globals(), f"Missing: {name}"


def build_c2_pruned_ddl(db_id, selected_tables):
    schema = schema_by_id[db_id]

    names = schema["table_names_original"]
    columns = schema["column_names_original"]
    types = schema["column_types"]

    selected = set(selected_tables)

    assert selected
    assert selected.issubset(set(names))

    type_map = {
        "text": "TEXT",
        "number": "REAL",
        "time": "TEXT",
        "boolean": "INTEGER",
        "others": "TEXT"
    }

    _, relationships = combined_fk_graph(db_id)

    # Deduplicate case-insensitive FK definitions.
    fk_seen = set()
    fk_records = []

    for fk in relationships:
        key = (
            fk["source_table"].casefold(),
            fk["source_column"].casefold(),
            fk["target_table"].casefold(),
            fk["target_column"].casefold()
        )

        if key not in fk_seen:
            fk_seen.add(key)
            fk_records.append(fk)

    primary_keys = set(schema["primary_keys"])

    statements = []

    for table_idx, table_name in enumerate(names):

        if table_name not in selected:
            continue

        definitions = []

        for col_idx, (owner, col_name) in enumerate(columns):
            if owner != table_idx:
                continue

            sql_type = type_map.get(
                types[col_idx], "TEXT"
            )

            definitions.append(
                f"    {quote_identifier(col_name)} {sql_type}"
            )

        pk_indices = sorted(
            col_idx
            for col_idx in primary_keys
            if columns[col_idx][0] == table_idx
        )

        if pk_indices:
            pk_sql = ", ".join(
                quote_identifier(columns[i][1])
                for i in pk_indices
            )

            definitions.append(
                f"    PRIMARY KEY ({pk_sql})"
            )

        for fk in fk_records:
            if fk["source_table"] != table_name:
                continue

            if fk["target_table"] not in selected:
                continue

            definitions.append(
                "    FOREIGN KEY "
                f'({quote_identifier(fk["source_column"])}) '
                "REFERENCES "
                f'{quote_identifier(fk["target_table"])}'
                f'({quote_identifier(fk["target_column"])})'
            )

        statements.append(
            f"CREATE TABLE {quote_identifier(table_name)} (\n"
            + ",\n".join(definitions)
            + "\n);"
        )

    return "\n\n".join(statements)


def validate_c2_ddl(ddl, expected_tables):
    conn = sqlite3.connect(":memory:")

    try:
        conn.executescript(ddl)

        actual_tables = {
            row[0]
            for row in conn.execute(
                "SELECT name FROM sqlite_master "
                "WHERE type='table'"
            )
        }

        assert actual_tables == set(expected_tables), (
            "DDL table mismatch"
        )

        fk_count = 0

        for table in expected_tables:
            escaped = table.replace('"', '""')

            fk_rows = conn.execute(
                f'PRAGMA foreign_key_list("{escaped}")'
            ).fetchall()

            fk_count += len(fk_rows)

        return fk_count

    finally:
        conn.close()


def c2_token_count(text):
    return len(
        tokenizer(
            text,
            add_special_tokens=False
        )["input_ids"]
    )


c2_train30_pruning = []

for record in selector_train30_results:

    assert record["parse_status"] == "PASS"

    db_id = record["db_id"]

    audit = audit_combined_fk(
        db_id,
        record["selected_tables"]
    )

    assert audit["status"] in {
        "PASS", "UNRESOLVED"
    }

    expanded_tables = audit["expanded_tables"]

    # Every originally selected table must survive.
    assert set(record["selected_tables"]).issubset(
        set(expanded_tables)
    )

    ddl = build_c2_pruned_ddl(
        db_id,
        expanded_tables
    )

    fk_count = validate_c2_ddl(
        ddl,
        expanded_tables
    )

    full_ddl = spider_schema_to_ddl(db_id)

    full_tokens = c2_token_count(full_ddl)
    pruned_tokens = c2_token_count(ddl)

    reduction = (
        100 * (full_tokens - pruned_tokens)
        / full_tokens
        if full_tokens else 0
    )

    result = {
        "sample_id": record["sample_id"],
        "db_id": db_id,
        "selected_tables": record["selected_tables"],
        "expanded_tables": expanded_tables,
        "join_status": audit["status"],
        "added_bridges": audit["added_bridges"],
        "full_schema_tokens": full_tokens,
        "pruned_schema_tokens": pruned_tokens,
        "reduction_percent": round(reduction, 2),
        "declared_fk_count": fk_count,
        "pruned_ddl": ddl
    }

    c2_train30_pruning.append(result)

    print(
        record["sample_id"],
        "|", audit["status"],
        "| tables:", len(expanded_tables),
        "| FK:", fk_count,
        "| reduction:", f"{reduction:.2f}%"
    )


assert len(c2_train30_pruning) == 30

statuses = Counter(
    r["join_status"]
    for r in c2_train30_pruning
)

print("\n===== SUMMARY =====")
print("Samples:", len(c2_train30_pruning))
print("DDL validation: 30/30 PASS")
print("Connectivity:", dict(statuses))

print(
    "Mean schema token reduction:",
    round(
        statistics.mean(
            r["reduction_percent"]
            for r in c2_train30_pruning
        ),
        2
    ),
    "%"
)

print(
    "Mean retained tables:",
    round(
        statistics.mean(
            len(r["expanded_tables"])
            for r in c2_train30_pruning
        ),
        2
    )
)

print("\n===== SPECIAL CASES =====")

for sample_id in ["train_00712", "train_04931"]:
    result = next(
        r for r in c2_train30_pruning
        if r["sample_id"] == sample_id
    )

    print("\nSample:", sample_id)
    print("Status:", result["join_status"])
    print("Tables:", result["expanded_tables"])
    print("Declared FK count:", result["declared_fk_count"])
    print("DDL:")
    print(result["pruned_ddl"])

print("\nCELL 48 HOÀN THÀNH!")


===== CELL 48 — RECONCILED SCHEMA PRUNING =====
train_00204 | PASS | tables: 2 | FK: 1 | reduction: 74.46%
train_00217 | PASS | tables: 1 | FK: 0 | reduction: 61.39%
train_00244 | PASS | tables: 1 | FK: 1 | reduction: 46.92%
train_00260 | PASS | tables: 1 | FK: 0 | reduction: 53.08%
train_00712 | UNRESOLVED | tables: 2 | FK: 0 | reduction: 52.46%
train_00767 | PASS | tables: 1 | FK: 0 | reduction: 57.80%
train_00839 | PASS | tables: 2 | FK: 1 | reduction: 83.49%
train_00912 | PASS | tables: 3 | FK: 2 | reduction: 63.01%
train_01143 | PASS | tables: 2 | FK: 1 | reduction: 0.00%
train_01628 | PASS | tables: 2 | FK: 1 | reduction: 81.13%
train_01791 | PASS | tables: 3 | FK: 2 | reduction: 0.00%
train_01828 | PASS | tables: 1 | FK: 0 | reduction: 79.08%
train_01905 | PASS | tables: 2 | FK: 1 | reduction: 26.91%
train_02006 | PASS | tables: 3 | FK: 2 | reduction: 0.00%
train_02253 | PASS | tables: 3 | FK: 2 | reduction: 23.11%
train_03456 | PASS | tables: 2 | FK: 1 | reduction: 75.98%
train

In [69]:

# CELL 49 — C2 SCHEMA INTEGRITY AUDIT
# MAC-SQL Lite v3.1 | Stage 4 | C2
# Train only | Read-only | No inference

import sqlite3
from pathlib import Path
from collections import Counter

print("===== CELL 49 — SCHEMA INTEGRITY AUDIT =====")

assert "c2_train30_pruning" in globals()
assert "schema_by_id" in globals()

DB_ROOT = Path(
    "/content/spider_data/spider_data/database"
)

def quote_sqlite_name(name):
    return '"' + name.replace('"', '""') + '"'


def get_table_structure(conn, table):
    quoted = quote_sqlite_name(table)

    columns = conn.execute(
        f"PRAGMA table_info({quoted})"
    ).fetchall()

    column_names = [r[1] for r in columns]

    # Preserve composite PK order.
    primary_key = [
        r[1]
        for r in sorted(
            (r for r in columns if r[5] > 0),
            key=lambda x: x[5]
        )
    ]

    foreign_keys = conn.execute(
        f"PRAGMA foreign_key_list({quoted})"
    ).fetchall()

    # Each FK component: source column, target table,
    # target column, and constraint ID.
    fk_components = [
        (
            str(r[3]).casefold(),
            str(r[2]).casefold(),
            str(r[4]).casefold() if r[4] else None,
            r[0],
            r[1]
        )
        for r in foreign_keys
    ]

    return {
        "columns": column_names,
        "primary_key": primary_key,
        "foreign_keys": fk_components
    }


def normalize_fk(fk):
    return (
        fk[0],
        fk[1],
        fk[2]
    )


integrity_results = []

for record in c2_train30_pruning:
    db_id = record["db_id"]
    selected_tables = record["expanded_tables"]

    db_path = DB_ROOT / db_id / f"{db_id}.sqlite"
    assert db_path.is_file()

    original = sqlite3.connect(
        f"file:{db_path}?mode=ro",
        uri=True
    )

    pruned = sqlite3.connect(":memory:")

    issues = []

    try:
        original.execute("PRAGMA query_only = ON")
        pruned.executescript(record["pruned_ddl"])

        selected_lower = {
            t.casefold()
            for t in selected_tables
        }

        for table in selected_tables:
            source = get_table_structure(
                original, table
            )
            reduced = get_table_structure(
                pruned, table
            )

            if [
                c.casefold() for c in source["columns"]
            ] != [
                c.casefold() for c in reduced["columns"]
            ]:
                issues.append({
                    "table": table,
                    "type": "COLUMN_MISMATCH",
                    "original": source["columns"],
                    "pruned": reduced["columns"]
                })

            if [
                c.casefold() for c in source["primary_key"]
            ] != [
                c.casefold() for c in reduced["primary_key"]
            ]:
                issues.append({
                    "table": table,
                    "type": "PRIMARY_KEY_MISMATCH",
                    "original": source["primary_key"],
                    "pruned": reduced["primary_key"]
                })

            # Compare only FKs whose target table survives.
            original_fk = {
                normalize_fk(fk)
                for fk in source["foreign_keys"]
                if fk[1] in selected_lower
            }

            reduced_fk = {
                normalize_fk(fk)
                for fk in reduced["foreign_keys"]
            }

            missing_fk = original_fk - reduced_fk

            if missing_fk:
                issues.append({
                    "table": table,
                    "type": "MISSING_DECLARED_FK",
                    "missing": sorted(
                        missing_fk,
                        key=str
                    )
                })

    finally:
        original.close()
        pruned.close()

    integrity_results.append({
        "sample_id": record["sample_id"],
        "db_id": db_id,
        "issues": issues
    })

    status = "PASS" if not issues else "REVIEW"

    print(
        record["sample_id"],
        "|", status,
        "| issues:", len(issues)
    )

    for issue in issues:
        print("   ", issue)


issue_counts = Counter(
    issue["type"]
    for result in integrity_results
    for issue in result["issues"]
)

passed = sum(
    not result["issues"]
    for result in integrity_results
)

print("\n===== SUMMARY =====")
print("Samples audited:", len(integrity_results))
print("Fully matched:", passed, "/30")
print("Samples requiring review:", 30 - passed)
print("Issue counts:", dict(issue_counts))

print("\nCELL 49 HOÀN THÀNH!")

===== CELL 49 — SCHEMA INTEGRITY AUDIT =====
train_00204 | PASS | issues: 0
train_00217 | PASS | issues: 0
train_00244 | PASS | issues: 0
train_00260 | PASS | issues: 0
train_00712 | PASS | issues: 0
train_00767 | PASS | issues: 0
train_00839 | PASS | issues: 0
train_00912 | REVIEW | issues: 1
    {'table': 'Customers_Policies', 'type': 'PRIMARY_KEY_MISMATCH', 'original': ['Customer_ID', 'Policy_ID'], 'pruned': ['Customer_ID']}
train_01143 | PASS | issues: 0
train_01628 | PASS | issues: 0
train_01791 | PASS | issues: 0
train_01828 | PASS | issues: 0
train_01905 | REVIEW | issues: 1
    {'table': 'budget', 'type': 'PRIMARY_KEY_MISMATCH', 'original': ['School_id', 'Year'], 'pruned': ['School_id']}
train_02006 | REVIEW | issues: 1
    {'table': 'station_company', 'type': 'PRIMARY_KEY_MISMATCH', 'original': ['Station_ID', 'Company_ID'], 'pruned': ['Station_ID']}
train_02253 | REVIEW | issues: 1
    {'table': 'repair_assignment', 'type': 'PRIMARY_KEY_MISMATCH', 'original': ['technician_id',

In [70]:

# CELL 50 — C2 SCHEMA PRUNING INTEGRITY FIX
# MAC-SQL Lite v3.1 | Stage 4 | C2
# Train only | Read-only source DB | No inference

import sqlite3
import statistics
from pathlib import Path
from collections import Counter

print("===== CELL 50 — SCHEMA INTEGRITY FIX =====")

assert "c2_train30_pruning" in globals()
assert "schema_by_id" in globals()
assert "tokenizer" in globals()
assert "combined_fk_graph" in globals()

DB_ROOT = Path(
    "/content/spider_data/spider_data/database"
)

def qname(name):
    return '"' + name.replace('"', '""') + '"'


def get_original_table_metadata(conn, table):
    info = conn.execute(
        f"PRAGMA table_info({qname(table)})"
    ).fetchall()

    fks = conn.execute(
        f"PRAGMA foreign_key_list({qname(table)})"
    ).fetchall()

    return info, fks


def build_c2_verified_ddl(db_id, selected_tables):
    db_path = DB_ROOT / db_id / f"{db_id}.sqlite"
    assert db_path.is_file()

    selected = set(selected_tables)

    conn = sqlite3.connect(
        f"file:{db_path}?mode=ro",
        uri=True
    )

    try:
        conn.execute("PRAGMA query_only = ON")

        statements = []

        for table in schema_by_id[db_id][
            "table_names_original"
        ]:
            if table not in selected:
                continue

            columns, sqlite_fks = (
                get_original_table_metadata(conn, table)
            )

            definitions = []

            for col in columns:
                name = col[1]
                dtype = col[2] or "TEXT"

                definitions.append(
                    f"    {qname(name)} {dtype}"
                )

            pk_columns = [
                col[1]
                for col in sorted(
                    (c for c in columns if c[5] > 0),
                    key=lambda c: c[5]
                )
            ]

            if pk_columns:
                definitions.append(
                    "    PRIMARY KEY ("
                    + ", ".join(qname(c) for c in pk_columns)
                    + ")"
                )

            # SQLite-declared FKs, only when both tables survive.
            seen = set()

            for fk in sqlite_fks:
                target_table = fk[2]

                if target_table.casefold() not in {
                    t.casefold() for t in selected
                }:
                    continue

                # This implementation handles single-column FKs.
                # Composite FKs require grouping by FK constraint ID.
                assert fk[1] == 0, (
                    f"Composite FK detected in {db_id}.{table}; "
                    "requires explicit handling"
                )

                key = (
                    fk[3].casefold(),
                    target_table.casefold(),
                    (fk[4] or "").casefold()
                )

                if key in seen:
                    continue

                seen.add(key)

                definitions.append(
                    f"    FOREIGN KEY ({qname(fk[3])}) "
                    f"REFERENCES {qname(target_table)}"
                    f"({qname(fk[4])})"
                )

            # Add Spider-declared FKs missing from SQLite.
            _, all_relationships = combined_fk_graph(db_id)

            for fk in all_relationships:
                if fk["source"] != "spider_metadata":
                    continue

                if fk["source_table"] != table:
                    continue

                if fk["target_table"] not in selected:
                    continue

                key = (
                    fk["source_column"].casefold(),
                    fk["target_table"].casefold(),
                    fk["target_column"].casefold()
                )

                if key in seen:
                    continue

                seen.add(key)

                definitions.append(
                    f'    FOREIGN KEY '
                    f'({qname(fk["source_column"])}) '
                    f'REFERENCES '
                    f'{qname(fk["target_table"])}'
                    f'({qname(fk["target_column"])})'
                )

            statements.append(
                f"CREATE TABLE {qname(table)} (\n"
                + ",\n".join(definitions)
                + "\n);"
            )

        return "\n\n".join(statements)

    finally:
        conn.close()


def verify_c2_schema(db_id, ddl, selected_tables):
    db_path = DB_ROOT / db_id / f"{db_id}.sqlite"

    original = sqlite3.connect(
        f"file:{db_path}?mode=ro",
        uri=True
    )
    reduced = sqlite3.connect(":memory:")

    try:
        original.execute("PRAGMA query_only = ON")
        reduced.executescript(ddl)

        for table in selected_tables:
            source_info, source_fk = (
                get_original_table_metadata(
                    original, table
                )
            )

            reduced_info, reduced_fk = (
                get_original_table_metadata(
                    reduced, table
                )
            )

            source_columns = [
                c[1].casefold()
                for c in source_info
            ]
            reduced_columns = [
                c[1].casefold()
                for c in reduced_info
            ]

            assert source_columns == reduced_columns, (
                f"Column mismatch: {db_id}.{table}"
            )

            def pk_order(info):
                return [
                    c[1].casefold()
                    for c in sorted(
                        (x for x in info if x[5] > 0),
                        key=lambda x: x[5]
                    )
                ]

            assert pk_order(source_info) == pk_order(
                reduced_info
            ), f"PK mismatch: {db_id}.{table}"

            selected_lower = {
                t.casefold() for t in selected_tables
            }

            def fk_set(rows):
                return {
                    (
                        r[3].casefold(),
                        r[2].casefold(),
                        (r[4] or "").casefold()
                    )
                    for r in rows
                    if r[2].casefold() in selected_lower
                }

            assert fk_set(source_fk).issubset(
                fk_set(reduced_fk)
            ), f"FK mismatch: {db_id}.{table}"

        return True

    finally:
        original.close()
        reduced.close()


c2_train30_verified = []

for record in c2_train30_pruning:
    db_id = record["db_id"]
    selected = record["expanded_tables"]

    ddl = build_c2_verified_ddl(
        db_id, selected
    )

    assert verify_c2_schema(
        db_id, ddl, selected
    )

    full_tokens = len(
        tokenizer(
            spider_schema_to_ddl(db_id),
            add_special_tokens=False
        )["input_ids"]
    )

    pruned_tokens = len(
        tokenizer(
            ddl,
            add_special_tokens=False
        )["input_ids"]
    )

    reduction = (
        100 * (full_tokens - pruned_tokens)
        / full_tokens
        if full_tokens else 0
    )

    c2_train30_verified.append({
        **record,
        "pruned_ddl": ddl,
        "pruned_schema_tokens": pruned_tokens,
        "reduction_percent": round(reduction, 2),
        "schema_integrity": "PASS"
    })

    print(
        record["sample_id"],
        "| VERIFIED",
        "| reduction:", f"{reduction:.2f}%"
    )


assert len(c2_train30_verified) == 30

print("\n===== SUMMARY =====")
print("Verified schemas:", len(c2_train30_verified), "/30")
print(
    "Mean token reduction:",
    round(
        statistics.mean(
            r["reduction_percent"]
            for r in c2_train30_verified
        ),
        2
    ),
    "%"
)

print("\n===== COMPOSITE PK CHECK =====")

for sample_id in [
    "train_00912",
    "train_01905",
    "train_02006",
    "train_02253",
    "train_04931",
    "train_05238"
]:
    r = next(
        x for x in c2_train30_verified
        if x["sample_id"] == sample_id
    )

    print(
        sample_id,
        "|", r["db_id"],
        "| PASS"
    )

print("\nCELL 50 HOÀN THÀNH!")

===== CELL 50 — SCHEMA INTEGRITY FIX =====
train_00204 | VERIFIED | reduction: 73.99%
train_00217 | VERIFIED | reduction: 61.39%
train_00244 | VERIFIED | reduction: 46.92%
train_00260 | VERIFIED | reduction: 53.08%
train_00712 | VERIFIED | reduction: 40.49%
train_00767 | VERIFIED | reduction: 57.80%
train_00839 | VERIFIED | reduction: 81.19%
train_00912 | VERIFIED | reduction: 59.44%
train_01143 | VERIFIED | reduction: 0.00%
train_01628 | VERIFIED | reduction: 77.61%
train_01791 | VERIFIED | reduction: -5.13%
train_01828 | VERIFIED | reduction: 79.08%
train_01905 | VERIFIED | reduction: 25.56%
train_02006 | VERIFIED | reduction: -1.90%
train_02253 | VERIFIED | reduction: 19.75%
train_03456 | VERIFIED | reduction: 70.19%
train_04139 | VERIFIED | reduction: 34.78%
train_04464 | VERIFIED | reduction: -9.47%
train_04467 | VERIFIED | reduction: -9.47%
train_04597 | VERIFIED | reduction: 79.93%
train_04837 | VERIFIED | reduction: 65.51%
train_04931 | VERIFIED | reduction: 60.14%
train_05238 

In [71]:

# CELL 51 — C2 FULL-SCHEMA FALLBACK AUDIT
# MAC-SQL Lite v3.1 | Stage 4 | C2
# Train only | No inference | No dev access

import statistics
from collections import Counter

print("===== CELL 51 — FULL-SCHEMA FALLBACK AUDIT =====")

assert "c2_train30_verified" in globals()
assert "schema_by_id" in globals()
assert "spider_schema_to_ddl" in globals()
assert "tokenizer" in globals()

def c2_final_schema_ddl(db_id, selected_tables):
    all_tables = schema_by_id[db_id][
        "table_names_original"
    ]

    if set(selected_tables) == set(all_tables):
        return (
            spider_schema_to_ddl(db_id),
            "FULL_SCHEMA_FALLBACK"
        )

    return (
        build_c2_verified_ddl(
            db_id, selected_tables
        ),
        "PRUNED_SCHEMA"
    )


c2_train30_final_audit = []

for record in c2_train30_verified:
    db_id = record["db_id"]
    selected = record["expanded_tables"]

    final_ddl, mode = c2_final_schema_ddl(
        db_id, selected
    )

    assert set(selected).issubset(
        set(
            schema_by_id[db_id][
                "table_names_original"
            ]
        )
    )

    full_ddl = spider_schema_to_ddl(db_id)

    full_tokens = len(
        tokenizer(
            full_ddl,
            add_special_tokens=False
        )["input_ids"]
    )

    final_tokens = len(
        tokenizer(
            final_ddl,
            add_special_tokens=False
        )["input_ids"]
    )

    reduction = (
        100 * (full_tokens - final_tokens)
        / full_tokens
        if full_tokens else 0
    )

    if mode == "FULL_SCHEMA_FALLBACK":
        assert final_ddl == full_ddl
        assert reduction == 0

    c2_train30_final_audit.append({
        "sample_id": record["sample_id"],
        "db_id": db_id,
        "selected_tables": record["selected_tables"],
        "expanded_tables": selected,
        "join_status": record["join_status"],
        "schema_mode": mode,
        "full_schema_tokens": full_tokens,
        "final_schema_tokens": final_tokens,
        "reduction_percent": round(reduction, 2),
        "final_ddl": final_ddl
    })

    print(
        record["sample_id"],
        "|", mode,
        "| reduction:", f"{reduction:.2f}%"
    )


assert len(c2_train30_final_audit) == 30

modes = Counter(
    r["schema_mode"]
    for r in c2_train30_final_audit
)

print("\n===== SUMMARY =====")
print("Samples:", len(c2_train30_final_audit))
print("Schema modes:", dict(modes))

print(
    "Mean token reduction:",
    round(
        statistics.mean(
            r["reduction_percent"]
            for r in c2_train30_final_audit
        ),
        2
    ),
    "%"
)

print(
    "Negative reductions:",
    sum(
        r["reduction_percent"] < 0
        for r in c2_train30_final_audit
    )
)

print(
    "Unresolved FK cases:",
    sum(
        r["join_status"] == "UNRESOLVED"
        for r in c2_train30_final_audit
    )
)

print("\nCELL 51 HOÀN THÀNH!")

===== CELL 51 — FULL-SCHEMA FALLBACK AUDIT =====
train_00204 | PRUNED_SCHEMA | reduction: 73.99%
train_00217 | PRUNED_SCHEMA | reduction: 61.39%
train_00244 | PRUNED_SCHEMA | reduction: 46.92%
train_00260 | PRUNED_SCHEMA | reduction: 53.08%
train_00712 | PRUNED_SCHEMA | reduction: 40.49%
train_00767 | PRUNED_SCHEMA | reduction: 57.80%
train_00839 | PRUNED_SCHEMA | reduction: 81.19%
train_00912 | PRUNED_SCHEMA | reduction: 59.44%
train_01143 | FULL_SCHEMA_FALLBACK | reduction: 0.00%
train_01628 | PRUNED_SCHEMA | reduction: 77.61%
train_01791 | FULL_SCHEMA_FALLBACK | reduction: 0.00%
train_01828 | PRUNED_SCHEMA | reduction: 79.08%
train_01905 | PRUNED_SCHEMA | reduction: 25.56%
train_02006 | FULL_SCHEMA_FALLBACK | reduction: 0.00%
train_02253 | PRUNED_SCHEMA | reduction: 19.75%
train_03456 | PRUNED_SCHEMA | reduction: 70.19%
train_04139 | PRUNED_SCHEMA | reduction: 34.78%
train_04464 | FULL_SCHEMA_FALLBACK | reduction: 0.00%
train_04467 | FULL_SCHEMA_FALLBACK | reduction: 0.00%
train_045

In [72]:

# CELL 52 — C2 PRE-FREEZE CONSISTENCY AUDIT
# MAC-SQL Lite v3.1 | Stage 4 | C2
# Train only | No model inference | No dev gold

import json
import hashlib
from collections import Counter

print("===== CELL 52 — C2 PRE-FREEZE AUDIT =====")

required = [
    "samples",
    "selector_train30_results",
    "c2_train30_final_audit",
    "c2_final_schema_ddl",
    "build_selector_messages",
    "build_c1_messages",
    "build_c2_sql_messages",
    "C1_SYSTEM_PROMPT",
    "SELECTOR_SYSTEM_PROMPT",
    "schema_by_id",
    "tokenizer"
]

for name in required:
    assert name in globals(), f"Missing: {name}"

assert len(samples) == 30
assert len(selector_train30_results) == 30
assert len(c2_train30_final_audit) == 30

selector_map = {
    r["sample_id"]: r
    for r in selector_train30_results
}

audit_map = {
    r["sample_id"]: r
    for r in c2_train30_final_audit
}

assert len(selector_map) == 30
assert len(audit_map) == 30


def canonical_hash(value):
    payload = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":")
    )
    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


issues = []
verified = []

for sample in samples:
    sid = sample["sample_id"]
    db_id = sample["db_id"]
    question = sample["question"]

    selector = selector_map[sid]
    audit = audit_map[sid]

    # Selector must receive question + schema,
    # not gold SQL.
    selector_messages = build_selector_messages(
        question, db_id
    )

    assert len(selector_messages) == 2
    assert selector_messages[0]["role"] == "system"
    assert selector_messages[1]["role"] == "user"

    # Check selected tables survive FK expansion.
    assert set(selector["selected_tables"]).issubset(
        set(audit["expanded_tables"])
    )

    # Rebuild final DDL independently.
    regenerated_ddl, regenerated_mode = (
        c2_final_schema_ddl(
            db_id,
            audit["expanded_tables"]
        )
    )

    assert regenerated_mode == audit["schema_mode"]
    assert regenerated_ddl == audit["final_ddl"]

    # C1 and C2 SQL-generation prompts
    # must differ only in schema text.
    c1_messages = build_c1_messages(
        question, db_id
    )

    c2_messages = build_c2_sql_messages(
        question, audit["final_ddl"]
    )

    assert c1_messages[0] == c2_messages[0]
    assert c1_messages[0]["content"] == C1_SYSTEM_PROMPT

    assert c1_messages[1]["role"] == "user"
    assert c2_messages[1]["role"] == "user"

    full_schema = spider_schema_to_ddl(db_id)

    expected_c1_user = (
        f"Database schema:\n{full_schema}\n\n"
        f"Question:\n{question}\n\n"
        "Write the SQLite query:"
    )

    expected_c2_user = (
        f"Database schema:\n{audit['final_ddl']}\n\n"
        f"Question:\n{question}\n\n"
        "Write the SQLite query:"
    )

    assert c1_messages[1]["content"] == expected_c1_user
    assert c2_messages[1]["content"] == expected_c2_user

    # Verify no gold SQL is inserted as a prompt field.
    assert set(selector_messages[0]) == {
        "role", "content"
    }
    assert set(selector_messages[1]) == {
        "role", "content"
    }

    verified.append({
        "sample_id": sid,
        "db_id": db_id,
        "schema_mode": audit["schema_mode"],
        "join_status": audit["join_status"],
        "selected_tables": audit["expanded_tables"],
        "ddl_sha256": hashlib.sha256(
            audit["final_ddl"].encode("utf-8")
        ).hexdigest(),
        "selector_prompt_sha256": canonical_hash(
            selector_messages
        ),
        "sql_prompt_sha256": canonical_hash(
            c2_messages
        )
    })


assert len(verified) == 30

modes = Counter(
    r["schema_mode"]
    for r in verified
)

statuses = Counter(
    r["join_status"]
    for r in verified
)

print("Train samples verified:", len(verified))
print("Selector prompt structure: PASS")
print("Selected-table preservation: PASS")
print("DDL regeneration: PASS")
print("C1/C2 SQL prompt parity: PASS")
print("Schema modes:", dict(modes))
print("FK statuses:", dict(statuses))

print("\n===== REPRODUCIBILITY =====")
print(
    "Train-30 audit SHA256:",
    canonical_hash(verified)
)

print("\nCELL 52 HOÀN THÀNH!")

===== CELL 52 — C2 PRE-FREEZE AUDIT =====
Train samples verified: 30
Selector prompt structure: PASS
Selected-table preservation: PASS
DDL regeneration: PASS
C1/C2 SQL prompt parity: PASS
Schema modes: {'PRUNED_SCHEMA': 25, 'FULL_SCHEMA_FALLBACK': 5}
FK statuses: {'PASS': 29, 'UNRESOLVED': 1}

===== REPRODUCIBILITY =====
Train-30 audit SHA256: 67b8eb131e01cb5041e99590891d2d564087732ff8a154cba6fe4a94fe76fcbc

CELL 52 HOÀN THÀNH!


In [73]:

# CELL 53 — FREEZE C2 SELECTOR CONFIGURATION
# MAC-SQL Lite v3.1 | Stage 4 | C2
# Train-only development | No model inference
# No dev evaluation | Immutable artifacts

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

print("===== CELL 53 — FREEZE C2 CONFIG =====")

required = [
    "BASE",
    "samples",
    "selector_train30_results",
    "c2_train30_final_audit",
    "SELECTOR_SYSTEM_PROMPT",
    "C1_SYSTEM_PROMPT",
    "verified",
    "canonical_hash",
    "tokenizer"
]

for name in required:
    assert name in globals(), f"Missing: {name}"

assert len(samples) == 30
assert len(selector_train30_results) == 30
assert len(c2_train30_final_audit) == 30
assert len(verified) == 30

BASE = Path(BASE)
config_dir = BASE / "configs"
result_dir = BASE / "results" / "c2"

config_dir.mkdir(parents=True, exist_ok=True)
result_dir.mkdir(parents=True, exist_ok=True)


def canonical_bytes(obj):
    return json.dumps(
        obj,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":")
    ).encode("utf-8")


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def save_immutable_json(path, obj):
    data = json.dumps(
        obj,
        ensure_ascii=False,
        sort_keys=True,
        indent=2
    ).encode("utf-8")

    if path.exists():
        existing = path.read_bytes()

        assert existing == data, (
            f"Immutable artifact conflict: {path}"
        )

        print("EXISTS — identical:", path.name)
        return

    path.write_bytes(data)
    print("SAVED:", path.name)


c1_manifest_path = (
    BASE / "results" / "c1"
    / "c1_final_manifest_v1.json"
)

assert c1_manifest_path.is_file(), (
    "Frozen C1 manifest not found"
)

c1_manifest_sha256 = sha256_file(
    c1_manifest_path
)

train_ids = [
    sample["sample_id"]
    for sample in samples
]

selector_audit = []

selector_by_id = {
    r["sample_id"]: r
    for r in selector_train30_results
}

final_by_id = {
    r["sample_id"]: r
    for r in c2_train30_final_audit
}

for sid in train_ids:
    selector = selector_by_id[sid]
    final = final_by_id[sid]

    selector_audit.append({
        "sample_id": sid,
        "db_id": selector["db_id"],
        "parse_status": selector["parse_status"],
        "selected_tables": selector["selected_tables"],
        "expanded_tables": final["expanded_tables"],
        "join_status": final["join_status"],
        "schema_mode": final["schema_mode"],
        "full_schema_tokens": final[
            "full_schema_tokens"
        ],
        "final_schema_tokens": final[
            "final_schema_tokens"
        ],
        "reduction_percent": final[
            "reduction_percent"
        ],
        "ddl_sha256": hashlib.sha256(
            final["final_ddl"].encode("utf-8")
        ).hexdigest()
    })


c2_config = {
    "experiment": "MAC-SQL Lite v3.1",
    "configuration": "C2_SELECTOR",
    "version": 1,
    "dataset": "Spider 1.0",
    "development_split": "train",
    "development_samples": 30,
    "evaluation_split": "dev",
    "evaluation_samples": 300,
    "model": "Qwen/Qwen2.5-Coder-3B-Instruct",
    "quantization": "none",
    "decoding": {
        "do_sample": False,
        "max_new_tokens": 256
    },
    "selector": {
        "system_prompt": SELECTOR_SYSTEM_PROMPT,
        "output_format": {
            "tables": ["table_name"]
        },
        "parsing": "JSON extraction and exact schema table canonicalization",
        "selection_granularity": "table_level",
        "column_pruning": False
    },
    "schema_processing": {
        "foreign_key_sources": [
            "spider_metadata",
            "sqlite_declared"
        ],
        "bridge_tables": "shortest declared FK paths",
        "unresolved_fk": (
            "retain selected tables; "
            "do not infer undeclared FK"
        ),
        "full_schema_fallback": (
            "use original C1 DDL when "
            "all database tables are retained"
        ),
        "primary_keys": "original SQLite metadata",
        "preserve_all_selected_columns": True
    },
    "sql_generator": {
        "system_prompt": C1_SYSTEM_PROMPT,
        "same_prompt_template_as_c1": True
    },
    "evaluation": {
        "metric": "Test Suite Execution Accuracy",
        "gold_sql_used_in_prompt": False,
        "development_gold_access": False
    },
    "baseline": {
        "configuration": "C1",
        "manifest_sha256": c1_manifest_sha256
    },
    "train30_audit_sha256": canonical_hash(
        verified
    )
}

config_path = (
    config_dir / "c2_selector_v1.json"
)

audit_path = (
    result_dir / "c2_train30_audit_v1.json"
)

save_immutable_json(
    config_path,
    c2_config
)

save_immutable_json(
    audit_path,
    selector_audit
)

c2_manifest = {
    "experiment": "MAC-SQL Lite v3.1",
    "configuration": "C2_SELECTOR",
    "version": 1,
    "artifacts": {
        "c2_selector_v1.json": sha256_file(
            config_path
        ),
        "c2_train30_audit_v1.json": sha256_file(
            audit_path
        ),
        "c1_final_manifest_v1.json": (
            c1_manifest_sha256
        )
    },
    "train30_audit_sha256": canonical_hash(
        verified
    )
}

manifest_path = (
    result_dir / "c2_config_manifest_v1.json"
)

save_immutable_json(
    manifest_path,
    c2_manifest
)

print("\n===== FREEZE SUMMARY =====")
print("C2 config:", config_path)
print("Train audit:", audit_path)
print("Manifest:", manifest_path)
print("C1 manifest SHA256:", c1_manifest_sha256)
print("Train audit SHA256:", canonical_hash(verified))
print("Status: CONFIGURATION FROZEN")

print("\nCELL 53 HOÀN THÀNH!")


===== CELL 53 — FREEZE C2 CONFIG =====
SAVED: c2_selector_v1.json
SAVED: c2_train30_audit_v1.json
SAVED: c2_config_manifest_v1.json

===== FREEZE SUMMARY =====
C2 config: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/configs/c2_selector_v1.json
Train audit: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/results/c2/c2_train30_audit_v1.json
Manifest: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/results/c2/c2_config_manifest_v1.json
C1 manifest SHA256: 3e01e4035bcdfbe7fbccd8b8b952ceff1c7f3906c258cbb30f2608323e61cac6
Train audit SHA256: 67b8eb131e01cb5041e99590891d2d564087732ff8a154cba6fe4a94fe76fcbc
Status: CONFIGURATION FROZEN

CELL 53 HOÀN THÀNH!


In [74]:

# CELL 54 — C2 DEV-300 PREFLIGHT & CHECKPOINT AUDIT
# MAC-SQL Lite v3.1 | Stage 4 | C2
# Read-only | No inference | No dev gold access

import json
import hashlib
from pathlib import Path
from collections import Counter

print("===== CELL 54 — C2 DEV-300 PREFLIGHT =====")

assert "BASE" in globals()
assert "model" in globals()
assert "tokenizer" in globals()

BASE = Path(BASE)

config_path = BASE / "configs/c2_selector_v1.json"
audit_path = BASE / "results/c2/c2_train30_audit_v1.json"
manifest_path = BASE / "results/c2/c2_config_manifest_v1.json"
c1_manifest_path = BASE / "results/c1/c1_final_manifest_v1.json"
dev_path = BASE / "data/dev_eval_300_seed42.json"

required_files = [
    config_path,
    audit_path,
    manifest_path,
    c1_manifest_path,
    dev_path
]

for path in required_files:
    assert path.is_file(), f"Missing file: {path}"
    print("FOUND:", path.relative_to(BASE))


def file_sha256(path):
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


with config_path.open("r", encoding="utf-8") as f:
    config = json.load(f)

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

with dev_path.open("r", encoding="utf-8") as f:
    dev_data = json.load(f)

expected = manifest["artifacts"]

assert file_sha256(config_path) == expected[
    "c2_selector_v1.json"
]

assert file_sha256(audit_path) == expected[
    "c2_train30_audit_v1.json"
]

assert file_sha256(c1_manifest_path) == expected[
    "c1_final_manifest_v1.json"
]

assert config["configuration"] == "C2_SELECTOR"
assert config["evaluation_samples"] == 300
assert config["decoding"]["do_sample"] is False
assert config["decoding"]["max_new_tokens"] == 256

print("\n===== CONFIG INTEGRITY =====")
print("C2 configuration SHA256: PASS")
print("C2 train audit SHA256: PASS")
print("Frozen C1 manifest SHA256: PASS")
print("Decoding configuration: PASS")

# Inspect the actual dev file structure without
# accessing the separate dev gold file.
if isinstance(dev_data, list):
    dev_samples = dev_data
    dev_container_type = "list"

elif isinstance(dev_data, dict):
    possible_keys = [
        "samples", "data", "examples", "dev_samples"
    ]

    matches = [
        k for k in possible_keys
        if isinstance(dev_data.get(k), list)
    ]

    assert len(matches) == 1, (
        "Unexpected dev JSON structure. "
        f"Top-level keys: {list(dev_data.keys())}"
    )

    dev_samples = dev_data[matches[0]]
    dev_container_type = f"dict[{matches[0]}]"

else:
    raise TypeError("Unsupported dev JSON structure")

assert len(dev_samples) == 300, (
    f"Expected 300 dev samples, got {len(dev_samples)}"
)

assert all(isinstance(s, dict) for s in dev_samples)

print("\n===== DEV INPUT INSPECTION =====")
print("Container:", dev_container_type)
print("Number of samples:", len(dev_samples))
print("First sample keys:", sorted(dev_samples[0].keys()))

# Never print questions or SQL here.
ids = [
    s.get("sample_id")
    for s in dev_samples
]

db_ids = [
    s.get("db_id")
    for s in dev_samples
]

assert all(isinstance(x, str) and x for x in ids)
assert len(set(ids)) == 300

assert all(
    isinstance(s.get("question"), str)
    and s["question"].strip()
    for s in dev_samples
)

assert all(
    isinstance(x, str) and x
    for x in db_ids
)

assert "schema_by_id" in globals()
assert all(db_id in schema_by_id for db_id in db_ids)

# Ensure no gold SQL is accidentally supplied
# in the inference records.
gold_like_keys = {
    "query", "gold_sql", "gold_query",
    "sql_gold", "reference_sql", "gold"
}

present_gold_keys = sorted({
    k
    for sample in dev_samples
    for k in sample
    if k.casefold() in gold_like_keys
})

print("Unique sample IDs: PASS")
print("Questions and DB IDs: PASS")
print("Known database schemas: PASS")
print("Gold-like fields:", present_gold_keys)

assert not present_gold_keys, (
    "Inference input contains possible gold SQL fields. "
    "Do not proceed until inspected."
)

print("Dev inference input isolation: PASS")

# Check existing C2 artifacts without modifying them.
c2_dir = BASE / "results/c2"

existing = sorted(
    str(p.relative_to(c2_dir))
    for p in c2_dir.rglob("*")
    if p.is_file()
)

print("\n===== EXISTING C2 ARTIFACTS =====")
for name in existing:
    print(" -", name)

print("\n===== PREFLIGHT SUMMARY =====")
print("Frozen configuration: VERIFIED")
print("Dev sample count: 300")
print("Dev gold SQL: NOT OPENED")
print("Model inference: NOT RUN")
print("Files modified: NONE")

print("\nCELL 54 HOÀN THÀNH!")

===== CELL 54 — C2 DEV-300 PREFLIGHT =====
FOUND: configs/c2_selector_v1.json
FOUND: results/c2/c2_train30_audit_v1.json
FOUND: results/c2/c2_config_manifest_v1.json
FOUND: results/c1/c1_final_manifest_v1.json
FOUND: data/dev_eval_300_seed42.json

===== CONFIG INTEGRITY =====
C2 configuration SHA256: PASS
C2 train audit SHA256: PASS
Frozen C1 manifest SHA256: PASS
Decoding configuration: PASS

===== DEV INPUT INSPECTION =====
Container: dict[samples]
Number of samples: 300
First sample keys: ['db_id', 'question', 'sample_id', 'source_index']
Unique sample IDs: PASS
Questions and DB IDs: PASS
Known database schemas: PASS
Gold-like fields: []
Dev inference input isolation: PASS

===== EXISTING C2 ARTIFACTS =====
 - c2_config_manifest_v1.json
 - c2_train30_audit_v1.json

===== PREFLIGHT SUMMARY =====
Frozen configuration: VERIFIED
Dev sample count: 300
Dev gold SQL: NOT OPENED
Model inference: NOT RUN
Files modified: NONE

CELL 54 HOÀN THÀNH!


In [75]:

# CELL 55 — C2 CHECKPOINT & RESUME INFRASTRUCTURE
# MAC-SQL Lite v3.1 | Stage 4 | C2
# No inference | No dev gold access

import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

print("===== CELL 55 — C2 CHECKPOINT SETUP =====")

assert "BASE" in globals()
assert "dev_samples" in globals()
assert len(dev_samples) == 300

BASE = Path(BASE)
C2_DIR = BASE / "results" / "c2"
C2_DIR.mkdir(parents=True, exist_ok=True)

C2_CHECKPOINT = C2_DIR / "c2_dev_checkpoint_v1.jsonl"
C2_RUN_META = C2_DIR / "c2_run_metadata_v1.json"

C2_CONFIG = BASE / "configs/c2_selector_v1.json"
C2_DEV_INPUT = BASE / "data/dev_eval_300_seed42.json"
C2_MANIFEST = C2_DIR / "c2_config_manifest_v1.json"


def c2_sha256(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def c2_atomic_json_write(path, obj):
    path = Path(path)
    temp_path = path.with_name(
        path.name + ".tmp"
    )

    payload = json.dumps(
        obj,
        ensure_ascii=False,
        indent=2,
        sort_keys=True
    )

    with temp_path.open("w", encoding="utf-8") as f:
        f.write(payload)
        f.write("\n")
        f.flush()
        os.fsync(f.fileno())

    os.replace(temp_path, path)


def c2_atomic_jsonl_append(path, record):
    """
    Atomic replacement of the complete checkpoint.

    This is intentionally conservative for a
    300-sample experiment.
    """
    path = Path(path)

    existing = b""

    if path.exists():
        existing = path.read_bytes()

    new_line = (
        json.dumps(
            record,
            ensure_ascii=False,
            sort_keys=True
        ) + "\n"
    ).encode("utf-8")

    temp_path = path.with_name(
        path.name + ".tmp"
    )

    with temp_path.open("wb") as f:
        f.write(existing)
        f.write(new_line)
        f.flush()
        os.fsync(f.fileno())

    os.replace(temp_path, path)


def c2_load_checkpoint(path):
    path = Path(path)

    if not path.exists():
        return {}

    records = {}

    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue

            record = json.loads(line)
            sid = record["sample_id"]

            if sid in records:
                raise ValueError(
                    f"Duplicate sample_id {sid} "
                    f"at line {line_number}"
                )

            records[sid] = record

    return records


# Freeze input and configuration identities.
expected_run_metadata = {
    "experiment": "MAC-SQL Lite v3.1",
    "configuration": "C2_SELECTOR",
    "version": 1,
    "sample_count": 300,
    "config_sha256": c2_sha256(C2_CONFIG),
    "dev_input_sha256": c2_sha256(C2_DEV_INPUT),
    "config_manifest_sha256": c2_sha256(
        C2_MANIFEST
    ),
    "checkpoint_format": "JSONL",
    "checkpoint_version": 1,
    "model": "Qwen/Qwen2.5-Coder-3B-Instruct",
    "decoding": {
        "do_sample": False,
        "max_new_tokens": 256
    }
}


if C2_RUN_META.exists():
    with C2_RUN_META.open("r", encoding="utf-8") as f:
        stored_metadata = json.load(f)

    assert stored_metadata == expected_run_metadata, (
        "Run metadata mismatch. "
        "Do not resume with changed inputs/config."
    )

    print("Existing run metadata: VERIFIED")

else:
    c2_atomic_json_write(
        C2_RUN_META,
        expected_run_metadata
    )
    print("Run metadata: CREATED")


# Check checkpoint integrity.
completed_records = c2_load_checkpoint(
    C2_CHECKPOINT
)

expected_ids = {
    s["sample_id"]
    for s in dev_samples
}

unknown_ids = set(completed_records) - expected_ids

assert not unknown_ids, (
    f"Checkpoint contains unknown IDs: {unknown_ids}"
)

remaining_samples = [
    s for s in dev_samples
    if s["sample_id"] not in completed_records
]

print("\n===== CHECKPOINT STATUS =====")
print("Checkpoint path:", C2_CHECKPOINT)
print("Completed:", len(completed_records))
print("Remaining:", len(remaining_samples))
print("Total:", len(dev_samples))

assert (
    len(completed_records)
    + len(remaining_samples)
    == 300
)

print("\n===== INTEGRITY =====")
print("Config SHA256:", expected_run_metadata["config_sha256"])
print("Dev SHA256:", expected_run_metadata["dev_input_sha256"])
print("Checkpoint duplicate IDs: NONE")
print("Input/config identities: VERIFIED")

print("\n===== SAFETY =====")
print("C1 artifacts: UNTOUCHED")
print("Dev gold SQL: NOT OPENED")
print("Model inference: NOT RUN")

print("\nCELL 55 HOÀN THÀNH!")

===== CELL 55 — C2 CHECKPOINT SETUP =====
Run metadata: CREATED

===== CHECKPOINT STATUS =====
Checkpoint path: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/results/c2/c2_dev_checkpoint_v1.jsonl
Completed: 0
Remaining: 300
Total: 300

===== INTEGRITY =====
Config SHA256: 5b7ec0ad716a975894ab0ea805bb9764971e7581f2494dd3c7b298a7659b09ea
Dev SHA256: 201438c5f0e2949db3af1baf0f28ea06db8fa845b9e46ed2c02ac8467b3b470b
Checkpoint duplicate IDs: NONE
Input/config identities: VERIFIED

===== SAFETY =====
C1 artifacts: UNTOUCHED
Dev gold SQL: NOT OPENED
Model inference: NOT RUN

CELL 55 HOÀN THÀNH!


In [76]:

# CELL 56 — C2 SELECTOR DEV INFERENCE (FIRST 5)
# MAC-SQL Lite v3.1 | Stage 4 | C2
# Inference only | No dev gold | Resume-safe

import json
import time
from pathlib import Path

print("===== CELL 56 — C2 FIRST 5 DEV SAMPLES =====")

required = [
    "dev_samples",
    "build_selector_messages",
    "parse_selector_tables",
    "audit_combined_fk",
    "c2_final_schema_ddl",
    "build_c2_sql_messages",
    "generate_c2_sql",
    "model",
    "tokenizer",
    "c2_load_checkpoint",
    "c2_atomic_jsonl_append",
    "C2_CHECKPOINT",
    "C2_RUN_META",
    "expected_run_metadata",
    "c2_sha256",
    "C2_CONFIG",
    "C2_DEV_INPUT",
    "C2_MANIFEST"
]

for name in required:
    assert name in globals(), f"Missing: {name}"

# Verify frozen inputs before inference.
assert c2_sha256(C2_CONFIG) == (
    expected_run_metadata["config_sha256"]
)

assert c2_sha256(C2_DEV_INPUT) == (
    expected_run_metadata["dev_input_sha256"]
)

assert c2_sha256(C2_MANIFEST) == (
    expected_run_metadata["config_manifest_sha256"]
)

with Path(C2_RUN_META).open(
    "r", encoding="utf-8"
) as f:
    assert json.load(f) == expected_run_metadata


def c2_generate_selector(messages):
    """
    Selector inference with the same model,
    greedy decoding, and 256-token limit.
    """
    import torch

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    input_tokens = inputs["input_ids"].shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=256,
            pad_token_id=tokenizer.eos_token_id
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    generated = output[0][input_tokens:]

    text = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    return {
        "raw_output": text,
        "input_tokens": int(input_tokens),
        "output_tokens": int(len(generated)),
        "inference_seconds": float(elapsed)
    }


def c2_run_one(sample):
    sid = sample["sample_id"]
    db_id = sample["db_id"]
    question = sample["question"]

    total_start = time.perf_counter()

    # Step 1: Selector
    selector_messages = build_selector_messages(
        question, db_id
    )

    selector_result = c2_generate_selector(
        selector_messages
    )

    selected_tables, parse_status = (
        parse_selector_tables(
            selector_result["raw_output"],
            db_id
        )
    )

    # Deterministic failure fallback:
    # retain the full schema rather than
    # guessing table names.
    if parse_status != "PASS" or not selected_tables:
        selected_tables = schema_by_id[db_id][
            "table_names_original"
        ]
        selector_fallback = "PARSE_FAILURE_FULL_SCHEMA"
    else:
        selector_fallback = None

    # Step 2: FK expansion
    audit = audit_combined_fk(
        db_id, selected_tables
    )

    expanded_tables = audit["expanded_tables"]

    # Step 3: Schema pruning
    final_ddl, schema_mode = c2_final_schema_ddl(
        db_id, expanded_tables
    )

    # Step 4: SQL generation
    sql_messages = build_c2_sql_messages(
        question, final_ddl
    )

    sql_result = generate_c2_sql(
        sql_messages
    )

    total_seconds = (
        time.perf_counter() - total_start
    )

    return {
        "sample_id": sid,
        "source_index": sample["source_index"],
        "db_id": db_id,
        "question": question,
        "configuration": "C2_SELECTOR",
        "selector": {
            "raw_output": selector_result["raw_output"],
            "parse_status": parse_status,
            "fallback": selector_fallback,
            "selected_tables": selected_tables,
            "expanded_tables": expanded_tables,
            "join_status": audit["status"],
            "schema_mode": schema_mode,
            "input_tokens": selector_result["input_tokens"],
            "output_tokens": selector_result["output_tokens"],
            "inference_seconds": selector_result[
                "inference_seconds"
            ]
        },
        "generator": {
            "sql": sql_result["sql"],
            "raw_output": sql_result["raw_output"],
            "input_tokens": sql_result["input_tokens"],
            "output_tokens": sql_result["output_tokens"],
            "inference_seconds": sql_result[
                "inference_seconds"
            ]
        },
        "total_pipeline_seconds": float(total_seconds)
    }


# Resume existing checkpoint.
completed = c2_load_checkpoint(C2_CHECKPOINT)

# First five positions in the fixed dev order.
target_samples = dev_samples[:5]

for index, sample in enumerate(target_samples, start=1):
    sid = sample["sample_id"]

    if sid in completed:
        print(
            f"[{index}/5] SKIP {sid} "
            "(already checkpointed)"
        )
        continue

    print(f"\n[{index}/5] RUN {sid}")

    result = c2_run_one(sample)

    # Persist before proceeding to the next sample.
    c2_atomic_jsonl_append(
        C2_CHECKPOINT, result
    )

    completed[sid] = result

    print(
        "Selector:",
        result["selector"]["selected_tables"]
    )

    print(
        "Schema mode:",
        result["selector"]["schema_mode"]
    )

    print(
        "SQL:",
        result["generator"]["sql"]
    )

    print(
        "Latency:",
        round(
            result["total_pipeline_seconds"], 3
        ),
        "seconds"
    )


# Verify saved records by reloading from disk.
saved = c2_load_checkpoint(C2_CHECKPOINT)

assert all(
    sample["sample_id"] in saved
    for sample in target_samples
)

print("\n===== CHECKPOINT SUMMARY =====")
print("Completed:", len(saved), "/300")
print("Remaining:", 300 - len(saved))
print("First 5 dev samples: SAVED")
print("Dev gold SQL: NOT ACCESSED")

print("\nCELL 56 HOÀN THÀNH!")


===== CELL 56 — C2 FIRST 5 DEV SAMPLES =====

[1/5] RUN dev_00000
Selector: ['singer']
Schema mode: PRUNED_SCHEMA
SQL: SELECT COUNT(*) FROM singer
Latency: 1.063 seconds

[2/5] RUN dev_00003
Selector: ['singer']
Schema mode: PRUNED_SCHEMA
SQL: SELECT Name, Country, Age FROM singer ORDER BY Age DESC
Latency: 1.232 seconds

[3/5] RUN dev_00006
Selector: ['singer', 'singer_in_concert', 'concert']
Schema mode: PRUNED_SCHEMA
SQL: SELECT T1.Name, T1.Song_release_year FROM singer AS T1 INNER JOIN singer_in_concert AS T2 ON T1.Singer_ID = T2.Singer_ID ORDER BY T1.Age ASC LIMIT 1
Latency: 3.017 seconds

[4/5] RUN dev_00007
Selector: ['singer', 'singer_in_concert']
Schema mode: PRUNED_SCHEMA
SQL: SELECT Song_Name, Song_release_year FROM singer ORDER BY Age DESC LIMIT 1
Latency: 1.676 seconds

[5/5] RUN dev_00011
Selector: ['singer']
Schema mode: PRUNED_SCHEMA
SQL: SELECT Country, COUNT(*) FROM singer GROUP BY Country
Latency: 1.153 seconds

===== CHECKPOINT SUMMARY =====
Completed: 5 /300
Remain

In [77]:

# CELL 57 — C2 FULL DEV-300 INFERENCE
# MAC-SQL Lite v3.1 | Stage 4 | C2
# Resume-safe | No dev gold | Checkpoint per sample

import json
import time
from pathlib import Path
from collections import Counter

print("===== CELL 57 — C2 FULL DEV-300 RUN =====")

required = [
    "dev_samples",
    "c2_run_one",
    "c2_load_checkpoint",
    "c2_atomic_jsonl_append",
    "C2_CHECKPOINT",
    "C2_RUN_META",
    "expected_run_metadata",
    "c2_sha256",
    "C2_CONFIG",
    "C2_DEV_INPUT",
    "C2_MANIFEST"
]

for name in required:
    assert name in globals(), f"Missing: {name}"

# Check that frozen inputs have not changed.
assert c2_sha256(C2_CONFIG) == (
    expected_run_metadata["config_sha256"]
)
assert c2_sha256(C2_DEV_INPUT) == (
    expected_run_metadata["dev_input_sha256"]
)
assert c2_sha256(C2_MANIFEST) == (
    expected_run_metadata["config_manifest_sha256"]
)

with Path(C2_RUN_META).open("r", encoding="utf-8") as f:
    assert json.load(f) == expected_run_metadata

assert len(dev_samples) == 300

expected_ids = [s["sample_id"] for s in dev_samples]
assert len(set(expected_ids)) == 300

completed = c2_load_checkpoint(C2_CHECKPOINT)

assert set(completed).issubset(set(expected_ids))

print("Already completed:", len(completed))
print("Remaining:", 300 - len(completed))
print("Starting/resuming C2 inference...\n")

session_start = time.perf_counter()
newly_completed = 0

for position, sample in enumerate(dev_samples, start=1):
    sid = sample["sample_id"]

    if sid in completed:
        continue

    try:
        result = c2_run_one(sample)

        assert result["sample_id"] == sid
        assert result["db_id"] == sample["db_id"]
        assert isinstance(result["generator"]["sql"], str)

        c2_atomic_jsonl_append(
            C2_CHECKPOINT,
            result
        )

        completed[sid] = result
        newly_completed += 1

    except Exception as exc:
        print(
            f"\nSTOP at {position}/300 | {sid}",
            f"| {type(exc).__name__}: {exc}"
        )
        print(
            "Previously completed records remain saved."
        )
        raise

    if (
        position % 25 == 0
        or position == 300
    ):
        elapsed = time.perf_counter() - session_start

        print(
            f"[{position}/300] "
            f"Completed={len(completed)} "
            f"Remaining={300-len(completed)} "
            f"New_this_session={newly_completed} "
            f"Elapsed={elapsed:.1f}s",
            flush=True
        )

# Reload checkpoint from disk.
saved = c2_load_checkpoint(C2_CHECKPOINT)

assert len(saved) == 300
assert set(saved) == set(expected_ids)

parse_statuses = Counter(
    r["selector"]["parse_status"]
    for r in saved.values()
)

schema_modes = Counter(
    r["selector"]["schema_mode"]
    for r in saved.values()
)

join_statuses = Counter(
    r["selector"]["join_status"]
    for r in saved.values()
)

selector_seconds = sum(
    r["selector"]["inference_seconds"]
    for r in saved.values()
)

generator_seconds = sum(
    r["generator"]["inference_seconds"]
    for r in saved.values()
)

pipeline_seconds = sum(
    r["total_pipeline_seconds"]
    for r in saved.values()
)

print("\n===== C2 DEV-300 INFERENCE SUMMARY =====")
print("Saved predictions:", len(saved), "/300")
print("Selector parse statuses:", dict(parse_statuses))
print("Schema modes:", dict(schema_modes))
print("FK join statuses:", dict(join_statuses))

print(
    "Selector inference time:",
    round(selector_seconds, 2), "s"
)
print(
    "Generator inference time:",
    round(generator_seconds, 2), "s"
)
print(
    "Total pipeline time:",
    round(pipeline_seconds, 2), "s"
)

print(
    "Empty SQL predictions:",
    sum(
        not r["generator"]["sql"].strip()
        for r in saved.values()
    )
)

print("Dev gold SQL: NOT ACCESSED")
print("\nCELL 57 HOÀN THÀNH!")


===== CELL 57 — C2 FULL DEV-300 RUN =====
Already completed: 5
Remaining: 295
Starting/resuming C2 inference...

[25/300] Completed=25 Remaining=275 New_this_session=20 Elapsed=55.2s
[50/300] Completed=50 Remaining=250 New_this_session=45 Elapsed=115.0s
[75/300] Completed=75 Remaining=225 New_this_session=70 Elapsed=178.9s
[100/300] Completed=100 Remaining=200 New_this_session=95 Elapsed=222.6s
[125/300] Completed=125 Remaining=175 New_this_session=120 Elapsed=271.5s
[150/300] Completed=150 Remaining=150 New_this_session=145 Elapsed=327.9s
[175/300] Completed=175 Remaining=125 New_this_session=170 Elapsed=392.2s
[200/300] Completed=200 Remaining=100 New_this_session=195 Elapsed=437.9s
[225/300] Completed=225 Remaining=75 New_this_session=220 Elapsed=497.6s
[250/300] Completed=250 Remaining=50 New_this_session=245 Elapsed=549.0s
[275/300] Completed=275 Remaining=25 New_this_session=270 Elapsed=603.3s
[300/300] Completed=300 Remaining=0 New_this_session=295 Elapsed=656.5s

===== C2 DEV-3

In [78]:

# CELL 58 — C2 OFFICIAL EVALUATION INPUT
# No inference | No prompt tuning | No SQL modification

import json
import hashlib
import inspect
import sys
from pathlib import Path
from collections import Counter

print("===== CELL 58 — C2 EVALUATION INPUT =====")

BASE = Path(BASE)
C2_DIR = BASE / "results" / "c2"

checkpoint_path = C2_DIR / "c2_dev_checkpoint_v1.jsonl"
prediction_path = C2_DIR / "c2_dev_predictions_v1.jsonl"
gold_path = BASE / "data/dev_gold_300_seed42.json"
c1_path = BASE / "results/c1/c1_dev_predictions_v1.jsonl"

assert checkpoint_path.is_file()
assert gold_path.is_file()
assert c1_path.is_file()

# Load and verify all C2 predictions.
saved = c2_load_checkpoint(checkpoint_path)

assert len(saved) == 300

expected_ids = [s["sample_id"] for s in dev_samples]

assert len(set(expected_ids)) == 300
assert set(saved) == set(expected_ids)

predictions = []

for sample in dev_samples:
    sid = sample["sample_id"]
    record = saved[sid]

    assert record["db_id"] == sample["db_id"]

    sql = record["generator"]["sql"]

    assert isinstance(sql, str)
    assert sql.strip()

    predictions.append({
        "sample_id": sid,
        "source_index": sample["source_index"],
        "db_id": sample["db_id"],
        "pred_sql": sql
    })

# Write once; on rerun, require identical bytes.
payload = "".join(
    json.dumps(
        row,
        ensure_ascii=False,
        sort_keys=True
    ) + "\n"
    for row in predictions
).encode("utf-8")

if prediction_path.exists():
    assert prediction_path.read_bytes() == payload, (
        "Existing prediction file differs. "
        "Refusing to overwrite."
    )
    print("Prediction file: EXISTING / VERIFIED")
else:
    prediction_path.write_bytes(payload)
    print("Prediction file: CREATED")

print("C2 predictions:", len(predictions))
print(
    "Prediction SHA256:",
    hashlib.sha256(payload).hexdigest()
)

# Inspect gold structure without printing SQL values.
with gold_path.open("r", encoding="utf-8") as f:
    gold_data = json.load(f)

print("\n===== GOLD STRUCTURE =====")
print("Gold container type:", type(gold_data).__name__)

if isinstance(gold_data, dict):
    print("Top-level keys:", list(gold_data.keys()))

    for key, value in gold_data.items():
        if isinstance(value, list):
            print("List field:", key, "| Length:", len(value))
            if value and isinstance(value[0], dict):
                print("Record keys:", sorted(value[0].keys()))

elif isinstance(gold_data, list):
    print("Gold records:", len(gold_data))
    if gold_data and isinstance(gold_data[0], dict):
        print("Record keys:", sorted(gold_data[0].keys()))

# Inspect C1 record format for parity.
with c1_path.open("r", encoding="utf-8") as f:
    first_c1 = json.loads(next(
        line for line in f if line.strip()
    ))

print("\n===== C1 FORMAT =====")
print("C1 record keys:", sorted(first_c1.keys()))

# Inspect official evaluator function.
eval_repo = Path("/content/test-suite-sql-eval")
assert eval_repo.is_dir()

if str(eval_repo) not in sys.path:
    sys.path.insert(0, str(eval_repo))

import evaluation

print("\n===== OFFICIAL EVALUATOR =====")
print("Evaluator path:", evaluation.__file__)
print(
    "evaluate signature:",
    inspect.signature(evaluation.evaluate)
)

# Ensure checkpoint and prediction order agree.
assert all(
    row["sample_id"] == expected_ids[i]
    for i, row in enumerate(predictions)
)

print("\n===== SUMMARY =====")
print("C2 SQL predictions: 300/300")
print("Empty SQL: 0")
print("Prediction order: VERIFIED")
print("C1 artifacts: UNMODIFIED")
print("No new inference: VERIFIED")
print("No dev-driven tuning performed")

print("\nCELL 58 HOÀN THÀNH!")


===== CELL 58 — C2 EVALUATION INPUT =====
Prediction file: CREATED
C2 predictions: 300
Prediction SHA256: 2d69d897dcbf74087caf20cd89a951c0b8f66a87cf8de7b1dea6087b8a0c818a

===== GOLD STRUCTURE =====
Gold container type: dict
Top-level keys: ['dataset', 'split', 'purpose', 'seed', 'sample_count', 'sha256', 'samples']
List field: samples | Length: 300
Record keys: ['db_id', 'gold_sql', 'sample_id']

===== C1 FORMAT =====
C1 record keys: ['db_id', 'inference_seconds', 'input_tokens', 'output_tokens', 'predicted_sql', 'run_signature', 'sample_id']

===== OFFICIAL EVALUATOR =====
Evaluator path: /content/test-suite-sql-eval/evaluation.py
evaluate signature: (gold, predict, db_dir, etype, kmaps, plug_value, keep_distinct, progress_bar_for_each_datapoint)

===== SUMMARY =====
C2 SQL predictions: 300/300
Empty SQL: 0
Prediction order: VERIFIED
C1 artifacts: UNMODIFIED
No new inference: VERIFIED
No dev-driven tuning performed

CELL 58 HOÀN THÀNH!


In [81]:

# CELL 59B — OFFICIAL C2 EVALUATION IN SUBPROCESS
# Fix: asyncio.run() inside Colab event loop
# No inference | No SQL modification

import sys
import json
import subprocess
import hashlib
from pathlib import Path

print("===== CELL 59B — C2 OFFICIAL EVALUATION =====")

BASE = Path(BASE)
C2_DIR = BASE / "results" / "c2"

gold_txt = C2_DIR / "c2_eval_gold_v1.txt"
pred_txt = C2_DIR / "c2_eval_predict_v1.txt"

eval_repo = Path("/content/test-suite-sql-eval")
db_dir = Path("/content/test_suite_databases/database")

assert gold_txt.is_file()
assert pred_txt.is_file()
assert eval_repo.is_dir()
assert db_dir.is_dir()

# Confirm input row counts.
with gold_txt.open("r", encoding="utf-8") as f:
    gold_lines = f.read().splitlines()

with pred_txt.open("r", encoding="utf-8") as f:
    pred_lines = f.read().splitlines()

assert len(gold_lines) == 300
assert len(pred_lines) == 300

# Check expected gold format.
assert all(
    "\t" in line
    for line in gold_lines
)

print("Gold samples:", len(gold_lines))
print("Predictions:", len(pred_lines))

print(
    "Prediction SHA256:",
    hashlib.sha256(
        pred_txt.read_bytes()
    ).hexdigest()
)

# Use a standalone Python interpreter so that
# asyncio.run() does not inherit Colab's event loop.
runner_code = r'''
import sys
from pathlib import Path

repo = Path(sys.argv[1])
gold = sys.argv[2]
predict = sys.argv[3]
db_dir = sys.argv[4]

sys.path.insert(0, str(repo))

import evaluation

result = evaluation.evaluate(
    gold=gold,
    predict=predict,
    db_dir=db_dir,
    etype="exec",
    kmaps=None,
    plug_value=False,
    keep_distinct=True,
    progress_bar_for_each_datapoint=False
)

print("\n===== EVALUATOR RETURN =====")
print(repr(result))
'''

command = [
    sys.executable,
    "-c",
    runner_code,
    str(eval_repo),
    str(gold_txt),
    str(pred_txt),
    str(db_dir)
]

print("\nRunning official evaluator...\n")

process = subprocess.run(
    command,
    cwd=str(eval_repo),
    capture_output=True,
    text=True,
    timeout=1800
)

print("===== EVALUATOR STDOUT =====")
print(process.stdout[-16000:])

if process.stderr.strip():
    print("\n===== EVALUATOR STDERR =====")
    print(process.stderr[-6000:])

print("\nExit code:", process.returncode)

if process.returncode != 0:
    raise RuntimeError(
        "Official evaluator failed. "
        "See STDERR above. "
        "Predictions remain unchanged."
    )

# Save raw official output without silently
# replacing a different previous result.
output_path = (
    C2_DIR / "c2_official_eval_output_v1.txt"
)

output_bytes = process.stdout.encode("utf-8")

if output_path.exists():
    assert output_path.read_bytes() == output_bytes, (
        "Existing evaluation output differs. "
        "Refusing overwrite."
    )
    print("Evaluator output: VERIFIED")
else:
    output_path.write_bytes(output_bytes)
    print("Evaluator output: SAVED")

print("\n===== FINAL STATUS =====")
print("Official evaluation process: SUCCESS")
print("Evaluated samples: 300")
print("Model inference: NOT RUN")
print("C1 predictions: UNMODIFIED")
print("\nCELL 59B HOÀN THÀNH!")

===== CELL 59B — C2 OFFICIAL EVALUATION =====
Gold samples: 300
Predictions: 300
Prediction SHA256: 27f9af09cb0e0632a5e1eee9d4f940fb1d3151025a6948aaf2722b7faba78b22

Running official evaluator...

===== EVALUATOR STDOUT =====
                     easy                 medium               hard                 extra                all                 
count                75                   131                  46                   48                   300                 
=====================   EXECUTION ACCURACY     =====================
execution            0.787                0.672                0.348                0.458                0.617               

===== EVALUATOR RETURN =====
None


Exit code: 0
Evaluator output: SAVED

===== FINAL STATUS =====
Official evaluation process: SUCCESS
Evaluated samples: 300
Model inference: NOT RUN
C1 predictions: UNMODIFIED

CELL 59B HOÀN THÀNH!


In [82]:

# CELL 60 — FREEZE C2 OFFICIAL RESULTS
# No inference | No reevaluation

import json
import re
import hashlib
from pathlib import Path

print("===== CELL 60 — FREEZE C2 RESULTS =====")

BASE = Path(BASE)
c2_dir = BASE / "results" / "c2"

eval_path = c2_dir / "c2_official_eval_output_v1.txt"
pred_path = c2_dir / "c2_dev_predictions_v1.jsonl"
checkpoint_path = c2_dir / "c2_dev_checkpoint_v1.jsonl"
summary_path = c2_dir / "c2_final_summary_v1.json"

assert all(p.is_file() for p in [
    eval_path, pred_path, checkpoint_path
])

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

eval_text = eval_path.read_text(encoding="utf-8")

# Parse actual official evaluator output.
count_match = re.search(
    r"(?m)^count\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)",
    eval_text
)

score_match = re.search(
    r"(?m)^execution\s+"
    r"([0-9.]+)\s+([0-9.]+)\s+"
    r"([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)",
    eval_text
)

assert count_match, "Cannot parse evaluator counts"
assert score_match, "Cannot parse evaluator EX"

labels = ["easy", "medium", "hard", "extra", "all"]

counts = {
    k: int(v)
    for k, v in zip(labels, count_match.groups())
}

scores = {
    k: float(v)
    for k, v in zip(labels, score_match.groups())
}

assert counts == {
    "easy": 75,
    "medium": 131,
    "hard": 46,
    "extra": 48,
    "all": 300
}

assert scores["all"] == 0.617

saved = c2_load_checkpoint(checkpoint_path)
assert len(saved) == 300

selector_seconds = sum(
    r["selector"]["inference_seconds"]
    for r in saved.values()
)

generator_seconds = sum(
    r["generator"]["inference_seconds"]
    for r in saved.values()
)

pipeline_seconds = sum(
    r["total_pipeline_seconds"]
    for r in saved.values()
)

summary = {
    "experiment": "MAC-SQL Lite v3.1",
    "configuration": "C2_SELECTOR",
    "dataset": "Spider 1.0",
    "split": "dev",
    "sample_count": 300,
    "model": "Qwen/Qwen2.5-Coder-3B-Instruct",
    "metric": "Test Suite Execution Accuracy",
    "evaluator": "test-suite-sql-eval",
    "plug_value": False,
    "keep_distinct": True,
    "counts_by_difficulty": counts,
    "execution_accuracy": scores,
    "c1_reference_ex": 0.567,
    "c2_minus_c1_percentage_points": round(
        (scores["all"] - 0.567) * 100, 3
    ),
    "timing_seconds": {
        "selector": round(selector_seconds, 4),
        "generator": round(generator_seconds, 4),
        "total_pipeline": round(pipeline_seconds, 4),
        "mean_pipeline_per_sample": round(
            pipeline_seconds / 300, 4
        )
    },
    "sha256": {
        "predictions": sha256_file(pred_path),
        "checkpoint": sha256_file(checkpoint_path),
        "official_evaluation": sha256_file(eval_path)
    },
    "notes": [
        "C2 uses table-level schema selection.",
        "No fine-tuning.",
        "No dev-gold-driven prompt tuning.",
        "Reported EX values are rounded by the evaluator.",
        "Timing includes Selector and SQL Generator."
    ]
}

payload = (
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2,
        sort_keys=True
    ) + "\n"
).encode("utf-8")

if summary_path.exists():
    assert summary_path.read_bytes() == payload, (
        "Frozen C2 summary differs. Refusing overwrite."
    )
    print("C2 summary: EXISTING / VERIFIED")
else:
    summary_path.write_bytes(payload)
    print("C2 summary: CREATED")

print("\n===== OFFICIAL C2 RESULTS =====")
for label in labels:
    print(
        f"{label:7s}: "
        f"{scores[label] * 100:.1f}% "
        f"(n={counts[label]})"
    )

print("\nC1 EX: 56.7%")
print(f"C2 EX: {scores['all'] * 100:.1f}%")
print(
    "Improvement:",
    f"{summary['c2_minus_c1_percentage_points']:+.1f}",
    "percentage points"
)

print(
    "Mean C2 pipeline latency:",
    round(pipeline_seconds / 300, 3),
    "seconds/sample"
)

print("\nSummary:", summary_path)
print("C1 artifacts: UNMODIFIED")
print("Model inference: NOT RUN")
print("\nCELL 60 HOÀN THÀNH!")

===== CELL 60 — FREEZE C2 RESULTS =====
C2 summary: CREATED

===== OFFICIAL C2 RESULTS =====
easy   : 78.7% (n=75)
medium : 67.2% (n=131)
hard   : 34.8% (n=46)
extra  : 45.8% (n=48)
all    : 61.7% (n=300)

C1 EX: 56.7%
C2 EX: 61.7%
Improvement: +5.0 percentage points
Mean C2 pipeline latency: 2.197 seconds/sample

Summary: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/results/c2/c2_final_summary_v1.json
C1 artifacts: UNMODIFIED
Model inference: NOT RUN

CELL 60 HOÀN THÀNH!


In [83]:

# CELL 61 — C3 DECOMPOSER TRAIN SMOKE
# MAC-SQL Lite v3.1 | Stage 4 | C3
# Train-only development | No dev gold

import json
import time
import torch

print("===== CELL 61 — C3 DECOMPOSER TRAIN SMOKE =====")

required = [
    "model",
    "tokenizer",
    "samples",
    "spider_schema_to_ddl",
    "C1_SYSTEM_PROMPT"
]

for name in required:
    assert name in globals(), f"Missing: {name}"

assert len(samples) == 30

C3_DECOMPOSER_SYSTEM_PROMPT = (
    "You are a database query planning agent. "
    "Analyze the natural language question and database schema. "
    "Produce a concise plan for constructing a SQL query. "
    "Identify the required tables, joins, filters, "
    "aggregations, grouping, ordering, and limits "
    "only when relevant. "
    "Do not generate SQL. "
    "Do not invent tables or columns. "
    "Return only a numbered list of reasoning steps."
)

C3_GENERATOR_SYSTEM_PROMPT = C1_SYSTEM_PROMPT


def c3_decomposer_messages(question, db_id):
    schema = spider_schema_to_ddl(db_id)

    return [
        {
            "role": "system",
            "content": C3_DECOMPOSER_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Database schema:\n{schema}\n\n"
                f"Question:\n{question}\n\n"
                "Produce a concise SQL construction plan."
            )
        }
    ]


def c3_generator_messages(question, db_id, plan):
    schema = spider_schema_to_ddl(db_id)

    return [
        {
            "role": "system",
            "content": C3_GENERATOR_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Database schema:\n{schema}\n\n"
                f"Question:\n{question}\n\n"
                f"Query construction plan:\n{plan}\n\n"
                "Generate the SQL query."
            )
        }
    ]


def c3_generate(messages, max_new_tokens=256):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    input_count = inputs["input_ids"].shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    generated = outputs[0][input_count:]

    return {
        "text": tokenizer.decode(
            generated,
            skip_special_tokens=True
        ).strip(),
        "input_tokens": int(input_count),
        "output_tokens": int(len(generated)),
        "inference_seconds": float(elapsed)
    }


# Use first three fixed train samples.
c3_smoke_results = []

for sample in samples[:3]:
    sid = sample["sample_id"]
    db_id = sample["db_id"]
    question = sample["question"]

    print(f"\n===== TRAIN SAMPLE {sid} =====")

    plan_result = c3_generate(
        c3_decomposer_messages(question, db_id)
    )

    sql_result = c3_generate(
        c3_generator_messages(
            question,
            db_id,
            plan_result["text"]
        )
    )

    record = {
        "sample_id": sid,
        "db_id": db_id,
        "question": question,
        "plan": plan_result["text"],
        "raw_sql_output": sql_result["text"],
        "plan_input_tokens": plan_result["input_tokens"],
        "plan_output_tokens": plan_result["output_tokens"],
        "sql_input_tokens": sql_result["input_tokens"],
        "sql_output_tokens": sql_result["output_tokens"],
        "total_inference_seconds": (
            plan_result["inference_seconds"]
            + sql_result["inference_seconds"]
        )
    }

    c3_smoke_results.append(record)

    print("Question:", question)
    print("Plan:", plan_result["text"])
    print("SQL output:", sql_result["text"])
    print(
        "Inference time:",
        round(record["total_inference_seconds"], 3),
        "seconds"
    )

assert len(c3_smoke_results) == 3

print("\n===== C3 SMOKE SUMMARY =====")
print("Train samples tested:", len(c3_smoke_results))
print("Dev gold accessed: NO")
print("C1/C2 modified: NO")
print("\nCELL 61 HOÀN THÀNH!")


===== CELL 61 — C3 DECOMPOSER TRAIN SMOKE =====

===== TRAIN SAMPLE train_00204 =====
Question: What are the different ids and names of the stations that have had more than 12 bikes available?
Plan: 1. **Identify Relevant Tables**: The question involves stations and their availability status. The `station` table contains station information, and the `status` table contains the availability status of each station.

2. **Determine Join Condition**: To link stations with their availability status, we need to join the `station` table with the `status` table on the `id` column.

3. **Filter Availability**: We need to filter the results to include only those stations where the number of bikes available is greater than 12.

4. **Select Required Columns**: We are interested in the `id` and `name` of the stations.

5. **Construct the Query**:
   - Start with the `station` table as the primary table.
   - Join the `status` table on the `id` column.
   - Apply the filter condition to select stati

In [84]:

# CELL 62 — C3 PROMPT FIX & TRAIN SMOKE
# Train-only | No dev inference | No dev gold

import re
import time
import json

print("===== CELL 62 — C3 DECOMPOSER FIX =====")

C3_DECOMPOSER_SYSTEM_PROMPT = (
    "You are a SQL query planning agent. "
    "Given a question and database schema, produce "
    "a short, structured query plan. "
    "Use only tables and columns present in the schema. "
    "Specify: tables, joins, filters, aggregation, "
    "grouping, distinct, ordering, and limit when needed. "
    "Preserve the exact meaning of the question. "
    "Do not write SQL statements, SQL code blocks, "
    "or explanations. "
    "Return at most 6 concise numbered steps."
)

def c3_decomposer_messages(question, db_id):
    schema = spider_schema_to_ddl(db_id)

    return [
        {
            "role": "system",
            "content": C3_DECOMPOSER_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Database schema:\n{schema}\n\n"
                f"Question:\n{question}\n\n"
                "Provide a concise query construction plan."
            )
        }
    ]


def c3_generator_messages(question, db_id, plan):
    schema = spider_schema_to_ddl(db_id)

    return [
        {
            "role": "system",
            "content": C1_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Database schema:\n{schema}\n\n"
                f"Question:\n{question}\n\n"
                f"Query construction plan:\n{plan}\n\n"
                "Generate only the final SQL query."
            )
        }
    ]


def c3_extract_sql(text):
    text = text.strip()

    match = re.fullmatch(
        r"```(?:sql)?\s*(.*?)\s*```",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if match:
        text = match.group(1).strip()

    return text


c3_fixed_smoke_results = []

for sample in samples[:3]:
    sid = sample["sample_id"]
    db_id = sample["db_id"]
    question = sample["question"]

    print(f"\n===== {sid} =====")
    print("Question:", question)

    plan_result = c3_generate(
        c3_decomposer_messages(question, db_id)
    )

    sql_result = c3_generate(
        c3_generator_messages(
            question,
            db_id,
            plan_result["text"]
        )
    )

    clean_sql = c3_extract_sql(
        sql_result["text"]
    )

    record = {
        "sample_id": sid,
        "db_id": db_id,
        "plan": plan_result["text"],
        "raw_sql": sql_result["text"],
        "sql": clean_sql,
        "plan_input_tokens": plan_result["input_tokens"],
        "plan_output_tokens": plan_result["output_tokens"],
        "sql_input_tokens": sql_result["input_tokens"],
        "sql_output_tokens": sql_result["output_tokens"],
        "total_seconds": (
            plan_result["inference_seconds"]
            + sql_result["inference_seconds"]
        )
    }

    c3_fixed_smoke_results.append(record)

    print("\nPlan:", record["plan"])
    print("\nSQL:", record["sql"])
    print(
        "\nTime:",
        round(record["total_seconds"], 2),
        "seconds"
    )

assert len(c3_fixed_smoke_results) == 3
assert all(r["sql"].strip() for r in c3_fixed_smoke_results)

print("\n===== CELL 62 SUMMARY =====")
print("Train samples:", len(c3_fixed_smoke_results))
print("SQL extraction: PASS")
print("Dev inference: NOT RUN")
print("Dev gold: NOT ACCESSED")
print("C1/C2: UNMODIFIED")
print("\nCELL 62 HOÀN THÀNH!")


===== CELL 62 — C3 DECOMPOSER FIX =====

===== train_00204 =====
Question: What are the different ids and names of the stations that have had more than 12 bikes available?

Plan: 1. **Identify Relevant Tables**: The question involves stations and their status, so `station` and `status` tables are relevant.
2. **Join Tables**: Join `station` with `status` on `station.id` to link each station with its status data.
3. **Filter Data**: Filter the joined table to include only rows where `bikes_available` is greater than 12.
4. **Select Distinct IDs and Names**: Select distinct `id` and `name` from the filtered table to get unique station identifiers and names.
5. **Order Results (Optional)**: If needed, order the results by `id` or `name` for better readability.
6. **Limit Results (Optional)**: If there are many stations, limit the output to a certain number of results.

SQL: SELECT DISTINCT T1.id, T1.name FROM station AS T1 INNER JOIN status AS T2 ON T1.id = T2.station_id WHERE T2.bikes_av

In [85]:

# CELL 63 — C3 FINAL TRAIN VALIDATION
# Train-only | No dev gold | No dev inference

print("===== CELL 63 — C3 FINAL TRAIN VALIDATION =====")

C3_DECOMPOSER_SYSTEM_PROMPT = (
    "You are a database query planning agent. "
    "Given a natural language question and database schema, "
    "produce a concise SQL construction plan. "
    "Use only tables and columns present in the schema. "
    "Identify the required tables, join keys, filters, "
    "aggregations, grouping, distinct, ordering, and limits. "
    "Do not introduce joins, filters, ordering, or limits "
    "unless they are necessary to answer the question. "
    "For superlatives such as oldest, youngest, highest, "
    "or lowest, identify the comparison column and whether "
    "the question asks for a global or grouped extreme. "
    "Do not use pairwise self-joins to find global extremes "
    "when ordering or aggregation is sufficient. "
    "Never invent foreign keys or relationships. "
    "Do not write SQL code or SQL statements. "
    "Return at most 6 short numbered steps."
)

c3_final_smoke_results = []

for sample in samples[:3]:
    sid = sample["sample_id"]
    db_id = sample["db_id"]
    question = sample["question"]

    print(f"\n===== {sid} =====")

    plan_result = c3_generate(
        c3_decomposer_messages(question, db_id)
    )

    sql_result = c3_generate(
        c3_generator_messages(
            question,
            db_id,
            plan_result["text"]
        )
    )

    sql = c3_extract_sql(
        sql_result["text"]
    )

    record = {
        "sample_id": sid,
        "db_id": db_id,
        "question": question,
        "plan": plan_result["text"],
        "sql": sql,
        "plan_input_tokens": plan_result["input_tokens"],
        "plan_output_tokens": plan_result["output_tokens"],
        "sql_input_tokens": sql_result["input_tokens"],
        "sql_output_tokens": sql_result["output_tokens"],
        "total_seconds": (
            plan_result["inference_seconds"]
            + sql_result["inference_seconds"]
        )
    }

    c3_final_smoke_results.append(record)

    print("Plan:", record["plan"])
    print("SQL:", record["sql"])
    print(
        "Time:",
        round(record["total_seconds"], 2),
        "s"
    )

assert len(c3_final_smoke_results) == 3

assert all(
    r["sql"].strip()
    for r in c3_final_smoke_results
)

print("\n===== SUMMARY =====")
print("Train smoke:", len(c3_final_smoke_results))
print("Empty SQL:", sum(
    not r["sql"].strip()
    for r in c3_final_smoke_results
))
print("Dev gold: NOT ACCESSED")
print("C1/C2: UNMODIFIED")
print("\nCELL 63 HOÀN THÀNH!")


===== CELL 63 — C3 FINAL TRAIN VALIDATION =====

===== train_00204 =====
Plan: 1. **Identify Required Tables**: The question involves stations and their status, so we need to access the `station` table and the `status` table.

2. **Join Tables**: Since we need to link each station with its status, we will perform an inner join between the `station` table (aliased as T1) and the `status` table (aliased as T2) on the `id` column.

3. **Filter Data**: We need to filter the results to include only those stations where the number of bikes available (`bikes_available`) is greater than 12.

4. **Select Distinct Columns**: To get unique station IDs and names, we will select the `id` and `name` columns from the `station` table.

5. **Construct SQL Query**: Combine the above steps into a SQL query to retrieve the desired information.

6. **Final Query**:
   ```sql
   SELECT DISTINCT T1.id, T1.name
   FROM station AS T1
   INNER JOIN status AS T2 ON T1.id = T2.station_id
   WHERE T2.bikes_availab

In [86]:

# CELL 64 — FREEZE C3 CONFIG & CHECKPOINT
# MAC-SQL Lite v3.1 | Stage 4 | C3
# No inference | No dev gold access

import json
import hashlib
import os
from pathlib import Path

print("===== CELL 64 — FREEZE C3 =====")

BASE = Path(BASE)
C3_DIR = BASE / "results" / "c3"
C3_DIR.mkdir(parents=True, exist_ok=True)

C3_CONFIG_PATH = BASE / "configs/c3_decomposer_v1.json"
C3_CHECKPOINT = C3_DIR / "c3_dev_checkpoint_v1.jsonl"
C3_RUN_META = C3_DIR / "c3_run_metadata_v1.json"

assert len(dev_samples) == 300
assert len({s["sample_id"] for s in dev_samples}) == 300

assert "model" in globals()
assert "tokenizer" in globals()
assert "c3_generate" in globals()
assert "c3_decomposer_messages" in globals()
assert "c3_generator_messages" in globals()
assert "c3_extract_sql" in globals()

def c3_hash_bytes(data):
    return hashlib.sha256(data).hexdigest()

def c3_hash_file(path):
    return c3_hash_bytes(Path(path).read_bytes())

def c3_write_once(path, obj):
    path = Path(path)
    payload = (
        json.dumps(
            obj,
            ensure_ascii=False,
            sort_keys=True,
            indent=2
        ) + "\n"
    ).encode("utf-8")

    if path.exists():
        assert path.read_bytes() == payload, (
            f"Frozen artifact differs: {path}"
        )
        print("VERIFIED:", path.name)
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        temp = path.with_name(path.name + ".tmp")

        with temp.open("wb") as f:
            f.write(payload)
            f.flush()
            os.fsync(f.fileno())

        os.replace(temp, path)
        print("CREATED:", path.name)

C3_CONFIG = {
    "experiment": "MAC-SQL Lite v3.1",
    "configuration": "C3_DECOMPOSER",
    "version": 1,
    "model": "Qwen/Qwen2.5-Coder-3B-Instruct",
    "model_precision": "FP16",
    "fine_tuning": False,
    "schema": "FULL_SCHEMA",
    "decomposer_system_prompt": C3_DECOMPOSER_SYSTEM_PROMPT,
    "generator_system_prompt": C1_SYSTEM_PROMPT,
    "decomposer_user_template": (
        "Database schema:\\n{schema}\\n\\n"
        "Question:\\n{question}\\n\\n"
        "Provide a concise query construction plan."
    ),
    "generator_user_template": (
        "Database schema:\\n{schema}\\n\\n"
        "Question:\\n{question}\\n\\n"
        "Query construction plan:\\n{plan}\\n\\n"
        "Generate only the final SQL query."
    ),
    "decoding": {
        "do_sample": False,
        "max_new_tokens": 256
    },
    "sql_postprocessing": "strip_outer_markdown_code_fence",
    "development": "train_only",
    "evaluation": {
        "dataset": "Spider 1.0",
        "split": "dev",
        "sample_count": 300,
        "metric": "Test Suite Execution Accuracy",
        "plug_value": False,
        "keep_distinct": True
    }
}

c3_write_once(C3_CONFIG_PATH, C3_CONFIG)

dev_input_path = BASE / "data/dev_eval_300_seed42.json"
c1_manifest_path = (
    BASE / "results/c1/c1_final_manifest_v1.json"
)

assert dev_input_path.is_file()
assert c1_manifest_path.is_file()

C3_METADATA = {
    "configuration": "C3_DECOMPOSER",
    "version": 1,
    "config_sha256": c3_hash_file(C3_CONFIG_PATH),
    "dev_input_sha256": c3_hash_file(dev_input_path),
    "c1_manifest_sha256": c3_hash_file(c1_manifest_path),
    "checkpoint_format": "JSONL",
    "sample_count": 300
}

c3_write_once(C3_RUN_META, C3_METADATA)

# Read-only checkpoint validation.
if C3_CHECKPOINT.exists():
    c3_existing = {}

    with C3_CHECKPOINT.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            record = json.loads(line)
            sid = record["sample_id"]

            assert sid not in c3_existing
            c3_existing[sid] = record
else:
    c3_existing = {}

expected_ids = {s["sample_id"] for s in dev_samples}

assert set(c3_existing).issubset(expected_ids)

print("\n===== C3 STATUS =====")
print("Config SHA256:", C3_METADATA["config_sha256"])
print("Completed:", len(c3_existing))
print("Remaining:", 300 - len(c3_existing))
print("Schema: FULL_SCHEMA")
print("Decoding: GREEDY / 256 tokens")
print("Dev gold: NOT ACCESSED")
print("C1/C2: UNMODIFIED")
print("\nCELL 64 HOÀN THÀNH!")


===== CELL 64 — FREEZE C3 =====
CREATED: c3_decomposer_v1.json
CREATED: c3_run_metadata_v1.json

===== C3 STATUS =====
Config SHA256: a75cc7790f4f473563b68d3fe1782e180e62a353b150f8dd9d72737e904297fd
Completed: 0
Remaining: 300
Schema: FULL_SCHEMA
Decoding: GREEDY / 256 tokens
Dev gold: NOT ACCESSED
C1/C2: UNMODIFIED

CELL 64 HOÀN THÀNH!


In [88]:

# CELL 65 — C3 FULL DEV-300 INFERENCE
# MAC-SQL Lite v3.1 | C3 DECOMPOSER
# Resume-safe | No dev gold | No SQL execution

import json
import time
import os
import hashlib
from pathlib import Path
from collections import Counter

print("===== CELL 65 — C3 FULL DEV-300 RUN =====")

BASE = Path(BASE)
C3_DIR = BASE / "results" / "c3"

C3_CONFIG_PATH = BASE / "configs/c3_decomposer_v1.json"
C3_RUN_META = C3_DIR / "c3_run_metadata_v1.json"
C3_CHECKPOINT = C3_DIR / "c3_dev_checkpoint_v1.jsonl"

DEV_INPUT = BASE / "data/dev_eval_300_seed42.json"
C1_MANIFEST = BASE / "results/c1/c1_final_manifest_v1.json"

# ------------------------------------------
# 1. Verify frozen configuration
# ------------------------------------------

required_functions = [
    "c3_generate",
    "c3_decomposer_messages",
    "c3_generator_messages",
    "c3_extract_sql"
]

for name in required_functions:
    assert name in globals(), f"Missing function: {name}"

assert len(dev_samples) == 300
assert len({s["sample_id"] for s in dev_samples}) == 300

with C3_CONFIG_PATH.open("r", encoding="utf-8") as f:
    frozen_config = json.load(f)

with C3_RUN_META.open("r", encoding="utf-8") as f:
    frozen_meta = json.load(f)

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)
    return h.hexdigest()

assert sha256_file(C3_CONFIG_PATH) == frozen_meta["config_sha256"]
assert sha256_file(DEV_INPUT) == frozen_meta["dev_input_sha256"]
assert sha256_file(C1_MANIFEST) == frozen_meta["c1_manifest_sha256"]

assert frozen_config["configuration"] == "C3_DECOMPOSER"
assert frozen_config["schema"] == "FULL_SCHEMA"
assert frozen_config["decoding"]["do_sample"] is False
assert frozen_config["decoding"]["max_new_tokens"] == 256

# Restore frozen prompt to avoid accidental changes.
C3_DECOMPOSER_SYSTEM_PROMPT = frozen_config[
    "decomposer_system_prompt"
]

assert C1_SYSTEM_PROMPT == frozen_config[
    "generator_system_prompt"
]

# ------------------------------------------
# 2. Checkpoint utilities
# ------------------------------------------

def c3_load_checkpoint(path):
    path = Path(path)
    records = {}

    if not path.exists():
        return records

    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue

            record = json.loads(line)
            sid = record["sample_id"]

            if sid in records:
                raise ValueError(
                    f"Duplicate sample_id: {sid}, "
                    f"line {line_number}"
                )

            records[sid] = record

    return records


def c3_append_checkpoint(path, record):
    """
    Atomic checkpoint replacement.
    Safe for a 300-sample experiment.
    """
    path = Path(path)

    existing = (
        path.read_bytes()
        if path.exists()
        else b""
    )

    new_line = (
        json.dumps(
            record,
            ensure_ascii=False,
            sort_keys=True
        ) + "\n"
    ).encode("utf-8")

    temp = path.with_name(path.name + ".tmp")

    with temp.open("wb") as f:
        f.write(existing)
        f.write(new_line)
        f.flush()
        os.fsync(f.fileno())

    os.replace(temp, path)


# ------------------------------------------
# 3. Single-sample C3 pipeline
# ------------------------------------------

def c3_run_one(sample):
    sid = sample["sample_id"]
    db_id = sample["db_id"]
    question = sample["question"]

    total_start = time.perf_counter()

    # Decomposer: full schema -> plan
    plan_result = c3_generate(
        c3_decomposer_messages(question, db_id),
        max_new_tokens=256
    )

    plan = plan_result["text"]

    # Generator: full schema + plan -> SQL
    sql_result = c3_generate(
        c3_generator_messages(
            question, db_id, plan
        ),
        max_new_tokens=256
    )

    sql = c3_extract_sql(sql_result["text"])

    total_seconds = (
        time.perf_counter() - total_start
    )

    return {
        "sample_id": sid,
        "source_index": sample["source_index"],
        "db_id": db_id,
        "question": question,
        "configuration": "C3_DECOMPOSER",
        "decomposer": {
            "plan": plan,
            "input_tokens": plan_result["input_tokens"],
            "output_tokens": plan_result["output_tokens"],
            "inference_seconds": plan_result[
                "inference_seconds"
            ]
        },
        "generator": {
            "raw_output": sql_result["text"],
            "sql": sql,
            "input_tokens": sql_result["input_tokens"],
            "output_tokens": sql_result["output_tokens"],
            "inference_seconds": sql_result[
                "inference_seconds"
            ]
        },
        "total_pipeline_seconds": float(total_seconds)
    }


# ------------------------------------------
# 4. Resume and run all 300 samples
# ------------------------------------------

completed = c3_load_checkpoint(C3_CHECKPOINT)

expected_ids = {
    s["sample_id"]
    for s in dev_samples
}

assert set(completed).issubset(expected_ids)

print("Already completed:", len(completed))
print("Remaining:", 300 - len(completed))
print("Starting/resuming C3 inference...\n")

session_start = time.perf_counter()
newly_completed = 0

for position, sample in enumerate(dev_samples, start=1):
    sid = sample["sample_id"]

    if sid in completed:
        continue

    try:
        result = c3_run_one(sample)

        assert result["sample_id"] == sid
        assert result["db_id"] == sample["db_id"]
        assert isinstance(
            result["generator"]["sql"], str
        )

        # Save before starting next sample.
        c3_append_checkpoint(
            C3_CHECKPOINT,
            result
        )

        completed[sid] = result
        newly_completed += 1

    except Exception as exc:
        print(
            f"\nSTOP at {position}/300 | {sid}",
            f"| {type(exc).__name__}: {exc}"
        )
        print(
            "Previously completed results remain saved."
        )
        raise

    if position % 25 == 0 or position == 300:
        elapsed = time.perf_counter() - session_start

        print(
            f"[{position}/300] "
            f"Completed={len(completed)} "
            f"Remaining={300-len(completed)} "
            f"New_this_session={newly_completed} "
            f"Elapsed={elapsed:.1f}s",
            flush=True
        )


# ------------------------------------------
# 5. Verify final checkpoint and summarize
# ------------------------------------------

saved = c3_load_checkpoint(C3_CHECKPOINT)

assert len(saved) == 300
assert set(saved) == expected_ids

empty_sql = sum(
    not r["generator"]["sql"].strip()
    for r in saved.values()
)

decomposer_seconds = sum(
    r["decomposer"]["inference_seconds"]
    for r in saved.values()
)

generator_seconds = sum(
    r["generator"]["inference_seconds"]
    for r in saved.values()
)

pipeline_seconds = sum(
    r["total_pipeline_seconds"]
    for r in saved.values()
)

plan_tokens = sum(
    r["decomposer"]["output_tokens"]
    for r in saved.values()
)

print("\n===== C3 DEV-300 INFERENCE SUMMARY =====")
print("Saved predictions:", len(saved), "/300")
print("Empty SQL:", empty_sql)

print(
    "Decomposer inference:",
    round(decomposer_seconds, 2), "s"
)
print(
    "Generator inference:",
    round(generator_seconds, 2), "s"
)
print(
    "Total pipeline:",
    round(pipeline_seconds, 2), "s"
)
print(
    "Mean pipeline latency:",
    round(pipeline_seconds / 300, 3),
    "s/sample"
)
print(
    "Mean decomposer output tokens:",
    round(plan_tokens / 300, 2)
)

print("Dev gold: NOT ACCESSED")
print("C1/C2: UNMODIFIED")
print("\nCELL 65 HOÀN THÀNH!")


===== CELL 65 — C3 FULL DEV-300 RUN =====
Already completed: 265
Remaining: 35
Starting/resuming C3 inference...

[275/300] Completed=275 Remaining=25 New_this_session=10 Elapsed=158.3s
[300/300] Completed=300 Remaining=0 New_this_session=35 Elapsed=432.0s

===== C3 DEV-300 INFERENCE SUMMARY =====
Saved predictions: 300 /300
Empty SQL: 0
Decomposer inference: 2825.17 s
Generator inference: 600.36 s
Total pipeline: 3427.65 s
Mean pipeline latency: 11.425 s/sample
Mean decomposer output tokens: 214.08
Dev gold: NOT ACCESSED
C1/C2: UNMODIFIED

CELL 65 HOÀN THÀNH!


In [89]:

# CELL 66 — C3 OFFICIAL TEST SUITE EX
# No model inference | Official evaluator subprocess

import json
import sys
import re
import subprocess
import hashlib
from pathlib import Path

print("===== CELL 66 — C3 OFFICIAL EVALUATION =====")

BASE = Path(BASE)
C3_DIR = BASE / "results" / "c3"

C3_CHECKPOINT = C3_DIR / "c3_dev_checkpoint_v1.jsonl"
C3_GOLD_TXT = C3_DIR / "c3_eval_gold_v1.txt"
C3_PRED_TXT = C3_DIR / "c3_eval_predict_v1.txt"
C3_EVAL_OUTPUT = C3_DIR / "c3_official_eval_output_v1.txt"

EVAL_REPO = Path("/content/test-suite-sql-eval")
SUITE_DB = Path("/content/test_suite_databases/database")

assert C3_CHECKPOINT.is_file()
assert EVAL_REPO.is_dir()
assert SUITE_DB.is_dir()

# -----------------------------------
# 1. Load 300 frozen C3 predictions
# -----------------------------------

records = {}

with C3_CHECKPOINT.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        r = json.loads(line)
        sid = r["sample_id"]

        assert sid not in records
        records[sid] = r

assert len(records) == 300
assert len(dev_samples) == 300

# -----------------------------------
# 2. Load frozen gold ONLY for eval
# -----------------------------------

gold_path = BASE / "data/dev_gold_300_seed42.json"

assert gold_path.is_file()

with gold_path.open("r", encoding="utf-8") as f:
    gold_data = json.load(f)

# Support list or common wrapped formats.
if isinstance(gold_data, dict):
    for key in ("samples", "data", "records"):
        if key in gold_data:
            gold_data = gold_data[key]
            break

assert isinstance(gold_data, list)
assert len(gold_data) == 300

gold_by_id = {
    r["sample_id"]: r
    for r in gold_data
}

assert len(gold_by_id) == 300

# -----------------------------------
# 3. Build evaluator input files
# -----------------------------------

gold_lines = []
pred_lines = []

for sample in dev_samples:
    sid = sample["sample_id"]
    db_id = sample["db_id"]

    assert sid in records
    assert sid in gold_by_id

    prediction = records[sid]
    gold = gold_by_id[sid]

    assert prediction["db_id"] == db_id
    assert gold["db_id"] == db_id

    gold_sql = gold.get("gold_sql", gold.get("query"))
    pred_sql = prediction["generator"]["sql"]

    assert isinstance(gold_sql, str) and gold_sql.strip()
    assert isinstance(pred_sql, str) and pred_sql.strip()

    assert "\n" not in gold_sql
    assert "\t" not in gold_sql

    # Keep SQL unchanged except flattening line breaks
    # required by the official line-based evaluator.
    pred_sql_line = " ".join(pred_sql.splitlines())

    assert "\t" not in pred_sql_line

    gold_lines.append(f"{gold_sql}\t{db_id}")
    pred_lines.append(pred_sql_line)

C3_GOLD_TXT.write_text(
    "\n".join(gold_lines) + "\n",
    encoding="utf-8"
)

C3_PRED_TXT.write_text(
    "\n".join(pred_lines) + "\n",
    encoding="utf-8"
)

assert len(C3_GOLD_TXT.read_text(
    encoding="utf-8"
).splitlines()) == 300

assert len(C3_PRED_TXT.read_text(
    encoding="utf-8"
).splitlines()) == 300

print("Gold lines:", len(gold_lines))
print("Prediction lines:", len(pred_lines))

# -----------------------------------
# 4. Official evaluator subprocess
# -----------------------------------

runner_code = r'''
import sys
from pathlib import Path

repo = Path(sys.argv[1])
gold = sys.argv[2]
predict = sys.argv[3]
db_dir = sys.argv[4]

sys.path.insert(0, str(repo))
import evaluation

result = evaluation.evaluate(
    gold=gold,
    predict=predict,
    db_dir=db_dir,
    etype="exec",
    kmaps=None,
    plug_value=False,
    keep_distinct=True,
    progress_bar_for_each_datapoint=False
)

print("\n===== EVALUATOR RETURN =====")
print(repr(result))
'''

process = subprocess.run(
    [
        sys.executable,
        "-c",
        runner_code,
        str(EVAL_REPO),
        str(C3_GOLD_TXT),
        str(C3_PRED_TXT),
        str(SUITE_DB)
    ],
    cwd=str(EVAL_REPO),
    capture_output=True,
    text=True,
    timeout=1800
)

combined_output = (
    process.stdout
    + "\n===== STDERR =====\n"
    + process.stderr
)

C3_EVAL_OUTPUT.write_text(
    combined_output,
    encoding="utf-8"
)

print("\n===== EVALUATOR OUTPUT =====")
print(process.stdout[-6000:])

if process.returncode != 0:
    print("===== STDERR =====")
    print(process.stderr[-4000:])
    raise RuntimeError(
        f"Evaluator failed: {process.returncode}"
    )

# -----------------------------------
# 5. Parse official accuracy
# -----------------------------------

match = re.search(
    r"^\s*execution\s+"
    r"([0-9.]+)\s+"
    r"([0-9.]+)\s+"
    r"([0-9.]+)\s+"
    r"([0-9.]+)\s+"
    r"([0-9.]+)\s*$",
    process.stdout,
    flags=re.MULTILINE
)

assert match, "Could not parse official EX."

values = [float(x) for x in match.groups()]

print("\n===== OFFICIAL C3 TEST SUITE EX =====")

for name, value in zip(
    ["easy", "medium", "hard", "extra", "all"],
    values
):
    print(f"{name:7s}: {value * 100:.1f}%")

print("\n===== COMPARISON =====")
print("C1 Baseline : 56.7%")
print("C2 Selector : 61.7%")
print(f"C3 Decomposer: {values[-1] * 100:.1f}%")

print(
    "C3 vs C1:",
    f"{(values[-1] - 0.567) * 100:+.1f}",
    "percentage points"
)

print(
    "C3 vs C2:",
    f"{(values[-1] - 0.617) * 100:+.1f}",
    "percentage points"
)

print("\nModel inference: NOT RUN")
print("C1/C2: UNMODIFIED")
print("\nCELL 66 HOÀN THÀNH!")


===== CELL 66 — C3 OFFICIAL EVALUATION =====
Gold lines: 300
Prediction lines: 300

===== EVALUATOR OUTPUT =====
                     easy                 medium               hard                 extra                all                 
count                75                   131                  46                   48                   300                 
=====================   EXECUTION ACCURACY     =====================
execution            0.720                0.542                0.261                0.354                0.513               

===== EVALUATOR RETURN =====
None


===== OFFICIAL C3 TEST SUITE EX =====
easy   : 72.0%
medium : 54.2%
hard   : 26.1%
extra  : 35.4%
all    : 51.3%

===== COMPARISON =====
C1 Baseline : 56.7%
C2 Selector : 61.7%
C3 Decomposer: 51.3%
C3 vs C1: -5.4 percentage points
C3 vs C2: -10.4 percentage points

Model inference: NOT RUN
C1/C2: UNMODIFIED

CELL 66 HOÀN THÀNH!


In [90]:

# CELL 67 — FREEZE C3 OFFICIAL RESULTS
# No inference | No evaluation

import json
import hashlib
import re
import os
from pathlib import Path

print("===== CELL 67 — FREEZE C3 RESULTS =====")

BASE = Path(BASE)
C3_DIR = BASE / "results" / "c3"

checkpoint_path = C3_DIR / "c3_dev_checkpoint_v1.jsonl"
config_path = BASE / "configs/c3_decomposer_v1.json"
metadata_path = C3_DIR / "c3_run_metadata_v1.json"
eval_path = C3_DIR / "c3_official_eval_output_v1.txt"
summary_path = C3_DIR / "c3_final_summary_v1.json"

for path in [
    checkpoint_path,
    config_path,
    metadata_path,
    eval_path
]:
    assert path.is_file(), f"Missing: {path}"

def c3_file_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

records = [
    json.loads(line)
    for line in checkpoint_path.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

assert len(records) == 300
assert len({r["sample_id"] for r in records}) == 300

eval_text = eval_path.read_text(encoding="utf-8")

match = re.search(
    r"^\s*execution\s+"
    r"([0-9.]+)\s+"
    r"([0-9.]+)\s+"
    r"([0-9.]+)\s+"
    r"([0-9.]+)\s+"
    r"([0-9.]+)\s*$",
    eval_text,
    flags=re.MULTILINE
)

assert match, "Official EX not found"

easy, medium, hard, extra, overall = [
    float(x) for x in match.groups()
]

assert abs(overall - 0.513) < 0.001

total_seconds = sum(
    r["total_pipeline_seconds"]
    for r in records
)

decomposer_seconds = sum(
    r["decomposer"]["inference_seconds"]
    for r in records
)

generator_seconds = sum(
    r["generator"]["inference_seconds"]
    for r in records
)

summary = {
    "configuration": "C3_DECOMPOSER",
    "dataset": "Spider 1.0",
    "split": "dev",
    "sample_count": 300,
    "metric": "Official Test Suite EX",
    "execution_accuracy": {
        "easy": easy,
        "medium": medium,
        "hard": hard,
        "extra": extra,
        "all": overall
    },
    "comparison": {
        "c1_baseline_ex": 0.567,
        "c2_selector_ex": 0.617,
        "c3_vs_c1_percentage_points": round(
            (overall - 0.567) * 100, 2
        ),
        "c3_vs_c2_percentage_points": round(
            (overall - 0.617) * 100, 2
        )
    },
    "inference": {
        "decomposer_seconds": decomposer_seconds,
        "generator_seconds": generator_seconds,
        "total_pipeline_seconds": total_seconds,
        "mean_pipeline_seconds": total_seconds / 300,
        "mean_plan_output_tokens": sum(
            r["decomposer"]["output_tokens"]
            for r in records
        ) / 300
    },
    "artifacts_sha256": {
        "checkpoint": c3_file_hash(checkpoint_path),
        "config": c3_file_hash(config_path),
        "run_metadata": c3_file_hash(metadata_path),
        "official_evaluator_output": c3_file_hash(eval_path)
    },
    "interpretation": (
        "The current Decomposer-only adaptation "
        "underperformed the baseline on this "
        "300-example Spider dev subset."
    )
}

payload = (
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2,
        sort_keys=True
    ) + "\n"
).encode("utf-8")

if summary_path.exists():
    assert summary_path.read_bytes() == payload, (
        "Existing frozen C3 summary differs"
    )
    print("C3 summary: VERIFIED")
else:
    temp = summary_path.with_name(
        summary_path.name + ".tmp"
    )

    with temp.open("wb") as f:
        f.write(payload)
        f.flush()
        os.fsync(f.fileno())

    os.replace(temp, summary_path)
    print("C3 summary: CREATED")

print("\n===== OFFICIAL C3 RESULTS =====")
print(f"Easy   : {easy * 100:.1f}%")
print(f"Medium : {medium * 100:.1f}%")
print(f"Hard   : {hard * 100:.1f}%")
print(f"Extra  : {extra * 100:.1f}%")
print(f"All    : {overall * 100:.1f}%")

print("\nC3 vs C1: -5.4 percentage points")
print("C3 vs C2: -10.4 percentage points")
print(
    "Mean latency:",
    round(total_seconds / 300, 3),
    "seconds/sample"
)

print("\nSummary:", summary_path)
print("C1/C2 artifacts: UNMODIFIED")
print("Model inference: NOT RUN")
print("\nCELL 67 HOÀN THÀNH!")


===== CELL 67 — FREEZE C3 RESULTS =====
C3 summary: CREATED

===== OFFICIAL C3 RESULTS =====
Easy   : 72.0%
Medium : 54.2%
Hard   : 26.1%
Extra  : 35.4%
All    : 51.3%

C3 vs C1: -5.4 percentage points
C3 vs C2: -10.4 percentage points
Mean latency: 11.425 seconds/sample

Summary: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/results/c3/c3_final_summary_v1.json
C1/C2 artifacts: UNMODIFIED
Model inference: NOT RUN

CELL 67 HOÀN THÀNH!


In [91]:

# CELL 68 — C4 BASELINE SQL EXECUTION AUDIT
# No model inference | No dev gold | Read-only SQLite

import json
import sqlite3
import time
from pathlib import Path
from collections import Counter

print("===== CELL 68 — C4 EXECUTION AUDIT =====")

BASE = Path(BASE)
C1_PRED_PATH = (
    BASE / "results/c1/c1_dev_predictions_v1.jsonl"
)

DB_ROOT = Path("/content/spider_data/spider_data/database")

assert C1_PRED_PATH.is_file()
assert DB_ROOT.is_dir()
assert len(dev_samples) == 300

c1_predictions = {}

with C1_PRED_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)
        sid = record["sample_id"]

        assert sid not in c1_predictions
        c1_predictions[sid] = record

assert len(c1_predictions) == 300

def c4_find_db(db_id):
    db_path = DB_ROOT / db_id / f"{db_id}.sqlite"
    assert db_path.is_file(), f"Missing DB: {db_path}"
    return db_path


def c4_check_sql(sql, db_id, timeout_seconds=5):
    if not sql or not sql.strip():
        return {
            "status": "EMPTY_SQL",
            "error": "Empty SQL",
            "seconds": 0.0
        }

    db_path = c4_find_db(db_id)

    start = time.perf_counter()
    connection = None

    try:
        connection = sqlite3.connect(
            f"file:{db_path.as_posix()}?mode=ro",
            uri=True,
            timeout=1
        )

        connection.execute("PRAGMA query_only = ON")

        # Interrupt queries exceeding the time budget.
        def progress_handler():
            if time.perf_counter() - start > timeout_seconds:
                return 1
            return 0

        connection.set_progress_handler(
            progress_handler,
            1000
        )

        cursor = connection.execute(sql)

        # Fetch all rows to detect runtime errors
        # beyond the initial execute() call.
        cursor.fetchall()

        return {
            "status": "EXECUTION_OK",
            "error": None,
            "seconds": time.perf_counter() - start
        }

    except sqlite3.Error as exc:
        elapsed = time.perf_counter() - start

        message = str(exc)

        if (
            "interrupted" in message.lower()
            and elapsed >= timeout_seconds
        ):
            status = "TIMEOUT"
        else:
            status = "EXECUTION_ERROR"

        return {
            "status": status,
            "error": message,
            "seconds": elapsed
        }

    finally:
        if connection is not None:
            connection.close()


c4_execution_audit = []

for position, sample in enumerate(dev_samples, 1):
    sid = sample["sample_id"]
    db_id = sample["db_id"]

    assert sid in c1_predictions

    prediction = c1_predictions[sid]
    assert prediction["db_id"] == db_id

    sql = prediction["predicted_sql"]

    result = c4_check_sql(sql, db_id)

    c4_execution_audit.append({
        "sample_id": sid,
        "db_id": db_id,
        "predicted_sql": sql,
        **result
    })

    if position % 50 == 0:
        print(f"Checked {position}/300", flush=True)

assert len(c4_execution_audit) == 300

counts = Counter(
    r["status"]
    for r in c4_execution_audit
)

print("\n===== C4 EXECUTION AUDIT RESULTS =====")

for status in [
    "EXECUTION_OK",
    "EXECUTION_ERROR",
    "TIMEOUT",
    "EMPTY_SQL"
]:
    print(
        f"{status:17s}:",
        counts.get(status, 0)
    )

print("\n===== FIRST 10 NON-OK EXAMPLES =====")

problematic = [
    r for r in c4_execution_audit
    if r["status"] != "EXECUTION_OK"
]

for r in problematic[:10]:
    print("\nSample:", r["sample_id"])
    print("DB:", r["db_id"])
    print("SQL:", r["predicted_sql"][:300])
    print("Status:", r["status"])
    print("Error:", r["error"])

print("\nTotal problematic:", len(problematic))
print("Dev gold: NOT ACCESSED")
print("Model inference: NOT RUN")
print("C1/C2/C3: UNMODIFIED")
print("\nCELL 68 HOÀN THÀNH!")


===== CELL 68 — C4 EXECUTION AUDIT =====
Checked 50/300
Checked 100/300
Checked 150/300
Checked 200/300
Checked 250/300
Checked 300/300

===== C4 EXECUTION AUDIT RESULTS =====
EXECUTION_OK     : 257
EXECUTION_ERROR  : 43
TIMEOUT          : 0
EMPTY_SQL        : 0

===== FIRST 10 NON-OK EXAMPLES =====

Sample: dev_00006
DB: concert_singer
SQL: SELECT T2.Name, T2.Song_release_year FROM singer AS T1 INNER JOIN singer_in_concert AS T3 ON T1.Singer_ID = T3.Singer_ID INNER JOIN concert AS T4 ON T3.concert_ID = T4.concert_ID INNER JOIN stadium AS T5 ON T4.Stadium_ID = T5.Stadium_ID ORDER BY T1.Age DESC LIMIT 1
Status: EXECUTION_ERROR
Error: no such column: T2.Name

Sample: dev_00007
DB: concert_singer
SQL: SELECT T2.Song_Name, T2.Song_release_year FROM singer AS T1 INNER JOIN singer_in_concert AS T3 ON T1.Singer_ID = T3.Singer_ID INNER JOIN concert AS T4 ON T3.concert_ID = T4.concert_ID INNER JOIN stadium AS T5 ON T4.Stadium_ID = T5.Stadium_ID WHERE T1.Age = (SELECT MIN(Age) FROM singer) ORDER

In [92]:

# CELL 69 — C4 REFINER CONFIG & ERROR AUDIT FREEZE
# No model inference | No dev gold

import json
import hashlib
import os
from pathlib import Path
from collections import Counter

print("===== CELL 69 — C4 REFINER SETUP =====")

BASE = Path(BASE)
C4_DIR = BASE / "results" / "c4"
C4_DIR.mkdir(parents=True, exist_ok=True)

C4_AUDIT_PATH = C4_DIR / "c4_execution_audit_v1.jsonl"
C4_CONFIG_PATH = BASE / "configs/c4_refiner_v1.json"
C4_CHECKPOINT = C4_DIR / "c4_refiner_checkpoint_v1.jsonl"

assert len(c4_execution_audit) == 300
assert len({r["sample_id"] for r in c4_execution_audit}) == 300

C4_REFINER_SYSTEM_PROMPT = (
    "You are a SQLite SQL repair agent. "
    "Given a natural language question, database schema, "
    "a previously generated SQL query, and its SQLite error, "
    "repair the query so that it executes correctly and "
    "preserves the meaning of the question. "
    "Use only existing tables and columns from the schema. "
    "Correct invalid aliases, column references, joins, "
    "and syntax errors when necessary. "
    "Make the smallest semantically appropriate correction. "
    "Do not add unrelated joins, filters, ordering, or limits. "
    "Return only the corrected SQL query, without explanations "
    "or Markdown formatting."
)

def c4_refiner_messages(question, db_id, sql, error):
    schema = spider_schema_to_ddl(db_id)

    return [
        {
            "role": "system",
            "content": C4_REFINER_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Database schema:\n{schema}\n\n"
                f"Question:\n{question}\n\n"
                f"Original SQL:\n{sql}\n\n"
                f"SQLite error:\n{error}\n\n"
                "Return the repaired SQL query."
            )
        }
    ]


def c4_sha256(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


def c4_write_once(path, data):
    path = Path(path)

    if isinstance(data, (dict, list)):
        content = (
            json.dumps(
                data,
                ensure_ascii=False,
                indent=2,
                sort_keys=True
            ) + "\n"
        ).encode("utf-8")
    else:
        content = data

    if path.exists():
        assert path.read_bytes() == content, (
            f"Frozen artifact differs: {path}"
        )
        print("VERIFIED:", path.name)
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        temp = path.with_name(path.name + ".tmp")

        with temp.open("wb") as f:
            f.write(content)
            f.flush()
            os.fsync(f.fileno())

        os.replace(temp, path)
        print("CREATED:", path.name)


# Preserve the exact order of the 300 dev samples.
audit_by_id = {
    r["sample_id"]: r
    for r in c4_execution_audit
}

ordered_audit = [
    audit_by_id[s["sample_id"]]
    for s in dev_samples
]

audit_bytes = "".join(
    json.dumps(
        r,
        ensure_ascii=False,
        sort_keys=True
    ) + "\n"
    for r in ordered_audit
).encode("utf-8")

c4_write_once(C4_AUDIT_PATH, audit_bytes)

counts = Counter(
    r["status"]
    for r in ordered_audit
)

assert counts["EXECUTION_ERROR"] == 43
assert counts["EXECUTION_OK"] == 257
assert counts["TIMEOUT"] == 0
assert counts["EMPTY_SQL"] == 0

C4_CONFIG = {
    "configuration": "C4_REFINER",
    "version": 1,
    "model": "Qwen/Qwen2.5-Coder-3B-Instruct",
    "precision": "FP16",
    "schema": "FULL_SCHEMA",
    "baseline": "C1_FROZEN_PREDICTIONS",
    "refinement_trigger": "SQLITE_EXECUTION_ERROR",
    "max_repair_attempts": 1,
    "decoding": {
        "do_sample": False,
        "max_new_tokens": 256
    },
    "refiner_system_prompt": C4_REFINER_SYSTEM_PROMPT,
    "audit_sha256": c4_sha256(C4_AUDIT_PATH),
    "development": "TRAIN_INDEPENDENT_FIXED_ERROR_FEEDBACK",
    "gold_sql_used_for_refinement": False
}

c4_write_once(C4_CONFIG_PATH, C4_CONFIG)

# Check existing checkpoint without modifying it.
if C4_CHECKPOINT.exists():
    checkpoint_records = [
        json.loads(line)
        for line in C4_CHECKPOINT.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]
else:
    checkpoint_records = []

assert len({
    r["sample_id"]
    for r in checkpoint_records
}) == len(checkpoint_records)

error_ids = {
    r["sample_id"]
    for r in ordered_audit
    if r["status"] == "EXECUTION_ERROR"
}

assert {
    r["sample_id"]
    for r in checkpoint_records
}.issubset(error_ids)

print("\n===== C4 CONFIG =====")
print("Execution OK:", counts["EXECUTION_OK"])
print("Execution errors:", counts["EXECUTION_ERROR"])
print("Repair attempts per error:", 1)
print("Existing repairs:", len(checkpoint_records))
print("Remaining repairs:", 43 - len(checkpoint_records))
print("Audit SHA256:", c4_sha256(C4_AUDIT_PATH))
print("Dev gold: NOT ACCESSED")
print("Model inference: NOT RUN")
print("C1/C2/C3: UNMODIFIED")
print("\nCELL 69 HOÀN THÀNH!")


===== CELL 69 — C4 REFINER SETUP =====
CREATED: c4_execution_audit_v1.jsonl
CREATED: c4_refiner_v1.json

===== C4 CONFIG =====
Execution OK: 257
Execution errors: 43
Repair attempts per error: 1
Existing repairs: 0
Remaining repairs: 43
Audit SHA256: e598b2ff09bb7293ad62f2ee81a317138935726fb3229bdf23e5ceb0581b08e5
Dev gold: NOT ACCESSED
Model inference: NOT RUN
C1/C2/C3: UNMODIFIED

CELL 69 HOÀN THÀNH!


In [93]:

# CELL 70 — C4 REFINER INFERENCE (43 ERRORS)
# Resume-safe | No dev gold | One repair per error

import json
import os
import time
import hashlib
from pathlib import Path
from collections import Counter

print("===== CELL 70 — C4 REFINER RUN =====")

BASE = Path(BASE)
C4_DIR = BASE / "results" / "c4"

C4_CONFIG_PATH = BASE / "configs/c4_refiner_v1.json"
C4_AUDIT_PATH = C4_DIR / "c4_execution_audit_v1.jsonl"
C4_CHECKPOINT = C4_DIR / "c4_refiner_checkpoint_v1.jsonl"

assert C4_CONFIG_PATH.is_file()
assert C4_AUDIT_PATH.is_file()

with C4_CONFIG_PATH.open("r", encoding="utf-8") as f:
    frozen_config = json.load(f)

assert frozen_config["configuration"] == "C4_REFINER"
assert frozen_config["max_repair_attempts"] == 1
assert frozen_config["refinement_trigger"] == "SQLITE_EXECUTION_ERROR"

assert (
    hashlib.sha256(C4_AUDIT_PATH.read_bytes()).hexdigest()
    == frozen_config["audit_sha256"]
)

# Restore frozen prompt.
C4_REFINER_SYSTEM_PROMPT = frozen_config[
    "refiner_system_prompt"
]

audit = [
    json.loads(line)
    for line in C4_AUDIT_PATH.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

assert len(audit) == 300

errors = [
    r for r in audit
    if r["status"] == "EXECUTION_ERROR"
]

assert len(errors) == 43

dev_by_id = {
    s["sample_id"]: s
    for s in dev_samples
}

assert len(dev_by_id) == 300

# -----------------------------------
# 1. Checkpoint helpers
# -----------------------------------

def c4_load_checkpoint(path):
    records = {}
    path = Path(path)

    if not path.exists():
        return records

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            r = json.loads(line)
            sid = r["sample_id"]

            if sid in records:
                raise ValueError(
                    f"Duplicate checkpoint ID: {sid}"
                )

            records[sid] = r

    return records


def c4_append_checkpoint(path, record):
    path = Path(path)

    existing = (
        path.read_bytes()
        if path.exists()
        else b""
    )

    new_line = (
        json.dumps(
            record,
            ensure_ascii=False,
            sort_keys=True
        ) + "\n"
    ).encode("utf-8")

    temp = path.with_name(path.name + ".tmp")

    with temp.open("wb") as f:
        f.write(existing)
        f.write(new_line)
        f.flush()
        os.fsync(f.fileno())

    os.replace(temp, path)


# -----------------------------------
# 2. Resume completed repairs
# -----------------------------------

completed = c4_load_checkpoint(C4_CHECKPOINT)

error_ids = {
    r["sample_id"]
    for r in errors
}

assert set(completed).issubset(error_ids)

print("Total execution errors:", len(errors))
print("Already repaired:", len(completed))
print("Remaining:", len(errors) - len(completed))

# -----------------------------------
# 3. Run Refiner
# -----------------------------------

session_start = time.perf_counter()
new_count = 0

for position, item in enumerate(errors, start=1):
    sid = item["sample_id"]

    if sid in completed:
        continue

    sample = dev_by_id[sid]

    assert sample["db_id"] == item["db_id"]

    question = sample["question"]
    db_id = item["db_id"]
    original_sql = item["predicted_sql"]
    original_error = item["error"]

    try:
        # One model call only.
        result = c3_generate(
            c4_refiner_messages(
                question,
                db_id,
                original_sql,
                original_error
            ),
            max_new_tokens=256
        )

        repaired_sql = c3_extract_sql(
            result["text"]
        )

        # Validate execution without gold SQL.
        if repaired_sql.strip():
            validation = c4_check_sql(
                repaired_sql,
                db_id,
                timeout_seconds=5
            )
        else:
            validation = {
                "status": "EMPTY_SQL",
                "error": "Refiner produced empty SQL",
                "seconds": 0.0
            }

        record = {
            "sample_id": sid,
            "db_id": db_id,
            "configuration": "C4_REFINER",
            "original_sql": original_sql,
            "original_error": original_error,
            "repaired_sql": repaired_sql,
            "raw_refiner_output": result["text"],
            "refiner_input_tokens": result["input_tokens"],
            "refiner_output_tokens": result["output_tokens"],
            "refiner_inference_seconds": result[
                "inference_seconds"
            ],
            "validation": validation,
            "repair_attempts": 1
        }

        c4_append_checkpoint(
            C4_CHECKPOINT,
            record
        )

        completed[sid] = record
        new_count += 1

    except Exception as exc:
        print(
            f"\nSTOP at {position}/43 ({sid}):",
            type(exc).__name__,
            str(exc)
        )
        print("Checkpoint remains saved.")
        raise

    if (
        position % 5 == 0
        or position == len(errors)
    ):
        print(
            f"[{position}/43] "
            f"Saved={len(completed)} "
            f"Remaining={43-len(completed)} "
            f"Session={time.perf_counter()-session_start:.1f}s",
            flush=True
        )

# -----------------------------------
# 4. Final validation
# -----------------------------------

saved = c4_load_checkpoint(C4_CHECKPOINT)

assert len(saved) == 43
assert set(saved) == error_ids

statuses = Counter(
    r["validation"]["status"]
    for r in saved.values()
)

total_refiner_seconds = sum(
    r["refiner_inference_seconds"]
    for r in saved.values()
)

print("\n===== C4 REFINER SUMMARY =====")
print("Total errors:", 43)
print("Repair attempts:", len(saved))
print(
    "Execution OK after repair:",
    statuses.get("EXECUTION_OK", 0)
)
print(
    "Still execution error:",
    statuses.get("EXECUTION_ERROR", 0)
)
print(
    "Timeout:",
    statuses.get("TIMEOUT", 0)
)
print(
    "Empty SQL:",
    statuses.get("EMPTY_SQL", 0)
)
print(
    "Refiner inference time:",
    round(total_refiner_seconds, 2), "s"
)
print(
    "Mean time per repair:",
    round(total_refiner_seconds / 43, 3), "s"
)

print("\nDev gold: NOT ACCESSED")
print("C1/C2/C3: UNMODIFIED")
print("\nCELL 70 HOÀN THÀNH!")


===== CELL 70 — C4 REFINER RUN =====
Total execution errors: 43
Already repaired: 0
Remaining: 43
[5/43] Saved=5 Remaining=38 Session=24.7s
[10/43] Saved=10 Remaining=33 Session=34.5s
[15/43] Saved=15 Remaining=28 Session=46.0s
[20/43] Saved=20 Remaining=23 Session=59.7s
[25/43] Saved=25 Remaining=18 Session=73.3s
[30/43] Saved=30 Remaining=13 Session=86.6s
[35/43] Saved=35 Remaining=8 Session=98.6s
[40/43] Saved=40 Remaining=3 Session=111.1s
[43/43] Saved=43 Remaining=0 Session=119.4s

===== C4 REFINER SUMMARY =====
Total errors: 43
Repair attempts: 43
Execution OK after repair: 13
Still execution error: 30
Timeout: 0
Empty SQL: 0
Refiner inference time: 118.31 s
Mean time per repair: 2.751 s

Dev gold: NOT ACCESSED
C1/C2/C3: UNMODIFIED

CELL 70 HOÀN THÀNH!


In [94]:

# CELL 71 — C4 MERGE + OFFICIAL TEST SUITE EX
# No model inference | No gold-guided repair

import json
import re
import sys
import subprocess
import hashlib
from pathlib import Path

print("===== CELL 71 — C4 OFFICIAL EVALUATION =====")

BASE = Path(BASE)
C4_DIR = BASE / "results" / "c4"

C1_PATH = BASE / "results/c1/c1_dev_predictions_v1.jsonl"
C4_AUDIT = C4_DIR / "c4_execution_audit_v1.jsonl"
C4_REPAIRS = C4_DIR / "c4_refiner_checkpoint_v1.jsonl"

C4_PRED_JSONL = C4_DIR / "c4_dev_predictions_v1.jsonl"
C4_GOLD_TXT = C4_DIR / "c4_eval_gold_v1.txt"
C4_PRED_TXT = C4_DIR / "c4_eval_predict_v1.txt"
C4_EVAL_OUTPUT = C4_DIR / "c4_official_eval_output_v1.txt"

EVAL_REPO = Path("/content/test-suite-sql-eval")
SUITE_DB = Path("/content/test_suite_databases/database")

def load_jsonl(path):
    records = {}
    for line in Path(path).read_text(
        encoding="utf-8"
    ).splitlines():
        if not line.strip():
            continue
        r = json.loads(line)
        sid = r["sample_id"]
        assert sid not in records, f"Duplicate ID: {sid}"
        records[sid] = r
    return records

c1 = load_jsonl(C1_PATH)
audit = load_jsonl(C4_AUDIT)
repairs = load_jsonl(C4_REPAIRS)

assert len(c1) == 300
assert len(audit) == 300
assert len(repairs) == 43
assert len(dev_samples) == 300

error_ids = {
    sid for sid, r in audit.items()
    if r["status"] == "EXECUTION_ERROR"
}
assert set(repairs) == error_ids

# Reuse the exact frozen C3 gold input to maintain
# sample alignment across C1/C2/C3/C4.
C3_GOLD_TXT = (
    BASE / "results/c3/c3_eval_gold_v1.txt"
)
assert C3_GOLD_TXT.is_file()

gold_lines = C3_GOLD_TXT.read_text(
    encoding="utf-8"
).splitlines()
assert len(gold_lines) == 300

merged = []
pred_lines = []

for index, sample in enumerate(dev_samples):
    sid = sample["sample_id"]
    db_id = sample["db_id"]

    assert sid in c1
    assert sid in audit
    assert c1[sid]["db_id"] == db_id
    assert audit[sid]["db_id"] == db_id
    assert gold_lines[index].endswith("\t" + db_id)

    original_sql = c1[sid]["predicted_sql"]

    if sid in error_ids:
        final_sql = repairs[sid]["repaired_sql"]
        source = "REFINER"
        validation_status = repairs[sid][
            "validation"
        ]["status"]
    else:
        final_sql = original_sql
        source = "BASELINE_UNCHANGED"
        validation_status = "EXECUTION_OK"

    assert isinstance(final_sql, str)
    assert final_sql.strip()

    merged.append({
        "sample_id": sid,
        "source_index": sample["source_index"],
        "db_id": db_id,
        "predicted_sql": final_sql,
        "prediction_source": source,
        "validation_status": validation_status
    })

    pred_lines.append(
        " ".join(final_sql.splitlines())
    )

assert len(merged) == 300
assert len(pred_lines) == 300

prediction_payload = "".join(
    json.dumps(
        r,
        ensure_ascii=False,
        sort_keys=True
    ) + "\n"
    for r in merged
)

def write_once(path, content):
    path = Path(path)
    content = content.encode("utf-8")

    if path.exists():
        assert path.read_bytes() == content, (
            f"Frozen artifact differs: {path}"
        )
        print("VERIFIED:", path.name)
    else:
        path.write_bytes(content)
        print("CREATED:", path.name)

write_once(C4_PRED_JSONL, prediction_payload)

write_once(
    C4_GOLD_TXT,
    "\n".join(gold_lines) + "\n"
)

write_once(
    C4_PRED_TXT,
    "\n".join(pred_lines) + "\n"
)

print("\nMerged predictions:", len(merged))
print("Unchanged C1 SQL:", 300 - len(error_ids))
print("Refiner SQL:", len(error_ids))

# Official evaluator must run in subprocess
# to avoid asyncio.run() conflicts in Colab.
runner_code = r'''
import sys
from pathlib import Path

repo = Path(sys.argv[1])
sys.path.insert(0, str(repo))

import evaluation

evaluation.evaluate(
    gold=sys.argv[2],
    predict=sys.argv[3],
    db_dir=sys.argv[4],
    etype="exec",
    kmaps=None,
    plug_value=False,
    keep_distinct=True,
    progress_bar_for_each_datapoint=False
)
'''

process = subprocess.run(
    [
        sys.executable,
        "-c",
        runner_code,
        str(EVAL_REPO),
        str(C4_GOLD_TXT),
        str(C4_PRED_TXT),
        str(SUITE_DB)
    ],
    cwd=str(EVAL_REPO),
    capture_output=True,
    text=True,
    timeout=1800
)

output = (
    process.stdout
    + "\n===== STDERR =====\n"
    + process.stderr
)

C4_EVAL_OUTPUT.write_text(
    output,
    encoding="utf-8"
)

print("\n===== OFFICIAL EVALUATOR OUTPUT =====")
print(process.stdout[-5000:])

if process.returncode != 0:
    print(process.stderr[-3000:])
    raise RuntimeError(
        f"Evaluator failed: {process.returncode}"
    )

match = re.search(
    r"^\s*execution\s+"
    r"([0-9.]+)\s+"
    r"([0-9.]+)\s+"
    r"([0-9.]+)\s+"
    r"([0-9.]+)\s+"
    r"([0-9.]+)\s*$",
    process.stdout,
    flags=re.MULTILINE
)

assert match, "Official EX not found"

easy, medium, hard, extra, overall = [
    float(x) for x in match.groups()
]

print("\n===== OFFICIAL C4 TEST SUITE EX =====")
print(f"Easy   : {easy * 100:.1f}%")
print(f"Medium : {medium * 100:.1f}%")
print(f"Hard   : {hard * 100:.1f}%")
print(f"Extra  : {extra * 100:.1f}%")
print(f"All    : {overall * 100:.1f}%")

print("\n===== COMPARISON =====")
print("C1 Baseline   : 56.7%")
print("C2 Selector   : 61.7%")
print("C3 Decomposer : 51.3%")
print(f"C4 Refiner    : {overall * 100:.1f}%")
print(
    "C4 vs C1:",
    f"{(overall - 0.567) * 100:+.1f}",
    "percentage points"
)

print("\nModel inference: NOT RUN")
print("C1/C2/C3: UNMODIFIED")
print("\nCELL 71 HOÀN THÀNH!")


===== CELL 71 — C4 OFFICIAL EVALUATION =====
CREATED: c4_dev_predictions_v1.jsonl
CREATED: c4_eval_gold_v1.txt
CREATED: c4_eval_predict_v1.txt

Merged predictions: 300
Unchanged C1 SQL: 257
Refiner SQL: 43

===== OFFICIAL EVALUATOR OUTPUT =====
                     easy                 medium               hard                 extra                all                 
count                75                   131                  46                   48                   300                 
=====================   EXECUTION ACCURACY     =====================
execution            0.760                0.603                0.391                0.500                0.593               


===== OFFICIAL C4 TEST SUITE EX =====
Easy   : 76.0%
Medium : 60.3%
Hard   : 39.1%
Extra  : 50.0%
All    : 59.3%

===== COMPARISON =====
C1 Baseline   : 56.7%
C2 Selector   : 61.7%
C3 Decomposer : 51.3%
C4 Refiner    : 59.3%
C4 vs C1: +2.6 percentage points

Model inference: NOT RUN
C1/C2/C3: UNMODIFIED



In [96]:

# CELL 72 FIXED — FREEZE C4 OFFICIAL RESULTS
# No inference | No evaluation

import json
import hashlib
import re
import os
from pathlib import Path
from collections import Counter

print("===== CELL 72 FIXED — FREEZE C4 RESULTS =====")

BASE = Path(BASE)
C4_DIR = BASE / "results" / "c4"

paths = {
    "config": BASE / "configs/c4_refiner_v1.json",
    "audit": C4_DIR / "c4_execution_audit_v1.jsonl",
    "repairs": C4_DIR / "c4_refiner_checkpoint_v1.jsonl",
    "predictions": C4_DIR / "c4_dev_predictions_v1.jsonl",
    "eval_gold": C4_DIR / "c4_eval_gold_v1.txt",
    "eval_predict": C4_DIR / "c4_eval_predict_v1.txt",
    "eval_output": C4_DIR / "c4_official_eval_output_v1.txt"
}

for name, path in paths.items():
    assert path.is_file(), f"Missing {name}: {path}"

def file_sha256(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

def load_jsonl(path):
    return [
        json.loads(line)
        for line in Path(path).read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]

audit = load_jsonl(paths["audit"])
repairs = load_jsonl(paths["repairs"])
predictions = load_jsonl(paths["predictions"])

assert len(audit) == 300
assert len(repairs) == 43
assert len(predictions) == 300

assert len({
    r["sample_id"] for r in predictions
}) == 300

repair_statuses = Counter(
    r["validation"]["status"]
    for r in repairs
)

assert repair_statuses["EXECUTION_OK"] == 13
assert repair_statuses["EXECUTION_ERROR"] == 30

# FIX: Parse all 5 evaluator accuracy values.
eval_text = paths["eval_output"].read_text(
    encoding="utf-8"
)

match = re.search(
    r"^\s*execution\s+"
    r"([0-9.]+)\s+"
    r"([0-9.]+)\s+"
    r"([0-9.]+)\s+"
    r"([0-9.]+)\s+"
    r"([0-9.]+)\s*$",
    eval_text,
    flags=re.MULTILINE
)

assert match, "Official EX not found"

easy, medium, hard, extra, overall = [
    float(x) for x in match.groups()
]

assert abs(overall - 0.593) < 0.001

refiner_seconds = sum(
    r["refiner_inference_seconds"]
    for r in repairs
)

summary = {
    "configuration": "C4_REFINER",
    "dataset": "Spider 1.0",
    "split": "dev",
    "sample_count": 300,
    "metric": "Official Test Suite EX",
    "execution_accuracy": {
        "easy": easy,
        "medium": medium,
        "hard": hard,
        "extra": extra,
        "all": overall
    },
    "repair": {
        "original_execution_errors": 43,
        "repair_attempts": 43,
        "execution_ok_after_repair": 13,
        "execution_error_after_repair": 30,
        "repair_success_rate": 13 / 43,
        "total_refiner_inference_seconds": refiner_seconds,
        "mean_refiner_seconds_per_repair": refiner_seconds / 43,
        "mean_additional_seconds_per_sample": refiner_seconds / 300
    },
    "comparison": {
        "c1_ex": 0.567,
        "c4_ex": overall,
        "improvement_percentage_points": round(
            (overall - 0.567) * 100, 2
        )
    },
    "artifacts_sha256": {
        name: file_sha256(path)
        for name, path in paths.items()
    }
}

summary_path = C4_DIR / "c4_final_summary_v1.json"

payload = (
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2,
        sort_keys=True
    ) + "\n"
).encode("utf-8")

if summary_path.exists():
    assert summary_path.read_bytes() == payload, (
        "Frozen C4 summary differs"
    )
    print("C4 summary: VERIFIED")
else:
    temp = summary_path.with_name(
        summary_path.name + ".tmp"
    )

    with temp.open("wb") as f:
        f.write(payload)
        f.flush()
        os.fsync(f.fileno())

    os.replace(temp, summary_path)
    print("C4 summary: CREATED")

print("\n===== OFFICIAL C4 RESULTS =====")
print(f"Easy   : {easy * 100:.1f}%")
print(f"Medium : {medium * 100:.1f}%")
print(f"Hard   : {hard * 100:.1f}%")
print(f"Extra  : {extra * 100:.1f}%")
print(f"All    : {overall * 100:.1f}%")

print("\nC4 vs C1: +2.6 percentage points")
print(f"Repair success: {13/43*100:.1f}%")
print(
    "Additional inference:",
    round(refiner_seconds / 300, 3),
    "s/sample"
)

print("\nSummary:", summary_path)
print("C1/C2/C3: UNMODIFIED")
print("Model inference: NOT RUN")
print("\nCELL 72 HOÀN THÀNH!")


===== CELL 72 FIXED — FREEZE C4 RESULTS =====
C4 summary: CREATED

===== OFFICIAL C4 RESULTS =====
Easy   : 76.0%
Medium : 60.3%
Hard   : 39.1%
Extra  : 50.0%
All    : 59.3%

C4 vs C1: +2.6 percentage points
Repair success: 30.2%
Additional inference: 0.394 s/sample

Summary: /content/drive/MyDrive/0. Classroom/3. NĂM 3/HK1(3)/Nhập môn Xử lý ngôn ngữ tự nhiên/bài báo nghiên cứu/mac_sql_lite_v3/results/c4/c4_final_summary_v1.json
C1/C2/C3: UNMODIFIED
Model inference: NOT RUN

CELL 72 HOÀN THÀNH!


In [97]:

# CELL 73 — C5 COMPONENT INTERFACE INSPECTION
# No inference | No dev gold | No file changes

import inspect
import json
from pathlib import Path

print("===== CELL 73 — C5 COMPONENT INSPECTION =====")

components = [
    "c2_generate_selector",
    "parse_selector_tables",
    "c2_final_schema_ddl",
    "build_c2_verified_ddl",
    "c3_decomposer_messages",
    "c3_generator_messages",
    "c3_generate",
    "c3_extract_sql",
    "c4_refiner_messages",
    "c4_check_sql",
]

for name in components:
    obj = globals().get(name)

    print(f"\n--- {name} ---")

    if obj is None:
        print("STATUS: MISSING")
        continue

    try:
        print("SIGNATURE:", inspect.signature(obj))
    except (TypeError, ValueError):
        print("SIGNATURE: unavailable")

    try:
        source = inspect.getsource(obj)
        print("SOURCE:")
        print(source[:4500])
        if len(source) > 4500:
            print("... SOURCE TRUNCATED")
    except (OSError, TypeError):
        print("SOURCE: unavailable")

print("\n===== FROZEN CONFIG FILES =====")

BASE = Path(BASE)

config_paths = [
    BASE / "configs/c2_selector_v1.json",
    BASE / "configs/c3_decomposer_v1.json",
    BASE / "configs/c4_refiner_v1.json",
]

for path in config_paths:
    print(f"\n{path.name}: {'FOUND' if path.exists() else 'NOT FOUND'}")

    if path.exists():
        data = json.loads(path.read_text(encoding="utf-8"))
        print("KEYS:", list(data.keys()))
        print(
            "PROMPT FIELDS:",
            [
                k for k in data
                if "prompt" in k.lower()
            ]
        )

print("\n===== C5 READINESS =====")
print("Model inference: NOT RUN")
print("Dev gold: NOT ACCESSED")
print("Frozen C1/C2/C3/C4: UNMODIFIED")
print("\nCELL 73 HOÀN THÀNH!")


===== CELL 73 — C5 COMPONENT INSPECTION =====

--- c2_generate_selector ---
SIGNATURE: (messages)
SOURCE:
def c2_generate_selector(messages):
    """
    Selector inference with the same model,
    greedy decoding, and 256-token limit.
    """
    import torch

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    input_tokens = inputs["input_ids"].shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=256,
            pad_token_id=tokenizer.eos_token_id
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    generated = output[0][input_tokens:]

    tex

In [98]:

# CELL 74 — C5 FULL PIPELINE SETUP
# No inference | No dev gold | Freeze before evaluation

import json
import hashlib
import inspect
import os
from pathlib import Path

print("===== CELL 74 — C5 FULL PIPELINE SETUP =====")

BASE = Path(BASE)
C5_DIR = BASE / "results/c5"
C5_DIR.mkdir(parents=True, exist_ok=True)

C5_CONFIG_PATH = BASE / "configs/c5_full_pipeline_v1.json"
C5_CHECKPOINT = C5_DIR / "c5_dev_checkpoint_v1.jsonl"

C2_CONFIG_PATH = BASE / "configs/c2_selector_v1.json"
C3_CONFIG_PATH = BASE / "configs/c3_decomposer_v1.json"
C4_CONFIG_PATH = BASE / "configs/c4_refiner_v1.json"

def c5_sha256(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

with C2_CONFIG_PATH.open(encoding="utf-8") as f:
    c2_frozen = json.load(f)

with C3_CONFIG_PATH.open(encoding="utf-8") as f:
    c3_frozen = json.load(f)

with C4_CONFIG_PATH.open(encoding="utf-8") as f:
    c4_frozen = json.load(f)

# Ensure prompts remain identical to frozen C3/C4.
assert (
    C3_DECOMPOSER_SYSTEM_PROMPT
    == c3_frozen["decomposer_system_prompt"]
)
assert (
    C1_SYSTEM_PROMPT
    == c3_frozen["generator_system_prompt"]
)
assert (
    C4_REFINER_SYSTEM_PROMPT
    == c4_frozen["refiner_system_prompt"]
)

# C3 adaptation: only replace full schema with
# Selector-provided schema; preserve prompt wording.
def c5_decomposer_messages(question, schema_ddl):
    return [
        {
            "role": "system",
            "content": C3_DECOMPOSER_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Database schema:\n{schema_ddl}\n\n"
                f"Question:\n{question}\n\n"
                "Provide a concise query construction plan."
            )
        }
    ]


def c5_generator_messages(question, schema_ddl, plan):
    return [
        {
            "role": "system",
            "content": C1_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Database schema:\n{schema_ddl}\n\n"
                f"Question:\n{question}\n\n"
                f"Query construction plan:\n{plan}\n\n"
                "Generate only the final SQL query."
            )
        }
    ]


# C4 adaptation: keep full schema for repair,
# consistent with the frozen C4 design.
def c5_refiner_messages(question, db_id, sql, error):
    return c4_refiner_messages(
        question, db_id, sql, error
    )


# Confirm adapters preserve the original C3
# prompts when supplied the full schema.
test_question = "How many records are there?"
test_db_id = dev_samples[0]["db_id"]
full_schema = spider_schema_to_ddl(test_db_id)
test_plan = "Count all matching records."

assert c5_decomposer_messages(
    test_question, full_schema
) == c3_decomposer_messages(
    test_question, test_db_id
)

assert c5_generator_messages(
    test_question, full_schema, test_plan
) == c3_generator_messages(
    test_question, test_db_id, test_plan
)

C5_CONFIG = {
    "configuration": "C5_FULL_PIPELINE",
    "version": 1,
    "dataset": "Spider 1.0",
    "split": "dev",
    "sample_count": 300,
    "model": "Qwen/Qwen2.5-Coder-3B-Instruct",
    "precision": "FP16",
    "fine_tuning": False,
    "pipeline": [
        "SELECTOR",
        "DECOMPOSER",
        "SQL_GENERATOR",
        "EXECUTION_CHECK",
        "REFINER_ON_SQLITE_ERROR"
    ],
    "selector": {
        "source": "C2_FROZEN",
        "schema": "FK_EXPANDED_PRUNED_SCHEMA",
        "fallback": "FULL_SCHEMA"
    },
    "decomposer": {
        "source": "C3_FROZEN",
        "schema": "SELECTOR_OUTPUT"
    },
    "generator": {
        "source": "C3_FROZEN",
        "schema": "SELECTOR_OUTPUT",
        "plan": "DECOMPOSER_OUTPUT"
    },
    "refiner": {
        "source": "C4_FROZEN",
        "schema": "FULL_SCHEMA",
        "trigger": "SQLITE_EXECUTION_ERROR",
        "max_attempts": 1
    },
    "decoding": {
        "do_sample": False,
        "max_new_tokens": 256
    },
    "source_config_sha256": {
        "c2": c5_sha256(C2_CONFIG_PATH),
        "c3": c5_sha256(C3_CONFIG_PATH),
        "c4": c5_sha256(C4_CONFIG_PATH)
    },
    "dev_gold_used_for_design": False
}

payload = (
    json.dumps(
        C5_CONFIG,
        ensure_ascii=False,
        indent=2,
        sort_keys=True
    ) + "\n"
).encode("utf-8")

if C5_CONFIG_PATH.exists():
    assert C5_CONFIG_PATH.read_bytes() == payload, (
        "Frozen C5 config differs"
    )
    print("C5 config: VERIFIED")
else:
    temp = C5_CONFIG_PATH.with_name(
        C5_CONFIG_PATH.name + ".tmp"
    )

    with temp.open("wb") as f:
        f.write(payload)
        f.flush()
        os.fsync(f.fileno())

    os.replace(temp, C5_CONFIG_PATH)
    print("C5 config: CREATED")

print("\n===== C5 CONFIGURATION =====")
print("Selector: C2 frozen")
print("Decomposer: C3 frozen")
print("SQL Generator: C3 frozen")
print("Refiner: C4 frozen")
print("Schema propagation: VERIFIED")
print("Repair attempts: 1")
print("Evaluation samples: 300")
print("Config SHA256:", c5_sha256(C5_CONFIG_PATH))

print("\nModel inference: NOT RUN")
print("Dev gold: NOT ACCESSED")
print("C1/C2/C3/C4: UNMODIFIED")
print("\nCELL 74 HOÀN THÀNH!")


===== CELL 74 — C5 FULL PIPELINE SETUP =====
C5 config: CREATED

===== C5 CONFIGURATION =====
Selector: C2 frozen
Decomposer: C3 frozen
SQL Generator: C3 frozen
Refiner: C4 frozen
Schema propagation: VERIFIED
Repair attempts: 1
Evaluation samples: 300
Config SHA256: 4a8aaf0d07b17a316cd543fe3e2e16557983a0ea2704235f0d95f6a453d300e6

Model inference: NOT RUN
Dev gold: NOT ACCESSED
C1/C2/C3/C4: UNMODIFIED

CELL 74 HOÀN THÀNH!


In [ ]:

# CELL 75 — C5 FULL PIPELINE INFERENCE
# 300 Spider dev samples | Resume-safe | No dev gold

import json
import os
import time
import hashlib
from pathlib import Path
from collections import Counter

print("===== CELL 75 — C5 FULL PIPELINE =====")

BASE = Path(BASE)
C5_DIR = BASE / "results/c5"
C5_DIR.mkdir(parents=True, exist_ok=True)

C5_CONFIG_PATH = BASE / "configs/c5_full_pipeline_v1.json"
C5_CHECKPOINT = C5_DIR / "c5_dev_checkpoint_v1.jsonl"

with C5_CONFIG_PATH.open(encoding="utf-8") as f:
    c5_config = json.load(f)

assert c5_config["configuration"] == "C5_FULL_PIPELINE"
assert c5_config["sample_count"] == 300

# Confirm frozen source configurations have not changed.
for name, path in {
    "c2": BASE / "configs/c2_selector_v1.json",
    "c3": BASE / "configs/c3_decomposer_v1.json",
    "c4": BASE / "configs/c4_refiner_v1.json"
}.items():
    actual = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    assert actual == c5_config[
        "source_config_sha256"
    ][name], f"{name} frozen config changed"


def c5_load_checkpoint(path):
    records = {}
    if not Path(path).exists():
        return records

    for line in Path(path).read_text(
        encoding="utf-8"
    ).splitlines():
        if not line.strip():
            continue

        record = json.loads(line)
        sid = record["sample_id"]

        assert sid not in records, (
            f"Duplicate checkpoint: {sid}"
        )

        records[sid] = record

    return records


def c5_append_checkpoint(path, record):
    path = Path(path)
    previous = (
        path.read_bytes()
        if path.exists()
        else b""
    )

    new_line = (
        json.dumps(
            record,
            ensure_ascii=False,
            sort_keys=True
        ) + "\n"
    ).encode("utf-8")

    temp = path.with_name(path.name + ".tmp")

    with temp.open("wb") as f:
        f.write(previous)
        f.write(new_line)
        f.flush()
        os.fsync(f.fileno())

    os.replace(temp, path)


# Use the frozen C2 selector prompt and FK expansion.
def c5_select_schema(question, db_id):
    selector_result = c2_generate_selector(
        build_selector_messages(question, db_id)
    )

    selected_tables, parse_status = parse_selector_tables(
        selector_result["raw_output"],
        db_id
    )

    if selected_tables is None:
        schema_ddl = spider_schema_to_ddl(db_id)
        schema_mode = "FULL_SCHEMA_FALLBACK"
        final_tables = schema_by_id[db_id][
            "table_names_original"
        ]
        fk_status = "NOT_APPLICABLE"
    else:
        fk_result = audit_combined_fk(
            db_id, selected_tables
        )

        # C2's FK audit may expose an expanded table set.
        # Use the frozen C2 schema-processing function
        # rather than reimplementing FK expansion.
        schema_ddl, schema_mode = c2_final_schema_ddl(
            db_id, selected_tables
        )

        final_tables = selected_tables
        fk_status = (
            fk_result.get("status", "UNKNOWN")
            if isinstance(fk_result, dict)
            else "AUDITED"
        )

    return {
        "schema_ddl": schema_ddl,
        "schema_mode": schema_mode,
        "selected_tables": final_tables,
        "parse_status": parse_status,
        "fk_status": fk_status,
        "selector": selector_result
    }


def c5_run_one(sample):
    sid = sample["sample_id"]
    question = sample["question"]
    db_id = sample["db_id"]

    start = time.perf_counter()

    # 1. Selector
    selected = c5_select_schema(
        question, db_id
    )

    schema_ddl = selected["schema_ddl"]

    # 2. Decomposer
    decomposition = c3_generate(
        c5_decomposer_messages(
            question, schema_ddl
        ),
        max_new_tokens=256
    )

    plan = decomposition["text"]

    # 3. SQL Generator
    generation = c3_generate(
        c5_generator_messages(
            question, schema_ddl, plan
        ),
        max_new_tokens=256
    )

    generated_sql = c3_extract_sql(
        generation["text"]
    )

    # 4. SQLite execution check
    initial_check = c4_check_sql(
        generated_sql, db_id,
        timeout_seconds=5
    )

    # 5. Refiner (one attempt, only on execution error)
    repair = None
    final_sql = generated_sql
    final_check = initial_check

    if initial_check["status"] == "EXECUTION_ERROR":
        repair = c3_generate(
            c5_refiner_messages(
                question,
                db_id,
                generated_sql,
                initial_check["error"]
            ),
            max_new_tokens=256
        )

        final_sql = c3_extract_sql(
            repair["text"]
        )

        final_check = c4_check_sql(
            final_sql, db_id,
            timeout_seconds=5
        )

    elapsed = time.perf_counter() - start

    return {
        "sample_id": sid,
        "source_index": sample["source_index"],
        "db_id": db_id,
        "question": question,
        "schema_mode": selected["schema_mode"],
        "selected_tables": selected["selected_tables"],
        "selector_parse_status": selected["parse_status"],
        "fk_status": selected["fk_status"],
        "selector_raw_output": selected[
            "selector"
        ]["raw_output"],
        "plan": plan,
        "generated_sql": generated_sql,
        "predicted_sql": final_sql,
        "initial_execution_status": initial_check["status"],
        "initial_execution_error": initial_check["error"],
        "final_execution_status": final_check["status"],
        "refiner_triggered": repair is not None,
        "refiner_raw_output": (
            repair["text"] if repair else None
        ),
        "selector_seconds": selected[
            "selector"
        ]["inference_seconds"],
        "decomposer_seconds": decomposition[
            "inference_seconds"
        ],
        "generator_seconds": generation[
            "inference_seconds"
        ],
        "refiner_seconds": (
            repair["inference_seconds"]
            if repair else 0.0
        ),
        "pipeline_seconds": elapsed,
        "selector_output_tokens": selected[
            "selector"
        ]["output_tokens"],
        "decomposer_output_tokens": decomposition[
            "output_tokens"
        ],
        "generator_output_tokens": generation[
            "output_tokens"
        ],
        "refiner_output_tokens": (
            repair["output_tokens"]
            if repair else 0
        )
    }


completed = c5_load_checkpoint(C5_CHECKPOINT)

expected_ids = {
    s["sample_id"] for s in dev_samples
}

assert len(dev_samples) == 300
assert set(completed).issubset(expected_ids)

print("Total samples:", len(dev_samples))
print("Already completed:", len(completed))
print("Remaining:", 300 - len(completed))

session_start = time.perf_counter()

for index, sample in enumerate(dev_samples, start=1):
    sid = sample["sample_id"]

    if sid in completed:
        continue

    try:
        result = c5_run_one(sample)

        c5_append_checkpoint(
            C5_CHECKPOINT, result
        )

        completed[sid] = result

    except Exception as exc:
        print(
            f"\nSTOP at {index}/300 ({sid}):",
            type(exc).__name__,
            str(exc)
        )
        print("Checkpoint remains saved.")
        raise

    if index % 10 == 0 or index == 300:
        elapsed = time.perf_counter() - session_start

        print(
            f"[{index}/300] "
            f"Saved={len(completed)} "
            f"Remaining={300-len(completed)} "
            f"Session={elapsed:.1f}s",
            flush=True
        )


saved = c5_load_checkpoint(C5_CHECKPOINT)

assert len(saved) == 300
assert set(saved) == expected_ids

status_counts = Counter(
    r["final_execution_status"]
    for r in saved.values()
)

schema_counts = Counter(
    r["schema_mode"]
    for r in saved.values()
)

repair_count = sum(
    r["refiner_triggered"]
    for r in saved.values()
)

total_time = sum(
    r["pipeline_seconds"]
    for r in saved.values()
)

print("\n===== C5 FULL PIPELINE SUMMARY =====")
print("Completed:", len(saved))
print("Schema modes:", dict(schema_counts))
print("Refiner triggered:", repair_count)
print("Final execution statuses:", dict(status_counts))
print("Total pipeline time:", round(total_time, 2), "s")
print("Mean time per sample:", round(total_time / 300, 3), "s")
print("Checkpoint:", C5_CHECKPOINT)

print("\nDev gold: NOT ACCESSED")
print("C1/C2/C3/C4: UNMODIFIED")
print("\nCELL 75 HOÀN THÀNH!")


===== CELL 75 — C5 FULL PIPELINE =====
Total samples: 300
Already completed: 0
Remaining: 300
[10/300] Saved=10 Remaining=290 Session=123.6s
[20/300] Saved=20 Remaining=280 Session=252.1s
[30/300] Saved=30 Remaining=270 Session=367.0s
[40/300] Saved=40 Remaining=260 Session=519.8s
[50/300] Saved=50 Remaining=250 Session=658.0s
[60/300] Saved=60 Remaining=240 Session=791.3s
[70/300] Saved=70 Remaining=230 Session=928.0s
[80/300] Saved=80 Remaining=220 Session=1055.7s
[90/300] Saved=90 Remaining=210 Session=1178.0s
[100/300] Saved=100 Remaining=200 Session=1281.6s
[110/300] Saved=110 Remaining=190 Session=1396.7s
[120/300] Saved=120 Remaining=180 Session=1500.0s
[130/300] Saved=130 Remaining=170 Session=1613.8s
